# π0.5 occlusion features

Colab notebook for one question: which layer-5 transcoder features are about the bowl being painted over, and do those features move the action.

Use an L4 or A100 with high RAM. Run the cells from top to bottom.

1. Controls, Drive, install, caches, and the task-0 demonstrations.
2. Write `scripts/collect_layer5_replay.py`. Later cells import it.
3. Controlled contrasts. This trains the transcoder once and writes `outputs/permanence/controlled_contrasts/`.
4. Occlusion features. Reloads that transcoder. A feature is kept only if it rises for gray paint on the bowl and stays quiet for the same paint off the bowl, for the color swap, and for removing the bowl. The cell then patches those features and compares them with random features and with the color features. Writes `outputs/permanence/occlusion_features/`.

5. Paper gaps. Run this after the occlusion-feature cell, in the same runtime. It keeps one feature rule, scans the demos for gripper contact, writes a zoomed agent and wrist figure, and adds a second noise seed, a bootstrap interval, and a 10-step rollout. Writes `outputs/permanence/paper_gaps/`.

6. Cover autopsy. Run this after the paper-gaps cell, in the same runtime. It splits the full cover by camera, counts leftover pixels, forwards the same covered image twice, and checks whether the first 10 actions differ from the rest of the chunk and whether the arm moves. Writes `outputs/permanence/cover_autopsy/`.

7. Paper claim. Run this after the cover-autopsy cell, in the same runtime. It rebuilds the wrist mask without the 20% cap, paints leftover pixels, places another object in front of the bowl, rescores the feature rule on more frames, and rolls out full episodes with a 10-step re-query. Writes `outputs/permanence/paper_claim/`.

8. Remaining gaps. Run this after the paper-claim cell, in the same runtime. It places a second object on the wrist-camera ray, forwards that scene and a composite of the two camera renders, and steps closed-loop actions through the policy postprocessor. Writes `outputs/permanence/remaining_gaps/`.

9. Tight cover. Run this after the remaining-gaps cell, in the same runtime. It forwards the smallest plate that covers the bowl in both cameras, records the wrist-camera distance when the held bowl still fills the frame, and rolls out the visible bowl, the gray paint, the removed bowl, and that plate after a fresh reset. Writes `outputs/permanence/tight_cover/`.

10. Conclusion. Run this after the tight-cover cell, in the same runtime. It reuses those plate placements, counts leftover bowl pixels, paints them, and compares that action with full gray paint and with removal. Writes `outputs/permanence/conclusion/`.

11. Research zip. Run this after the experiment cells, in the same runtime. It writes `/content/pi05_permanence_research.zip` with the notebook, source, tests, figures, and summary JSON, and it skips caches, archives, secrets, and policy weights.

If this runtime already finished the experiment cells, run only the zip cell.


In [ ]:
# @title Controls

# Shared Drive folder that holds hf_home.tar and the LIBERO archives.
DRIVE_ROOT = "/content/drive/MyDrive/groot-run-shared-programmer908"  # @param {type:"string"}
DRIVE_FOLDER_ID = "13z_Qh91Ww0jdXoju2kQsG12AaEPKZsnH"  # @param {type:"string"}
AUTO_CREATE_DRIVE_SHORTCUT = True  # @param {type:"boolean"}
SHARED_HF_TOKEN_FILE = "secrets/HF_TOKEN.txt"  # @param {type:"string"}

# Pi0.5 does not load on a T4.
REQUIRED_GPU = "A100"  # @param ["L4", "A100", "Any"]
MIN_GPU_MEMORY_GB = 20  # @param {type:"integer"}
MIN_SYSTEM_RAM_GB = 24  # @param {type:"integer"}

# Task. The later cells inherit these.
SUITE = "libero_spatial"  # @param ["libero_spatial", "libero_object", "libero_goal", "libero_10"]
TASK_IDS = "[0]"  # @param {type:"string"}

# Cache. Archive mode avoids copying thousands of small Drive files.
CACHE_TRANSFER_MODE = "archive"  # @param ["archive", "folders"]
ALLOW_AUTH_REFRESH = True  # @param {type:"boolean"}
FORCE_AUTH_REFRESH = False  # @param {type:"boolean"}
HF_OFFLINE = True  # @param {type:"boolean"}

import os
os.environ["SUITE"] = SUITE
os.environ["TASK_IDS"] = TASK_IDS
print("suite", SUITE, "tasks", TASK_IDS)


In [ ]:
# @title Mount Drive And Validate Runtime

from pathlib import Path
import os
import subprocess
import time

from google.colab import drive


def parse_gib_from_meminfo() -> int:
    with open("/proc/meminfo", "r", encoding="utf-8") as handle:
        for line in handle:
            if line.startswith("MemTotal:"):
                kb = int(line.split()[1])
                return (kb + 1024 * 1024 - 1) // (1024 * 1024)
    return 0


def validate_colab_runtime():
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader,nounits"],
        text=True,
        capture_output=True,
    )
    print(result.stdout or result.stderr)
    if result.returncode != 0:
        raise RuntimeError("No NVIDIA GPU visible. In Colab: Runtime -> Change runtime type -> GPU.")

    gpu_line = result.stdout.strip().splitlines()[0]
    parts = [part.strip() for part in gpu_line.split(",")]
    gpu_name = parts[0]
    gpu_mem_gb = int((int(parts[2]) + 1023) // 1024) if len(parts) >= 3 and parts[2].isdigit() else 0
    system_ram_gb = parse_gib_from_meminfo()
    print(f"Runtime memory: GPU={gpu_name} ~{gpu_mem_gb} GiB VRAM | system RAM ~{system_ram_gb} GiB")

    if REQUIRED_GPU != "Any" and REQUIRED_GPU.lower() not in gpu_name.lower():
        raise RuntimeError(
            f"Requested {REQUIRED_GPU}, but Colab allocated {gpu_name}. "
            "Use Runtime -> Change runtime type -> GPU -> L4/A100, then rerun from this cell."
        )
    if int(MIN_GPU_MEMORY_GB) > 0 and gpu_mem_gb < int(MIN_GPU_MEMORY_GB):
        raise RuntimeError(
            f"GPU VRAM is too small for Pi0.5 LIBERO: {gpu_name} has ~{gpu_mem_gb} GiB, "
            f"need >= {MIN_GPU_MEMORY_GB} GiB. T4 usually exits 137 while loading the policy. "
            "Switch to L4 or A100 before running eval."
        )
    if int(MIN_SYSTEM_RAM_GB) > 0 and system_ram_gb < int(MIN_SYSTEM_RAM_GB):
        raise RuntimeError(
            f"System RAM is too small for Pi0.5 LIBERO: runtime has ~{system_ram_gb} GiB, "
            f"need >= {MIN_SYSTEM_RAM_GB} GiB. In Colab, choose a high-RAM L4/A100 runtime."
        )


validate_colab_runtime()

drive.mount("/content/drive")


def ensure_drive_root_visible(root_value: str) -> Path:
    root = Path(root_value)
    if root.exists():
        return root

    folder_id = str(globals().get("DRIVE_FOLDER_ID", "")).strip()
    auto_shortcut = bool(globals().get("AUTO_CREATE_DRIVE_SHORTCUT", True))
    mydrive_prefixes = ("/content/drive/MyDrive/", "/content/drive/My Drive/")
    if folder_id and auto_shortcut and str(root).startswith(mydrive_prefixes):
        print("Drive root is not visible yet; creating a My Drive shortcut to the shared folder.", flush=True)
        try:
            from google.colab import auth
            auth.authenticate_user()
            try:
                from googleapiclient.discovery import build
            except Exception:
                subprocess.run(["python3", "-m", "pip", "install", "-q", "-U", "google-api-python-client"], check=True)
                from googleapiclient.discovery import build
            service = build("drive", "v3")
            metadata = {
                "name": root.name,
                "mimeType": "application/vnd.google-apps.shortcut",
                "shortcutDetails": {"targetId": folder_id},
                "parents": ["root"],
            }
            created = service.files().create(body=metadata, fields="id,name").execute()
            print(f"Created Drive shortcut: {created.get('name')} ({created.get('id')})", flush=True)
            for _ in range(12):
                if root.exists():
                    return root
                time.sleep(5)
        except Exception as exc:
            raise RuntimeError(
                "Could not auto-create the Drive shortcut. Confirm the folder was shared with this Google account "
                "and DRIVE_FOLDER_ID is the shared folder ID."
            ) from exc
        if root.exists():
            return root
        raise RuntimeError(
            f"Drive shortcut was created, but {root} is not visible in the mounted filesystem yet. "
            "Rerun this cell, or use a Google Workspace Shared Drive path."
        )

    if folder_id and str(root).startswith("/content/drive/Shareddrives/"):
        raise RuntimeError(
            f"Shared Drive path is not visible: {root}. Confirm the teammate has access to the Shared Drive."
        )

    if not folder_id:
        print(
            "DRIVE_FOLDER_ID is blank and DRIVE_ROOT does not exist. Creating a new folder at DRIVE_ROOT; "
            "this is intended only for initial owner setup. For zero teammate setup, set DRIVE_FOLDER_ID in the notebook.",
            flush=True,
        )
    return root


DRIVE_ROOT = ensure_drive_root_visible(DRIVE_ROOT)
DRIVE_ARCHIVES = DRIVE_ROOT / "archives"
DRIVE_HF_HOME = DRIVE_ROOT / "hf_home"
DRIVE_LIBERO_CACHE = DRIVE_ROOT / "libero_cache"
DRIVE_LIBERO_DATASETS = DRIVE_ROOT / "libero_datasets"
DRIVE_OUTPUTS = DRIVE_ROOT / "outputs"
DRIVE_NOTEBOOK_META = DRIVE_ROOT / "notebook_meta"
DRIVE_SECRETS = DRIVE_ROOT / "secrets"
DRIVE_HF_TOKEN_FILE = DRIVE_ROOT / SHARED_HF_TOKEN_FILE
for path in [DRIVE_ROOT, DRIVE_ARCHIVES, DRIVE_HF_HOME, DRIVE_LIBERO_CACHE, DRIVE_LIBERO_DATASETS, DRIVE_OUTPUTS, DRIVE_NOTEBOOK_META, DRIVE_SECRETS]:
    path.mkdir(parents=True, exist_ok=True)

HF_HOME_ARCHIVE = DRIVE_ARCHIVES / "hf_home.tar"
LIBERO_CACHE_ARCHIVE = DRIVE_ARCHIVES / "libero_cache.tar"
LIBERO_DATASETS_ARCHIVE = DRIVE_ARCHIVES / "libero_datasets.tar"

print("Drive root:", DRIVE_ROOT)
print("Archive dir:", DRIVE_ARCHIVES)
print("Optional shared HF token file:", DRIVE_HF_TOKEN_FILE)


In [ ]:
# @title Install Native Runtime Equivalent To cloud/libero/Dockerfile

import os
import subprocess
from pathlib import Path

VENV = Path("/content/lerobot-venv")
PYTHON = VENV / "bin/python"
UV_BIN_DIR = Path("/content/uv-bin")
UV = str(UV_BIN_DIR / "uv")


def run(cmd, *, env=None):
    cmd = list(map(str, cmd))
    print("$", " ".join(cmd), flush=True)
    try:
        return subprocess.run(cmd, env=env, check=True)
    except subprocess.CalledProcessError as exc:
        print(f"Command failed with exit code {exc.returncode}: {' '.join(cmd)}", flush=True)
        raise


def install_uv() -> str:
    # Colab may already have /usr/local/bin/uv, but its version/behavior can differ.
    # Use a notebook-owned binary so every clean runtime follows the same path.
    UV_BIN_DIR.mkdir(parents=True, exist_ok=True)
    installer = Path("/tmp/install-uv.sh")
    run(["curl", "-LsSf", "https://astral.sh/uv/install.sh", "-o", installer])
    env = os.environ.copy()
    env["UV_INSTALL_DIR"] = str(UV_BIN_DIR)
    run(["sh", installer], env=env)
    os.environ["PATH"] = f"{UV_BIN_DIR}:" + os.environ["PATH"]
    run([UV, "--version"])
    return UV


apt_packages = [
    "build-essential", "ca-certificates", "cmake", "curl", "ffmpeg", "git", "pv", "rsync",
    "libegl1", "libgl1", "libglib2.0-0", "libglvnd0", "libglx0", "libopengl0",
    "libosmesa6-dev", "libsm6", "libxext6", "libxrender1", "pkg-config",
]
apt_env = os.environ.copy()
apt_env["DEBIAN_FRONTEND"] = "noninteractive"
run(["apt-get", "update", "-qq"], env=apt_env)
run(["apt-get", "install", "-y", "-qq", *apt_packages], env=apt_env)
UV = install_uv()

# Managed Python avoids system-package/venv quirks in Colab images.
run([UV, "python", "install", "3.12"])
run([UV, "venv", "--clear", str(VENV), "--python", "3.12"])

run([
    UV, "pip", "install", "--python", str(PYTHON), "--torch-backend", "cu128",
    "lerobot[evaluation,libero,pi]", "hf-transfer", "opencv-python", "numpy",
])

os.environ["PATH"] = f"{VENV / 'bin'}:{UV_BIN_DIR}:" + os.environ["PATH"]
print("Python:", PYTHON)
run([str(PYTHON), "-c", "import torch; print('torch', torch.__version__); print('cuda', torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu')"])


In [ ]:
# @title Local workspace

from pathlib import Path
import os

LOCAL_REPO = Path("/content/groot-run")
LOCAL_REPO.mkdir(parents=True, exist_ok=True)
(LOCAL_REPO / "scripts").mkdir(parents=True, exist_ok=True)
os.environ["LOCAL_REPO"] = str(LOCAL_REPO)
print("Local workspace:", LOCAL_REPO)


In [ ]:
# @title Prepare Drive Caches And Offline Mode

from pathlib import Path
import os
import shutil
import shlex
import subprocess
import time

LOCAL_HF_HOME = Path("/content/hf_home")
LOCAL_LIBERO_CACHE = Path.home() / ".cache/libero"
LOCAL_OUTPUT_ROOT = LOCAL_REPO / "outputs/eval/pi05_libero"
LOCAL_DATA_ROOT = LOCAL_REPO / "data/libero/datasets"
LOCAL_LIBERO_CONFIG = LOCAL_REPO / ".libero"
for path in [LOCAL_OUTPUT_ROOT, LOCAL_LIBERO_CONFIG]:
    path.mkdir(parents=True, exist_ok=True)

HF_CACHE_REFRESHED = False

repos = ["lerobot/pi05_libero_finetuned", "google/paligemma-3b-pt-224", "lerobot/libero-assets"]


def hf_repo_type(repo_id: str) -> str:
    return "dataset" if repo_id == "lerobot/libero-assets" else "model"


def now_text() -> str:
    return time.strftime("%H:%M:%S")


def format_duration(seconds: float | None) -> str:
    if seconds is None or seconds != seconds or seconds < 0:
        return "unknown"
    seconds = int(seconds)
    hours, rem = divmod(seconds, 3600)
    minutes, secs = divmod(rem, 60)
    if hours:
        return f"{hours}h{minutes:02d}m{secs:02d}s"
    if minutes:
        return f"{minutes}m{secs:02d}s"
    return f"{secs}s"


def format_bytes(value: float | int | None) -> str:
    if value is None:
        return "unknown"
    value = float(value)
    units = ["B", "KiB", "MiB", "GiB", "TiB"]
    for unit in units:
        if abs(value) < 1024 or unit == units[-1]:
            return f"{value:.1f}{unit}" if unit != "B" else f"{value:.0f}{unit}"
        value /= 1024
    return f"{value:.1f}PiB"


def timed(label, fn):
    print(f"\n== {label} | start {now_text()} ==", flush=True)
    start = time.time()
    result = fn()
    elapsed = time.time() - start
    print(f"== done: {label} | elapsed {format_duration(elapsed)} | finish {now_text()} ==", flush=True)
    return result


def run(cmd, **kwargs):
    print("+", " ".join(map(str, cmd)), flush=True)
    start = time.time()
    result = subprocess.run([str(x) for x in cmd], check=True, **kwargs)
    print(f"+ done in {format_duration(time.time() - start)}", flush=True)
    return result


def reset_dir(path: Path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def dir_has_anything(path: Path) -> bool:
    return path.exists() and any(path.iterdir())


def hf_cache_has(repo_id: str, root: Path) -> bool:
    namespace, name = repo_id.split("/", 1)
    prefix = "datasets" if hf_repo_type(repo_id) == "dataset" else "models"
    return (root / "hub" / f"{prefix}--{namespace}--{name}").exists()


def get_hf_token(required: bool = False) -> str:
    token = ""
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN") or ""
        if token:
            print("Using private Colab Secret HF_TOKEN.", flush=True)
            return token
    except Exception:
        pass
    token_file = globals().get("DRIVE_HF_TOKEN_FILE")
    if token_file and Path(token_file).exists():
        token = Path(token_file).read_text().strip()
        if token:
            print(f"Using shared Drive HF token file: {token_file}", flush=True)
            return token
    if required:
        raise RuntimeError(
            "ALLOW_AUTH_REFRESH=True requires either private Colab Secret HF_TOKEN "
            f"or shared Drive token file at {globals().get('DRIVE_HF_TOKEN_FILE', 'DRIVE_ROOT/secrets/HF_TOKEN.txt')}."
        )
    return ""



def path_size_bytes(path: Path) -> int:
    if not path.exists():
        return 0
    total = 0
    for root, dirs, files in os.walk(path):
        dirs[:] = [name for name in dirs if not Path(root, name).is_symlink()]
        for name in files:
            file_path = Path(root, name)
            try:
                total += file_path.lstat().st_size
            except OSError:
                pass
    return total


def du(path: Path):
    if path.exists():
        run(["du", "-sh", path])


def extract_tar(archive: Path, dst: Path):
    reset_dir(dst)
    size = archive.stat().st_size
    print(f"Archive: {archive} ({format_bytes(size)}) -> {dst}", flush=True)
    cmd = f"set -euo pipefail; pv -ptebarf -s {size} {shlex.quote(str(archive))} | tar -C {shlex.quote(str(dst))} -xf -"
    run(["bash", "-lc", cmd])
    du(dst)


def create_tar(src: Path, archive: Path):
    if not dir_has_anything(src):
        print(f"Skipping archive for empty directory: {src}", flush=True)
        return
    archive.parent.mkdir(parents=True, exist_ok=True)
    tmp = archive.with_suffix(archive.suffix + ".tmp")
    if tmp.exists():
        tmp.unlink()
    size_bytes = path_size_bytes(src)
    print(f"Archiving: {src} ({format_bytes(size_bytes)}) -> {archive}", flush=True)
    cmd = f"set -euo pipefail; tar -C {shlex.quote(str(src))} -cf - . | pv -ptebarf -s {size_bytes} > {shlex.quote(str(tmp))}"
    run(["bash", "-lc", cmd])
    tmp.replace(archive)
    run(["ls", "-lh", archive])


def rsync_tree(src: Path, dst: Path, reset: bool = True):
    src.mkdir(parents=True, exist_ok=True)
    if reset:
        reset_dir(dst)
    else:
        dst.mkdir(parents=True, exist_ok=True)
    print(f"Rsync: {src} ({format_bytes(path_size_bytes(src))}) -> {dst}", flush=True)
    run(["rsync", "-a", "--human-readable", "--info=progress2", "--stats", f"{src}/", f"{dst}/"])
    du(dst)


if CACHE_TRANSFER_MODE == "archive" and HF_HOME_ARCHIVE.exists() and not FORCE_AUTH_REFRESH:
    timed("extract HF cache archive from Drive to /content", lambda: extract_tar(HF_HOME_ARCHIVE, LOCAL_HF_HOME))
elif ALLOW_AUTH_REFRESH:
    token = get_hf_token(required=True)
    reset_dir(LOCAL_HF_HOME)
    run([str(PYTHON), "-c", "import huggingface_hub; print('huggingface_hub OK')"])
    refresh_code = """
from huggingface_hub import HfApi, snapshot_download
from huggingface_hub.utils import enable_progress_bars
import os
import subprocess
import threading
import time

repos = os.environ['REFRESH_REPOS'].split(',')
token = os.environ.get('HF_TOKEN') or None
cache_dir = os.environ['LOCAL_HF_HUB_CACHE']


def repo_type(repo):
    return 'dataset' if repo == 'lerobot/libero-assets' else 'model'
heartbeat_seconds = int(os.environ.get('HF_PROGRESS_HEARTBEAT_SECONDS', '15'))


def now_text():
    return time.strftime('%H:%M:%S')


def format_duration(seconds):
    if seconds is None or seconds != seconds or seconds < 0:
        return 'unknown'
    seconds = int(seconds)
    hours, rem = divmod(seconds, 3600)
    minutes, secs = divmod(rem, 60)
    if hours:
        return f'{hours}h{minutes:02d}m{secs:02d}s'
    if minutes:
        return f'{minutes}m{secs:02d}s'
    return f'{secs}s'


def format_bytes(value):
    if value is None:
        return 'unknown'
    value = float(value)
    units = ['B', 'KiB', 'MiB', 'GiB', 'TiB']
    for unit in units:
        if abs(value) < 1024 or unit == units[-1]:
            return f'{value:.1f}{unit}' if unit != 'B' else f'{value:.0f}{unit}'
        value /= 1024
    return f'{value:.1f}PiB'


def path_size_bytes(path):
    if not os.path.exists(path):
        return 0
    total = 0
    for root, dirs, files in os.walk(path):
        dirs[:] = [name for name in dirs if not os.path.islink(os.path.join(root, name))]
        for name in files:
            file_path = os.path.join(root, name)
            try:
                total += os.lstat(file_path).st_size
            except OSError:
                pass
    return total


def estimate_repo_size(api, repo):
    try:
        if repo_type(repo) == 'dataset':
            info = api.dataset_info(repo_id=repo, token=token, files_metadata=True)
        else:
            info = api.model_info(repo_id=repo, token=token, files_metadata=True)
        sizes = [getattr(sibling, 'size', None) for sibling in getattr(info, 'siblings', [])]
        total = sum(size for size in sizes if isinstance(size, int))
        count = len(sizes)
        return count, total or None
    except Exception as exc:
        print(f'[{repo}] could not estimate size before download: {type(exc).__name__}: {exc}', flush=True)
        return None, None


def heartbeat(repo, root, start_bytes, estimated_total, stop_event):
    last_time = time.time()
    last_bytes = start_bytes
    start_time = last_time
    while not stop_event.wait(heartbeat_seconds):
        now = time.time()
        current_bytes = path_size_bytes(root)
        delta = max(0, current_bytes - start_bytes)
        recent_rate = max(0, current_bytes - last_bytes) / max(0.001, now - last_time)
        avg_rate = delta / max(0.001, now - start_time)
        if estimated_total and avg_rate > 0:
            remaining = max(0, estimated_total - delta)
            eta = remaining / avg_rate
            pct = min(100.0, (delta / estimated_total) * 100.0)
            estimate_text = f'{pct:5.1f}% of est. {format_bytes(estimated_total)} | ETA {format_duration(eta)}'
        else:
            estimate_text = 'ETA unknown'
        print(
            f'[{repo}] progress {format_bytes(delta)} | recent {format_bytes(recent_rate)}/s | '
            f'avg {format_bytes(avg_rate)}/s | elapsed {format_duration(now - start_time)} | {estimate_text}',
            flush=True,
        )
        last_time = now
        last_bytes = current_bytes


os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '0'
os.environ['TQDM_DISABLE'] = '0'
os.environ['TQDM_MININTERVAL'] = '1'
try:
    enable_progress_bars()
except Exception as exc:
    print(f'Could not force-enable Hugging Face progress bars: {type(exc).__name__}: {exc}', flush=True)

api = HfApi()
for index, repo in enumerate(repos, 1):
    print(f'\\n[{index}/{len(repos)}] preparing {repo} at {now_text()}', flush=True)
    file_count, estimated_total = estimate_repo_size(api, repo)
    if file_count is not None:
        print(f'[{repo}] estimated remote files={file_count} size={format_bytes(estimated_total)}', flush=True)
    before = path_size_bytes(cache_dir)
    stop_event = threading.Event()
    monitor = threading.Thread(target=heartbeat, args=(repo, cache_dir, before, estimated_total, stop_event), daemon=True)
    start = time.time()
    monitor.start()
    try:
        path = snapshot_download(
            repo_id=repo,
            repo_type=repo_type(repo),
            cache_dir=cache_dir,
            token=token,
            max_workers=8,
        )
    finally:
        stop_event.set()
        monitor.join(timeout=2)
    elapsed = time.time() - start
    after = path_size_bytes(cache_dir)
    delta = max(0, after - before)
    avg_rate = delta / max(0.001, elapsed)
    print(f'[{repo}] -> {path}', flush=True)
    print(
        f'[{repo}] done at {now_text()} | elapsed {format_duration(elapsed)} | '
        f'cache delta {format_bytes(delta)} | avg {format_bytes(avg_rate)}/s | total cache {format_bytes(after)}',
        flush=True,
    )
"""
    env = os.environ.copy()
    for key in ["HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE"]:
        env.pop(key, None)
    env.update({
        "PYTHONUNBUFFERED": "1",
        "HF_TOKEN": token,
        "HF_HOME": str(LOCAL_HF_HOME),
        "HF_HUB_CACHE": str(LOCAL_HF_HOME / "hub"),
        "LOCAL_HF_HUB_CACHE": str(LOCAL_HF_HOME / "hub"),
        "REFRESH_REPOS": ",".join(repos),
        "HF_HUB_ENABLE_HF_TRANSFER": "1",
        "HF_XET_HIGH_PERFORMANCE": "1",
        "HF_HUB_DISABLE_PROGRESS_BARS": "0",
        "TQDM_DISABLE": "0",
        "TQDM_MININTERVAL": "1",
        "HF_PROGRESS_HEARTBEAT_SECONDS": "15",
    })
    timed("download gated HF models to fast local disk", lambda: subprocess.run([str(PYTHON), "-u", "-c", refresh_code], env=env, check=True))
    HF_CACHE_REFRESHED = True
elif CACHE_TRANSFER_MODE == "folders" or dir_has_anything(DRIVE_HF_HOME):
    print("Archive missing; falling back to Drive folder rsync. This can be very slow for HF caches.", flush=True)
    timed("copy HF cache folder from Drive to /content", lambda: rsync_tree(DRIVE_HF_HOME, LOCAL_HF_HOME))
else:
    raise RuntimeError(
        "No HF cache archive or folder found in Drive. Set ALLOW_AUTH_REFRESH=True once with private Colab Secret HF_TOKEN or DRIVE_ROOT/secrets/HF_TOKEN.txt."
    )

missing_local = [repo for repo in repos if not hf_cache_has(repo, LOCAL_HF_HOME)]
if missing_local and ALLOW_AUTH_REFRESH:
    token = get_hf_token(required=False)
    refresh_missing_code = """
from huggingface_hub import snapshot_download
import os

repos = os.environ['MISSING_REPOS'].split(',')
token = os.environ.get('HF_TOKEN') or None
cache_dir = os.environ['LOCAL_HF_HUB_CACHE']


def repo_type(repo):
    return 'dataset' if repo == 'lerobot/libero-assets' else 'model'

for repo in repos:
    print('refreshing missing cache entry', repo, 'repo_type', repo_type(repo), flush=True)
    path = snapshot_download(
        repo_id=repo,
        repo_type=repo_type(repo),
        cache_dir=cache_dir,
        token=token,
        max_workers=8,
    )
    print(repo, '->', path, flush=True)
"""
    env = os.environ.copy()
    for key in ["HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE"]:
        env.pop(key, None)
    env.update({
        "PYTHONUNBUFFERED": "1",
        "HF_TOKEN": token or "",
        "HF_HOME": str(LOCAL_HF_HOME),
        "HF_HUB_CACHE": str(LOCAL_HF_HOME / "hub"),
        "LOCAL_HF_HUB_CACHE": str(LOCAL_HF_HOME / "hub"),
        "MISSING_REPOS": ",".join(missing_local),
        "HF_HUB_ENABLE_HF_TRANSFER": "1",
        "HF_XET_HIGH_PERFORMANCE": "1",
        "HF_HUB_DISABLE_PROGRESS_BARS": "0",
        "TQDM_DISABLE": "0",
        "TQDM_MININTERVAL": "1",
    })
    timed("refresh missing HF cache entries", lambda: subprocess.run([str(PYTHON), "-u", "-c", refresh_missing_code], env=env, check=True))
    HF_CACHE_REFRESHED = True
    missing_local = [repo for repo in repos if not hf_cache_has(repo, LOCAL_HF_HOME)]

if missing_local:
    raise RuntimeError(
        "Local HF cache is missing: " + ", ".join(missing_local) + "\n"
        "Run once with ALLOW_AUTH_REFRESH=True plus private Colab Secret HF_TOKEN or DRIVE_ROOT/secrets/HF_TOKEN.txt, "
        "or populate DRIVE_ROOT/archives/hf_home.tar."
    )

if CACHE_TRANSFER_MODE == "archive" and LIBERO_CACHE_ARCHIVE.exists():
    timed("extract LIBERO cache archive from Drive", lambda: extract_tar(LIBERO_CACHE_ARCHIVE, LOCAL_LIBERO_CACHE))
else:
    timed("copy LIBERO cache folder from Drive", lambda: rsync_tree(DRIVE_LIBERO_CACHE, LOCAL_LIBERO_CACHE))

if CACHE_TRANSFER_MODE == "archive" and LIBERO_DATASETS_ARCHIVE.exists():
    timed("extract LIBERO datasets archive from Drive", lambda: extract_tar(LIBERO_DATASETS_ARCHIVE, LOCAL_DATA_ROOT))
else:
    timed("copy LIBERO datasets folder from Drive", lambda: rsync_tree(DRIVE_LIBERO_DATASETS, LOCAL_DATA_ROOT))

os.environ.update({
    "PATH": f"{VENV / 'bin'}:" + os.environ["PATH"],
    "PYTHON": str(PYTHON),
    "HF_HOME": str(LOCAL_HF_HOME),
    "HF_HUB_CACHE": str(LOCAL_HF_HOME / "hub"),
    "HF_HUB_ENABLE_HF_TRANSFER": "1",
    "HF_XET_HIGH_PERFORMANCE": "1",
    "HF_HUB_DISABLE_PROGRESS_BARS": "0",
    "TQDM_DISABLE": "0",
    "TQDM_MININTERVAL": "1",
    "MUJOCO_GL": "egl",
    "PYOPENGL_PLATFORM": "egl",
    "MUJOCO_EGL_DEVICE_ID": "0",
    "MPLBACKEND": "Agg",
    "LIBERO_CONFIG_PATH": str(LOCAL_LIBERO_CONFIG),
    "LIBERO_DATASET_DIR": str(LOCAL_DATA_ROOT),
    "OUTPUT_ROOT": str(LOCAL_OUTPUT_ROOT),
})
if HF_OFFLINE:
    os.environ.update({"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "HF_DATASETS_OFFLINE": "1"})
else:
    os.environ.pop("HF_HUB_OFFLINE", None)
    os.environ.pop("TRANSFORMERS_OFFLINE", None)
    os.environ.pop("HF_DATASETS_OFFLINE", None)

validate_code = """
from huggingface_hub import snapshot_download
import os


def repo_type(repo):
    return 'dataset' if repo == 'lerobot/libero-assets' else 'model'

for repo in os.environ['VALIDATE_REPOS'].split(','):
    path = snapshot_download(repo_id=repo, repo_type=repo_type(repo), cache_dir=os.environ['HF_HUB_CACHE'], local_files_only=True)
    print(repo, '->', path, flush=True)
"""
env = os.environ.copy()
env["VALIDATE_REPOS"] = ",".join(repos)
timed("validate local HF cache with local_files_only=True", lambda: subprocess.run([str(PYTHON), "-u", "-c", validate_code], env=env, check=True))

install_assets_code = """
from pathlib import Path
import os
import shutil
import site

from huggingface_hub import snapshot_download

roots = [Path(path) for path in site.getsitepackages()]
user_site = site.getusersitepackages()
if user_site:
    roots.append(Path(user_site))
for root in roots:
    candidate = root / "libero" / "libero"
    if candidate.exists():
        libero_root = candidate
        break
else:
    raise SystemExit("Could not find installed libero package path")

assets_dir = libero_root / "assets"
required = assets_dir / "scenes" / "libero_tabletop_base_style.xml"
snapshot = Path(snapshot_download(
    repo_id="lerobot/libero-assets",
    repo_type="dataset",
    cache_dir=os.environ["HF_HUB_CACHE"],
    local_files_only=True,
))
if not required.exists():
    print(f"Installing LIBERO assets: {snapshot} -> {assets_dir}", flush=True)
    assets_dir.mkdir(parents=True, exist_ok=True)
    for child in snapshot.iterdir():
        if child.name == ".gitattributes":
            continue
        target = assets_dir / child.name
        if child.is_dir():
            shutil.copytree(child, target, dirs_exist_ok=True)
        else:
            shutil.copy2(child, target)
if not required.exists():
    raise SystemExit(f"LIBERO asset install failed; missing {required}")
print("libero_assets OK", required, flush=True)
"""
timed("install LIBERO assets into package", lambda: subprocess.run([str(PYTHON), "-u", "-c", install_assets_code], env=os.environ.copy(), check=True))

print("\nLocal cache summary:")
du(LOCAL_HF_HOME)
du(LOCAL_LIBERO_CACHE)
du(LOCAL_DATA_ROOT)


In [ ]:
# @title Fetch LIBERO demonstrations
# Downloads the hdf5 demos for SUITE / TASK_IDS when the Drive archive is empty.
import os, sys, subprocess, time
from pathlib import Path

LOCAL_REPO = Path(globals().get("LOCAL_REPO", "/content/groot-run"))
PYTHON     = Path(globals().get("PYTHON", "/content/lerobot-venv/bin/python"))

SUITE_TO_FETCH   = SUITE
FETCH_TASK_IDS   = TASK_IDS
SAVE_TO_DRIVE    = True               # tar the result to Drive so cell 6 finds it next time

FETCH_SRC = r'''#!/usr/bin/env python
"""Download the raw LIBERO demonstration files this suite/task needs."""

import json
import os
from pathlib import Path

import h5py
from huggingface_hub import HfApi, hf_hub_download

def ensure_libero_config() -> Path:
    """LIBERO asks on stdin the first time it is imported, which fails in a subprocess.

    Writing its config.yaml first skips the prompt entirely.
    """
    import importlib.util

    config_dir = Path(os.environ.get("LIBERO_CONFIG_PATH") or (Path.home() / ".libero"))
    config_dir.mkdir(parents=True, exist_ok=True)
    os.environ["LIBERO_CONFIG_PATH"] = str(config_dir)
    config_file = config_dir / "config.yaml"

    spec = importlib.util.find_spec("libero")
    if spec is None or not spec.submodule_search_locations:
        raise SystemExit("libero is not installed in this interpreter.")
    package = Path(list(spec.submodule_search_locations)[0]) / "libero"
    datasets = Path(os.environ.get("LIBERO_DATASET_DIR") or (package / "datasets"))
    datasets.mkdir(parents=True, exist_ok=True)

    wanted = {
        "benchmark_root": str(package),
        "bddl_files": str(package / "bddl_files"),
        "init_states": str(package / "init_files"),
        "datasets": str(datasets),
        "assets": str(package / "assets"),
    }
    current = {}
    if config_file.is_file():
        try:
            import yaml

            current = yaml.safe_load(config_file.read_text()) or {}
        except Exception:
            current = {}
    merged = dict(current)
    merged.update(wanted)
    if merged != current:
        try:
            import yaml

            config_file.write_text(yaml.safe_dump(merged))
        except Exception:
            config_file.write_text("".join(f"{k}: {v}\n" for k, v in merged.items()))
        print(f"wrote LIBERO config {config_file}", flush=True)
    return config_file


ensure_libero_config()


from libero.libero import benchmark

REPO = os.environ.get("LIBERO_DEMO_REPO", "yifengzhu-hf/LIBERO-datasets")
SUITE = os.environ.get("SUITE", "libero_spatial")
WANT = json.loads(os.environ.get("FETCH_TASK_IDS", "[0]"))
DEST = Path(os.environ["LIBERO_DATASET_DIR"])
DEST.mkdir(parents=True, exist_ok=True)

suite = benchmark.get_benchmark_dict()[SUITE]()
n_tasks = suite.n_tasks
ids = WANT if WANT else list(range(n_tasks))
print(f"{SUITE}: {n_tasks} tasks, fetching {ids}", flush=True)

api = HfApi()
listing = [f for f in api.list_repo_files(REPO, repo_type="dataset") if f.endswith(".hdf5")]
print(f"{REPO}: {len(listing)} hdf5 files in the repo", flush=True)

got = []
for task_id in ids:
    task = suite.get_task(int(task_id))
    wanted = [
        f"{task.problem_folder}/{task.name}_demo.hdf5",
        f"{SUITE}/{task.name}_demo.hdf5",
    ]
    match = next((f for f in wanted if f in listing), None)
    if match is None:
        near = [f for f in listing if task.name in f]
        if len(near) == 1:
            match = near[0]
        else:
            raise SystemExit(
                f"Cannot find a demo file for task {task_id} ({task.name}).\n"
                f"Tried: {wanted}\nClose matches: {near[:5]}"
            )

    target = DEST / match
    if target.is_file() and target.stat().st_size > 1_000_000:
        print(f"[{task_id}] already present: {target} "
              f"({target.stat().st_size / 1e6:.0f} MB)", flush=True)
    else:
        print(f"[{task_id}] downloading {match}", flush=True)
        hf_hub_download(
            repo_id=REPO,
            repo_type="dataset",
            filename=match,
            local_dir=str(DEST),
        )
        print(f"[{task_id}] -> {target} ({target.stat().st_size / 1e6:.0f} MB)", flush=True)

    with h5py.File(target, "r") as handle:
        demos = sorted(handle["data"].keys())
        group = handle["data"][demos[0]]
        keys = sorted(group.keys())
        if "states" not in keys:
            raise SystemExit(
                f"{target} has no 'states' in {demos[0]} (keys: {keys}). "
                "This is the wrong dataset format for replay."
            )
        print(f"[{task_id}] {len(demos)} demos | {demos[0]} keys={keys} | "
              f"states {group['states'].shape}", flush=True)
    got.append(str(target))

print("\nReady:", flush=True)
for path in got:
    print("  ", path, flush=True)
'''

datasets = Path(os.environ.get("LIBERO_DATASET_DIR", LOCAL_REPO / "data/libero/datasets"))
scripts = LOCAL_REPO / "scripts"
scripts.mkdir(parents=True, exist_ok=True)
fetch = scripts / "fetch_libero_demos.py"
fetch.write_text(FETCH_SRC)

env = os.environ.copy()
env.setdefault("HF_HOME", "/content/hf_home")
env.setdefault("HF_HUB_CACHE", "/content/hf_home/hub")
env.setdefault("LIBERO_CONFIG_PATH", str(LOCAL_REPO / ".libero"))
env["LIBERO_DATASET_DIR"] = str(datasets)
env["SUITE"] = SUITE_TO_FETCH
env["FETCH_TASK_IDS"] = FETCH_TASK_IDS
env["PYTHONUNBUFFERED"] = "1"
env["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
# The demo mirror is public, but it has to be fetched online.
for key in ("HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE"):
    env.pop(key, None)

def stream(cmd, cwd, env, label):
    print(f"\n$ {' '.join(map(str, cmd))}", flush=True)
    proc = subprocess.Popen(
        [str(c) for c in cmd], cwd=str(cwd), env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    tail = []
    for line in proc.stdout:
        line = line.rstrip()
        print(line, flush=True)
        tail.append(line)
        del tail[:-40]
    code = proc.wait()
    if code != 0:
        raise SystemExit(f"{label} exited with code {code}. Last lines:\n" + "\n".join(tail))

start = time.time()
stream([PYTHON, "-u", fetch], LOCAL_REPO, env, "fetch_libero_demos.py")
print(f"\nfetch took {time.time() - start:.0f}s", flush=True)

found = sorted(datasets.rglob("*.hdf5"))
total = sum(p.stat().st_size for p in found)
print(f"\n{len(found)} hdf5 files, {total / 1e9:.2f} GB under {datasets}", flush=True)
for path in found:
    print("   ", path.relative_to(datasets), f"{path.stat().st_size / 1e6:.0f} MB", flush=True)

# Cache to Drive so the next runtime takes the fast archive path in cell 6.
archive = globals().get("LIBERO_DATASETS_ARCHIVE")
if SAVE_TO_DRIVE and archive is not None and found:
    archive = Path(archive)
    if archive.exists():
        print(f"\n{archive} already exists - leaving it alone.", flush=True)
    else:
        archive.parent.mkdir(parents=True, exist_ok=True)
        tmp = archive.with_suffix(".tar.tmp")
        print(f"\nArchiving {datasets} -> {archive} ({total / 1e9:.2f} GB, this is slow)", flush=True)
        subprocess.run(["tar", "-C", str(datasets), "-cf", str(tmp), "."], check=True)
        tmp.replace(archive)
        print("wrote", archive, f"{archive.stat().st_size / 1e9:.2f} GB", flush=True)
elif SAVE_TO_DRIVE and archive is None:
    print("\n(LIBERO_DATASETS_ARCHIVE is not defined - run cell 3 first to archive to Drive.)", flush=True)

print("\nDemos are ready. Run the collector cell, then the contrast cell.", flush=True)


In [ ]:
# @title Write the layer-5 replay library
# The contrast cell imports this module. It does not replay the old observational run.

import os, sys, subprocess
from pathlib import Path

LOCAL_REPO = Path(globals().get("LOCAL_REPO", "/content/groot-run"))
PYTHON = Path(globals().get("PYTHON", "/content/lerobot-venv/bin/python"))
if not PYTHON.exists():
    PYTHON = Path(sys.executable)

SCRIPT_SRC = r'''#!/usr/bin/env python
"""Replay saved LIBERO demonstrations and save one PaliGemma layer-5 vector per frame.

The recorded demonstration moves the arm. Pi0.5 only looks. The simulator
labels each frame from finger contact and camera geometry. A contact sheet is
written so those labels can be checked by eye before any transcoder training.

Run after the Colab setup cells, with the lerobot venv on PATH:

    SUITE=libero_spatial TASK_IDS='[0]' MAX_EPISODES=2 MAX_STEPS=60 \
        python scripts/collect_layer5_replay.py

Raise MAX_EPISODES and MAX_STEPS only after the contact sheet looks right.
"""

from __future__ import annotations

import json
import os

# Headless rendering must be chosen before mujoco / libero are imported.
os.environ.setdefault("MUJOCO_GL", "egl")
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
os.environ.setdefault("MPLBACKEND", "Agg")

import re
import traceback
from pathlib import Path

import cv2
import h5py
import numpy as np
import torch

def ensure_libero_config() -> Path:
    """LIBERO asks on stdin the first time it is imported, which fails in a subprocess.

    Writing its config.yaml first skips the prompt entirely.
    """
    import importlib.util

    config_dir = Path(os.environ.get("LIBERO_CONFIG_PATH") or (Path.home() / ".libero"))
    config_dir.mkdir(parents=True, exist_ok=True)
    os.environ["LIBERO_CONFIG_PATH"] = str(config_dir)
    config_file = config_dir / "config.yaml"

    spec = importlib.util.find_spec("libero")
    if spec is None or not spec.submodule_search_locations:
        raise SystemExit("libero is not installed in this interpreter.")
    package = Path(list(spec.submodule_search_locations)[0]) / "libero"
    datasets = Path(os.environ.get("LIBERO_DATASET_DIR") or (package / "datasets"))
    datasets.mkdir(parents=True, exist_ok=True)

    wanted = {
        "benchmark_root": str(package),
        "bddl_files": str(package / "bddl_files"),
        "init_states": str(package / "init_files"),
        "datasets": str(datasets),
        "assets": str(package / "assets"),
    }
    current = {}
    if config_file.is_file():
        try:
            import yaml

            current = yaml.safe_load(config_file.read_text()) or {}
        except Exception:
            current = {}
    merged = dict(current)
    merged.update(wanted)
    if merged != current:
        try:
            import yaml

            config_file.write_text(yaml.safe_dump(merged))
        except Exception:
            config_file.write_text("".join(f"{k}: {v}\n" for k, v in merged.items()))
        print(f"wrote LIBERO config {config_file}", flush=True)
    return config_file


ensure_libero_config()


SUITE = os.environ.get("SUITE", "libero_spatial")
TASK_IDS = json.loads(os.environ.get("TASK_IDS", "[0]"))
MAX_EPISODES = int(os.environ.get("MAX_EPISODES", "2"))
MAX_STEPS = int(os.environ.get("MAX_STEPS", "60"))
GONE_PER_EPISODE = int(os.environ.get("GONE_PER_EPISODE", "2"))
SHEET_N = int(os.environ.get("SHEET_N", "20"))
LAYER_INDEX = int(os.environ.get("LAYER_INDEX", "5"))
LAYER_NAME = os.environ.get("LAYER_NAME", "").strip()
GRIP_RADIUS = float(os.environ.get("GRIP_RADIUS", "0.03"))
DEPTH_MARGIN = float(os.environ.get("DEPTH_MARGIN", "0.02"))
MIN_PER_CLASS = int(os.environ.get("MIN_PER_CLASS", "20"))
POLICY_PATH = os.environ.get(
    "PROBE_POLICY_PATH",
    os.environ.get("POLICY_PATH", "lerobot/pi05_libero_finetuned"),
)


def repo_root() -> Path:
    return Path(os.environ.get("LOCAL_REPO", Path(__file__).resolve().parents[1]))


def out_dir() -> Path:
    path = repo_root() / "outputs" / "permanence" / "layer5_replay"
    path.mkdir(parents=True, exist_ok=True)
    return path


def quat_xyzw_to_axisangle(quat: np.ndarray) -> np.ndarray:
    quat = np.asarray(quat, dtype=np.float64)
    norm = np.linalg.norm(quat)
    if norm < 1e-8:
        return np.zeros(3, dtype=np.float32)
    quat = quat / norm
    w = float(np.clip(quat[3], -1.0, 1.0))
    den = float(np.sqrt(max(1.0 - w * w, 0.0)))
    if den < 1e-8:
        return np.zeros(3, dtype=np.float32)
    return (quat[:3] * 2.0 * np.arccos(w) / den).astype(np.float32)


def libero_state(raw: dict) -> np.ndarray:
    """8 numbers: gripper position (3), axis-angle (3), gripper opening (2)."""
    state = np.concatenate(
        [
            np.asarray(raw["robot0_eef_pos"], dtype=np.float32).reshape(3),
            quat_xyzw_to_axisangle(raw["robot0_eef_quat"]),
            np.asarray(raw["robot0_gripper_qpos"], dtype=np.float32).reshape(2),
        ]
    )
    if state.shape != (8,):
        raise RuntimeError(f"Expected an 8-number arm state, got {state.shape}")
    return state


def image_hwc(raw: dict, *keys: str) -> np.ndarray:
    for key in keys:
        if key in raw:
            image = np.ascontiguousarray(raw[key])
            # Match the LeRobot LIBERO convention: rotate the simulator image 180 degrees.
            return image[::-1, ::-1].copy()
    raise KeyError(f"None of {keys} are in the simulator observation. Keys: {sorted(raw)}")


def find_demo_file(task) -> Path:
    name = task.name
    folder = task.problem_folder
    roots = []
    if os.environ.get("LIBERO_DATASET_DIR"):
        roots.append(Path(os.environ["LIBERO_DATASET_DIR"]))
    roots.append(repo_root() / "data" / "libero" / "datasets")
    try:
        from libero.libero import get_libero_path

        roots.append(Path(get_libero_path("datasets")))
    except Exception:
        pass
    exact_names = [f"{name}_demo.hdf5", f"{name}.hdf5"]
    tried = []
    for root in roots:
        for exact in exact_names:
            candidate = root / folder / exact
            tried.append(candidate)
            if candidate.is_file():
                return candidate
        if root.is_dir():
            for hit in sorted(root.rglob("*.hdf5")):
                if name in hit.stem or (folder in hit.parts and name.split("_")[0] in hit.stem):
                    return hit
    raise FileNotFoundError(
        "No LIBERO demonstration file found. Cell 6 should have extracted "
        "libero_datasets.tar. Looked for:\n" + "\n".join(str(p) for p in tried[:12])
    )


def load_demo(path: Path, index: int) -> tuple[np.ndarray, np.ndarray]:
    with h5py.File(path, "r") as handle:
        demos = sorted(
            handle["data"].keys(),
            key=lambda name: int(re.search(r"(\d+)$", name).group(1)),
        )
        if index >= len(demos):
            raise IndexError(f"{path} has {len(demos)} demos, asked for index {index}")
        group = handle["data"][demos[index]]
        actions = np.asarray(group["actions"])
        if "states" not in group:
            raise KeyError(
                f"{path} demo {demos[index]} has no 'states'. Replay needs the simulator state. "
                f"Keys present: {sorted(group.keys())}"
            )
        states = np.asarray(group["states"])
    if len(states) == len(actions) + 1:
        states = states[:-1]
    steps = min(len(states), len(actions), MAX_STEPS)
    return actions[:steps], states[:steps]


def observe_state(env, state: np.ndarray) -> dict:
    state = np.asarray(state, dtype=np.float64)
    if hasattr(env, "set_init_state"):
        return env.set_init_state(state)
    env.reset()
    env.sim.set_state_from_flattened(state)
    env.sim.forward()
    inner = env.env if hasattr(env, "env") else env
    return inner._get_observations()


def sim_of(env):
    return env.sim


def target_words(sentence: str) -> list[str]:
    low = sentence.lower()
    verbs = ("pick up", "pick", "grasp", "take", "put", "place", "move", "open", "close", "turn")
    rels = (" between ", " next to ", " on top of ", " on the ", " in the ", " and ", " near ", " into ", " onto ")
    segment = low
    for verb in verbs:
        at = segment.find(verb)
        if at >= 0:
            segment = segment[at + len(verb) :]
            break
    cut = len(segment)
    for rel in rels:
        at = segment.find(rel)
        if 0 <= at < cut:
            cut = at
    stop = {"the", "a", "an", "and", "that", "with", "your", "from", "into", "onto"}
    words = [w for w in re.findall(r"[a-z]+", segment[:cut]) if len(w) > 3 and w not in stop]
    return words or [w for w in re.findall(r"[a-z]+", low) if len(w) > 3 and w not in stop]


def candidate_bodies(sim, sentence: str) -> dict[int, list[int]]:
    model = sim.model
    try:
        import mujoco

        free = int(mujoco.mjtJoint.mjJNT_FREE)
    except Exception:
        free = 0
    movable = set()
    for body in range(model.nbody):
        start = int(model.body_jntadr[body])
        count = int(model.body_jntnum[body])
        for joint in range(start, start + count):
            if joint >= 0 and int(model.jnt_type[joint]) == free:
                movable.add(body)
                break
    by_body: dict[int, list[int]] = {}
    for geom in range(model.ngeom):
        body = int(model.geom_bodyid[geom])
        if body in movable:
            by_body.setdefault(body, []).append(geom)
    words = target_words(sentence)
    named = {}
    for body, geoms in by_body.items():
        blob = " ".join((model.geom_id2name(g) or "") for g in geoms).lower()
        blob += " " + (model.body_id2name(body) or "").lower()
        if any(word in blob for word in words):
            named[body] = geoms
    return named or by_body


def finger_geoms(sim) -> list[int]:
    model = sim.model
    return [i for i in range(model.ngeom) if "finger" in (model.geom_id2name(i) or "").lower()]


def contact_body(sim, fingers: list[int], bodies: dict[int, list[int]]):
    if not fingers or not bodies:
        return None, None
    geom_to_body = {geom: body for body, geoms in bodies.items() for geom in geoms}
    finger_set = set(fingers)
    data = sim.data
    for index in range(int(data.ncon)):
        contact = data.contact[index]
        a, b = int(contact.geom1), int(contact.geom2)
        if a in finger_set and b in geom_to_body:
            return True, geom_to_body[b]
        if b in finger_set and a in geom_to_body:
            return True, geom_to_body[a]
    return False, None


def site_id(sim) -> int | None:
    for name in ("gripper0_grip_site", "gripper0_eef", "robot0_eef", "grip_site"):
        try:
            return int(sim.model.site_name2id(name))
        except Exception:
            continue
    return None


def camera_id(sim) -> int:
    model = sim.model
    for index in range(model.ncam):
        if (model.camera_id2name(index) or "") == "agentview":
            return index
    return 0


def occlusion(sim, geoms: list[int], cam: int, site: int | None):
    """Return ('hidden'|'visible', depth_gap, angle_ratio), or None if unmeasurable.

    depth_gap  = how far in front of the object the grip site sits, in metres.
    angle_ratio = angular separation object-vs-gripper divided by the gripper's
                  own angular radius. Below 1.0 the gripper covers the object.

    While the object is grasped the grip site and the object centroid nearly
    coincide, so angle_ratio is ~0 on every held frame. The depth margin is what
    separates "the gripper is between the camera and the object" from "they are
    at the same depth", and it is the knob to tune from the printed table.
    """
    if not geoms or site is None:
        return None
    data = sim.data
    camera = np.asarray(data.cam_xpos[cam], dtype=np.float64)
    obj = np.mean([np.asarray(data.geom_xpos[g], dtype=np.float64) for g in geoms], axis=0)
    grip = np.asarray(data.site_xpos[site], dtype=np.float64)
    to_obj = obj - camera
    to_grip = grip - camera
    obj_dist = float(np.linalg.norm(to_obj))
    grip_dist = float(np.linalg.norm(to_grip))
    if obj_dist < 1e-6 or grip_dist < 1e-6:
        return None
    cosine = float(np.clip(np.dot(to_obj, to_grip) / (obj_dist * grip_dist), -1.0, 1.0))
    angle = float(np.arccos(cosine))
    grip_angle = float(np.arctan2(GRIP_RADIUS, grip_dist))
    depth_gap = obj_dist - grip_dist
    ratio = angle / grip_angle if grip_angle > 1e-9 else float("inf")
    hidden = depth_gap > DEPTH_MARGIN and angle < grip_angle
    return ("hidden" if hidden else "visible"), depth_gap, ratio


def label_frame(sim, sentence: str, bodies: dict[int, list[int]]):
    """Return (label, body, geometry) - geometry is None unless the object is held."""
    fingers = finger_geoms(sim)
    touching, body = contact_body(sim, fingers, bodies)
    if touching is None:
        return None, None, None
    if not touching:
        return "not_held", None, None
    geoms = bodies.get(body) if body is not None else None
    if not geoms:
        return None, body, None
    measured = occlusion(sim, geoms, camera_id(sim), site_id(sim))
    if measured is None:
        return None, body, None
    seen, depth_gap, ratio = measured
    return f"held_{seen}", body, {"depth_gap": depth_gap, "angle_ratio": ratio}


def object_qpos_span(sim, body: int) -> tuple[int, int] | None:
    model = sim.model
    try:
        import mujoco

        free = int(mujoco.mjtJoint.mjJNT_FREE)
    except Exception:
        free = 0
    start = int(model.body_jntadr[body])
    count = int(model.body_jntnum[body])
    for joint in range(start, start + count):
        if joint >= 0 and int(model.jnt_type[joint]) == free:
            address = int(model.jnt_qposadr[joint])
            return address, address + 7
    return None


def move_object_away(sim, body: int):
    span = object_qpos_span(sim, body)
    if span is None:
        return None
    data = sim.data
    saved = data.qpos.copy()
    start, stop = span
    data.qpos[start : start + 3] = np.array([5.0, 5.0, -1.0])
    data.qpos[start + 3 : stop] = np.array([1.0, 0.0, 0.0, 0.0])
    data.qvel[:] = 0
    sim.forward()
    return saved


def restore_qpos(sim, saved: np.ndarray) -> None:
    sim.data.qpos[:] = saved
    sim.data.qvel[:] = 0
    sim.forward()


def render_after_edit(env) -> dict:
    """Re-render AFTER the object was moved.

    robosuite caches observables and only refreshes them on a sim step, so a
    plain _get_observations() hands back the image from before the edit. That
    silently makes every 'gone' frame a duplicate of its held frame.
    """
    inner = env.env if hasattr(env, "env") else env
    try:
        return inner._get_observations(force_update=True)
    except TypeError:
        if hasattr(inner, "_update_observables"):
            inner._update_observables(force=True)
        return inner._get_observations()


def pick_layer(policy):
    """Find the one PaliGemma language-model layer to hook, and say why if it cannot.

    Every module here sits under 'paligemma_with_expert', so the wrapper name
    contains the word 'expert'. The action expert must be excluded by its own
    module name ('gemma_expert'), never by a substring search on the full path.
    """
    named = dict(policy.named_modules())
    if LAYER_NAME:
        if LAYER_NAME not in named:
            print(f"\nLAYER_NAME={LAYER_NAME!r} is not a module. Modules ending in "
                  f"layers.{LAYER_INDEX}:", flush=True)
            for name in [n for n in named if n.endswith(f"layers.{LAYER_INDEX}")][:40]:
                print("   ", name, flush=True)
            raise RuntimeError(f"LAYER_NAME={LAYER_NAME!r} is not a module of this policy.")
        print("Hooking (from LAYER_NAME)", LAYER_NAME, flush=True)
        return LAYER_NAME, named[LAYER_NAME]

    every = [(name, module) for name, module in named.items()
             if name.endswith(f".layers.{LAYER_INDEX}") or name == f"layers.{LAYER_INDEX}"]
    print(f"\nModules ending in layers.{LAYER_INDEX}:", flush=True)
    for name, _ in every:
        print("   ", name, flush=True)

    def is_vision(name: str) -> bool:
        return "vision_tower" in name or "vision_model" in name

    def is_action_expert(name: str) -> bool:
        return "gemma_expert" in name

    body = [item for item in every if not is_vision(item[0]) and not is_action_expert(item[0])]
    language = [item for item in body if "language_model" in item[0]]
    chosen = language or body
    if len(chosen) != 1:
        raise RuntimeError(
            f"Expected exactly one PaliGemma layer {LAYER_INDEX}, found {len(chosen)}: "
            f"{[name for name, _ in chosen]}. Set LAYER_NAME to one of the module "
            "names printed above."
        )
    print("Hooking", chosen[0][0], flush=True)
    return chosen[0]


def load_policy():
    from lerobot.configs.policies import PreTrainedConfig
    from lerobot.envs.configs import LiberoEnv
    from lerobot.policies.factory import make_policy, make_pre_post_processors

    offline = os.environ.get("HF_HUB_OFFLINE") == "1"
    cache = os.environ.get("HF_HUB_CACHE")
    print(f"Loading policy {POLICY_PATH} (offline={offline})", flush=True)
    cfg = PreTrainedConfig.from_pretrained(POLICY_PATH, cache_dir=cache, local_files_only=offline)
    cfg.pretrained_path = Path(POLICY_PATH)
    cfg.device = "cuda"
    cfg.dtype = "bfloat16"
    cfg.compile_model = False
    cfg.gradient_checkpointing = False
    cfg.n_action_steps = 10
    env_cfg = LiberoEnv(task=SUITE, task_ids=[int(TASK_IDS[0])])
    policy = make_policy(cfg=cfg, env_cfg=env_cfg, rename_map={})
    policy.eval()
    pre, post = make_pre_post_processors(
        policy_cfg=cfg,
        pretrained_path=POLICY_PATH,
        preprocessor_overrides={
            "device_processor": {"device": "cuda"},
            "rename_observations_processor": {"rename_map": {}},
        },
    )
    _name, layer = pick_layer(policy)
    return policy, pre, post, layer


def as_image_tensor(picture: np.ndarray) -> torch.Tensor:
    tensor = torch.from_numpy(np.ascontiguousarray(picture)).permute(2, 0, 1).float().div(255.0)
    return tensor.unsqueeze(0)


def policy_batch(pre, raw: dict, sentence: str, policy) -> dict:
    batch = {
        "observation.images.image": as_image_tensor(image_hwc(raw, "agentview_image", "agentview_rgb")),
        "observation.images.image2": as_image_tensor(
            image_hwc(raw, "robot0_eye_in_hand_image", "eye_in_hand_rgb", "robot0_eye_in_hand_rgb")
        ),
        "observation.state": torch.from_numpy(libero_state(raw)).unsqueeze(0),
        "task": [sentence],
    }
    features = getattr(policy.config, "input_features", {}) or {}
    reference = batch["observation.images.image"]
    for key in features:
        if key.startswith("observation.images.") and key not in batch:
            batch[key] = torch.zeros_like(reference)
    return pre(batch)


def action_chunk(policy, batch):
    if hasattr(policy, "predict_action_chunk"):
        return policy.predict_action_chunk(batch)
    return policy.select_action(batch)


def first_action(chunk) -> np.ndarray:
    if isinstance(chunk, (tuple, list)):
        chunk = chunk[0]
    if isinstance(chunk, dict):
        chunk = chunk.get("action", next(iter(chunk.values())))
    action = chunk.detach().float().cpu().numpy()
    while action.ndim > 2:
        action = action[0]
    if action.ndim == 2:
        action = action[0]
    return np.asarray(action, dtype=np.float32)


def layer_vector(module) -> dict:
    box = {}

    def hook(_module, _inputs, output):
        tensor = output[0] if isinstance(output, (tuple, list)) else output
        if not torch.is_tensor(tensor):
            return
        if tensor.ndim == 3:
            tensor = tensor.mean(dim=1)
        elif tensor.ndim == 2:
            pass
        else:
            raise RuntimeError(f"Unexpected layer output shape {tuple(tensor.shape)}")
        box["vector"] = tensor[0].detach().float().cpu().numpy()

    handle = module.register_forward_hook(hook)
    return {"handle": handle, "box": box}


def thumb(picture: np.ndarray, label: str) -> np.ndarray:
    tile = cv2.resize(picture, (192, 192), interpolation=cv2.INTER_AREA)
    tile = cv2.copyMakeBorder(tile, 28, 0, 0, 0, cv2.BORDER_CONSTANT, value=(255, 255, 255))
    cv2.putText(tile, label, (6, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 0), 1, cv2.LINE_AA)
    return tile


def write_sheet(buckets: dict[str, list[np.ndarray]], path: Path) -> None:
    classes = ["held_visible", "held_hidden", "not_held", "gone"]
    rows = []
    blank = np.full((220, 192, 3), 255, dtype=np.uint8)
    for name in classes:
        tiles = buckets.get(name, [])[:SHEET_N]
        if not tiles:
            missing = blank.copy()
            cv2.putText(missing, f"{name}: none", (6, 110), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)
            tiles = [missing]
        row = list(tiles)
        if len(row) < 5:
            row = row + [blank.copy() for _ in range(5 - len(row))]
        # Pack up to SHEET_N tiles as several strips of 5.
        strips = []
        for start in range(0, len(row), 5):
            strip = row[start : start + 5]
            while len(strip) < 5:
                strip.append(blank.copy())
            strips.append(np.concatenate(strip, axis=1))
        rows.append(np.concatenate(strips, axis=0))
    width = max(row.shape[1] for row in rows)
    padded = []
    for row in rows:
        if row.shape[1] < width:
            pad = np.full((row.shape[0], width - row.shape[1], 3), 255, dtype=np.uint8)
            row = np.concatenate([row, pad], axis=1)
        padded.append(row)
    sheet = np.concatenate(padded, axis=0)
    bgr = cv2.cvtColor(sheet, cv2.COLOR_RGB2BGR)
    cv2.imwrite(str(path), bgr)


def main() -> None:
    from libero.libero import benchmark, get_libero_path
    from libero.libero.envs import OffScreenRenderEnv

    if not torch.cuda.is_available():
        raise SystemExit("This script needs the GPU. Run it in the Colab runtime after cell 6.")
    print(f"torch {torch.__version__} | cuda {torch.version.cuda} | {torch.cuda.get_device_name(0)}", flush=True)

    suite = benchmark.get_benchmark_dict()[SUITE]()
    policy, pre, post, layer = load_policy()
    captured = layer_vector(layer)
    records = []
    vectors = []
    actions = []
    sheet: dict[str, list[np.ndarray]] = {
        name: [] for name in ("held_visible", "held_hidden", "not_held", "gone")
    }
    both_class_episodes = 0
    geometry = []      # one row per held frame, for the calibration table
    gone_deltas = []   # mean |pixel change| the object removal produced

    try:
        for task_id in TASK_IDS:
            task = suite.get_task(int(task_id))
            sentence = task.language
            demo_path = find_demo_file(task)
            bddl = os.path.join(get_libero_path("bddl_files"), task.problem_folder, task.bddl_file)
            print(f"\nTask {task_id}: {sentence}", flush=True)
            print(f"  demos: {demo_path}", flush=True)
            env = OffScreenRenderEnv(
                bddl_file_name=bddl,
                camera_heights=256,
                camera_widths=256,
            )
            env.seed(0)
            try:
                with h5py.File(demo_path, "r") as handle:
                    n_demos = len(handle["data"].keys())
                n_demos = min(n_demos, MAX_EPISODES)
                for episode in range(n_demos):
                    _actions, states = load_demo(demo_path, episode)
                    observe_state(env, states[0])
                    bodies = candidate_bodies(sim_of(env), sentence)
                    counts = {"held_visible": 0, "held_hidden": 0, "not_held": 0, "dropped": 0}
                    gone_left = GONE_PER_EPISODE
                    for step, state in enumerate(states):
                        raw = observe_state(env, state)
                        sim = sim_of(env)
                        label, body, geom_stats = label_frame(sim, sentence, bodies)
                        if geom_stats is not None:
                            geometry.append(
                                {"label": label, "step": step, "episode": episode, **geom_stats}
                            )
                        if label is None:
                            counts["dropped"] += 1
                            continue
                        if hasattr(policy, "reset"):
                            policy.reset()
                        captured["box"].clear()
                        with torch.inference_mode():
                            chunk = action_chunk(policy, policy_batch(pre, raw, sentence, policy))
                        if "vector" not in captured["box"]:
                            raise RuntimeError(
                                "PaliGemma layer hook did not fire. The hooked module is not on the "
                                "forward path; set LAYER_NAME to a different module."
                            )
                        vectors.append(captured["box"]["vector"].astype(np.float16))
                        actions.append(first_action(chunk))
                        records.append(
                            {
                                "suite": SUITE,
                                "task_id": int(task_id),
                                "episode": episode,
                                "step": step,
                                "label": label,
                                "sentence": sentence,
                                "gone_of": -1,
                            }
                        )
                        counts[label] += 1
                        if len(sheet[label]) < SHEET_N:
                            picture = image_hwc(raw, "agentview_image", "agentview_rgb")
                            sheet[label].append(thumb(picture, f"{label} t{task_id}e{episode}:{step}"))
                        if gone_left > 0 and label.startswith("held_") and body is not None:
                            span = object_qpos_span(sim, body)
                            saved = move_object_away(sim, body)
                            if saved is not None and span is not None:
                                start, stop = span
                                arm_before = np.concatenate([saved[:start], saved[stop:]])
                                arm_after = np.concatenate(
                                    [sim.data.qpos[:start], sim.data.qpos[stop:]]
                                )
                                arm_same = np.allclose(arm_before, arm_after, atol=1e-5)
                                if arm_same:
                                    gone_raw = render_after_edit(env)
                                    before_pic = image_hwc(raw, "agentview_image", "agentview_rgb")
                                    after_pic = image_hwc(gone_raw, "agentview_image", "agentview_rgb")
                                    pixel_delta = float(
                                        np.abs(after_pic.astype(np.float32) - before_pic.astype(np.float32)).mean()
                                    )
                                    gone_deltas.append(pixel_delta)
                                    if pixel_delta < 1e-6:
                                        raise RuntimeError(
                                            "The 'gone' edit did not reach the renderer: the image is "
                                            "byte-identical to the held frame, so the gone vectors would "
                                            "be duplicates. render_after_edit() must force an observable "
                                            "refresh."
                                        )
                                    if hasattr(policy, "reset"):
                                        policy.reset()
                                    captured["box"].clear()
                                    with torch.inference_mode():
                                        gone_chunk = action_chunk(
                                            policy, policy_batch(pre, gone_raw, sentence, policy)
                                        )
                                    if "vector" not in captured["box"]:
                                        raise RuntimeError(
                                            "PaliGemma layer hook did not fire on the gone frame."
                                        )
                                    vectors.append(captured["box"]["vector"].astype(np.float16))
                                    actions.append(first_action(gone_chunk))
                                    records.append(
                                        {
                                            "suite": SUITE,
                                            "task_id": int(task_id),
                                            "episode": episode,
                                            "step": step,
                                            "label": "gone",
                                            "sentence": sentence,
                                            "gone_of": len(records) - 1,
                                        }
                                    )
                                    if len(sheet["gone"]) < SHEET_N:
                                        picture = image_hwc(gone_raw, "agentview_image", "agentview_rgb")
                                        sheet["gone"].append(
                                            thumb(picture, f"gone t{task_id}e{episode}:{step}")
                                        )
                                    gone_left -= 1
                                restore_qpos(sim, saved)
                    if counts["held_hidden"] > 0 and counts["not_held"] > 0:
                        both_class_episodes += 1
                    print(
                        f"  ep {episode}: "
                        + " ".join(f"{key}={value}" for key, value in counts.items()),
                        flush=True,
                    )
            finally:
                env.close()
    finally:
        captured["handle"].remove()

    if not records:
        raise SystemExit("No frames were labeled. Check that the demonstration files contain states.")

    destination = out_dir()
    np.savez_compressed(
        destination / "layer5_frames.npz",
        vectors=np.stack(vectors),
        actions=np.stack(actions),
        labels=np.array([row["label"] for row in records]),
        task_ids=np.array([row["task_id"] for row in records], dtype=np.int32),
        episodes=np.array([row["episode"] for row in records], dtype=np.int32),
        steps=np.array([row["step"] for row in records], dtype=np.int32),
        gone_of=np.array([row["gone_of"] for row in records], dtype=np.int32),
    )
    (destination / "frames.json").write_text(json.dumps(records, indent=2))
    sheet_path = destination / "contact_sheet.png"
    write_sheet(sheet, sheet_path)
    counts = {}
    for row in records:
        counts[row["label"]] = counts.get(row["label"], 0) + 1
    held = [row for row in geometry if row["label"].startswith("held_")]
    gaps = np.array([row["depth_gap"] for row in held], dtype=np.float64)
    ratios = np.array([row["angle_ratio"] for row in held], dtype=np.float64)
    table = []
    if len(gaps):
        print("\n--- occlusion calibration (held frames only) ---", flush=True)
        print(f"depth_gap percentiles (m): " + "  ".join(
            f"p{p}={np.percentile(gaps, p):+.4f}" for p in (5, 10, 25, 50, 75, 90, 95)), flush=True)
        print(f"angle_ratio percentiles:   " + "  ".join(
            f"p{p}={np.percentile(ratios, p):.3f}" for p in (5, 10, 25, 50, 75, 90, 95)), flush=True)
        print(f"\nheld frames = {len(gaps)}. Split at the current "
              f"GRIP_RADIUS={GRIP_RADIUS} DEPTH_MARGIN={DEPTH_MARGIN}:", flush=True)
        print("  margin   hidden  visible", flush=True)
        covered = ratios < 1.0
        for margin in (0.00, 0.01, 0.02, 0.03, 0.04, 0.06, 0.08):
            hidden_n = int(((gaps > margin) & covered).sum())
            table.append({"depth_margin": margin, "hidden": hidden_n,
                          "visible": int(len(gaps) - hidden_n)})
            mark = "  <== current" if abs(margin - DEPTH_MARGIN) < 1e-9 else ""
            print(f"  {margin:.2f}   {hidden_n:6d}  {len(gaps) - hidden_n:7d}{mark}", flush=True)

    audit = {
        "counts": counts,
        "min_per_class": MIN_PER_CLASS,
        "grip_radius": GRIP_RADIUS,
        "depth_margin": DEPTH_MARGIN,
        "held_frames_measured": len(held),
        "depth_gap_percentiles": {f"p{p}": float(np.percentile(gaps, p))
                                  for p in (5, 25, 50, 75, 95)} if len(gaps) else {},
        "angle_ratio_percentiles": {f"p{p}": float(np.percentile(ratios, p))
                                    for p in (5, 25, 50, 75, 95)} if len(ratios) else {},
        "hidden_visible_split_by_margin": table,
        "gone_pixel_change_mean": float(np.mean(gone_deltas)) if gone_deltas else 0.0,
        "gone_pixel_change_min": float(np.min(gone_deltas)) if gone_deltas else 0.0,
        "classes_below_minimum": [name for name in
                                  ("held_visible", "held_hidden", "not_held", "gone")
                                  if counts.get(name, 0) < MIN_PER_CLASS],
    }

    summary = {
        "policy": POLICY_PATH,
        "suite": SUITE,
        "task_ids": TASK_IDS,
        "frames": len(records),
        "vector_dim": int(vectors[0].shape[0]),
        "counts": counts,
        "episodes_with_both_held_hidden_and_not_held": both_class_episodes,
        "sheet": str(sheet_path),
        "npz": str(destination / "layer5_frames.npz"),
        "label_audit": audit,
    }
    (destination / "summary.json").write_text(json.dumps(summary, indent=2))
    print("\n" + json.dumps(summary, indent=2), flush=True)
    print("\nOpen contact_sheet.png. Each tile's label must match what you see.", flush=True)
    print("If a class is often wrong, do not train the transcoder. Fix the rule and rerun.", flush=True)

    if audit["classes_below_minimum"]:
        print("\n===== label audit FAILED =====", flush=True)
        for name in audit["classes_below_minimum"]:
            print(f"  {name}: {counts.get(name, 0)} frames, need {MIN_PER_CLASS}", flush=True)
        print("summary.json holds the audit. Pick a DEPTH_MARGIN from the table above "
              "that splits held frames into two classes of at least "
              f"{MIN_PER_CLASS}, or raise MAX_EPISODES / GONE_PER_EPISODE.", flush=True)
        raise SystemExit(1)
    print(f"\nlabel audit passed: every class has at least {MIN_PER_CLASS} frames.", flush=True)


if __name__ == "__main__":
    try:
        main()
    except SystemExit:
        raise
    except BaseException:
        print("\n===== collect_layer5_replay.py failed =====", flush=True)
        traceback.print_exc()
        raise SystemExit(1)
'''

scripts = LOCAL_REPO / "scripts"
scripts.mkdir(parents=True, exist_ok=True)
script = scripts / "collect_layer5_replay.py"
script.write_text(SCRIPT_SRC)
print("wrote", script, script.stat().st_size, "bytes", flush=True)

datasets = Path(os.environ.get("LIBERO_DATASET_DIR", LOCAL_REPO / "data/libero/datasets"))
hdf5 = sorted(datasets.rglob("*.hdf5")) if datasets.is_dir() else []
print(f"dataset dir: {datasets}  ({len(hdf5)} hdf5 files)", flush=True)
if not hdf5:
    raise SystemExit(f"No .hdf5 demos under {datasets}. Run the fetch cell first.")

env = os.environ.copy()
env["PYTHONPATH"] = str(scripts) + os.pathsep + env.get("PYTHONPATH", "")
env["PYTHONUNBUFFERED"] = "1"
env.setdefault("MUJOCO_GL", "egl")
env.setdefault("PYOPENGL_PLATFORM", "egl")
env.setdefault("LIBERO_DATASET_DIR", str(datasets))
probe = """
import collect_layer5_replay as collect
needed = [
    "load_policy", "find_demo_file", "load_demo", "observe_state", "sim_of",
    "candidate_bodies", "label_frame", "object_qpos_span", "move_object_away",
    "restore_qpos", "render_after_edit", "image_hwc", "libero_state", "as_image_tensor",
]
missing = [name for name in needed if not hasattr(collect, name)]
if missing:
    raise SystemExit("collect_layer5_replay is missing: " + ", ".join(missing))
print("collect_layer5_replay OK", collect.SUITE, collect.TASK_IDS)
"""
proc = subprocess.run([str(PYTHON), "-u", "-c", probe], cwd=str(scripts), env=env, text=True)
if proc.returncode != 0:
    raise SystemExit(f"collect_layer5_replay import failed with code {proc.returncode}")


## Occlusion features

The contrast cell above has to finish first. It writes `transcoder.pt`. The next cell does not train again.

Read the feature table, then the circuit table.

- A feature in the first table rose for paint on the bowl and stayed quiet for paint off the bowl, color, and absence.
- A count in the circuit table counts only if it closes at least 10 points more of the occlusion action gap than the random patch and at least 10 points more than the color features.
- If the first table says none, no occlusion feature passed. The second table shows the largest occlusion scores so you can see what they also responded to.


In [ ]:
# ONE CELL — controlled recolor / occlusion / absence, per-token transcoder, circuit trace
# Paste into the existing Colab runtime after setup. Does not overwrite layer5_replay or fidelity outputs.

import os, sys, subprocess
from pathlib import Path

os.environ["CTRL_MAX_EPISODES"] = "2"
os.environ["CTRL_MAX_STEPS"] = "24"
os.environ["CTRL_PROBE_FRAMES"] = "4"
os.environ["CTRL_TRAIN_EXTRA"] = "3"
os.environ["CTRL_N_FEATURES"] = "512"
os.environ["CTRL_K"] = "16"
os.environ["CTRL_TRAIN_STEPS"] = "400"
os.environ["CTRL_TOP_FEATURES"] = "8"
os.environ["CTRL_SEED"] = "0"
os.environ.setdefault("SUITE", "libero_spatial")
os.environ.setdefault("TASK_IDS", "[0]")
os.environ.setdefault(
    "LAYER_NAME",
    "model.paligemma_with_expert.paligemma.model.language_model.layers.5",
)
os.environ.setdefault("MUJOCO_GL", "egl")
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

LOCAL_REPO = Path(os.environ.get("LOCAL_REPO", "/content/groot-run"))
PYTHON = Path("/content/lerobot-venv/bin/python")
if not PYTHON.exists():
    PYTHON = Path(sys.executable)

SCRIPT = r'''#!/usr/bin/env python
"""Controlled color, occlusion, and absence contrasts for a per-token transcoder.

The task object stands in for the apple. At a frozen arm state, language
prompt, and noise seed, each condition changes one thing:

  base              original render
  recolor           object color only. Pixels that change are the object mask.
  absent            object teleported away; arm qpos unchanged
  occluded          gray paint on half of those pixels; every other pixel matches base
  slab_miss         the same gray shape, translated off the object
  occluded_absent   the occluded disk painted on the absent render

A TopK transcoder is fit on layer-5 tokens and, when the module exists, predicts
layer-6 tokens. Circuit tracing patches only the contrast-specific features and
compares them with norm-matched controls, plus full-token, pooled, and
token-structure patches on the same pairs.

Run inside the Colab runtime that already has scripts/collect_layer5_replay.py.
"""

from __future__ import annotations

import json
import os
import random
from pathlib import Path
from typing import Literal, assert_never

import cv2
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F

CONDITION_NAMES = (
    "base",
    "recolor",
    "absent",
    "occluded",
    "slab_miss",
    "occluded_absent",
)
CONTRAST_NAMES = ("color", "absence", "occlusion")
ContrastName = Literal["color", "absence", "occlusion"]
GRAY = (128, 128, 128)
GREEN = (20, 170, 40)
RED = (210, 30, 30)


def seed_all(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def changed_pixels(before: np.ndarray, after: np.ndarray) -> np.ndarray:
    return np.any(before.astype(np.int16) != after.astype(np.int16), axis=-1)


def changed_fraction(before: np.ndarray, after: np.ndarray) -> float:
    return float(changed_pixels(before, after).mean())


def pixel_rmse(before: np.ndarray, after: np.ndarray) -> float:
    delta = before.astype(np.float32) - after.astype(np.float32)
    return float(np.sqrt(np.mean(delta * delta)))


def outside_fraction(before: np.ndarray, after: np.ndarray, mask: np.ndarray) -> float:
    changed = changed_pixels(before, after)
    return float(np.logical_and(changed, ~mask).mean())


def paint_mask(image: np.ndarray, mask: np.ndarray, color: tuple[int, int, int]) -> np.ndarray:
    painted = image.copy()
    painted[mask] = color
    return painted


def disk_mask(height: int, width: int, cx: float, cy: float, radius: float) -> np.ndarray:
    yy, xx = np.ogrid[:height, :width]
    return (xx - cx) ** 2 + (yy - cy) ** 2 <= radius ** 2


def dilate_mask(mask: np.ndarray, radius: int) -> np.ndarray:
    out = mask.copy()
    height, width = mask.shape
    for dy in range(-radius, radius + 1):
        for dx in range(-radius, radius + 1):
            if dx * dx + dy * dy > radius * radius:
                continue
            src_y0, src_y1 = max(0, -dy), min(height, height - dy)
            src_x0, src_x1 = max(0, -dx), min(width, width - dx)
            dst_y0, dst_x0 = max(0, dy), max(0, dx)
            dst_y1 = dst_y0 + (src_y1 - src_y0)
            dst_x1 = dst_x0 + (src_x1 - src_x0)
            out[dst_y0:dst_y1, dst_x0:dst_x1] |= mask[src_y0:src_y1, src_x0:src_x1]
    return out


def half_mask(full: np.ndarray) -> np.ndarray:
    ys, xs = np.nonzero(full)
    if len(xs) == 0:
        return full.copy()
    cx = float(np.median(xs))
    xx = np.arange(full.shape[1], dtype=np.float64)[None, :]
    left = full & (xx < cx)
    if int(left.sum()) < 10:
        left = full & (xx <= cx)
    return left


def shift_mask(mask: np.ndarray, dy: int, dx: int) -> np.ndarray:
    out = np.zeros_like(mask)
    height, width = mask.shape
    src_y0, src_y1 = max(0, -dy), min(height, height - dy)
    src_x0, src_x1 = max(0, -dx), min(width, width - dx)
    dst_y0, dst_x0 = max(0, dy), max(0, dx)
    dst_y1 = dst_y0 + (src_y1 - src_y0)
    dst_x1 = dst_x0 + (src_x1 - src_x0)
    out[dst_y0:dst_y1, dst_x0:dst_x1] = mask[src_y0:src_y1, src_x0:src_x1]
    return out


def translated_copy(mask: np.ndarray, avoid: np.ndarray) -> np.ndarray | None:
    """Copy `mask` to a corner. The copy keeps every pixel and misses `avoid`."""
    ys, xs = np.nonzero(mask)
    if len(xs) == 0:
        return None
    height, width = mask.shape
    box_h = int(ys.max() - ys.min())
    box_w = int(xs.max() - xs.min())
    dilated = dilate_mask(avoid, 4)
    corners = (
        (2, 2),
        (2, width - box_w - 3),
        (height - box_h - 3, 2),
        (height - box_h - 3, width - box_w - 3),
    )
    for top, left in corners:
        if top < 0 or left < 0:
            continue
        moved = shift_mask(mask, int(top - ys.min()), int(left - xs.min()))
        if int(moved.sum()) != int(mask.sum()):
            continue
        if np.logical_and(moved, dilated).any():
            continue
        return moved
    return None


def union_disks(height: int, width: int, disks: list[tuple[float, float, float]]) -> np.ndarray:
    mask = np.zeros((height, width), dtype=bool)
    for cx, cy, radius in disks:
        mask |= disk_mask(height, width, cx, cy, radius)
    return mask


def world_to_camera(xpos: np.ndarray, xmat: np.ndarray, point: np.ndarray) -> np.ndarray:
    rotation = np.asarray(xmat, dtype=np.float64).reshape(3, 3)
    return rotation.T @ (np.asarray(point, dtype=np.float64) - np.asarray(xpos, dtype=np.float64))


def project_point(
    xpos: np.ndarray,
    xmat: np.ndarray,
    fovy_deg: float,
    point: np.ndarray,
    height: int,
    width: int,
    flip180: bool,
) -> tuple[float, float, float] | None:
    """Project a world point. Returns (u, v, depth) in the image the policy sees."""
    camera = world_to_camera(xpos, xmat, point)
    depth = float(-camera[2])
    if depth <= 1e-6:
        return None
    fovy = np.deg2rad(float(fovy_deg))
    fy = 0.5 * height / np.tan(0.5 * fovy)
    fx = fy
    u = fx * (camera[0] / depth) + width / 2.0
    v = fy * (-camera[1] / depth) + height / 2.0
    if flip180:
        u = (width - 1) - u
        v = (height - 1) - v
    return float(u), float(v), depth


def pixel_radius(fovy_deg: float, height: int, depth: float, rbound: float) -> float:
    fovy = np.deg2rad(float(fovy_deg))
    fy = 0.5 * height / np.tan(0.5 * fovy)
    return float(fy * rbound / max(depth, 1e-6))


def swapped_rgba(current: np.ndarray) -> np.ndarray:
    """Red-dominant paint becomes green. Green-dominant paint becomes red."""
    rgb = np.asarray(current, dtype=np.float64)[:, :3].mean(axis=0)
    alpha = np.asarray(current, dtype=np.float32)[:, 3:4]
    if rgb[0] >= rgb[1]:
        color = np.array([0.05, 0.72, 0.12], dtype=np.float32)
    else:
        color = np.array([0.82, 0.08, 0.08], dtype=np.float32)
    tiled = np.repeat(color[None, :], len(current), axis=0)
    return np.concatenate([tiled, alpha], axis=1)


def mean_object_color(image: np.ndarray, mask: np.ndarray) -> np.ndarray:
    pixels = image[mask]
    if len(pixels) == 0:
        return np.zeros(3, dtype=np.float64)
    return pixels.astype(np.float64).mean(axis=0)


def pixel_recolor(image: np.ndarray, mask: np.ndarray) -> np.ndarray:
    rgb = mean_object_color(image, mask)
    color = GREEN if rgb[0] >= rgb[1] else RED
    return paint_mask(image, mask, color)


def action_array(chunk) -> np.ndarray:
    if isinstance(chunk, (tuple, list)):
        chunk = chunk[0]
    if isinstance(chunk, dict):
        chunk = chunk.get("action", next(iter(chunk.values())))
    action = chunk.detach().float().cpu().numpy()
    if action.ndim == 3:
        if action.shape[0] != 1:
            raise RuntimeError(f"Expected a batch of 1, got {action.shape}")
        action = action[0]
    if action.ndim != 2 or action.shape[0] < 2:
        raise RuntimeError("Need a full action chunk, not a single action.")
    return np.asarray(action, dtype=np.float32)


def action_metrics(edited: np.ndarray, source: np.ndarray, donor: np.ndarray) -> dict:
    edited64 = edited.astype(np.float64)
    source64 = source.astype(np.float64)
    donor64 = donor.astype(np.float64)
    baseline = float(np.sqrt(np.mean((source64 - donor64) ** 2)))
    error = float(np.sqrt(np.mean((edited64 - donor64) ** 2)))
    gap = None if baseline <= 1e-4 else (baseline - error) / baseline
    return {
        "source_to_donor_rmse": baseline,
        "edited_to_donor_rmse": error,
        "fraction_gap_closed": gap,
    }


def structure_delta(donor: np.ndarray, source: np.ndarray) -> np.ndarray:
    """Remove the token-mean from a [tokens, dim] difference."""
    delta = donor.astype(np.float32) - source.astype(np.float32)
    return delta - delta.mean(axis=0, keepdims=True)


def moving_token_mask(
    base_tokens: np.ndarray, edited_tokens: np.ndarray, quantile: float = 0.85
) -> np.ndarray:
    delta = np.linalg.norm(
        edited_tokens.astype(np.float32) - base_tokens.astype(np.float32), axis=-1
    )
    if not np.isfinite(delta).all():
        raise RuntimeError("Nonfinite token delta.")
    cut = float(np.quantile(delta, quantile))
    mask = delta >= cut
    if int(mask.sum()) < 8:
        mask = np.zeros(len(delta), dtype=bool)
        mask[np.argsort(-delta)[:8]] = True
    return mask


def contrast_score(base_code: np.ndarray, edited_code: np.ndarray, mask: np.ndarray) -> np.ndarray:
    return (edited_code[mask] - base_code[mask]).mean(axis=0)


def pick_features(
    primary: np.ndarray,
    others: list[np.ndarray],
    k: int,
    min_specificity: float = 0.5,
) -> tuple[np.ndarray, bool]:
    primary64 = np.asarray(primary, dtype=np.float64)
    other64 = [np.asarray(other, dtype=np.float64) for other in others]
    denominator = np.abs(primary64) + sum(np.abs(other) for other in other64) + 1e-8
    specificity = np.abs(primary64) / denominator
    magnitude_cut = float(np.quantile(np.abs(primary64), 0.90))
    eligible = np.flatnonzero(
        (specificity > min_specificity) & (np.abs(primary64) >= magnitude_cut)
    )
    specific = True
    if eligible.size < min(4, k):
        specific = False
        eligible = np.arange(primary64.size)
    order = eligible[np.argsort(-np.abs(primary64[eligible]))]
    return order[:k].astype(np.int64), specific


def off_contrast(name: ContrastName) -> ContrastName:
    match name:
        case "color":
            return "absence"
        case "absence":
            return "color"
        case "occlusion":
            return "color"
        case _ as unexpected:
            assert_never(unexpected)


def remap_features(code_delta: np.ndarray, src_ids: np.ndarray, dst_ids: np.ndarray) -> np.ndarray:
    remapped = np.zeros_like(code_delta)
    remapped[:, dst_ids] = code_delta[:, src_ids]
    return remapped


def keep_features(code_delta: np.ndarray, feature_ids: np.ndarray) -> np.ndarray:
    kept = np.zeros_like(code_delta)
    kept[:, feature_ids] = code_delta[:, feature_ids]
    return kept


def match_l2(delta: np.ndarray, reference: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    ref = float(np.linalg.norm(reference))
    cur = float(np.linalg.norm(delta))
    if cur < eps or ref < eps:
        return delta
    return delta * (ref / cur)


def mean_std(chunks: list[np.ndarray]) -> tuple[np.ndarray, np.ndarray]:
    count = 0
    total = None
    for chunk in chunks:
        flat = chunk.reshape(-1, chunk.shape[-1]).astype(np.float64)
        total = flat.sum(axis=0) if total is None else total + flat.sum(axis=0)
        count += len(flat)
    if total is None or count < 2:
        raise RuntimeError("Not enough tokens to fit the transcoder.")
    mean = total / count
    var = np.zeros_like(mean)
    for chunk in chunks:
        flat = chunk.reshape(-1, chunk.shape[-1]).astype(np.float64)
        var += ((flat - mean) ** 2).sum(axis=0)
    std = np.sqrt(var / (count - 1)).clip(min=1e-6)
    return mean.astype(np.float32), std.astype(np.float32)


def r2_score(pred: np.ndarray, target: np.ndarray) -> float:
    resid = float(np.mean((pred - target) ** 2))
    var = float(np.mean((target - target.mean(axis=0)) ** 2))
    if var < 1e-12:
        return 0.0
    return 1.0 - resid / var


def cosine(a: np.ndarray, b: np.ndarray) -> float:
    denom = float(np.linalg.norm(a) * np.linalg.norm(b))
    if denom < 1e-12:
        return 0.0
    return float(np.dot(a, b) / denom)


def evenly(items: list, count: int) -> list:
    if len(items) <= count:
        return list(items)
    indexes = np.linspace(0, len(items) - 1, count).round().astype(int)
    chosen = []
    seen: set[int] = set()
    for index in indexes:
        key = int(index)
        if key not in seen:
            seen.add(key)
            chosen.append(items[key])
    return chosen


def choose_frames(records: list[dict], count: int) -> list[dict]:
    held = [row for row in records if str(row["label"]).startswith("held_")]
    rest = [row for row in records if not str(row["label"]).startswith("held_")]
    picked = evenly(held, count)
    if len(picked) < count:
        picked.extend(evenly(rest, count - len(picked)))
    return picked


def full_cover_interpretation(outside: float, action_rmse: float) -> str:
    """Describe the full-cover diagnostic.

    Images agree when fewer than 0.2% of pixels still differ. Action chunks
    agree below an RMSE of 1e-3. A larger RMSE on agreeing images is the
    residual floor for near-matched inputs. The identical-input re-forward is
    the determinism check, and this diagnostic does not replace it.
    """
    images_agree = outside < 0.002
    actions_agree = action_rmse < 1e-3
    if images_agree and actions_agree:
        return (
            "Full cover makes the present and absent images agree, and the action chunks agree. "
            "A single forward has no remaining evidence the object is behind the cover."
        )
    if images_agree:
        return (
            f"The covered images agree, and the action chunks differ by RMSE {action_rmse:.5f}. "
            "The identical-input re-forward already matched, so this gap is the residual floor "
            "for near-matched images. Read later gap-closed numbers against this floor."
        )
    return (
        "Pixels outside the cover still differ between present and absent. "
        "The partial-occlusion contrast and the absence contrast are the controlled comparisons."
    )


class TokenTranscoder(nn.Module):
    """Per-token TopK transcoder from layer 5, optionally predicting layer 6."""

    def __init__(self, dim: int, n_features: int, k: int, predict_next: bool) -> None:
        super().__init__()
        if k >= n_features:
            raise ValueError("k must be smaller than n_features")
        self.k = k
        self.predict_next = predict_next
        self.encoder = nn.Linear(dim, n_features)
        self.decoder = nn.Linear(n_features, dim, bias=False)
        if predict_next:
            self.next_decoder = nn.Linear(n_features, dim, bias=False)
        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.kaiming_uniform_(self.encoder.weight, a=5**0.5)
        nn.init.zeros_(self.encoder.bias)
        with torch.no_grad():
            decoder = self.encoder.weight.detach().T
            decoder = decoder / decoder.norm(dim=0, keepdim=True).clamp_min(1e-8)
            self.decoder.weight.copy_(decoder)
            if self.predict_next:
                self.next_decoder.weight.copy_(decoder)

    def encode(self, normalized: torch.Tensor) -> torch.Tensor:
        preactivations = self.encoder(normalized)
        values, indices = preactivations.topk(self.k, dim=-1)
        code = torch.zeros_like(preactivations)
        code.scatter_(-1, indices, F.relu(values))
        return code

    def forward(self, normalized: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor | None, torch.Tensor]:
        code = self.encode(normalized)
        recon = self.decoder(code)
        nxt = self.next_decoder(code) if self.predict_next else None
        return recon, nxt, code

    def renorm_decoder(self) -> None:
        with torch.no_grad():
            norms = self.decoder.weight.norm(dim=0).clamp_min(1e-8)
            self.decoder.weight.div_(norms)
            self.encoder.weight.mul_(norms[:, None])


def train_transcoder(
    layer5: list[np.ndarray],
    layer6: list[np.ndarray] | None,
    n_features: int,
    k: int,
    steps: int,
    batch: int,
    seed: int,
) -> tuple[TokenTranscoder, np.ndarray, np.ndarray, np.ndarray | None, np.ndarray | None]:
    mean5, std5 = mean_std(layer5)
    x = np.concatenate([chunk.reshape(-1, chunk.shape[-1]) for chunk in layer5], axis=0)
    x_norm = ((x - mean5) / std5).astype(np.float32)
    predict_next = layer6 is not None
    y_norm = None
    mean6 = None
    std6 = None
    if layer6 is not None:
        mean6, std6 = mean_std(layer6)
        y = np.concatenate([chunk.reshape(-1, chunk.shape[-1]) for chunk in layer6], axis=0)
        y_norm = ((y - mean6) / std6).astype(np.float32)
        if len(y_norm) != len(x_norm):
            raise RuntimeError("Layer 5 and layer 6 token counts differ.")
    model = TokenTranscoder(x_norm.shape[1], n_features, k, predict_next)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    rng = np.random.default_rng(seed)
    model.train()
    for step in range(steps):
        take = min(batch, len(x_norm))
        idx = rng.integers(0, len(x_norm), size=take)
        xb = torch.from_numpy(np.ascontiguousarray(x_norm[idx]))
        optimizer.zero_grad(set_to_none=True)
        recon, nxt, _code = model(xb)
        loss = F.mse_loss(recon, xb)
        if nxt is not None and y_norm is not None:
            yb = torch.from_numpy(np.ascontiguousarray(y_norm[idx]))
            loss = loss + F.mse_loss(nxt, yb)
        loss.backward()
        optimizer.step()
        model.renorm_decoder()
        if step == 0 or (step + 1) % 100 == 0 or step + 1 == steps:
            print(f"  transcoder step {step + 1}/{steps} loss={float(loss.detach()):.5f}", flush=True)
    model.eval()
    return model, mean5, std5, mean6, std6


def encode_tokens(model: TokenTranscoder, tokens: np.ndarray, mean: np.ndarray, std: np.ndarray) -> np.ndarray:
    normalized = (tokens.astype(np.float32) - mean) / std
    with torch.no_grad():
        code = model.encode(torch.from_numpy(np.ascontiguousarray(normalized)))
    return code.numpy()


def decode_delta(model: TokenTranscoder, code_delta: np.ndarray, std: np.ndarray) -> np.ndarray:
    with torch.no_grad():
        recon = model.decoder(torch.from_numpy(np.ascontiguousarray(code_delta.astype(np.float32))))
        raw = recon * torch.from_numpy(std.astype(np.float32))
    return raw.numpy().astype(np.float32)


def reconstruction(model: TokenTranscoder, tokens: np.ndarray, mean: np.ndarray, std: np.ndarray) -> np.ndarray:
    code = encode_tokens(model, tokens, mean, std)
    with torch.no_grad():
        recon = model.decoder(torch.from_numpy(code))
        raw = recon * torch.from_numpy(std) + torch.from_numpy(mean)
    return raw.numpy().astype(np.float32)


def next_prediction(
    model: TokenTranscoder,
    tokens: np.ndarray,
    mean5: np.ndarray,
    std5: np.ndarray,
    mean6: np.ndarray,
    std6: np.ndarray,
) -> np.ndarray:
    code = encode_tokens(model, tokens, mean5, std5)
    with torch.no_grad():
        pred = model.next_decoder(torch.from_numpy(code))
        raw = pred * torch.from_numpy(std6) + torch.from_numpy(mean6)
    return raw.numpy().astype(np.float32)


def load_collect_module():
    # Imported here so unit tests can load this module without LIBERO, and so
    # MUJOCO_GL / MAX_STEPS are set before collect_layer5_replay reads them.
    import collect_layer5_replay as collect

    return collect


def env_int(name: str, default: int) -> int:
    return int(os.environ.get(name, str(default)))


def configure_environment() -> None:
    os.environ.setdefault("MUJOCO_GL", "egl")
    os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
    os.environ.setdefault("MPLBACKEND", "Agg")
    os.environ.setdefault("SUITE", "libero_spatial")
    os.environ.setdefault("TASK_IDS", "[0]")
    os.environ.setdefault(
        "LAYER_NAME",
        "model.paligemma_with_expert.paligemma.model.language_model.layers.5",
    )
    # These two override any earlier collection cell so this run stays on the short controlled set.
    os.environ["MAX_STEPS"] = os.environ.get("CTRL_MAX_STEPS", "24")
    os.environ["MAX_EPISODES"] = os.environ.get("CTRL_MAX_EPISODES", "2")
    os.environ.setdefault("LOCAL_REPO", "/content/groot-run")


def unpack_output(output):
    tensor = output[0] if isinstance(output, (tuple, list)) else output
    if not torch.is_tensor(tensor) or tensor.ndim != 3 or tensor.shape[0] != 1:
        shape = tuple(tensor.shape) if torch.is_tensor(tensor) else type(tensor)
        raise RuntimeError(f"Expected layer output [1, tokens, dim], got {shape}")
    return tensor


def repack_output(output, edited):
    if isinstance(output, tuple):
        return (edited,) + output[1:]
    if isinstance(output, list):
        return [edited] + output[1:]
    return edited


def camera_index(sim, needle: str) -> int | None:
    model = sim.model
    for index in range(int(model.ncam)):
        name = model.camera_id2name(index) or ""
        if needle in name:
            return index
    return None


def project_geoms(sim, geoms: list[int], cam: int, height: int, width: int) -> list[tuple[float, float, float]]:
    data = sim.data
    model = sim.model
    xpos = np.asarray(data.cam_xpos[cam], dtype=np.float64)
    xmat = np.asarray(data.cam_xmat[cam], dtype=np.float64)
    fovy = float(model.cam_fovy[cam])
    disks = []
    for geom in geoms:
        point = np.asarray(data.geom_xpos[geom], dtype=np.float64)
        projected = project_point(xpos, xmat, fovy, point, height, width, flip180=True)
        if projected is None:
            continue
        u, v, depth = projected
        radius = pixel_radius(fovy, height, depth, float(model.geom_rbound[geom]))
        if radius < 6 or u < -radius or v < -radius or u >= width + radius or v >= height + radius:
            continue
        disks.append((u, v, radius))
    return disks


def masks_from_disks(height: int, width: int, disks: list[tuple[float, float, float]]):
    if not disks:
        return None, None, None
    full = union_disks(height, width, disks)
    if int(full.sum()) < 80:
        return None, None, None
    radius = float(np.median([disk[2] for disk in disks]))
    if radius > 0.45 * min(height, width):
        return None, None, None
    partial = half_mask(full)
    if int(partial.sum()) < 40:
        return None, None, None
    control = translated_copy(partial, full)
    if control is None:
        return None, None, None
    return full, partial, control


def images_of(collect, raw: dict) -> tuple[np.ndarray, np.ndarray]:
    agent = collect.image_hwc(raw, "agentview_image", "agentview_rgb")
    wrist = collect.image_hwc(
        raw, "robot0_eye_in_hand_image", "eye_in_hand_rgb", "robot0_eye_in_hand_rgb"
    )
    return agent, wrist


def write_rgba(sim, geoms: list[int], rgba: np.ndarray) -> None:
    for index, geom in enumerate(geoms):
        sim.model.geom_rgba[geom, :] = rgba[index]


def render_rgba_images(collect, env, sim, geoms: list[int]):
    current = np.array(sim.model.geom_rgba[geoms], dtype=np.float32).copy()
    edited = swapped_rgba(current)
    write_rgba(sim, geoms, edited)
    sim.forward()
    try:
        return images_of(collect, collect.render_after_edit(env))
    finally:
        write_rgba(sim, geoms, current)
        sim.forward()


def masks_from_change(before: np.ndarray, after: np.ndarray):
    """Object mask is the set of pixels the color swap actually changed."""
    changed = changed_pixels(before, after)
    count = int(changed.sum())
    fraction = float(changed.mean())
    if count < 80 or fraction < 0.001 or fraction > 0.2:
        return None
    partial = half_mask(changed)
    if int(partial.sum()) < 40:
        return None
    control = translated_copy(partial, changed)
    if control is None:
        return None
    return changed, partial, control


def pixel_recolor_wrist(wrist: np.ndarray, mask: np.ndarray | None) -> np.ndarray:
    if mask is None:
        return wrist.copy()
    return pixel_recolor(wrist, mask)


def arm_unchanged(saved: np.ndarray, sim, span: tuple[int, int]) -> bool:
    start, stop = span
    before = np.concatenate([saved[:start], saved[stop:]])
    after = np.concatenate([np.asarray(sim.data.qpos[:start]), np.asarray(sim.data.qpos[stop:])])
    return bool(np.allclose(before, after, atol=1e-5))


def render_absent(collect, env, sim, body: int):
    span = collect.object_qpos_span(sim, body)
    saved = collect.move_object_away(sim, body)
    if saved is None or span is None:
        return None
    try:
        if not arm_unchanged(saved, sim, span):
            return None
        return images_of(collect, collect.render_after_edit(env))
    finally:
        collect.restore_qpos(sim, saved)


def paint_pair(agent, wrist, agent_mask, wrist_mask, color):
    agent_out = paint_mask(agent, agent_mask, color)
    wrist_out = wrist.copy() if wrist_mask is None else paint_mask(wrist, wrist_mask, color)
    return agent_out, wrist_out


def projected_masks(sim, geoms: list[int], image: np.ndarray, needle: str):
    cam = camera_index(sim, needle)
    if cam is None:
        return None
    disks = project_geoms(sim, geoms, cam, image.shape[0], image.shape[1])
    full, partial, control = masks_from_disks(image.shape[0], image.shape[1], disks)
    if full is None or partial is None or control is None:
        return None
    return full, partial, control


def build_conditions(collect, env, sim, body: int, geoms: list[int], raw: dict) -> dict | None:
    base_agent, base_wrist = images_of(collect, raw)
    rgba_agent, rgba_wrist = render_rgba_images(collect, env, sim, geoms)
    empirical = masks_from_change(base_agent, rgba_agent)
    wrist_empirical = masks_from_change(base_wrist, rgba_wrist)
    if empirical is not None:
        full, partial, control = empirical
        recolor_agent, recolor_wrist = rgba_agent, rgba_wrist
        recolor_method = "rgba"
        if wrist_empirical is not None:
            wrist_full, wrist_partial, wrist_control = wrist_empirical
        else:
            wrist_full = wrist_partial = wrist_control = None
    else:
        projected = projected_masks(sim, geoms, base_agent, "agentview")
        if projected is None:
            return None
        full, partial, control = projected
        wrist_projected = projected_masks(sim, geoms, base_wrist, "eye_in_hand")
        if wrist_projected is None:
            wrist_full = wrist_partial = wrist_control = None
        else:
            wrist_full, wrist_partial, wrist_control = wrist_projected
        recolor_agent = pixel_recolor(base_agent, full)
        recolor_wrist = pixel_recolor_wrist(base_wrist, wrist_full)
        recolor_method = "pixel_disk"
    absent = render_absent(collect, env, sim, body)
    if absent is None:
        return None
    absent_agent, absent_wrist = absent
    occluded = paint_pair(base_agent, base_wrist, partial, wrist_partial, GRAY)
    slab = paint_pair(base_agent, base_wrist, control, wrist_control, GRAY)
    occluded_absent = paint_pair(absent_agent, absent_wrist, partial, wrist_partial, GRAY)
    if np.logical_and(partial, control).any():
        raise RuntimeError("Occlusion disk and control disk overlap.")
    if changed_fraction(base_agent, recolor_agent) < 1e-6:
        return None
    if not mask_edit_is_exact(base_agent, occluded[0], partial):
        return None
    if not mask_edit_is_exact(base_agent, slab[0], control):
        return None
    if not mask_edit_is_exact(absent_agent, occluded_absent[0], partial):
        return None
    if wrist_partial is not None:
        mask_edit_is_exact(base_wrist, occluded[1], wrist_partial)
    if wrist_control is not None:
        mask_edit_is_exact(base_wrist, slab[1], wrist_control)
    images = {
        "base": (base_agent, base_wrist),
        "recolor": (recolor_agent, recolor_wrist),
        "absent": (absent_agent, absent_wrist),
        "occluded": occluded,
        "slab_miss": slab,
        "occluded_absent": occluded_absent,
    }
    return {
        "images": images,
        "recolor_method": recolor_method,
        "agent_full": full,
        "agent_partial": partial,
        "wrist_full": wrist_full,
        "wrist_partial": wrist_partial,
        "wrist_painted": wrist_partial is not None,
    }


def mask_edit_is_exact(before: np.ndarray, after: np.ndarray, mask: np.ndarray) -> bool:
    changed = changed_pixels(before, after)
    if np.logical_and(changed, ~mask).any():
        raise RuntimeError("A controlled paint changed pixels outside its mask.")
    return bool(changed.any())


def batch_from_images(collect, pre, agent, wrist, state: np.ndarray, sentence: str, policy):
    batch = {
        "observation.images.image": collect.as_image_tensor(agent),
        "observation.images.image2": collect.as_image_tensor(wrist),
        "observation.state": torch.from_numpy(np.asarray(state, dtype=np.float32)).unsqueeze(0),
        "task": [sentence],
    }
    features = getattr(policy.config, "input_features", {}) or {}
    reference = batch["observation.images.image"]
    for key in features:
        if key.startswith("observation.images.") and key not in batch:
            batch[key] = torch.zeros_like(reference)
    return pre(batch)


def find_layer6(policy, layer5):
    names = [name for name, module in policy.named_modules() if module is layer5]
    if len(names) != 1:
        raise RuntimeError(f"Layer 5 module identity matched {names}")
    layer5_name = names[0]
    named = dict(policy.named_modules())
    candidate = layer5_name.replace(".layers.5", ".layers.6") if layer5_name.endswith(".layers.5") else ""
    if candidate in named:
        print("Hooking layer 6", candidate, flush=True)
        return named[candidate]
    hits = [
        name
        for name in named
        if name.endswith("language_model.layers.6") and "gemma_expert" not in name and "vision" not in name
    ]
    if len(hits) == 1:
        print("Hooking layer 6", hits[0], flush=True)
        return named[hits[0]]
    print("Layer 6 was not found. The transcoder reconstructs layer 5 only.", flush=True)
    return None


def forward_policy(policy, layer5, layer6, batch, noise_seed: int, payload: torch.Tensor | None, mode: str):
    seed_all(noise_seed)
    if hasattr(policy, "reset"):
        policy.reset()
    captured5 = []
    captured6 = []

    def hook5(_module, _inputs, output):
        tensor = unpack_output(output)
        match mode:
            case "base":
                edited = tensor
            case "add":
                if payload is None or payload.shape != tensor.shape:
                    raise RuntimeError("Add-mode payload does not match the layer output.")
                edited = (tensor.float() + payload).to(dtype=tensor.dtype)
            case "replace":
                if payload is None or payload.shape != tensor.shape:
                    raise RuntimeError("Replace-mode payload does not match the layer output.")
                edited = payload.to(device=tensor.device, dtype=tensor.dtype)
            case _ as unexpected:
                assert_never(unexpected)
        captured5.append(tensor.detach())
        return repack_output(output, edited)

    def hook6(_module, _inputs, output):
        captured6.append(unpack_output(output).detach())
        return output

    handle5 = layer5.register_forward_hook(hook5)
    handle6 = layer6.register_forward_hook(hook6) if layer6 is not None else None
    try:
        with torch.inference_mode():
            chunk = policy.predict_action_chunk(batch) if hasattr(policy, "predict_action_chunk") else policy.select_action(batch)
    finally:
        handle5.remove()
        if handle6 is not None:
            handle6.remove()
    if len(captured5) != 1:
        raise RuntimeError(f"Expected one layer-5 call, found {len(captured5)}.")
    tokens6 = None
    if layer6 is not None:
        if len(captured6) != 1:
            raise RuntimeError(f"Expected one layer-6 call, found {len(captured6)}.")
        tokens6 = captured6[0][0].float().cpu().numpy()
    tokens5 = captured5[0][0].float().cpu().numpy()
    return tokens5, tokens6, action_array(chunk)


def to_gpu(payload: np.ndarray, device) -> torch.Tensor:
    tensor = torch.from_numpy(np.ascontiguousarray(payload.astype(np.float32)))
    return tensor.unsqueeze(0).to(device)


def thumb(picture: np.ndarray, label: str) -> np.ndarray:
    tile = cv2.resize(picture, (160, 160), interpolation=cv2.INTER_AREA)
    tile = cv2.copyMakeBorder(tile, 22, 0, 0, 0, cv2.BORDER_CONSTANT, value=(255, 255, 255))
    cv2.putText(tile, label, (4, 16), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 0), 1, cv2.LINE_AA)
    return tile


def write_sheet(rows: list[tuple[str, list[np.ndarray]]], path: Path) -> None:
    strips = []
    for label, frames in rows:
        tiles = [thumb(frame, f"{label[:12]} {index}") for index, frame in enumerate(frames)]
        strips.append(np.concatenate(tiles, axis=1))
    width = max(strip.shape[1] for strip in strips)
    padded = []
    for strip in strips:
        if strip.shape[1] < width:
            pad = np.full((strip.shape[0], width - strip.shape[1], 3), 255, dtype=np.uint8)
            strip = np.concatenate([strip, pad], axis=1)
        padded.append(strip)
    sheet = np.concatenate(padded, axis=0)
    bgr = cv2.cvtColor(sheet, cv2.COLOR_RGB2BGR)
    cv2.imwrite(str(path), bgr)


def json_ready(value):
    if isinstance(value, dict):
        return {str(key): json_ready(val) for key, val in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(val) for val in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value


def mean_defined(values: list[float | None]) -> float | None:
    kept = [value for value in values if value is not None]
    if not kept:
        return None
    return float(np.mean(kept))


def print_circuit(rows: list[dict]) -> None:
    print("\nCIRCUIT  fraction of action-chunk RMSE closed (base -> edited)", flush=True)
    print(f"{'contrast':<12}{'patch':<24}{'gap_closed':>12}{'edit_l2':>12}{'base_rmse':>12}{'n':>6}", flush=True)
    for row in rows:
        gap = row["mean_gap_closed"]
        gap_text = "undefined" if gap is None else f"{100 * gap:.2f}%"
        print(
            f"{row['contrast']:<12}{row['patch']:<24}{gap_text:>12}"
            f"{row['mean_edit_l2']:>12.3f}{row['mean_baseline_rmse']:>12.4f}{row['n_defined']:>6}",
            flush=True,
        )


def feature_table(scores: dict[str, np.ndarray], picks: dict[str, np.ndarray]) -> list[dict]:
    rows = []
    for contrast, ids in picks.items():
        others = [scores[name] for name in CONTRAST_NAMES if name != contrast]
        denominator = np.abs(scores[contrast]) + sum(np.abs(other) for other in others) + 1e-8
        specificity = np.abs(scores[contrast]) / denominator
        for feature in ids:
            index = int(feature)
            rows.append(
                {
                    "contrast": contrast,
                    "feature": index,
                    "specificity": float(specificity[index]),
                    "color": float(scores["color"][index]),
                    "absence": float(scores["absence"][index]),
                    "occlusion": float(scores["occlusion"][index]),
                    "slab": float(scores["slab"][index]),
                }
            )
    return rows


def print_features(rows: list[dict]) -> None:
    print("\nFEATURES  mean code difference on the tokens that moved", flush=True)
    print(f"{'contrast':<12}{'feature':>8}{'specificity':>13}{'color':>10}{'absence':>10}{'occlusion':>10}{'slab':>10}", flush=True)
    for row in rows:
        print(
            f"{row['contrast']:<12}{row['feature']:>8}{row['specificity']:>13.3f}"
            f"{row['color']:>10.3f}{row['absence']:>10.3f}{row['occlusion']:>10.3f}{row['slab']:>10.3f}",
            flush=True,
        )


def frame_label(result) -> tuple:
    """Accept both label_frame shapes: (label, body) and (label, body, geometry)."""
    if isinstance(result, tuple):
        label = result[0] if result else None
        body = result[1] if len(result) > 1 else None
        return label, body
    return result, None


def inspect_candidate(collect, env, sim, sentence: str, episode: int, step: int, sim_state: np.ndarray):
    raw = collect.observe_state(env, sim_state)
    bodies = collect.candidate_bodies(sim, sentence)
    label, touched = frame_label(collect.label_frame(sim, sentence, bodies))
    body = touched if touched is not None else None
    if body is None and bodies:
        site = None
        for name in ("gripper0_grip_site", "gripper0_eef", "robot0_eef", "grip_site"):
            try:
                site = int(sim.model.site_name2id(name))
                break
            except Exception:
                site = None
        if site is not None:
            grip = np.asarray(sim.data.site_xpos[site], dtype=np.float64)
            body = min(
                bodies,
                key=lambda item: np.linalg.norm(np.mean(sim.data.geom_xpos[bodies[item]], axis=0) - grip),
            )
    if body is None or body not in bodies:
        return None
    if collect.object_qpos_span(sim, body) is None:
        return None
    built = build_conditions(collect, env, sim, body, bodies[body], raw)
    if built is None:
        return None
    return {
        "episode": episode,
        "step": step,
        "sim_state": np.asarray(sim_state),
        "label": label or "visible",
        "body": int(body),
        "geoms": list(bodies[body]),
        "built": built,
        "state": collect.libero_state(raw),
        "sentence": sentence,
    }


def capture(policy, layer5, layer6, collect, pre, frame, condition: str, noise_seed: int):
    agent, wrist = frame["built"]["images"][condition]
    batch = batch_from_images(collect, pre, agent, wrist, frame["state"], frame["sentence"], policy)
    device = next(policy.parameters()).device
    tokens5, tokens6, action = forward_policy(
        policy, layer5, layer6, batch, noise_seed, payload=None, mode="base"
    )
    return tokens5, tokens6, action, device


def score_probe(model, mean5, std5, probe_tokens: dict) -> dict[str, np.ndarray]:
    """probe_tokens[frame][condition] = layer5 float32 [T, D]."""
    buckets = {name: [] for name in ("color", "absence", "occlusion", "slab")}
    edited_of = {"color": "recolor", "absence": "absent", "occlusion": "occluded", "slab": "slab_miss"}
    for frame in probe_tokens:
        base = frame["base"]
        base_code = encode_tokens(model, base, mean5, std5)
        for contrast, condition in edited_of.items():
            edited = frame[condition]
            mask = moving_token_mask(base, edited)
            code = encode_tokens(model, edited, mean5, std5)
            buckets[contrast].append(contrast_score(base_code, code, mask))
    return {name: np.mean(np.stack(rows, axis=0), axis=0) for name, rows in buckets.items()}


def main() -> None:
    configure_environment()
    if not torch.cuda.is_available():
        raise SystemExit("CUDA GPU required. Use the Colab L4 or A100 runtime.")
    collect = load_collect_module()
    max_episodes = env_int("CTRL_MAX_EPISODES", 2)
    probe_count = env_int("CTRL_PROBE_FRAMES", 4)
    train_extra = env_int("CTRL_TRAIN_EXTRA", 3)
    n_features = env_int("CTRL_N_FEATURES", 512)
    k = env_int("CTRL_K", 16)
    steps = env_int("CTRL_TRAIN_STEPS", 400)
    top_k_features = env_int("CTRL_TOP_FEATURES", 8)
    seed = env_int("CTRL_SEED", 0)
    repo = Path(os.environ.get("LOCAL_REPO", "/content/groot-run"))
    dest = repo / "outputs" / "permanence" / "controlled_contrasts"
    dest.mkdir(parents=True, exist_ok=True)

    from libero.libero import benchmark, get_libero_path
    from libero.libero.envs import OffScreenRenderEnv

    suite = benchmark.get_benchmark_dict()[collect.SUITE]()
    task_id = int(collect.TASK_IDS[0])
    task = suite.get_task(task_id)
    sentence = task.language
    demo_path = collect.find_demo_file(task)
    bddl = os.path.join(get_libero_path("bddl_files"), task.problem_folder, task.bddl_file)
    print(f"Task {task_id}: {sentence}", flush=True)
    print(f"Demos: {demo_path}", flush=True)
    print("Loading policy", flush=True)
    policy, pre, _post, layer5 = collect.load_policy()
    policy.eval()
    layer6 = find_layer6(policy, layer5)

    episodes = []
    for index in range(max_episodes):
        try:
            _actions, states = collect.load_demo(demo_path, index)
        except IndexError:
            break
        episodes.append((index, states))
    if not episodes:
        raise SystemExit("No demonstration states were loaded.")
    same_episode = len(episodes) == 1
    train_eps = {episodes[0][0]} if same_episode else {item[0] for item in episodes[:-1]}
    probe_eps = {episodes[0][0]} if same_episode else {episodes[-1][0]}
    print(f"Train episodes {sorted(train_eps)} | probe episodes {sorted(probe_eps)}", flush=True)

    env = OffScreenRenderEnv(bddl_file_name=bddl, camera_heights=256, camera_widths=256)
    env.seed(0)
    candidates = []
    try:
        for episode, states in episodes:
            print(f"Scanning episode {episode} ({len(states)} steps)", flush=True)
            for step, sim_state in enumerate(states):
                row = inspect_candidate(collect, env, collect.sim_of(env), sentence, episode, step, sim_state)
                if row is not None:
                    candidates.append(row)
        if len(candidates) < 2:
            raise SystemExit(
                "No controllable frames. The task object did not project into the agent camera "
                "with room for a matched off-object disk."
            )
        if same_episode:
            mid = sorted(row["step"] for row in candidates)[len(candidates) // 2]
            train_candidates = [row for row in candidates if row["step"] < mid]
            probe_candidates = [row for row in candidates if row["step"] >= mid]
        else:
            train_candidates = [row for row in candidates if row["episode"] in train_eps]
            probe_candidates = [row for row in candidates if row["episode"] in probe_eps]
        probe_frames = choose_frames(probe_candidates, probe_count)
        extra_frames = choose_frames(
            [row for row in train_candidates if (row["episode"], row["step"]) not in {(f["episode"], f["step"]) for f in probe_frames}],
            train_extra,
        )
        if len(probe_frames) < 2:
            raise SystemExit(
                f"Only {len(probe_frames)} controllable probe frames. Raise CTRL_MAX_STEPS or CTRL_MAX_EPISODES."
            )
        print(
            f"Controllable frames: train-extra {len(extra_frames)} | probe {len(probe_frames)} "
            f"(scanned {len(candidates)})",
            flush=True,
        )

        train_l5: list[np.ndarray] = []
        train_l6: list[np.ndarray] = []
        probe_cutoff = min(int(frame["step"]) for frame in probe_frames) if same_episode else None
        kept_frames = {(int(frame["episode"]), int(frame["step"])) for frame in probe_frames + extra_frames}
        for row in candidates:
            if (int(row["episode"]), int(row["step"])) not in kept_frames:
                row.pop("built", None)
        for episode, states in episodes:
            for step, sim_state in enumerate(states):
                if episode not in train_eps:
                    continue
                if probe_cutoff is not None and step >= probe_cutoff:
                    continue
                raw = collect.observe_state(env, sim_state)
                state = collect.libero_state(raw)
                agent, wrist = images_of(collect, raw)
                batch = batch_from_images(collect, pre, agent, wrist, state, sentence, policy)
                noise = 1000 + episode * 100 + step
                tokens5, tokens6, _action = forward_policy(
                    policy, layer5, layer6, batch, noise, None, "base"
                )
                train_l5.append(tokens5)
                if tokens6 is not None:
                    train_l6.append(tokens6)
                if step % 8 == 0:
                    print(f"  train base ep {episode} step {step}", flush=True)

        def capture_conditions(frame, conditions: tuple[str, ...]) -> dict:
            noise = 5000 + int(frame["episode"]) * 100 + int(frame["step"])
            packed = {}
            for condition in conditions:
                tokens5, tokens6, action, _device = capture(
                    policy, layer5, layer6, collect, pre, frame, condition, noise
                )
                packed[condition] = {"l5": tokens5, "l6": tokens6, "action": action}
                print(
                    f"  captured ep {frame['episode']} step {frame['step']} {condition} "
                    f"recolor={frame['built']['recolor_method']}",
                    flush=True,
                )
            return packed

        for frame in extra_frames:
            packed = capture_conditions(frame, ("recolor", "absent", "occluded", "slab_miss"))
            for condition in packed:
                train_l5.append(packed[condition]["l5"])
                if packed[condition]["l6"] is not None:
                    train_l6.append(packed[condition]["l6"])

        probe_packed = [capture_conditions(frame, CONDITION_NAMES) for frame in probe_frames]

        # Determinism: a second base forward must reproduce the saved tokens and actions.
        check = probe_frames[0]
        noise = 5000 + int(check["episode"]) * 100 + int(check["step"])
        again5, _again6, again_action, device = capture(
            policy, layer5, layer6, collect, pre, check, "base", noise
        )
        saved5 = probe_packed[0]["base"]["l5"]
        max_token = float(np.max(np.abs(again5 - saved5)))
        if max_token > 1e-3 or not np.array_equal(again_action, probe_packed[0]["base"]["action"]):
            raise RuntimeError(
                f"Repeating the same inputs changed the forward (token max abs {max_token}). "
                "Circuit tracing needs a deterministic forward."
            )
        print(f"Determinism check passed (token max abs {max_token:.3e})", flush=True)

        pixel_rows = []
        for frame in probe_frames:
            base_agent = frame["built"]["images"]["base"][0]
            base_wrist = frame["built"]["images"]["base"][1]
            row = {
                "episode": int(frame["episode"]),
                "step": int(frame["step"]),
                "label": frame["label"],
                "recolor_method": frame["built"]["recolor_method"],
            }
            for condition in CONDITION_NAMES:
                if condition == "base":
                    continue
                row[condition] = changed_fraction(base_agent, frame["built"]["images"][condition][0])
            row["wrist_recolor"] = changed_fraction(base_wrist, frame["built"]["images"]["recolor"][1])
            pixel_rows.append(row)
        print("\nPIXELS  fraction of agent-view pixels that differ from base", flush=True)
        for row in pixel_rows:
            print(
                f"  ep {row['episode']} step {row['step']} {row['label']} via {row['recolor_method']}: "
                f"recolor={row['recolor']:.4f} wrist_recolor={row['wrist_recolor']:.4f} "
                f"absent={row['absent']:.4f} occluded={row['occluded']:.4f} slab_miss={row['slab_miss']:.4f}",
                flush=True,
            )

        # Full-cover diagnostic on the first probe frame. Both cameras are covered when the
        # object projects into them, then the two forwards see those covered images.
        first = probe_frames[0]
        cover = first["built"]["agent_full"]
        present_img = paint_mask(first["built"]["images"]["base"][0], cover, GRAY)
        absent_img = paint_mask(first["built"]["images"]["absent"][0], cover, GRAY)
        present_wrist = first["built"]["images"]["base"][1]
        absent_wrist = first["built"]["images"]["absent"][1]
        agent_outside = outside_fraction(present_img, absent_img, cover)
        wrist_cover = first["built"]["wrist_full"]
        if wrist_cover is not None:
            present_wrist = paint_mask(present_wrist, wrist_cover, GRAY)
            absent_wrist = paint_mask(absent_wrist, wrist_cover, GRAY)
            wrist_outside = outside_fraction(present_wrist, absent_wrist, wrist_cover)
        else:
            wrist_outside = changed_fraction(present_wrist, absent_wrist)
        outside = max(agent_outside, wrist_outside)

        def covered_forward(agent_image, wrist_image):
            batch = batch_from_images(
                collect, pre, agent_image, wrist_image, first["state"], first["sentence"], policy
            )
            _t5, _t6, action = forward_policy(policy, layer5, layer6, batch, noise, None, "base")
            return action

        present_action = covered_forward(present_img, present_wrist)
        absent_action = covered_forward(absent_img, absent_wrist)
        cover_rmse = float(np.sqrt(np.mean((present_action - absent_action) ** 2)))
        cover_text = full_cover_interpretation(outside, cover_rmse)
        print(
            f"\nFULL COVER  outside-pixel fraction={outside:.5f}  action RMSE={cover_rmse:.5f}",
            flush=True,
        )
        print(cover_text, flush=True)
        print("Continuing into transcoder training and circuit tracing.", flush=True)

        sheet_rows = []
        for condition in CONDITION_NAMES:
            sheet_rows.append(
                (condition, [frame["built"]["images"][condition][0] for frame in probe_frames[:4]])
            )
        sheet_rows.append(("cover_present", [present_img]))
        sheet_rows.append(("cover_absent", [absent_img]))
        sheet_path = dest / "contact_sheet.png"
        write_sheet(sheet_rows, sheet_path)
        print("Wrote", sheet_path, flush=True)

        print("\nTraining transcoder on CPU so the policy can stay loaded", flush=True)
        model, mean5, std5, mean6, std6 = train_transcoder(
            train_l5,
            train_l6 if train_l6 else None,
            n_features,
            k,
            steps,
            batch=2048,
            seed=seed,
        )
        probe_base = np.concatenate([item["base"]["l5"] for item in probe_packed], axis=0)
        recon = reconstruction(model, probe_base, mean5, std5)
        r2_5 = r2_score(recon, probe_base)
        r2_6 = None
        if mean6 is not None and std6 is not None and probe_packed[0]["base"]["l6"] is not None:
            probe_next = np.concatenate([item["base"]["l6"] for item in probe_packed], axis=0)
            predicted = next_prediction(model, probe_base, mean5, std5, mean6, std6)
            r2_6 = r2_score(predicted, probe_next)
        print(f"Probe base R^2 layer5={r2_5:.3f} layer6={r2_6}", flush=True)

        probe_token_maps = [{name: item[name]["l5"] for name in CONDITION_NAMES} for item in probe_packed]
        scores = score_probe(model, mean5, std5, probe_token_maps)
        picks = {}
        specific_flags = {}
        for contrast in CONTRAST_NAMES:
            others = [scores[name] for name in ("color", "absence", "occlusion", "slab") if name != contrast]
            ids, flag = pick_features(scores[contrast], others, top_k_features)
            picks[contrast] = ids
            specific_flags[contrast] = flag
        feature_rows = feature_table(scores, picks)
        print_features(feature_rows)
        print(
            "Score cosines  color-absence={:.3f} color-occlusion={:.3f} absence-occlusion={:.3f}".format(
                cosine(scores["color"], scores["absence"]),
                cosine(scores["color"], scores["occlusion"]),
                cosine(scores["absence"], scores["occlusion"]),
            ),
            flush=True,
        )

        rng = np.random.default_rng(seed)
        random_ids = {}
        for contrast in CONTRAST_NAMES:
            pool = np.setdiff1d(np.arange(n_features), picks[contrast])
            random_ids[contrast] = rng.choice(pool, size=len(picks[contrast]), replace=False)

        circuit_records = []
        patch_names = (
            "transcoder_features",
            "random_remap",
            "off_contrast",
            "full_tokens",
            "pooled",
            "token_structure",
        )
        edited_of = {"color": "recolor", "absence": "absent", "occlusion": "occluded"}
        for frame_index, (frame, packed) in enumerate(zip(probe_frames, probe_packed)):
            noise = 5000 + int(frame["episode"]) * 100 + int(frame["step"])
            source_tokens = packed["base"]["l5"]
            source_action = packed["base"]["action"]
            for contrast in CONTRAST_NAMES:
                condition = edited_of[contrast]
                donor_tokens = packed[condition]["l5"]
                donor_action = packed[condition]["action"]
                source_code = encode_tokens(model, source_tokens, mean5, std5)
                donor_code = encode_tokens(model, donor_tokens, mean5, std5)
                code_delta = donor_code - source_code
                selected = decode_delta(model, keep_features(code_delta, picks[contrast]), std5)
                random_delta = decode_delta(
                    model,
                    remap_features(keep_features(code_delta, picks[contrast]), picks[contrast], random_ids[contrast]),
                    std5,
                )
                other = off_contrast(contrast)  # type: ignore[arg-type]
                off_delta = decode_delta(model, keep_features(code_delta, picks[other]), std5)
                random_delta = match_l2(random_delta, selected)
                off_delta = match_l2(off_delta, selected)
                pooled = (donor_tokens - source_tokens).mean(axis=0, keepdims=True)
                pooled = np.repeat(pooled, source_tokens.shape[0], axis=0)
                payloads = {
                    "transcoder_features": ("add", selected),
                    "random_remap": ("add", random_delta),
                    "off_contrast": ("add", off_delta),
                    "full_tokens": ("replace", donor_tokens),
                    "pooled": ("add", pooled),
                    "token_structure": ("add", structure_delta(donor_tokens, source_tokens)),
                }
                agent, wrist = frame["built"]["images"]["base"]
                batch = batch_from_images(collect, pre, agent, wrist, frame["state"], frame["sentence"], policy)
                for patch in patch_names:
                    mode, payload_np = payloads[patch]
                    _tokens, _tokens6, action = forward_policy(
                        policy,
                        layer5,
                        None,
                        batch,
                        noise,
                        to_gpu(payload_np, device),
                        mode,
                    )
                    metrics = action_metrics(action, source_action, donor_action)
                    if mode == "replace":
                        realized = float(np.linalg.norm(payload_np - source_tokens))
                    else:
                        realized = float(np.linalg.norm(payload_np))
                    circuit_records.append(
                        {
                            "episode": int(frame["episode"]),
                            "step": int(frame["step"]),
                            "contrast": contrast,
                            "patch": patch,
                            "edit_l2": realized,
                            **metrics,
                        }
                    )
            print(f"Circuit frame {frame_index + 1}/{len(probe_frames)} done", flush=True)

        summary_rows = []
        for contrast in CONTRAST_NAMES:
            for patch in patch_names:
                selected_rows = [
                    row for row in circuit_records if row["contrast"] == contrast and row["patch"] == patch
                ]
                summary_rows.append(
                    {
                        "contrast": contrast,
                        "patch": patch,
                        "mean_gap_closed": mean_defined([row["fraction_gap_closed"] for row in selected_rows]),
                        "mean_edit_l2": float(np.mean([row["edit_l2"] for row in selected_rows])),
                        "mean_baseline_rmse": float(np.mean([row["source_to_donor_rmse"] for row in selected_rows])),
                        "n_defined": int(sum(row["fraction_gap_closed"] is not None for row in selected_rows)),
                        "n_pairs": len(selected_rows),
                    }
                )
        print_circuit(summary_rows)

        checkpoint = {
            "state_dict": model.state_dict(),
            "dim": int(mean5.shape[0]),
            "n_features": n_features,
            "k": k,
            "predict_next": model.predict_next,
            "mean5": mean5,
            "std5": std5,
            "mean6": mean6,
            "std6": std6,
            "features": {name: picks[name].tolist() for name in picks},
        }
        torch.save(checkpoint, dest / "transcoder.pt")
        np.savez_compressed(
            dest / "feature_scores.npz",
            **{f"score_{name}": value for name, value in scores.items()},
            **{f"features_{name}": picks[name] for name in picks},
        )
        summary = {
            "task": sentence,
            "task_id": task_id,
            "probe_frames": [
                {"episode": int(frame["episode"]), "step": int(frame["step"]), "label": frame["label"]}
                for frame in probe_frames
            ],
            "pixel_changed_fraction": pixel_rows,
            "full_cover": {
                "outside_pixel_fraction": outside,
                "action_rmse": cover_rmse,
                "images_agree": outside < 0.002,
                "actions_agree": cover_rmse < 1e-3,
                "residual_floor_rmse": cover_rmse if outside < 0.002 else None,
                "interpretation": cover_text,
            },
            "transcoder": {
                "r2_layer5_probe_base": r2_5,
                "r2_layer6_probe_base": r2_6,
                "n_features": n_features,
                "k": k,
                "steps": steps,
                "train_tokens": int(sum(chunk.shape[0] for chunk in train_l5)),
                "specific_selection": specific_flags,
            },
            "features": feature_rows,
            "score_cosines": {
                "color_absence": cosine(scores["color"], scores["absence"]),
                "color_occlusion": cosine(scores["color"], scores["occlusion"]),
                "absence_occlusion": cosine(scores["absence"], scores["occlusion"]),
            },
            "circuit": summary_rows,
            "limitations": [
                "Development diagnostic on a few frozen frames.",
                "Occlusion paints half of the pixels the color swap changed. The control is that same shape moved off the object.",
                "Absence teleports the object. The arm qpos, language, and proprioceptive state stay fixed.",
                "Recolor is a 3D material edit when that edit stays in a compact region, and a mask paint otherwise.",
                "Sparse patches are norm-matched to each other. Full-token, pooled, and token-structure patches keep their natural magnitude.",
                "A full cover that makes present and absent images agree cannot leave a hidden-object signal in one forward.",
                "A leftover action RMSE on that covered pair is the residual floor for near-matched images. The identical-input re-forward is the determinism check.",
            ],
        }
        (dest / "summary.json").write_text(json.dumps(json_ready(summary), indent=2))
        (repo / "outputs" / "permanence" / "controlled_contrasts_latest.txt").write_text(str(dest))
        print("\nWrote", dest / "summary.json", flush=True)
        print("SHEET", sheet_path, flush=True)
        print("RESULT_DIR", dest, flush=True)
    finally:
        env.close()


if __name__ == "__main__":
    main()
'''

scripts = LOCAL_REPO / "scripts"
scripts.mkdir(parents=True, exist_ok=True)
path = scripts / "controlled_contrasts.py"
path.write_text(SCRIPT)
if not (scripts / "collect_layer5_replay.py").is_file():
    raise SystemExit(
        "scripts/collect_layer5_replay.py is missing. Run the collection cell once in this runtime first."
    )

env = os.environ.copy()
env["PYTHONPATH"] = str(scripts) + os.pathsep + env.get("PYTHONPATH", "")
env["LOCAL_REPO"] = str(LOCAL_REPO)
env["PYTHONUNBUFFERED"] = "1"

def stream(cmd):
    proc = subprocess.Popen(
        cmd,
        cwd=str(scripts),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    tail = []
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
        tail.append(line.rstrip())
        del tail[:-40]
    code = proc.wait()
    if code != 0:
        raise SystemExit(
            "controlled_contrasts.py exited with code %s. Last lines:\n%s" % (code, "\n".join(tail))
        )

stream([str(PYTHON), "-u", str(path)])

sheet = LOCAL_REPO / "outputs" / "permanence" / "controlled_contrasts" / "contact_sheet.png"
summary = sheet.parent / "summary.json"
print("summary", summary)
try:
    from IPython.display import Image, display
    display(Image(filename=str(sheet)))
except Exception:
    print("contact sheet:", sheet)


In [ ]:
# Occlusion features — keep the specific ones, then circuit-trace them
# Run after the contrast cell. Reloads transcoder.pt. Does not retrain.
# Does not overwrite outputs/permanence/controlled_contrasts/.

import os, sys, subprocess
from pathlib import Path

os.environ.setdefault("CTRL_OCCLUSION_COUNTS", "8,32,128")
os.environ.setdefault("CTRL_MIN_SPECIFICITY", "0.5")
os.environ.setdefault("CTRL_SEED", "0")
os.environ.setdefault("SUITE", "libero_spatial")
os.environ.setdefault("TASK_IDS", "[0]")
os.environ.setdefault(
    "LAYER_NAME",
    "model.paligemma_with_expert.paligemma.model.language_model.layers.5",
)
os.environ.setdefault("MUJOCO_GL", "egl")
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

LOCAL_REPO = Path(os.environ.get("LOCAL_REPO", "/content/groot-run"))
PYTHON = Path("/content/lerobot-venv/bin/python")
if not PYTHON.exists():
    PYTHON = Path(sys.executable)

SCRIPT = r'''"""Find occlusion features in the saved per-token transcoder and circuit-trace them.

Reloads the transcoder from the contrast run. A feature is kept only when its
code rises on gray paint covering the bowl and stays quiet on the same paint
moved off the bowl, on the color swap, and on removing the bowl. Those features
are patched from the base frame toward the occluded frame and compared with a
same-size random patch and with the color features.

Does not retrain the transcoder and does not rewrite the contrast run.
"""

from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Literal, assert_never

import numpy as np
import torch

import controlled_contrasts as cc

QUIET_FOR = {
    "occlusion": ("slab", "color", "absence"),
    "color": ("slab", "occlusion", "absence"),
}
CAPTURE = ("base", "recolor", "absent", "occluded", "slab_miss")
PATCHES = ("occlusion_features", "random_remap", "color_features")
REFERENCES = ("full_tokens", "token_structure")
PatchName = Literal["occlusion_features", "random_remap", "color_features"]


def parse_counts(text: str) -> list[int]:
    counts: list[int] = []
    seen: set[int] = set()
    for part in text.split(","):
        piece = part.strip()
        if not piece:
            continue
        value = int(piece)
        if value < 1:
            raise ValueError(f"Feature count must be positive, got {value}")
        if value not in seen:
            seen.add(value)
            counts.append(value)
    if not counts:
        raise ValueError("CTRL_OCCLUSION_COUNTS is empty")
    return counts


def prefix_counts(n_specific: int, requested: list[int]) -> list[int]:
    """Nested prefixes of the features that passed. Never invents a fallback set."""
    counts: list[int] = []
    for value in requested:
        take = min(int(value), int(n_specific))
        if take >= 1 and take not in counts:
            counts.append(take)
    return counts


def specificity_of(primary: np.ndarray, others: list[np.ndarray]) -> np.ndarray:
    primary64 = np.asarray(primary, dtype=np.float64)
    rest = [np.asarray(other, dtype=np.float64) for other in others]
    denominator = np.abs(primary64) + sum(np.abs(other) for other in rest) + 1e-8
    return np.abs(primary64) / denominator


def specific_ids(
    scores: dict[str, np.ndarray],
    primary: str,
    quiet: tuple[str, ...],
    min_specificity: float,
) -> np.ndarray:
    """Features that rise on `primary` and stay quiet on every contrast in `quiet`.

    An empty result means none passed. There is no magnitude fallback.
    """
    primary64 = np.asarray(scores[primary], dtype=np.float64)
    others = [np.asarray(scores[name], dtype=np.float64) for name in quiet]
    specificity = specificity_of(primary64, others)
    magnitude_cut = float(np.quantile(np.abs(primary64), 0.90))
    eligible = np.flatnonzero(
        (primary64 > 0) & (specificity > min_specificity) & (np.abs(primary64) >= magnitude_cut)
    )
    order = eligible[np.argsort(-primary64[eligible], kind="mergesort")]
    return order.astype(np.int64)


def feature_rows(
    feature_ids: np.ndarray,
    scores: dict[str, np.ndarray],
    primary: str,
    quiet: tuple[str, ...],
) -> list[dict]:
    primary64 = np.asarray(scores[primary], dtype=np.float64)
    others = [np.asarray(scores[name], dtype=np.float64) for name in quiet]
    specificity = specificity_of(primary64, others)
    rows = []
    for feature_id in np.asarray(feature_ids, dtype=np.int64):
        index = int(feature_id)
        row = {
            "feature": index,
            "occlusion": float(scores["occlusion"][index]),
            "slab": float(scores["slab"][index]),
            "color": float(scores["color"][index]),
            "absence": float(scores["absence"][index]),
            "specificity": float(specificity[index]),
        }
        rows.append(row)
    return rows


def nearest_rejected(
    scores: dict[str, np.ndarray],
    quiet: tuple[str, ...],
    limit: int = 8,
) -> list[dict]:
    """Largest positive occlusion scores, so a failed filter can be read."""
    primary = np.asarray(scores["occlusion"], dtype=np.float64)
    order = np.argsort(-primary, kind="mergesort")
    positive = order[primary[order] > 0][:limit]
    return feature_rows(positive.astype(np.int64), scores, "occlusion", quiet)


def load_transcoder(path: Path) -> tuple[cc.TokenTranscoder, np.ndarray, np.ndarray, int]:
    try:
        checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        checkpoint = torch.load(path, map_location="cpu")
    n_features = int(checkpoint["n_features"])
    model = cc.TokenTranscoder(
        int(checkpoint["dim"]),
        n_features,
        int(checkpoint["k"]),
        bool(checkpoint["predict_next"]),
    )
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()
    mean5 = np.asarray(checkpoint["mean5"], dtype=np.float32)
    std5 = np.asarray(checkpoint["std5"], dtype=np.float32)
    return model, mean5, std5, n_features


def summarize(records: list[dict], count: int | None, patch: str) -> dict:
    chosen = [row for row in records if row["count"] == count and row["patch"] == patch]
    return {
        "count": count,
        "patch": patch,
        "mean_gap_closed": cc.mean_defined([row["fraction_gap_closed"] for row in chosen]),
        "mean_edit_l2": float(np.mean([row["edit_l2"] for row in chosen])) if chosen else None,
        "mean_baseline_rmse": float(np.mean([row["source_to_donor_rmse"] for row in chosen])) if chosen else None,
        "n_defined": int(sum(row["fraction_gap_closed"] is not None for row in chosen)),
        "n_pairs": len(chosen),
    }


def gap_of(rows: list[dict], count: int | None, patch: str) -> float | None:
    return next(row["mean_gap_closed"] for row in rows if row["count"] == count and row["patch"] == patch)


def occlusion_verdict(rows: list[dict], counts: list[int], n_specific: int) -> str:
    if n_specific == 0 or not counts:
        return (
            "No occlusion feature passed. None rose on paint covering the bowl while staying "
            "quiet on paint off the bowl, the color swap, and removal."
        )
    passing: list[int] = []
    lines: list[str] = []
    for count in counts:
        feature_gap = gap_of(rows, count, "occlusion_features")
        random_gap = gap_of(rows, count, "random_remap")
        color_gap = gap_of(rows, count, "color_features")
        if feature_gap is None or random_gap is None or color_gap is None:
            lines.append(f"{count} features: a control patch is missing.")
            continue
        lines.append(
            f"{count} features close {100 * feature_gap:.1f}% "
            f"(random {100 * random_gap:.1f}%, color features {100 * color_gap:.1f}%)."
        )
        if feature_gap - random_gap >= 0.10 and feature_gap - color_gap >= 0.10:
            passing.append(count)
    if passing:
        return (
            f"Smallest occlusion set that beats random and the color features by 10 points: {passing[0]}. "
            + " ".join(lines)
        )
    return "No occlusion set beat both controls by 10 points. " + " ".join(lines)


def print_feature_table(title: str, rows: list[dict]) -> None:
    print(f"\n{title}", flush=True)
    print(
        f"{'feature':>8}{'occlusion':>12}{'slab':>10}{'color':>10}{'absence':>10}{'specificity':>13}",
        flush=True,
    )
    if not rows:
        print("  none", flush=True)
        return
    for row in rows:
        print(
            f"{row['feature']:>8}{row['occlusion']:>12.3f}{row['slab']:>10.3f}"
            f"{row['color']:>10.3f}{row['absence']:>10.3f}{row['specificity']:>13.3f}",
            flush=True,
        )


def print_circuit(rows: list[dict]) -> None:
    print("\nCIRCUIT  fraction of occlusion action-chunk RMSE closed (base -> occluded)", flush=True)
    print(
        f"{'count':>8}{'patch':<22}{'gap_closed':>12}{'edit_l2':>12}{'base_rmse':>12}{'n':>6}",
        flush=True,
    )
    for row in rows:
        gap = row["mean_gap_closed"]
        gap_text = "undefined" if gap is None else f"{100 * gap:.2f}%"
        count = "ref" if row["count"] is None else str(row["count"])
        edit = row["mean_edit_l2"]
        base = row["mean_baseline_rmse"]
        edit_text = "nan" if edit is None else f"{edit:.3f}"
        base_text = "nan" if base is None else f"{base:.4f}"
        print(
            f"{count:>8}{row['patch']:<22}{gap_text:>12}{edit_text:>12}{base_text:>12}{row['n_defined']:>6}",
            flush=True,
        )


def decoded_patch(
    model: cc.TokenTranscoder,
    code_delta: np.ndarray,
    feature_ids: np.ndarray,
    std5: np.ndarray,
    reference: np.ndarray | None,
) -> np.ndarray:
    payload = cc.decode_delta(model, cc.keep_features(code_delta, feature_ids), std5)
    if reference is not None:
        payload = cc.match_l2(payload, reference)
    return payload


def main() -> None:
    cc.configure_environment()
    if not torch.cuda.is_available():
        raise SystemExit("CUDA GPU required. Use the Colab L4 or A100 runtime.")
    collect = cc.load_collect_module()
    seed = cc.env_int("CTRL_SEED", 0)
    min_specificity = float(os.environ.get("CTRL_MIN_SPECIFICITY", "0.5"))
    requested = parse_counts(os.environ.get("CTRL_OCCLUSION_COUNTS", "8,32,128"))
    repo = Path(os.environ.get("LOCAL_REPO", "/content/groot-run"))
    source = repo / "outputs" / "permanence" / "controlled_contrasts"
    dest = repo / "outputs" / "permanence" / "occlusion_features"
    dest.mkdir(parents=True, exist_ok=True)
    summary_path = source / "summary.json"
    checkpoint_path = source / "transcoder.pt"
    if not summary_path.is_file() or not checkpoint_path.is_file():
        raise SystemExit(
            "Missing the controlled-contrast run. Run that cell first so "
            f"{summary_path} and {checkpoint_path} exist."
        )
    prior = json.loads(summary_path.read_text())
    wanted = [(int(row["episode"]), int(row["step"])) for row in prior["probe_frames"]]
    if len(wanted) < 2:
        raise SystemExit("The saved summary has fewer than two probe frames.")

    model, mean5, std5, n_features = load_transcoder(checkpoint_path)
    print(
        f"Loaded transcoder dim={mean5.shape[0]} features={n_features} "
        f"specificity>{min_specificity} counts={requested}",
        flush=True,
    )
    print("Probe frames", wanted, flush=True)

    # Imported here so unit tests can load this module without LIBERO.
    from libero.libero import benchmark, get_libero_path
    from libero.libero.envs import OffScreenRenderEnv

    suite = benchmark.get_benchmark_dict()[collect.SUITE]()
    task_id = int(collect.TASK_IDS[0])
    task = suite.get_task(task_id)
    sentence = task.language
    demo_path = collect.find_demo_file(task)
    bddl = os.path.join(get_libero_path("bddl_files"), task.problem_folder, task.bddl_file)
    print(f"Task {task_id}: {sentence}", flush=True)
    print("Loading policy", flush=True)
    policy, pre, _post, layer5 = collect.load_policy()
    policy.eval()
    layer6 = cc.find_layer6(policy, layer5)

    demos: dict[int, list] = {}
    for episode, _step in wanted:
        if episode not in demos:
            _actions, states = collect.load_demo(demo_path, episode)
            demos[episode] = states

    env = OffScreenRenderEnv(bddl_file_name=bddl, camera_heights=256, camera_widths=256)
    env.seed(0)
    try:
        probe_frames = []
        for episode, step in wanted:
            states = demos[episode]
            if step >= len(states):
                raise SystemExit(f"Episode {episode} has {len(states)} steps; cannot rebuild step {step}.")
            print(f"Rebuilding ep {episode} step {step}", flush=True)
            row = cc.inspect_candidate(
                collect, env, collect.sim_of(env), sentence, episode, step, states[step]
            )
            if row is None:
                raise SystemExit(f"Could not rebuild ep {episode} step {step}.")
            probe_frames.append(row)

        def capture_conditions(frame) -> dict:
            noise = 5000 + int(frame["episode"]) * 100 + int(frame["step"])
            packed = {}
            for condition in CAPTURE:
                tokens5, _tokens6, action, _device = cc.capture(
                    policy, layer5, layer6, collect, pre, frame, condition, noise
                )
                packed[condition] = {"l5": tokens5, "action": action}
            return packed

        probe_packed = [capture_conditions(frame) for frame in probe_frames]
        check = probe_frames[0]
        noise = 5000 + int(check["episode"]) * 100 + int(check["step"])
        again5, _again6, again_action, device = cc.capture(
            policy, layer5, layer6, collect, pre, check, "base", noise
        )
        max_token = float(np.max(np.abs(again5 - probe_packed[0]["base"]["l5"])))
        if max_token > 1e-3 or not np.array_equal(again_action, probe_packed[0]["base"]["action"]):
            raise RuntimeError(f"Repeating the same inputs changed the forward (token max abs {max_token}).")
        print(f"Determinism check passed (token max abs {max_token:.3e})", flush=True)

        print("\nPIXELS  fraction of agent-view pixels that differ from base", flush=True)
        for frame in probe_frames:
            base_agent = frame["built"]["images"]["base"][0]
            parts = []
            for condition in ("recolor", "absent", "occluded", "slab_miss"):
                fraction = cc.changed_fraction(base_agent, frame["built"]["images"][condition][0])
                parts.append(f"{condition}={fraction:.4f}")
            print(
                f"  ep {frame['episode']} step {frame['step']} {frame['label']}: " + " ".join(parts),
                flush=True,
            )

        probe_token_maps = [{name: item[name]["l5"] for name in CAPTURE} for item in probe_packed]
        scores = cc.score_probe(model, mean5, std5, probe_token_maps)
        occlusion_ids = specific_ids(scores, "occlusion", QUIET_FOR["occlusion"], min_specificity)
        color_ids = specific_ids(scores, "color", QUIET_FOR["color"], min_specificity)
        counts = prefix_counts(int(occlusion_ids.size), requested)
        print(
            f"\nPassed: occlusion {int(occlusion_ids.size)} | color {int(color_ids.size)} "
            f"| patch counts {counts or 'none'}",
            flush=True,
        )
        passed_rows = feature_rows(occlusion_ids, scores, "occlusion", QUIET_FOR["occlusion"])
        print_feature_table("OCCLUSION FEATURES that passed", passed_rows)
        if occlusion_ids.size == 0:
            print_feature_table(
                "LARGEST occlusion scores that did not pass",
                nearest_rejected(scores, QUIET_FOR["occlusion"]),
            )

        rng = np.random.default_rng(seed)
        random_plan: dict[int, np.ndarray] = {}
        for count in counts:
            chosen = occlusion_ids[:count]
            pool = np.setdiff1d(np.arange(n_features), chosen)
            random_plan[count] = rng.choice(pool, size=int(chosen.size), replace=False).astype(np.int64)

        records: list[dict] = []
        for frame_index, (frame, packed) in enumerate(zip(probe_frames, probe_packed)):
            noise = 5000 + int(frame["episode"]) * 100 + int(frame["step"])
            source_tokens = packed["base"]["l5"]
            source_action = packed["base"]["action"]
            donor_tokens = packed["occluded"]["l5"]
            donor_action = packed["occluded"]["action"]
            agent, wrist = frame["built"]["images"]["base"]
            batch = cc.batch_from_images(collect, pre, agent, wrist, frame["state"], frame["sentence"], policy)
            code_delta = cc.encode_tokens(model, donor_tokens, mean5, std5) - cc.encode_tokens(
                model, source_tokens, mean5, std5
            )

            def run_patch(patch: str, count: int | None, mode: str, payload_np: np.ndarray) -> None:
                _tokens, _tokens6, action = cc.forward_policy(
                    policy, layer5, None, batch, noise, cc.to_gpu(payload_np, device), mode
                )
                metrics = cc.action_metrics(action, source_action, donor_action)
                if mode == "replace":
                    realized = float(np.linalg.norm(payload_np.astype(np.float64) - source_tokens.astype(np.float64)))
                else:
                    realized = float(np.linalg.norm(payload_np.astype(np.float64)))
                records.append(
                    {
                        "episode": int(frame["episode"]),
                        "step": int(frame["step"]),
                        "count": count,
                        "patch": patch,
                        "edit_l2": realized,
                        **metrics,
                    }
                )

            run_patch("full_tokens", None, "replace", donor_tokens.astype(np.float32))
            run_patch("token_structure", None, "add", cc.structure_delta(donor_tokens, source_tokens))

            for count in counts:
                selected = occlusion_ids[:count]
                feature_payload = decoded_patch(model, code_delta, selected, std5, None)
                moved = cc.remap_features(
                    cc.keep_features(code_delta, selected), selected, random_plan[count]
                )
                random_payload = cc.match_l2(cc.decode_delta(model, moved, std5), feature_payload)
                color_take = color_ids[: min(count, int(color_ids.size))]
                color_payload = decoded_patch(model, code_delta, color_take, std5, feature_payload)
                built: dict[PatchName, np.ndarray] = {
                    "occlusion_features": feature_payload,
                    "random_remap": random_payload,
                    "color_features": color_payload,
                }
                for patch in PATCHES:
                    match patch:
                        case "occlusion_features" | "random_remap" | "color_features":
                            run_patch(patch, count, "add", built[patch])
                        case _ as unexpected:
                            assert_never(unexpected)
            print(f"Occlusion frame {frame_index + 1}/{len(probe_frames)} done", flush=True)

        rows: list[dict] = []
        for patch in REFERENCES:
            rows.append(summarize(records, None, patch))
        for count in counts:
            for patch in PATCHES:
                rows.append(summarize(records, count, patch))
        print_circuit(rows)
        verdict = occlusion_verdict(rows, counts, int(occlusion_ids.size))
        print("\n" + verdict, flush=True)

        summary = {
            "task": sentence,
            "task_id": task_id,
            "source_run": str(source),
            "probe_frames": [
                {"episode": int(frame["episode"]), "step": int(frame["step"]), "label": frame["label"]}
                for frame in probe_frames
            ],
            "min_specificity": min_specificity,
            "requested_counts": requested,
            "counts": counts,
            "n_features": n_features,
            "occlusion_features": passed_rows,
            "color_features": feature_rows(color_ids, scores, "color", QUIET_FOR["color"]),
            "rejected_preview": nearest_rejected(scores, QUIET_FOR["occlusion"]) if occlusion_ids.size == 0 else [],
            "rows": rows,
            "verdict": verdict,
        }
        (dest / "summary.json").write_text(json.dumps(cc.json_ready(summary), indent=2))
        print("\nWrote", dest / "summary.json", flush=True)
        print("RESULT_DIR", dest, flush=True)
    finally:
        env.close()


if __name__ == "__main__":
    main()
'''

scripts = LOCAL_REPO / "scripts"
scripts.mkdir(parents=True, exist_ok=True)
path = scripts / "occlusion_features.py"
path.write_text(SCRIPT)
for required in ("controlled_contrasts.py", "collect_layer5_replay.py"):
    if not (scripts / required).is_file():
        raise SystemExit(
            f"scripts/{required} is missing. Run the collector cell and the contrast cell in this runtime first."
        )

env = os.environ.copy()
env["PYTHONPATH"] = str(scripts) + os.pathsep + env.get("PYTHONPATH", "")
env["LOCAL_REPO"] = str(LOCAL_REPO)
env["PYTHONUNBUFFERED"] = "1"

def stream(cmd):
    proc = subprocess.Popen(
        cmd,
        cwd=str(scripts),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    tail = []
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
        tail.append(line.rstrip())
        del tail[:-40]
    code = proc.wait()
    if code != 0:
        raise SystemExit(
            "occlusion_features.py exited with code %s. Last lines:\n%s" % (code, "\n".join(tail))
        )

stream([str(PYTHON), "-u", str(path)])
print("summary", LOCAL_REPO / "outputs" / "permanence" / "occlusion_features" / "summary.json")


In [ ]:
# Paper gaps — one feature rule, gripper scan, zoomed figure, error bar, short rollout
# Paste into the same runtime after the contrast cell. Does not retrain.
# Does not overwrite outputs/permanence/controlled_contrasts/ or occlusion_features/.

import os, sys, subprocess
from pathlib import Path

os.environ.setdefault("CTRL_SCAN_EPISODES", "20")
os.environ.setdefault("CTRL_SCAN_STEPS", "200")
os.environ.setdefault("CTRL_SCAN_STRIDE", "2")
os.environ.setdefault("CTRL_HELD_FRAMES", "4")
os.environ.setdefault("CTRL_ROLLOUT_STEPS", "10")
os.environ.setdefault("CTRL_BOOTSTRAP", "2000")
os.environ.setdefault("CTRL_MIN_SPECIFICITY", "0.5")
os.environ.setdefault("CTRL_SEED", "0")
os.environ.setdefault("SUITE", "libero_spatial")
os.environ.setdefault("TASK_IDS", "[0]")
os.environ.setdefault(
    "LAYER_NAME",
    "model.paligemma_with_expert.paligemma.model.language_model.layers.5",
)
os.environ.setdefault("MUJOCO_GL", "egl")
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

LOCAL_REPO = Path(os.environ.get("LOCAL_REPO", "/content/groot-run"))
PYTHON = Path("/content/lerobot-venv/bin/python")
if not PYTHON.exists():
    PYTHON = Path(sys.executable)

SCRIPT = r'''"""Close the gaps that block reading the occlusion run as a paper.

One rule for features: the code must rise on paint over the bowl and stay quiet
on paint off the bowl, on color, and on absence. The largest feature that fails
is reported, including when it falls or also moves for absence.

The same frames get the causal ceiling (full prefix, token structure, pooled),
a second noise seed, and a bootstrap interval. A short rollout asks whether the
chunk changes task success. A scan of the demonstrations counts gripper contact
and gripper hiding. The figure shows the agent view, the wrist view, the mask,
and a zoom of the paint.

Does not retrain the transcoder and does not rewrite earlier output directories.
"""

from __future__ import annotations

import json
import os
import re
from pathlib import Path

import cv2
import h5py
import numpy as np
import torch

import controlled_contrasts as cc
import occlusion_features as oc

CAPTURE = ("base", "recolor", "absent", "occluded", "slab_miss")
HELD_LABELS = ("held_hidden", "held_visible")


def bootstrap_interval(values: list[float], rng: np.random.Generator, draws: int = 2000) -> dict:
    arr = np.asarray([value for value in values if value is not None and np.isfinite(value)], dtype=np.float64)
    if arr.size == 0:
        return {"mean": None, "low": None, "high": None, "n": 0}
    if arr.size == 1:
        value = float(arr[0])
        return {"mean": value, "low": value, "high": value, "n": 1}
    picks = rng.integers(0, int(arr.size), size=(int(draws), int(arr.size)))
    means = arr[picks].mean(axis=1)
    return {
        "mean": float(arr.mean()),
        "low": float(np.quantile(means, 0.025)),
        "high": float(np.quantile(means, 0.975)),
        "n": int(arr.size),
    }


def zoom_box(mask: np.ndarray, pad: int, height: int, width: int) -> tuple[int, int, int, int]:
    ys, xs = np.nonzero(np.asarray(mask))
    if ys.size == 0:
        return 0, int(height), 0, int(width)
    y0 = max(0, int(ys.min()) - int(pad))
    y1 = min(int(height), int(ys.max()) + int(pad) + 1)
    x0 = max(0, int(xs.min()) - int(pad))
    x1 = min(int(width), int(xs.max()) + int(pad) + 1)
    return y0, y1, x0, x1


def crop(image: np.ndarray, box: tuple[int, int, int, int]) -> np.ndarray:
    y0, y1, x0, x1 = box
    return np.asarray(image)[y0:y1, x0:x1]


def spread_frames(rows: list[dict], limit: int) -> list[dict]:
    if limit < 1 or not rows:
        return []
    buckets: dict[int, list[dict]] = {}
    for row in rows:
        buckets.setdefault(int(row["episode"]), []).append(row)
    picked: list[dict] = []
    while len(picked) < limit:
        grew = False
        for episode in sorted(buckets):
            bucket = buckets[episode]
            if bucket and len(picked) < limit:
                picked.append(bucket.pop(0))
                grew = True
        if not grew:
            break
    return picked


def rejection_reasons(row: dict, min_specificity: float) -> list[str]:
    reasons: list[str] = []
    if float(row["occlusion"]) <= 0:
        reasons.append("falls instead of rising")
    if float(row["specificity"]) <= float(min_specificity):
        reasons.append(f"specificity {float(row['specificity']):.2f} is at or below {float(min_specificity):.2f}")
    if abs(float(row["absence"])) > 0.5 * max(abs(float(row["occlusion"])), 1e-8):
        reasons.append("also moves for absence")
    if abs(float(row["color"])) > 0.5 * max(abs(float(row["occlusion"])), 1e-8):
        reasons.append("also moves for color")
    if abs(float(row["slab"])) > 0.5 * max(abs(float(row["occlusion"])), 1e-8):
        reasons.append("also moves for paint off the bowl")
    return reasons


def claim_line(r2: float | None, full_gap: float | None, feature_gap: float | None) -> str:
    r2_text = "unknown" if r2 is None else f"{r2:.3f}"
    full_text = "unknown" if full_gap is None else f"{100 * full_gap:.1f}%"
    if feature_gap is None:
        feature_text = "no feature passed the rule, so there is no feature effect to report"
    else:
        feature_text = f"the features that pass close {100 * feature_gap:.1f}%"
    return (
        f"Layer-5 reconstruction R2 is {r2_text}. "
        f"The causal ceiling on this paint edit is the full prefix at {full_text}. "
        f"{feature_text}."
    )


def interval_text(stats: dict) -> str:
    if stats["mean"] is None:
        return "undefined"
    if stats["n"] < 2:
        return f"{100 * stats['mean']:.1f}% (n={stats['n']})"
    return (
        f"{100 * stats['mean']:.1f}% "
        f"[{100 * stats['low']:.1f}%, {100 * stats['high']:.1f}%] n={stats['n']}"
    )


def read_demo_states(path: Path, max_steps: int) -> list[np.ndarray]:
    demos: list[np.ndarray] = []
    with h5py.File(path, "r") as handle:
        names = sorted(
            handle["data"].keys(),
            key=lambda name: int(re.search(r"(\d+)$", name).group(1)),
        )
        for name in names:
            group = handle["data"][name]
            states = np.asarray(group["states"])
            actions = np.asarray(group["actions"])
            if len(states) == len(actions) + 1:
                states = states[:-1]
            demos.append(states[:max_steps])
    return demos


def object_xyz(sim, body: int, span: tuple[int, int] | None) -> np.ndarray | None:
    if span is None:
        return None
    start = int(span[0])
    return np.asarray(sim.data.qpos[start : start + 3], dtype=np.float64).copy()


def task_succeeded(env) -> bool:
    for owner in (env, getattr(env, "env", None)):
        if owner is None:
            continue
        for name in ("check_success", "_check_success"):
            fn = getattr(owner, name, None)
            if callable(fn):
                return bool(fn())
    return False


def step_env(env, action: np.ndarray) -> None:
    vector = np.asarray(action, dtype=np.float64).reshape(-1)
    space = getattr(env, "action_space", None)
    width = int(space.shape[0]) if space is not None and getattr(space, "shape", None) else vector.size
    vector = vector[:width]
    outcome = env.step(vector)
    if not isinstance(outcome, tuple) or len(outcome) not in (4, 5):
        raise RuntimeError(f"env.step returned {type(outcome)}")


def overlay(image: np.ndarray, mask: np.ndarray) -> np.ndarray:
    out = np.array(image, copy=True)
    if mask is None or not np.asarray(mask).any():
        return out
    red = np.zeros_like(out)
    red[..., 0] = 255
    out[mask] = (0.45 * out[mask] + 0.55 * red[mask]).astype(np.uint8)
    return out


def enlarge(image: np.ndarray, factor: int = 4) -> np.ndarray:
    return np.repeat(np.repeat(np.asarray(image), factor, axis=0), factor, axis=1)


def write_edit_sheet(frame: dict, path: Path) -> None:
    images = frame["built"]["images"]
    partial = frame["built"].get("agent_partial")
    wrist_partial = frame["built"].get("wrist_partial")
    agent0 = images["base"][0]
    box = zoom_box(partial if partial is not None else np.zeros(agent0.shape[:2], dtype=bool), 36, agent0.shape[0], agent0.shape[1])
    wrist0 = images["base"][1]
    wrist_box = zoom_box(
        wrist_partial if wrist_partial is not None else np.zeros(wrist0.shape[:2], dtype=bool),
        36,
        wrist0.shape[0],
        wrist0.shape[1],
    )
    columns = ("base", "recolor", "occluded", "slab_miss", "absent")
    rows = []
    for kind, region in (("agent", box), ("wrist", wrist_box)):
        view = 0 if kind == "agent" else 1
        mask = partial if kind == "agent" else wrist_partial
        picture_row = []
        zoom_row = []
        mask_row = []
        for condition in columns:
            picture = images[condition][view]
            picture_row.append(picture)
            zoom_row.append(crop(picture, region if kind == "agent" else wrist_box))
            mask_row.append(overlay(picture, mask) if mask is not None else picture)
        rows.append((f"{kind}", picture_row))
        rows.append((f"{kind}_zoom", zoom_row))
        rows.append((f"{kind}_mask", mask_row))
    cc.write_sheet(rows, path)
    occluded = images["occluded"][0]
    zoom = enlarge(crop(occluded, box), 4)
    base_zoom = enlarge(crop(agent0, box), 4)
    pair = np.concatenate([base_zoom, zoom], axis=1)
    cv2.imwrite(str(path.with_name(path.stem + "_zoom.png")), cv2.cvtColor(pair, cv2.COLOR_RGB2BGR))


def cover_pair(frame: dict) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, float] | None:
    cover = frame["built"].get("agent_full")
    if cover is None:
        return None
    images = frame["built"]["images"]
    present = cc.paint_mask(images["base"][0], cover, cc.GRAY)
    absent = cc.paint_mask(images["absent"][0], cover, cc.GRAY)
    present_wrist = images["base"][1]
    absent_wrist = images["absent"][1]
    outside = cc.outside_fraction(present, absent, cover)
    wrist_cover = frame["built"].get("wrist_full")
    if wrist_cover is not None:
        present_wrist = cc.paint_mask(present_wrist, wrist_cover, cc.GRAY)
        absent_wrist = cc.paint_mask(absent_wrist, wrist_cover, cc.GRAY)
        outside = max(outside, cc.outside_fraction(present_wrist, absent_wrist, wrist_cover))
    return present, present_wrist, absent, absent_wrist, float(outside)


def per_frame_gaps(records: list[dict], patch: str, seed: int | None) -> list[float]:
    return [
        row["fraction_gap_closed"]
        for row in records
        if row["patch"] == patch and row["seed_tag"] == seed and row["fraction_gap_closed"] is not None
    ]


def main() -> None:
    cc.configure_environment()
    if not torch.cuda.is_available():
        raise SystemExit("CUDA GPU required. Use the Colab L4 or A100 runtime.")
    os.environ["MAX_STEPS"] = os.environ.get("CTRL_SCAN_STEPS", "200")
    collect = cc.load_collect_module()
    seed = cc.env_int("CTRL_SEED", 0)
    min_specificity = float(os.environ.get("CTRL_MIN_SPECIFICITY", "0.5"))
    scan_episodes = cc.env_int("CTRL_SCAN_EPISODES", 20)
    scan_steps = cc.env_int("CTRL_SCAN_STEPS", 200)
    stride = cc.env_int("CTRL_SCAN_STRIDE", 2)
    held_limit = cc.env_int("CTRL_HELD_FRAMES", 4)
    rollout_steps = cc.env_int("CTRL_ROLLOUT_STEPS", 10)
    draws = cc.env_int("CTRL_BOOTSTRAP", 2000)
    repo = Path(os.environ.get("LOCAL_REPO", "/content/groot-run"))
    source = repo / "outputs" / "permanence" / "controlled_contrasts"
    dest = repo / "outputs" / "permanence" / "paper_gaps"
    dest.mkdir(parents=True, exist_ok=True)
    summary_path = source / "summary.json"
    checkpoint_path = source / "transcoder.pt"
    if not summary_path.is_file() or not checkpoint_path.is_file():
        raise SystemExit(f"Missing {summary_path} or {checkpoint_path}. Run the contrast cell first.")
    prior = json.loads(summary_path.read_text())
    replicate = [(int(row["episode"]), int(row["step"])) for row in prior["probe_frames"]]
    r2 = (prior.get("transcoder") or {}).get("r2_layer5_probe_base")
    model, mean5, std5, n_features = oc.load_transcoder(checkpoint_path)
    print(f"Loaded transcoder features={n_features} R2={r2}", flush=True)

    # Imported here so unit tests can load this module without LIBERO.
    from libero.libero import benchmark, get_libero_path
    from libero.libero.envs import OffScreenRenderEnv

    suite = benchmark.get_benchmark_dict()[collect.SUITE]()
    task_id = int(collect.TASK_IDS[0])
    task = suite.get_task(task_id)
    sentence = task.language
    demo_path = collect.find_demo_file(task)
    bddl = os.path.join(get_libero_path("bddl_files"), task.problem_folder, task.bddl_file)
    demos = read_demo_states(demo_path, scan_steps)
    print(f"Task {task_id}: {sentence}", flush=True)
    print(f"Demos in file: {len(demos)}. Scanning {min(scan_episodes, len(demos))} at stride {stride}.", flush=True)

    env = OffScreenRenderEnv(bddl_file_name=bddl, camera_heights=256, camera_widths=256)
    env.seed(0)
    try:
        census: dict[str, int] = {}
        held_rows: list[dict] = []
        for episode, states in enumerate(demos[:scan_episodes]):
            print(f"Scanning episode {episode} ({len(states)} steps)", flush=True)
            for step in range(0, len(states), stride):
                collect.observe_state(env, states[step])
                sim = collect.sim_of(env)
                label, body = cc.frame_label(collect.label_frame(sim, sentence, collect.candidate_bodies(sim, sentence)))
                name = label or "unlabeled"
                census[name] = census.get(name, 0) + 1
                if name in HELD_LABELS and body is not None:
                    held_rows.append({"episode": episode, "step": step, "label": name, "body": int(body)})
        print("CENSUS", census, flush=True)
        hidden = [row for row in held_rows if row["label"] == "held_hidden"]
        visible = [row for row in held_rows if row["label"] == "held_visible"]
        chosen_held = spread_frames(hidden or visible, held_limit)
        print(
            f"Gripper contact frames: hidden={len(hidden)} visible={len(visible)}. "
            f"Using {len(chosen_held)} for the policy.",
            flush=True,
        )

        print("Loading policy", flush=True)
        policy, pre, _post, layer5 = collect.load_policy()
        policy.eval()
        layer6 = cc.find_layer6(policy, layer5)

        def rebuild(episode: int, step: int):
            states = demos[episode]
            if step >= len(states):
                raise SystemExit(f"Episode {episode} has {len(states)} scanned steps; cannot rebuild step {step}.")
            print(f"Rebuilding ep {episode} step {step}", flush=True)
            row = cc.inspect_candidate(collect, env, collect.sim_of(env), sentence, episode, step, states[step])
            return row

        replicate_frames = []
        for episode, step in replicate:
            row = rebuild(episode, step)
            if row is None:
                raise SystemExit(f"Could not rebuild replicate ep {episode} step {step}.")
            replicate_frames.append(row)
        held_frames = []
        for item in chosen_held:
            row = rebuild(int(item["episode"]), int(item["step"]))
            if row is None:
                print(f"Skipped held ep {item['episode']} step {item['step']}: controlled images failed.", flush=True)
                continue
            held_frames.append(row)

        sheet = dest / "edit_sheet.png"
        write_edit_sheet(replicate_frames[0], sheet)
        print("Wrote", sheet, "and", sheet.with_name(sheet.stem + "_zoom.png"), flush=True)
        if held_frames:
            held_sheet = dest / "held_sheet.png"
            write_edit_sheet(held_frames[0], held_sheet)
            print("Wrote", held_sheet, flush=True)

        def capture_frame(frame, noise: int) -> dict:
            packed = {}
            for condition in CAPTURE:
                tokens5, _tokens6, action, _device = cc.capture(
                    policy, layer5, layer6, collect, pre, frame, condition, noise
                )
                packed[condition] = {"l5": tokens5, "action": action}
            return packed

        groups = {"paint": replicate_frames, "gripper": held_frames}
        records: list[dict] = []
        rollouts: list[dict] = []
        covers: list[dict] = []
        passed_rows: list[dict] = []
        leader_row: dict | None = None
        leader_reasons: list[str] = []
        rng = np.random.default_rng(seed)

        for group_name, frames in groups.items():
            if not frames:
                continue
            packed_seed_a = []
            for frame in frames:
                noise_a = 5000 + int(frame["episode"]) * 100 + int(frame["step"])
                packed_seed_a.append(capture_frame(frame, noise_a))
            check = frames[0]
            noise_a = 5000 + int(check["episode"]) * 100 + int(check["step"])
            again5, _again6, again_action, device = cc.capture(
                policy, layer5, layer6, collect, pre, check, "base", noise_a
            )
            max_token = float(np.max(np.abs(again5 - packed_seed_a[0]["base"]["l5"])))
            if max_token > 1e-3 or not np.array_equal(again_action, packed_seed_a[0]["base"]["action"]):
                raise RuntimeError(f"{group_name} determinism failed (token max abs {max_token}).")
            print(f"{group_name} determinism passed (token max abs {max_token:.3e})", flush=True)

            token_maps = [{name: item[name]["l5"] for name in CAPTURE} for item in packed_seed_a]
            scores = cc.score_probe(model, mean5, std5, token_maps)
            occlusion_ids = oc.specific_ids(scores, "occlusion", oc.QUIET_FOR["occlusion"], min_specificity)
            color_ids = oc.specific_ids(scores, "color", oc.QUIET_FOR["color"], min_specificity)
            leader_id = int(np.argmax(np.abs(np.asarray(scores["occlusion"], dtype=np.float64))))
            leader_rows = oc.feature_rows(np.array([leader_id]), scores, "occlusion", oc.QUIET_FOR["occlusion"])
            leader = leader_rows[0]
            reasons = rejection_reasons(leader, min_specificity)
            if group_name == "paint":
                passed_rows = oc.feature_rows(occlusion_ids, scores, "occlusion", oc.QUIET_FOR["occlusion"])
                leader_row = leader
                leader_reasons = reasons
            print(
                f"{group_name}: passed {occlusion_ids.tolist() or 'none'} | "
                f"largest |occlusion| feature {leader_id} reasons={reasons or ['passes the rule']}",
                flush=True,
            )
            oc.print_feature_table(f"{group_name} PASSED", oc.feature_rows(occlusion_ids, scores, "occlusion", oc.QUIET_FOR["occlusion"]))

            pool = np.setdiff1d(np.arange(n_features), occlusion_ids) if occlusion_ids.size else np.arange(n_features)
            random_ids = (
                rng.choice(pool, size=int(occlusion_ids.size), replace=False).astype(np.int64)
                if occlusion_ids.size
                else np.array([], dtype=np.int64)
            )

            for frame, packed in zip(frames, packed_seed_a):
                noise_a = 5000 + int(frame["episode"]) * 100 + int(frame["step"])
                noise_b = 9000 + int(frame["episode"]) * 100 + int(frame["step"])
                source_tokens = packed["base"]["l5"]
                source_action = packed["base"]["action"]
                donor_tokens = packed["occluded"]["l5"]
                donor_action = packed["occluded"]["action"]
                agent, wrist = frame["built"]["images"]["base"]
                batch = cc.batch_from_images(collect, pre, agent, wrist, frame["state"], frame["sentence"], policy)
                code_delta = cc.encode_tokens(model, donor_tokens, mean5, std5) - cc.encode_tokens(
                    model, source_tokens, mean5, std5
                )

                def run_patch(patch: str, mode: str, payload_np: np.ndarray) -> np.ndarray:
                    _tokens, _tokens6, action = cc.forward_policy(
                        policy, layer5, None, batch, noise_a, cc.to_gpu(payload_np, device), mode
                    )
                    metrics = cc.action_metrics(action, source_action, donor_action)
                    records.append(
                        {
                            "group": group_name,
                            "episode": int(frame["episode"]),
                            "step": int(frame["step"]),
                            "label": frame["label"],
                            "patch": patch,
                            "seed_tag": 0,
                            "edit_l2": float(np.linalg.norm(payload_np.astype(np.float64))),
                            **metrics,
                        }
                    )
                    return action

                run_patch("full_tokens", "replace", donor_tokens.astype(np.float32))
                run_patch("token_structure", "add", cc.structure_delta(donor_tokens, source_tokens))
                pooled = np.repeat((donor_tokens - source_tokens).mean(axis=0, keepdims=True), source_tokens.shape[0], axis=0)
                run_patch("pooled", "add", pooled.astype(np.float32))
                feature_action = None
                if occlusion_ids.size:
                    feature_payload = oc.decoded_patch(model, code_delta, occlusion_ids, std5, None)
                    moved = cc.remap_features(cc.keep_features(code_delta, occlusion_ids), occlusion_ids, random_ids)
                    random_payload = cc.match_l2(cc.decode_delta(model, moved, std5), feature_payload)
                    color_take = color_ids[: int(occlusion_ids.size)]
                    color_payload = oc.decoded_patch(model, code_delta, color_take, std5, feature_payload)
                    feature_action = run_patch("occlusion_features", "add", feature_payload)
                    run_patch("random_remap", "add", random_payload)
                    run_patch("color_features", "add", color_payload)
                leader_payload = oc.decoded_patch(model, code_delta, np.array([leader_id]), std5, None)
                run_patch("rejected_leader", "add", leader_payload)
                absence_rmse = float(np.sqrt(np.mean((source_action - packed["absent"]["action"]) ** 2)))
                records.append(
                    {
                        "group": group_name,
                        "episode": int(frame["episode"]),
                        "step": int(frame["step"]),
                        "label": frame["label"],
                        "patch": "absence_rmse",
                        "seed_tag": 0,
                        "edit_l2": absence_rmse,
                        "source_to_donor_rmse": absence_rmse,
                        "edited_to_donor_rmse": None,
                        "fraction_gap_closed": None,
                    }
                )

                packed_b = capture_frame(frame, noise_b)
                source_b = packed_b["base"]["action"]
                donor_b = packed_b["occluded"]["action"]
                batch_b_tokens = packed_b["occluded"]["l5"]
                _tokens, _tokens6, action_b = cc.forward_policy(
                    policy, layer5, None, batch, noise_b, cc.to_gpu(batch_b_tokens.astype(np.float32), device), "replace"
                )
                records.append(
                    {
                        "group": group_name,
                        "episode": int(frame["episode"]),
                        "step": int(frame["step"]),
                        "label": frame["label"],
                        "patch": "full_tokens",
                        "seed_tag": 1,
                        "edit_l2": float(np.linalg.norm(batch_b_tokens.astype(np.float64) - packed_b["base"]["l5"].astype(np.float64))),
                        **cc.action_metrics(action_b, source_b, donor_b),
                    }
                )
                structure_b = cc.structure_delta(packed_b["occluded"]["l5"], packed_b["base"]["l5"])
                _tokens, _tokens6, action_s = cc.forward_policy(
                    policy, layer5, None, batch, noise_b, cc.to_gpu(structure_b, device), "add"
                )
                records.append(
                    {
                        "group": group_name,
                        "episode": int(frame["episode"]),
                        "step": int(frame["step"]),
                        "label": frame["label"],
                        "patch": "token_structure",
                        "seed_tag": 1,
                        "edit_l2": float(np.linalg.norm(structure_b.astype(np.float64))),
                        **cc.action_metrics(action_s, source_b, donor_b),
                    }
                )

                if group_name == "paint":
                    chunks = {
                        "base": source_action,
                        "occluded_image": donor_action,
                    }
                    if feature_action is not None:
                        chunks["occlusion_features"] = feature_action
                    body = int(frame["body"])
                    for name, chunk in chunks.items():
                        try:
                            collect.observe_state(env, frame["sim_state"])
                            sim = collect.sim_of(env)
                            span = collect.object_qpos_span(sim, body)
                            start = object_xyz(sim, body, span)
                            before = task_succeeded(env)
                            for action in np.asarray(chunk)[:rollout_steps]:
                                step_env(env, action)
                            end = object_xyz(collect.sim_of(env), body, span)
                            shift = None if start is None or end is None else float(np.linalg.norm(end - start))
                            rollouts.append(
                                {
                                    "episode": int(frame["episode"]),
                                    "step": int(frame["step"]),
                                    "chunk": name,
                                    "success_before": before,
                                    "success_after": task_succeeded(env),
                                    "bowl_shift_m": shift,
                                }
                            )
                        except Exception as exc:
                            rollouts.append(
                                {
                                    "episode": int(frame["episode"]),
                                    "step": int(frame["step"]),
                                    "chunk": name,
                                    "success_before": None,
                                    "success_after": None,
                                    "bowl_shift_m": None,
                                    "error": str(exc),
                                }
                            )

            for frame in frames:
                covered = cover_pair(frame)
                if covered is None:
                    continue
                present, present_wrist, absent, absent_wrist, outside = covered
                noise = 5000 + int(frame["episode"]) * 100 + int(frame["step"])

                def covered_forward(agent_image, wrist_image):
                    batch = cc.batch_from_images(
                        collect, pre, agent_image, wrist_image, frame["state"], frame["sentence"], policy
                    )
                    _t5, _t6, action = cc.forward_policy(policy, layer5, layer6, batch, noise, None, "base")
                    return action

                present_action = covered_forward(present, present_wrist)
                absent_action = covered_forward(absent, absent_wrist)
                cover_rmse = float(np.sqrt(np.mean((present_action - absent_action) ** 2)))
                covers.append(
                    {
                        "group": group_name,
                        "episode": int(frame["episode"]),
                        "step": int(frame["step"]),
                        "label": frame["label"],
                        "outside": outside,
                        "action_rmse": cover_rmse,
                        "text": cc.full_cover_interpretation(outside, cover_rmse),
                    }
                )

        def mean_gap(group: str, patch: str, seed_tag: int) -> float | None:
            chosen = [
                row["fraction_gap_closed"]
                for row in records
                if row["group"] == group and row["patch"] == patch and row["seed_tag"] == seed_tag
            ]
            return cc.mean_defined(chosen)

        paint_full = bootstrap_interval(per_frame_gaps([row for row in records if row["group"] == "paint"], "full_tokens", 0), rng, draws)
        paint_structure = bootstrap_interval(
            per_frame_gaps([row for row in records if row["group"] == "paint"], "token_structure", 0), rng, draws
        )
        paint_features = bootstrap_interval(
            per_frame_gaps([row for row in records if row["group"] == "paint"], "occlusion_features", 0), rng, draws
        )
        seed_a_full = mean_gap("paint", "full_tokens", 0)
        seed_b_full = mean_gap("paint", "full_tokens", 1)
        seed_a_structure = mean_gap("paint", "token_structure", 0)
        seed_b_structure = mean_gap("paint", "token_structure", 1)

        print("\nGAP 1  reconstruction is not causation", flush=True)
        line = claim_line(r2, seed_a_full, paint_features["mean"])
        print(line, flush=True)
        print(f"  full prefix {interval_text(paint_full)}", flush=True)
        print(f"  token structure {interval_text(paint_structure)}", flush=True)
        def pct(value: float | None) -> str:
            return "undefined" if value is None else f"{100 * value:.1f}%"

        print(f"  pooled {pct(mean_gap('paint', 'pooled', 0))}", flush=True)
        print(f"  strict features {interval_text(paint_features)}", flush=True)
        print(f"  random {pct(mean_gap('paint', 'random_remap', 0))}", flush=True)
        print(f"  color features {pct(mean_gap('paint', 'color_features', 0))}", flush=True)
        print(f"  rejected leader {pct(mean_gap('paint', 'rejected_leader', 0))}", flush=True)

        def absence_text(group: str) -> str:
            vals = [
                row["source_to_donor_rmse"]
                for row in records
                if row["group"] == group and row["patch"] == "absence_rmse"
            ]
            if not vals:
                return "none"
            return f"mean RMSE {float(np.mean(vals)):.4f} over {len(vals)} frames"

        print("\nGAP 2  gripper hiding, separate from the paint blob", flush=True)
        print("  census", census, flush=True)
        print("  removing the bowl, paint frames:", absence_text("paint"), flush=True)
        print("  removing the bowl, gripper frames:", absence_text("gripper"), flush=True)
        if not held_frames:
            print("  No held frame produced controlled images in this scan. The paint edit is still a paint blob.", flush=True)
        else:
            print(f"  held frames {[ (f['episode'], f['step'], f['label']) for f in held_frames ]}", flush=True)
            print(f"  held paint full prefix {pct(mean_gap('gripper', 'full_tokens', 0))}", flush=True)
            print(f"  held strict features {pct(mean_gap('gripper', 'occlusion_features', 0))}", flush=True)
        for cover in covers:
            print(
                f"  cover ep {cover['episode']} step {cover['step']} {cover['label']} "
                f"outside={cover['outside']:.5f} rmse={cover['action_rmse']:.5f}",
                flush=True,
            )
            print("   ", cover["text"], flush=True)

        print("\nGAP 3  more than a point estimate", flush=True)
        print(f"  paint frames {len(replicate_frames)} | held frames {len(held_frames)} | one task, task {task_id}", flush=True)
        print(f"  seed A full {pct(seed_a_full)} | seed B full {pct(seed_b_full)}", flush=True)
        print(f"  seed A structure {pct(seed_a_structure)} | seed B structure {pct(seed_b_structure)}", flush=True)
        print(f"  rollout steps {rollout_steps}", flush=True)
        for row in rollouts:
            print(
                f"  ep {row['episode']} step {row['step']} {row['chunk']}: "
                f"success {row['success_before']} -> {row['success_after']} bowl_shift_m={row.get('bowl_shift_m')} {row.get('error', '')}",
                flush=True,
            )

        print("\nGAP 4  figure", flush=True)
        wrist = replicate_frames[0]["built"]["images"]
        agent_recolor = cc.changed_fraction(wrist["base"][0], wrist["recolor"][0])
        wrist_recolor = cc.changed_fraction(wrist["base"][1], wrist["recolor"][1])
        agent_paint = cc.changed_fraction(wrist["base"][0], wrist["occluded"][0])
        print(f"  agent recolor {agent_recolor:.4f} | wrist recolor {wrist_recolor:.4f} | agent paint {agent_paint:.4f}", flush=True)
        print("  sheet", sheet, flush=True)
        print("  zoom", sheet.with_name(sheet.stem + "_zoom.png"), flush=True)

        print("\nGAP 5  one rule", flush=True)
        print(f"  passed {[row['feature'] for row in passed_rows] or 'none'}", flush=True)
        if leader_row is not None:
            print(
                f"  largest |occlusion| feature {leader_row['feature']} "
                f"occlusion={leader_row['occlusion']:.3f} slab={leader_row['slab']:.3f} "
                f"color={leader_row['color']:.3f} absence={leader_row['absence']:.3f} "
                f"specificity={leader_row['specificity']:.3f}",
                flush=True,
            )
            print("  rejected because:", "; ".join(leader_reasons) if leader_reasons else "it passes the rule", flush=True)

        summary = {
            "task": sentence,
            "task_id": task_id,
            "census": census,
            "replicate_frames": [
                {"episode": int(frame["episode"]), "step": int(frame["step"]), "label": frame["label"]}
                for frame in replicate_frames
            ],
            "held_frames": [
                {"episode": int(frame["episode"]), "step": int(frame["step"]), "label": frame["label"]}
                for frame in held_frames
            ],
            "passed": passed_rows,
            "leader": leader_row,
            "leader_reasons": leader_reasons,
            "claim": line,
            "intervals": {"full_tokens": paint_full, "token_structure": paint_structure, "occlusion_features": paint_features},
            "seed_replicate": {
                "full_a": seed_a_full,
                "full_b": seed_b_full,
                "structure_a": seed_a_structure,
                "structure_b": seed_b_structure,
            },
            "covers": covers,
            "rollouts": rollouts,
            "records": records,
            "figure": str(sheet),
        }
        (dest / "summary.json").write_text(json.dumps(cc.json_ready(summary), indent=2))
        print("\nWrote", dest / "summary.json", flush=True)
        print("RESULT_DIR", dest, flush=True)
    finally:
        env.close()


if __name__ == "__main__":
    main()
'''

scripts = LOCAL_REPO / "scripts"
scripts.mkdir(parents=True, exist_ok=True)
path = scripts / "paper_gaps.py"
path.write_text(SCRIPT)
for required in ("controlled_contrasts.py", "occlusion_features.py", "collect_layer5_replay.py"):
    if not (scripts / required).is_file():
        raise SystemExit(
            f"scripts/{required} is missing. Run the collector cell, the contrast cell, and the occlusion cell first."
        )

env = os.environ.copy()
env["PYTHONPATH"] = str(scripts) + os.pathsep + env.get("PYTHONPATH", "")
env["LOCAL_REPO"] = str(LOCAL_REPO)
env["PYTHONUNBUFFERED"] = "1"
env["MAX_STEPS"] = env.get("CTRL_SCAN_STEPS", "200")

def stream(cmd):
    proc = subprocess.Popen(
        cmd,
        cwd=str(scripts),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    tail = []
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
        tail.append(line.rstrip())
        del tail[:-60]
    code = proc.wait()
    if code != 0:
        raise SystemExit(
            "paper_gaps.py exited with code %s. Last lines:\n%s" % (code, "\n".join(tail))
        )

stream([str(PYTHON), "-u", str(path)])
print("summary", LOCAL_REPO / "outputs" / "permanence" / "paper_gaps" / "summary.json")


In [ ]:
# Cover autopsy — which camera, leftover pixels, and whether the arm moves
# Paste into the same runtime after the paper-gaps cell. Does not retrain.
# Does not overwrite outputs/permanence/paper_gaps/ or earlier directories.

import os, sys, subprocess
from pathlib import Path

os.environ.setdefault("CTRL_SCAN_EPISODES", "20")
os.environ.setdefault("CTRL_SCAN_STEPS", "200")
os.environ.setdefault("CTRL_GEOMETRY_STRIDE", "4")
os.environ.setdefault("CTRL_ROLLOUT_STEPS", "10")
os.environ.setdefault("CTRL_SEED", "0")
os.environ.setdefault("SUITE", "libero_spatial")
os.environ.setdefault("TASK_IDS", "[0]")
os.environ.setdefault(
    "LAYER_NAME",
    "model.paligemma_with_expert.paligemma.model.language_model.layers.5",
)
os.environ.setdefault("MUJOCO_GL", "egl")
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

LOCAL_REPO = Path(os.environ.get("LOCAL_REPO", "/content/groot-run"))
PYTHON = Path("/content/lerobot-venv/bin/python")
if not PYTHON.exists():
    PYTHON = Path(sys.executable)

SCRIPT = r'''
"""Explain the cover gaps and the inert rollout.

On each saved frame, compare three forwards: the natural covered pair, the same
covered image twice, and each camera swapped alone. A second pass records how
far the gripper is from hiding the bowl. A third pass compares the first 10
actions with the rest of the chunk, then steps the arm so a dead simulator
step is visible.

Does not retrain and does not rewrite earlier output directories.
"""

from __future__ import annotations

import json
import os
from pathlib import Path

import numpy as np
import torch

import controlled_contrasts as cc
import paper_gaps

PAINT_FRAMES = ((1, 0), (1, 8), (1, 15), (1, 23))
HELD_FRAMES = ((0, 44), (1, 44), (2, 40), (3, 46))


def cover_reading(outside: float, rmse: float) -> str:
    if outside >= 0.002:
        return "images still differ outside the cover"
    if rmse < 1e-3:
        return "images agree and the action agrees"
    if rmse < 0.05:
        return "images agree and the action gap is small"
    return "images agree and the action still moves"


def diff_stats(before: np.ndarray, after: np.ndarray) -> tuple[int, int]:
    delta = np.abs(np.asarray(before).astype(np.int16) - np.asarray(after).astype(np.int16))
    changed = delta.max(axis=-1) > 0
    peak = int(delta.max()) if changed.any() else 0
    return int(changed.sum()), peak


def window_rmse(source: np.ndarray, edited: np.ndarray, count: int) -> float:
    left = np.asarray(source, dtype=np.float64)[:count]
    right = np.asarray(edited, dtype=np.float64)[:count]
    if left.size == 0 or right.size == 0:
        return float("nan")
    return float(np.sqrt(np.mean((left - right) ** 2)))


def action_rmse(source: np.ndarray, edited: np.ndarray) -> float:
    return window_rmse(source, edited, max(np.asarray(source).shape[0], 1))


def main() -> None:
    cc.configure_environment()
    if not torch.cuda.is_available():
        raise SystemExit("CUDA GPU required. Use the Colab L4 or A100 runtime.")
    os.environ["MAX_STEPS"] = os.environ.get("CTRL_SCAN_STEPS", "200")
    collect = cc.load_collect_module()
    scan_episodes = cc.env_int("CTRL_SCAN_EPISODES", 20)
    scan_steps = cc.env_int("CTRL_SCAN_STEPS", 200)
    stride = cc.env_int("CTRL_GEOMETRY_STRIDE", 4)
    rollout_steps = cc.env_int("CTRL_ROLLOUT_STEPS", 10)
    repo = Path(os.environ.get("LOCAL_REPO", "/content/groot-run"))
    source = repo / "outputs" / "permanence" / "paper_gaps" / "summary.json"
    dest = repo / "outputs" / "permanence" / "cover_autopsy"
    dest.mkdir(parents=True, exist_ok=True)
    paint = list(PAINT_FRAMES)
    held = list(HELD_FRAMES)
    if source.is_file():
        prior = json.loads(source.read_text())
        if prior.get("replicate_frames"):
            paint = [(int(row["episode"]), int(row["step"])) for row in prior["replicate_frames"]]
        if prior.get("held_frames"):
            held = [(int(row["episode"]), int(row["step"])) for row in prior["held_frames"]]
    print("Paint frames", paint, flush=True)
    print("Held frames", held, flush=True)

    # Imported here so unit tests can load this module without LIBERO.
    from libero.libero import benchmark, get_libero_path
    from libero.libero.envs import OffScreenRenderEnv

    suite = benchmark.get_benchmark_dict()[collect.SUITE]()
    task_id = int(collect.TASK_IDS[0])
    task = suite.get_task(task_id)
    sentence = task.language
    demo_path = collect.find_demo_file(task)
    bddl = os.path.join(get_libero_path("bddl_files"), task.problem_folder, task.bddl_file)
    demos = paper_gaps.read_demo_states(demo_path, scan_steps)
    print(f"Task {task_id}: {sentence}", flush=True)

    env = OffScreenRenderEnv(bddl_file_name=bddl, camera_heights=256, camera_widths=256)
    env.seed(0)
    try:
        gaps: list[float] = []
        ratios: list[float] = []
        hidden_at_zero = 0
        held_n = 0
        print(f"Geometry scan: {min(scan_episodes, len(demos))} episodes, stride {stride}", flush=True)
        for episode, states in enumerate(demos[:scan_episodes]):
            for step in range(0, len(states), stride):
                collect.observe_state(env, states[step])
                sim = collect.sim_of(env)
                result = collect.label_frame(sim, sentence, collect.candidate_bodies(sim, sentence))
                label = result[0] if isinstance(result, tuple) else result
                geometry = result[2] if isinstance(result, tuple) and len(result) > 2 else None
                if label not in ("held_visible", "held_hidden") or not geometry:
                    continue
                held_n += 1
                depth_gap = float(geometry["depth_gap"])
                ratio = float(geometry["angle_ratio"])
                gaps.append(depth_gap)
                ratios.append(ratio)
                if depth_gap > 0.0 and ratio < 1.0:
                    hidden_at_zero += 1
        print(f"Held samples {held_n}. Hidden even with a zero depth margin: {hidden_at_zero}.", flush=True)
        if gaps:
            gap_arr = np.asarray(gaps)
            ratio_arr = np.asarray(ratios)
            print(
                f"  depth gap m: min {gap_arr.min():.4f} median {np.median(gap_arr):.4f} max {gap_arr.max():.4f}",
                flush=True,
            )
            print(
                f"  angle ratio: min {ratio_arr.min():.3f} median {np.median(ratio_arr):.3f} max {ratio_arr.max():.3f}",
                flush=True,
            )
            print(
                f"  depth gap > 0: {int((gap_arr > 0).sum())} | depth gap > 0.02: {int((gap_arr > 0.02).sum())} | "
                f"angle ratio < 1: {int((ratio_arr < 1).sum())}",
                flush=True,
            )

        print("Loading policy", flush=True)
        policy, pre, _post, layer5 = collect.load_policy()
        policy.eval()
        layer6 = cc.find_layer6(policy, layer5)

        def rebuild(episode: int, step: int):
            if episode >= len(demos) or step >= len(demos[episode]):
                raise SystemExit(f"Episode {episode} step {step} is outside the scanned demos.")
            print(f"Rebuilding ep {episode} step {step}", flush=True)
            row = cc.inspect_candidate(
                collect, env, collect.sim_of(env), sentence, episode, step, demos[episode][step]
            )
            if row is None:
                raise SystemExit(f"Could not rebuild ep {episode} step {step}.")
            return row

        frames = []
        for episode, step in paint:
            row = rebuild(episode, step)
            row["group"] = "paint"
            frames.append(row)
        for episode, step in held:
            row = rebuild(episode, step)
            row["group"] = "held"
            frames.append(row)

        def forward(frame, agent, wrist, noise: int):
            batch = cc.batch_from_images(collect, pre, agent, wrist, frame["state"], frame["sentence"], policy)
            _tokens, _tokens6, action = cc.forward_policy(policy, layer5, layer6, batch, noise, None, "base")
            return action

        rows = []
        print("\nCOVER  which camera still moves the action", flush=True)
        print(
            "agent_px and wrist_px are leftover pixels after the full cover, with the peak absolute change beside them.",
            flush=True,
        )
        print(
            "both = covered present vs covered absent. agent = agent camera only. wrist = wrist camera only.",
            flush=True,
        )
        print("same = the covered present image forwarded twice.", flush=True)
        print(
            f"{'group':<8}{'ep':>4}{'step':>6}{'agent_px':>10}{'peak':>6}{'wrist_px':>10}{'peak':>6}"
            f"{'both':>10}{'agent':>10}{'wrist':>10}{'same':>10}  reading",
            flush=True,
        )
        for frame in frames:
            covered = paper_gaps.cover_pair(frame)
            if covered is None:
                print(f"ep {frame['episode']} step {frame['step']}: no cover mask", flush=True)
                continue
            present, present_wrist, absent, absent_wrist, outside = covered
            agent_px, agent_peak = diff_stats(present, absent)
            wrist_px, wrist_peak = diff_stats(present_wrist, absent_wrist)
            noise = 5000 + int(frame["episode"]) * 100 + int(frame["step"])
            both_present = forward(frame, present, present_wrist, noise)
            both_absent = forward(frame, absent, absent_wrist, noise)
            agent_only = forward(frame, absent, present_wrist, noise)
            wrist_only = forward(frame, present, absent_wrist, noise)
            same_again = forward(frame, present, present_wrist, noise)
            both_rmse = action_rmse(both_present, both_absent)
            agent_rmse = action_rmse(both_present, agent_only)
            wrist_rmse = action_rmse(both_present, wrist_only)
            same_rmse = action_rmse(both_present, same_again)
            reading = cover_reading(outside, both_rmse)
            print(
                f"{frame['group']:<8}{int(frame['episode']):>4}{int(frame['step']):>6}"
                f"{agent_px:>10}{agent_peak:>6}{wrist_px:>10}{wrist_peak:>6}"
                f"{both_rmse:>10.4f}{agent_rmse:>10.4f}"
                f"{wrist_rmse:>10.4f}{same_rmse:>10.4f}  {reading}",
                flush=True,
            )
            rows.append(
                {
                    "group": frame["group"],
                    "episode": int(frame["episode"]),
                    "step": int(frame["step"]),
                    "label": frame["label"],
                    "outside": outside,
                    "agent_pixels": agent_px,
                    "agent_peak": agent_peak,
                    "wrist_pixels": wrist_px,
                    "wrist_peak": wrist_peak,
                    "both_rmse": both_rmse,
                    "agent_only_rmse": agent_rmse,
                    "wrist_only_rmse": wrist_rmse,
                    "identical_rmse": same_rmse,
                    "reading": reading,
                }
            )

        print("\nROLLOUT  first 10 actions versus the rest of the chunk", flush=True)
        print(
            "Read arm move against zeros. When base and occluded match zeros, those 10 steps leave the arm where it started.",
            flush=True,
        )
        print(
            "When the first-10 action RMSE is near 0 and the later RMSE is large, the chunk changes after the rollout window.",
            flush=True,
        )
        checks = [frame for frame in frames if (int(frame["episode"]), int(frame["step"])) in {(1, 0), (1, 44), (3, 46)}]
        rollout_rows = []
        for frame in checks:
            noise = 5000 + int(frame["episode"]) * 100 + int(frame["step"])
            base_tokens, _t6, base_action, _device = cc.capture(
                policy, layer5, layer6, collect, pre, frame, "base", noise
            )
            _occ_tokens, _t6b, occ_action, _device = cc.capture(
                policy, layer5, layer6, collect, pre, frame, "occluded", noise
            )
            del base_tokens
            early = window_rmse(base_action, occ_action, rollout_steps)
            late = window_rmse(
                np.asarray(base_action)[rollout_steps:],
                np.asarray(occ_action)[rollout_steps:],
                max(np.asarray(base_action).shape[0] - rollout_steps, 1),
            )
            motions = {}
            for name, chunk in (("base", base_action), ("occluded", occ_action), ("zeros", np.zeros_like(base_action))):
                collect.observe_state(env, frame["sim_state"])
                sim = collect.sim_of(env)
                start = np.asarray(sim.data.qpos[:7], dtype=np.float64).copy()
                for action in np.asarray(chunk)[:rollout_steps]:
                    paper_gaps.step_env(env, action)
                end = np.asarray(collect.sim_of(env).data.qpos[:7], dtype=np.float64).copy()
                motions[name] = float(np.linalg.norm(end - start))
            print(
                f"  ep {frame['episode']} step {frame['step']} {frame['group']}: "
                f"first {rollout_steps} action RMSE {early:.4f} | later RMSE {late:.4f} | "
                f"arm move base {motions['base']:.5f} occluded {motions['occluded']:.5f} zeros {motions['zeros']:.5f}",
                flush=True,
            )
            rollout_rows.append(
                {
                    "episode": int(frame["episode"]),
                    "step": int(frame["step"]),
                    "group": frame["group"],
                    "early_rmse": early,
                    "late_rmse": late,
                    "arm_move": motions,
                }
            )

        summary = {
            "task": sentence,
            "held_samples": held_n,
            "hidden_if_depth_margin_is_zero": hidden_at_zero,
            "depth_gap": {
                "min": float(np.min(gaps)) if gaps else None,
                "median": float(np.median(gaps)) if gaps else None,
                "max": float(np.max(gaps)) if gaps else None,
            },
            "angle_ratio": {
                "min": float(np.min(ratios)) if ratios else None,
                "median": float(np.median(ratios)) if ratios else None,
                "max": float(np.max(ratios)) if ratios else None,
            },
            "covers": rows,
            "rollouts": rollout_rows,
        }
        (dest / "summary.json").write_text(json.dumps(cc.json_ready(summary), indent=2))
        print("\nWrote", dest / "summary.json", flush=True)
        print("RESULT_DIR", dest, flush=True)
    finally:
        env.close()


if __name__ == "__main__":
    main()
'''

scripts = LOCAL_REPO / "scripts"
scripts.mkdir(parents=True, exist_ok=True)
path = scripts / "cover_autopsy.py"
path.write_text(SCRIPT)
for required in ("controlled_contrasts.py", "occlusion_features.py", "collect_layer5_replay.py", "paper_gaps.py"):
    if not (scripts / required).is_file():
        raise SystemExit(
            f"scripts/{required} is missing. Run the collector cell, the contrast cell, the occlusion cell, and the paper-gaps cell first."
        )

env = os.environ.copy()
env["PYTHONPATH"] = str(scripts) + os.pathsep + env.get("PYTHONPATH", "")
env["LOCAL_REPO"] = str(LOCAL_REPO)
env["PYTHONUNBUFFERED"] = "1"
env["MAX_STEPS"] = env.get("CTRL_SCAN_STEPS", "200")

def stream(cmd):
    proc = subprocess.Popen(
        cmd,
        cwd=str(scripts),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    tail = []
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
        tail.append(line.rstrip())
        del tail[:-60]
    code = proc.wait()
    if code != 0:
        raise SystemExit(
            "cover_autopsy.py exited with code %s. Last lines:\n%s" % (code, "\n".join(tail))
        )

stream([str(PYTHON), "-u", str(path)])
print("summary", LOCAL_REPO / "outputs" / "permanence" / "cover_autopsy" / "summary.json")


In [ ]:
# Paper claim — repaired wrist cover, occluder, feature stability, closed loop
# Paste into the same runtime after the cover-autopsy cell. Does not retrain.
# Does not overwrite outputs/permanence/cover_autopsy/ or earlier directories.

import os, sys, subprocess
from pathlib import Path

os.environ.setdefault("CTRL_SCAN_STEPS", "200")
os.environ.setdefault("CTRL_SCOPE_EPISODES", "3")
os.environ.setdefault("CTRL_SCOPE_STRIDE", "16")
os.environ.setdefault("CTRL_SCOPE_STEPS", "96")
os.environ.setdefault("CTRL_EXTRA_FRAMES", "4")
os.environ.setdefault("CTRL_ROLLOUT_HORIZON", "120")
os.environ.setdefault("CTRL_REPLAN", "10")
os.environ.setdefault("CTRL_ROLLOUT_EPISODES", "2")
os.environ.setdefault("CTRL_OCCLUDER_SCALE", "3")
os.environ.setdefault("CTRL_MIN_SPECIFICITY", "0.5")
os.environ.setdefault("CTRL_SEED", "0")
os.environ.setdefault("SUITE", "libero_spatial")
os.environ.setdefault("TASK_IDS", "[0]")
os.environ.setdefault(
    "LAYER_NAME",
    "model.paligemma_with_expert.paligemma.model.language_model.layers.5",
)
os.environ.setdefault("MUJOCO_GL", "egl")
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

LOCAL_REPO = Path(os.environ.get("LOCAL_REPO", "/content/groot-run"))
PYTHON = Path("/content/lerobot-venv/bin/python")
if not PYTHON.exists():
    PYTHON = Path(sys.executable)

SCRIPT = r'''
"""Close the remaining gaps on the occlusion run.

The wrist mask was refused when a color swap changed more than 20% of that
camera, so two held frames still showed the bowl. This run rebuilds the mask
without that cap, paints any pixels that still differ, places another object
on the camera-to-bowl ray, rescores the feature rule on more frames, and runs
a closed-loop episode with the normal 10-step re-query.

Does not retrain and does not rewrite earlier output directories.
"""

from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Literal, assert_never

import cv2
import numpy as np
import torch

import controlled_contrasts as cc
import occlusion_features as oc
import paper_gaps

PAINT_FRAMES = ((1, 0), (1, 8), (1, 15), (1, 23))
HELD_FRAMES = ((0, 44), (1, 44), (2, 40), (3, 46))
FIGURE_FRAMES = ((0, 44), (1, 44), (3, 46))
OCCLUDER_FRAMES = ((1, 0), (1, 44))
FROZEN_PAPER = (498, 366)
FROZEN_STRICT = (206, 366, 65)
EditName = Literal["base", "covered", "absent"]


def uncapped_change_mask(before: np.ndarray, after: np.ndarray, min_pixels: int = 20) -> np.ndarray | None:
    """Pixels the color swap changed, including when they fill more than 20% of the image."""
    changed = cc.changed_pixels(before, after)
    if int(changed.sum()) < int(min_pixels):
        return None
    return changed


def paint_camera(image: np.ndarray, mask: np.ndarray | None) -> np.ndarray:
    if mask is None:
        return np.array(image, copy=True)
    return cc.paint_mask(image, mask, cc.GRAY)


def diff_stats(before: np.ndarray, after: np.ndarray) -> tuple[int, int]:
    delta = np.abs(np.asarray(before).astype(np.int16) - np.asarray(after).astype(np.int16))
    changed = delta.max(axis=-1) > 0
    peak = int(delta.max()) if changed.any() else 0
    return int(changed.sum()), peak


def action_rmse(source: np.ndarray, edited: np.ndarray) -> float:
    left = np.asarray(source, dtype=np.float64)
    right = np.asarray(edited, dtype=np.float64)
    if left.size == 0 or right.size == 0:
        return float("nan")
    return float(np.sqrt(np.mean((left - right) ** 2)))


def residue_reading(pixels: int, peak: int, rmse: float) -> str:
    if pixels == 0 and rmse < 1e-3:
        return "images match and the action matches"
    if pixels == 0:
        return "images match and the action still moves"
    if peak >= 30:
        return "visible residue remains"
    if rmse < 0.05:
        return "a faint residue remains and the action gap is small"
    return "a faint residue remains and the action still moves"


def point_on_ray(camera: np.ndarray, target: np.ndarray, fraction: float) -> np.ndarray:
    origin = np.asarray(camera, dtype=np.float64)
    end = np.asarray(target, dtype=np.float64)
    return origin + float(fraction) * (end - origin)


def covering_ratio(camera: np.ndarray, front: np.ndarray, target: np.ndarray, radius: float) -> tuple[float, float]:
    to_target = np.asarray(target, dtype=np.float64) - np.asarray(camera, dtype=np.float64)
    to_front = np.asarray(front, dtype=np.float64) - np.asarray(camera, dtype=np.float64)
    target_dist = float(np.linalg.norm(to_target))
    front_dist = float(np.linalg.norm(to_front))
    if target_dist < 1e-6 or front_dist < 1e-6:
        return float("nan"), float("nan")
    cosine = float(np.clip(np.dot(to_target, to_front) / (target_dist * front_dist), -1.0, 1.0))
    angle = float(np.arccos(cosine))
    front_angle = float(np.arctan2(radius, front_dist))
    ratio = angle / front_angle if front_angle > 1e-9 else float("inf")
    return target_dist - front_dist, ratio


def prefer_occluder(names: list[tuple[int, str]], target: int) -> int | None:
    pool = [(body, name) for body, name in names if body != target]
    for needle in ("ramekin", "plate"):
        for body, name in pool:
            if needle in name:
                return body
    if not pool:
        return None
    return pool[0][0]


def overlap(found: list[int], frozen: tuple[int, ...]) -> list[int]:
    found_set = set(int(item) for item in found)
    return [int(item) for item in frozen if int(item) in found_set]


def mask_cover_fraction(before: np.ndarray, after: np.ndarray, bowl_mask: np.ndarray | None) -> float:
    if bowl_mask is None or not np.asarray(bowl_mask).any():
        return float("nan")
    changed = cc.changed_pixels(before, after)
    bowl = np.asarray(bowl_mask, dtype=bool)
    return float(np.logical_and(changed, bowl).sum() / max(int(bowl.sum()), 1))


def highlight(image: np.ndarray, mask: np.ndarray | None) -> np.ndarray:
    out = np.array(image, copy=True)
    if mask is None or not np.asarray(mask).any():
        return out
    red = np.zeros_like(out)
    red[..., 0] = 255
    chosen = np.asarray(mask, dtype=bool)
    out[chosen] = (0.45 * out[chosen] + 0.55 * red[chosen]).astype(np.uint8)
    return out


def pct(value: float | None) -> str:
    if value is None or not np.isfinite(value):
        return "undefined"
    return f"{100 * float(value):.1f}%"


def task_count(suite) -> int:
    value = getattr(suite, "n_tasks", None)
    if value is None:
        value = getattr(suite, "get_num_tasks", 10)
    if callable(value):
        value = value()
    return int(value)


def choose_placement(candidates: list[dict]) -> dict | None:
    usable = [row for row in candidates if np.isfinite(row["agent_cover"])]
    if not usable:
        return None
    bounded = [row for row in usable if row["scene_fraction"] < 0.45]
    pool = bounded or usable
    return max(pool, key=lambda row: float(row["agent_cover"]))


def scope_line(task_names: list[str], feature_frames: int, episodes: int, horizon: int) -> str:
    listed = "; ".join(task_names) if task_names else "none"
    return (
        f"Scope: lerobot/pi05_libero_finetuned, PaliGemma language-model layer 5, "
        f"proprioception taken from the unedited scene. "
        f"Feature rule scored on {feature_frames} task-0 frames. "
        f"Geometry scan: {listed}. "
        f"Closed loop: {episodes} episodes, {horizon} steps, re-query every 10 actions."
    )


def main() -> None:
    cc.configure_environment()
    if not torch.cuda.is_available():
        raise SystemExit("CUDA GPU required. Use the Colab L4 or A100 runtime.")
    os.environ["MAX_STEPS"] = os.environ.get("CTRL_SCAN_STEPS", "200")
    collect = cc.load_collect_module()
    # Simulator imports stay inside main so unit tests do not load LIBERO or MuJoCo.
    from libero.libero import benchmark, get_libero_path
    from libero.libero.envs import OffScreenRenderEnv

    try:
        import mujoco

        free_type = int(mujoco.mjtJoint.mjJNT_FREE)
    except Exception:
        free_type = 0

    scan_steps = cc.env_int("CTRL_SCAN_STEPS", 200)
    scope_episodes = cc.env_int("CTRL_SCOPE_EPISODES", 3)
    scope_stride = cc.env_int("CTRL_SCOPE_STRIDE", 16)
    scope_steps = cc.env_int("CTRL_SCOPE_STEPS", 96)
    extra_frames = cc.env_int("CTRL_EXTRA_FRAMES", 4)
    horizon = cc.env_int("CTRL_ROLLOUT_HORIZON", 120)
    replan = cc.env_int("CTRL_REPLAN", 10)
    rollout_episodes = cc.env_int("CTRL_ROLLOUT_EPISODES", 2)
    occluder_scale = float(os.environ.get("CTRL_OCCLUDER_SCALE", "3"))
    min_specificity = float(os.environ.get("CTRL_MIN_SPECIFICITY", "0.5"))
    repo = Path(os.environ.get("LOCAL_REPO", "/content/groot-run"))
    dest = repo / "outputs" / "permanence" / "paper_claim"
    dest.mkdir(parents=True, exist_ok=True)
    checkpoint_path = repo / "outputs" / "permanence" / "controlled_contrasts" / "transcoder.pt"
    if not checkpoint_path.is_file():
        raise SystemExit(f"Missing transcoder at {checkpoint_path}. Run the contrast cell first.")
    prior_path = repo / "outputs" / "permanence" / "paper_gaps" / "summary.json"
    paint = list(PAINT_FRAMES)
    held = list(HELD_FRAMES)
    if prior_path.is_file():
        prior = json.loads(prior_path.read_text())
        if prior.get("replicate_frames"):
            paint = [(int(row["episode"]), int(row["step"])) for row in prior["replicate_frames"]]
        if prior.get("held_frames"):
            held = [(int(row["episode"]), int(row["step"])) for row in prior["held_frames"]]

    suite = benchmark.get_benchmark_dict()[collect.SUITE]()
    n_tasks = task_count(suite)
    task_rows = []
    print(f"Geometry scan: {n_tasks} tasks, {scope_episodes} episodes, stride {scope_stride}", flush=True)
    for task_id in range(n_tasks):
        scan_env = None
        try:
            task = suite.get_task(task_id)
            sentence = task.language
            demo_path = collect.find_demo_file(task)
            bddl = os.path.join(get_libero_path("bddl_files"), task.problem_folder, task.bddl_file)
            demos = paper_gaps.read_demo_states(demo_path, scope_steps)
            scan_env = OffScreenRenderEnv(bddl_file_name=bddl, camera_heights=256, camera_widths=256)
            scan_env.seed(0)
            ratios: list[float] = []
            hidden = 0
            held_n = 0
            for states in demos[:scope_episodes]:
                for step in range(0, len(states), scope_stride):
                    collect.observe_state(scan_env, states[step])
                    sim = collect.sim_of(scan_env)
                    result = collect.label_frame(sim, sentence, collect.candidate_bodies(sim, sentence))
                    label = result[0] if isinstance(result, tuple) else result
                    geometry = result[2] if isinstance(result, tuple) and len(result) > 2 else None
                    if label not in ("held_visible", "held_hidden") or not geometry:
                        continue
                    held_n += 1
                    depth_gap = float(geometry["depth_gap"])
                    ratio = float(geometry["angle_ratio"])
                    ratios.append(ratio)
                    if depth_gap > 0.0 and ratio < 1.0:
                        hidden += 1
            ratio_min = float(np.min(ratios)) if ratios else None
            task_rows.append(
                {
                    "task_id": task_id,
                    "task": sentence,
                    "held_samples": held_n,
                    "hidden": hidden,
                    "angle_ratio_min": ratio_min,
                }
            )
            ratio_text = "none" if ratio_min is None else f"{ratio_min:.3f}"
            print(
                f"  task {task_id}: held {held_n} hidden {hidden} angle_ratio_min {ratio_text} | {sentence}",
                flush=True,
            )
        except Exception as exc:
            print(f"  task {task_id} scan failed: {exc}", flush=True)
            task_rows.append(
                {
                    "task_id": task_id,
                    "task": "",
                    "held_samples": 0,
                    "hidden": 0,
                    "angle_ratio_min": None,
                    "error": str(exc),
                }
            )
        finally:
            if scan_env is not None:
                scan_env.close()

    print("Loading policy", flush=True)
    policy, pre, _post, layer5 = collect.load_policy()
    policy.eval()
    layer6 = cc.find_layer6(policy, layer5)
    model, mean5, std5, n_features = oc.load_transcoder(checkpoint_path)

    task0 = suite.get_task(0)
    sentence0 = task0.language
    demo0 = paper_gaps.read_demo_states(collect.find_demo_file(task0), scan_steps)
    bddl0 = os.path.join(get_libero_path("bddl_files"), task0.problem_folder, task0.bddl_file)
    env = OffScreenRenderEnv(bddl_file_name=bddl0, camera_heights=256, camera_widths=256)
    env.seed(0)

    def rebuild(demos, sentence: str, episode: int, step: int):
        if episode >= len(demos) or step >= len(demos[episode]):
            print(f"Skip ep {episode} step {step}: outside the demos", flush=True)
            return None
        print(f"Rebuilding ep {episode} step {step}", flush=True)
        row = cc.inspect_candidate(
            collect, env, collect.sim_of(env), sentence, episode, step, demos[episode][step]
        )
        if row is None:
            print(f"Skip ep {episode} step {step}: no controllable object", flush=True)
            return None
        return row

    def forward(frame, agent, wrist, noise: int):
        batch = cc.batch_from_images(collect, pre, agent, wrist, frame["state"], frame["sentence"], policy)
        _tokens, _tokens6, action = cc.forward_policy(policy, layer5, layer6, batch, noise, None, "base")
        return action

    def camera_masks(frame) -> tuple[np.ndarray | None, np.ndarray | None, str, str]:
        images = frame["built"]["images"]
        agent_mask = uncapped_change_mask(images["base"][0], images["recolor"][0])
        wrist_mask = uncapped_change_mask(images["base"][1], images["recolor"][1])
        agent_source = "recolor" if agent_mask is not None else "none"
        wrist_source = "recolor" if wrist_mask is not None else "none"
        return agent_mask, wrist_mask, agent_source, wrist_source

    try:
        frames = []
        for episode, step in paint + held:
            row = rebuild(demo0, sentence0, episode, step)
            if row is None:
                continue
            row["group"] = "paint" if (episode, step) in set(paint) else "held"
            frames.append(row)

        print("\nCOVER  wrist mask without the 20% cap", flush=True)
        print(
            f"{'group':<8}{'ep':>4}{'step':>6}{'old_w':>8}{'new_w':>8}{'left':>8}{'peak':>6}"
            f"{'old_rmse':>10}{'new_rmse':>10}{'painted':>10}  reading",
            flush=True,
        )
        cover_rows = []
        figure_images: dict[tuple[int, int], dict] = {}
        for frame in frames:
            images = frame["built"]["images"]
            base_agent, base_wrist = images["base"]
            absent_agent, absent_wrist = images["absent"]
            old_agent = frame["built"].get("agent_full")
            old_wrist = frame["built"].get("wrist_full")
            new_agent, new_wrist, agent_source, wrist_source = camera_masks(frame)
            noise = 5000 + int(frame["episode"]) * 1000 + int(frame["step"])
            old_present = (paint_camera(base_agent, old_agent), paint_camera(base_wrist, old_wrist))
            old_absent = (paint_camera(absent_agent, old_agent), paint_camera(absent_wrist, old_wrist))
            new_present = (paint_camera(base_agent, new_agent), paint_camera(base_wrist, new_wrist))
            new_absent = (paint_camera(absent_agent, new_agent), paint_camera(absent_wrist, new_wrist))
            leftover = cc.changed_pixels(new_present[0], new_absent[0])
            leftover_wrist = cc.changed_pixels(new_present[1], new_absent[1])
            left_px, left_peak = diff_stats(new_present[0], new_absent[0])
            wrist_left, wrist_peak = diff_stats(new_present[1], new_absent[1])
            if wrist_left > left_px:
                left_px, left_peak = wrist_left, wrist_peak
            painted_present = (
                paint_camera(new_present[0], leftover),
                paint_camera(new_present[1], leftover_wrist),
            )
            painted_absent = (
                paint_camera(new_absent[0], leftover),
                paint_camera(new_absent[1], leftover_wrist),
            )
            old_action = forward(frame, old_present[0], old_present[1], noise)
            old_absent_action = forward(frame, old_absent[0], old_absent[1], noise)
            new_action = forward(frame, new_present[0], new_present[1], noise)
            new_absent_action = forward(frame, new_absent[0], new_absent[1], noise)
            same_action = forward(frame, new_present[0], new_present[1], noise)
            painted_action = forward(frame, painted_present[0], painted_present[1], noise)
            painted_absent_action = forward(frame, painted_absent[0], painted_absent[1], noise)
            old_rmse = action_rmse(old_action, old_absent_action)
            new_rmse = action_rmse(new_action, new_absent_action)
            painted_rmse = action_rmse(painted_action, painted_absent_action)
            same_rmse = action_rmse(new_action, same_action)
            reading = residue_reading(left_px, left_peak, new_rmse)
            old_wrist_px = 0 if old_wrist is None else int(np.asarray(old_wrist).sum())
            new_wrist_px = 0 if new_wrist is None else int(np.asarray(new_wrist).sum())
            print(
                f"{frame['group']:<8}{int(frame['episode']):>4}{int(frame['step']):>6}"
                f"{old_wrist_px:>8}{new_wrist_px:>8}{left_px:>8}{left_peak:>6}"
                f"{old_rmse:>10.4f}{new_rmse:>10.4f}{painted_rmse:>10.4f}  {reading}",
                flush=True,
            )
            print(
                f"    mask agent {agent_source} wrist {wrist_source} | identical re-forward {same_rmse:.4f} | "
                f"painted images differ by {diff_stats(painted_present[0], painted_absent[0])[0]} agent px and "
                f"{diff_stats(painted_present[1], painted_absent[1])[0]} wrist px",
                flush=True,
            )
            key = (int(frame["episode"]), int(frame["step"]))
            cover_rows.append(
                {
                    "group": frame["group"],
                    "episode": key[0],
                    "step": key[1],
                    "old_wrist_pixels": old_wrist_px,
                    "new_wrist_pixels": new_wrist_px,
                    "leftover_pixels": left_px,
                    "leftover_peak": left_peak,
                    "old_rmse": old_rmse,
                    "new_rmse": new_rmse,
                    "painted_rmse": painted_rmse,
                    "identical_rmse": same_rmse,
                    "agent_mask_source": agent_source,
                    "wrist_mask_source": wrist_source,
                    "reading": reading,
                }
            )
            if key in set(FIGURE_FRAMES):
                figure_images[key] = {
                    "base": base_wrist,
                    "absent": absent_wrist,
                    "old": old_absent[1],
                    "new": new_absent[1],
                    "leftover": leftover_wrist,
                }

        sheet = dest / "cover_sheet.png"
        sheet_rows = save_figure(figure_images, sheet)
        if sheet_rows:
            print("sheet", sheet, flush=True)

        print("\nOCCLUDER  another object on the camera-to-bowl ray", flush=True)
        occluder_rows = []
        by_key = {(int(frame["episode"]), int(frame["step"])): frame for frame in frames}
        for key in OCCLUDER_FRAMES:
            frame = by_key.get(key)
            if frame is None:
                continue
            collect.observe_state(env, frame["sim_state"])
            sim = collect.sim_of(env)
            bodies = free_body_map(sim, free_type)
            names = body_names(sim, bodies)
            occluder_body = prefer_occluder(names, int(frame["body"]))
            if occluder_body is None:
                print(f"  ep {key[0]} step {key[1]}: no second free body", flush=True)
                continue
            bowl_mask, wrist_bowl, _agent_source, _wrist_source = camera_masks(frame)
            base_agent, base_wrist = frame["built"]["images"]["base"]
            placed_rows = []
            for fraction in (0.50, 0.65, 0.80):
                placed = place_occluder(
                    collect, env, sim, occluder_body, bodies[occluder_body], frame["geoms"], fraction, occluder_scale
                )
                if placed is None:
                    continue
                placed_rows.append(
                    {
                        "fraction": fraction,
                        "agent": placed["agent"],
                        "wrist": placed["wrist"],
                        "agent_cover": mask_cover_fraction(base_agent, placed["agent"], bowl_mask),
                        "wrist_cover": mask_cover_fraction(base_wrist, placed["wrist"], wrist_bowl),
                        "scene_fraction": cc.changed_fraction(base_agent, placed["agent"]),
                        "depth_gap": placed["depth_gap"],
                        "angle_ratio": placed["angle_ratio"],
                    }
                )
            best = choose_placement(placed_rows)
            if best is None:
                print(f"  ep {key[0]} step {key[1]}: occluder could not be placed", flush=True)
                continue
            absent_agent, absent_wrist = frame["built"]["images"]["absent"]
            noise = 5000 + int(frame["episode"]) * 1000 + int(frame["step"])
            base_action = forward(frame, base_agent, base_wrist, noise)
            hidden_action = forward(frame, best["agent"], best["wrist"], noise)
            absent_action = forward(frame, absent_agent, absent_wrist, noise)
            to_hidden = action_rmse(base_action, hidden_action)
            to_absent = action_rmse(base_action, absent_action)
            closed = None if to_absent <= 1e-4 else (to_absent - action_rmse(hidden_action, absent_action)) / to_absent
            occluder_name = dict(names).get(occluder_body, str(occluder_body))
            print(
                f"  ep {key[0]} step {key[1]} {occluder_name}: along-ray {best['fraction']:.2f} "
                f"depth_gap {best['depth_gap']:.4f} angle_ratio {best['angle_ratio']:.3f} | "
                f"bowl pixels covered agent {best['agent_cover']:.3f} wrist {best['wrist_cover']:.3f} "
                f"scene {best['scene_fraction']:.3f} | "
                f"RMSE base→occluder {to_hidden:.4f} base→absent {to_absent:.4f} "
                f"absence gap closed by occluder {pct(closed)}",
                flush=True,
            )
            occluder_rows.append(
                {
                    "episode": key[0],
                    "step": key[1],
                    "occluder": occluder_name,
                    "fraction": best["fraction"],
                    "depth_gap": best["depth_gap"],
                    "angle_ratio": best["angle_ratio"],
                    "agent_cover": best["agent_cover"],
                    "wrist_cover": best["wrist_cover"],
                    "scene_fraction": best["scene_fraction"],
                    "rmse_base_to_occluder": to_hidden,
                    "rmse_base_to_absent": to_absent,
                    "absence_gap_closed": closed,
                }
            )

        print("\nFEATURES  same rule on more task-0 frames", flush=True)
        extra = find_extra_frames(collect, env, sentence0, demo0, frames, extra_frames)
        feature_frames = [frame for frame in frames if frame["group"] == "paint"] + extra
        probe = []
        packed_rows = []
        for frame in feature_frames:
            noise = 5000 + int(frame["episode"]) * 1000 + int(frame["step"])
            packed = {}
            for condition in oc.CAPTURE:
                tokens, _tokens6, action, device = cc.capture(
                    policy, layer5, layer6, collect, pre, frame, condition, noise
                )
                packed[condition] = {"l5": tokens, "action": action}
            probe.append({name: packed[name]["l5"] for name in oc.CAPTURE})
            packed_rows.append((frame, packed, device))
            print(
                f"  captured ep {frame['episode']} step {frame['step']} {frame['label']}",
                flush=True,
            )
        scores = cc.score_probe(model, mean5, std5, probe) if probe else {}
        passed = oc.specific_ids(scores, "occlusion", oc.QUIET_FOR["occlusion"], min_specificity) if scores else np.array([], dtype=np.int64)
        passed_list = [int(item) for item in passed.tolist()]
        print(f"  passed {passed_list or 'none'}", flush=True)
        print(f"  overlap with paper-gaps {list(FROZEN_PAPER)}: {overlap(passed_list, FROZEN_PAPER) or 'none'}", flush=True)
        print(f"  overlap with strict run {list(FROZEN_STRICT)}: {overlap(passed_list, FROZEN_STRICT) or 'none'}", flush=True)
        oc.print_feature_table(
            "PASSED",
            oc.feature_rows(passed, scores, "occlusion", oc.QUIET_FOR["occlusion"]) if scores else [],
        )
        feature_records = score_patches(
            policy, layer5, collect, pre, model, mean5, std5, n_features, packed_rows, passed
        )
        print_patch_means(feature_records)

        print("\nSECOND TASK  one covered frame outside task 0", flush=True)
        second = second_task_cover(
            collect, suite, pre, policy, layer5, layer6, scan_steps, get_libero_path, OffScreenRenderEnv
        )
        if second is None:
            print("  no second-task frame", flush=True)
        else:
            print(
                f"  task {second['task_id']}: {second['task']}\n"
                f"  ep {second['episode']} step {second['step']} new wrist px {second['new_wrist_pixels']} "
                f"leftover {second['leftover_pixels']} peak {second['leftover_peak']} "
                f"new RMSE {second['new_rmse']:.4f} painted RMSE {second['painted_rmse']:.4f} | {second['reading']}",
                flush=True,
            )

        print("\nCLOSED LOOP  re-query every 10 actions", flush=True)
        print(
            "The sim keeps the real bowl. covered paints the uncapped mask gray. "
            "absent shows the teleported bowl and then restores it before the step. "
            "Proprioception is the live unedited arm.",
            flush=True,
        )
        rollouts = run_closed_loop(
            collect,
            env,
            policy,
            pre,
            layer5,
            layer6,
            sentence0,
            demo0,
            rollout_episodes,
            horizon,
            replan,
        )

        summary = {
            "scope": scope_line([row["task"] for row in task_rows], len(feature_frames), rollout_episodes, horizon),
            "geometry": task_rows,
            "covers": cover_rows,
            "occluders": occluder_rows,
            "features": {
                "passed": passed_list,
                "overlap_paper_gaps": overlap(passed_list, FROZEN_PAPER),
                "overlap_strict": overlap(passed_list, FROZEN_STRICT),
                "frames": [
                    {"episode": int(frame["episode"]), "step": int(frame["step"]), "label": frame["label"]}
                    for frame in feature_frames
                ],
                "patches": patch_means(feature_records),
            },
            "second_task": second,
            "rollouts": rollouts,
            "figure": str(sheet) if sheet_rows else None,
        }
        (dest / "summary.json").write_text(json.dumps(cc.json_ready(summary), indent=2))
        print("\n" + summary["scope"], flush=True)
        print("Wrote", dest / "summary.json", flush=True)
        print("RESULT_DIR", dest, flush=True)
    finally:
        env.close()


def free_body_map(sim, free_type: int) -> dict[int, list[int]]:
    model = sim.model
    movable: set[int] = set()
    for body in range(int(model.nbody)):
        start = int(model.body_jntadr[body])
        count = int(model.body_jntnum[body])
        for joint in range(start, start + count):
            if joint >= 0 and int(model.jnt_type[joint]) == int(free_type):
                movable.add(body)
                break
    bodies: dict[int, list[int]] = {}
    for geom in range(int(model.ngeom)):
        body = int(model.geom_bodyid[geom])
        if body in movable:
            bodies.setdefault(body, []).append(geom)
    return bodies


def body_names(sim, bodies: dict[int, list[int]]) -> list[tuple[int, str]]:
    model = sim.model
    named = []
    for body, geoms in bodies.items():
        blob = " ".join((model.geom_id2name(geom) or "") for geom in geoms).lower()
        blob += " " + (model.body_id2name(body) or "").lower()
        named.append((body, blob))
    return named


def place_occluder(collect, env, sim, body: int, geoms: list[int], bowl_geoms: list[int], fraction: float, scale: float):
    span = collect.object_qpos_span(sim, body)
    if span is None or not geoms or not bowl_geoms:
        return None
    camera = np.asarray(sim.data.cam_xpos[collect.camera_id(sim)], dtype=np.float64)
    bowl = np.mean([np.asarray(sim.data.geom_xpos[geom], dtype=np.float64) for geom in bowl_geoms], axis=0)
    point = point_on_ray(camera, bowl, fraction)
    qpos = np.array(sim.data.qpos, copy=True)
    qvel = np.array(sim.data.qvel, copy=True)
    sizes = np.array(sim.model.geom_size, copy=True)
    start, stop = span
    try:
        sim.data.qpos[start : start + 3] = point
        sim.data.qpos[start + 3 : stop] = np.array([1.0, 0.0, 0.0, 0.0])
        sim.data.qvel[:] = 0
        for geom in geoms:
            sim.model.geom_size[geom] = sizes[geom] * float(scale)
        sim.forward()
        agent, wrist = cc.images_of(collect, collect.render_after_edit(env))
        front = np.mean([np.asarray(sim.data.geom_xpos[geom], dtype=np.float64) for geom in geoms], axis=0)
        depth_gap, ratio = covering_ratio(camera, front, bowl, 0.04 * float(scale))
        return {"agent": agent, "wrist": wrist, "depth_gap": depth_gap, "angle_ratio": ratio}
    finally:
        sim.model.geom_size[:] = sizes
        sim.data.qpos[:] = qpos
        sim.data.qvel[:] = qvel
        sim.forward()
        collect.render_after_edit(env)


def find_extra_frames(collect, env, sentence: str, demos, existing: list, limit: int) -> list:
    have = {(int(frame["episode"]), int(frame["step"])) for frame in existing}
    found = []
    for episode in (4, 6, 8, 10, 12):
        if episode >= len(demos):
            continue
        for step in (0, 24, 48):
            if len(found) >= limit or (episode, step) in have or step >= len(demos[episode]):
                continue
            row = cc.inspect_candidate(collect, env, collect.sim_of(env), sentence, episode, step, demos[episode][step])
            if row is None or row["label"] == "held_hidden":
                continue
            if str(row["label"]).startswith("held_"):
                continue
            wrist = row["built"].get("wrist_full")
            agent = row["built"].get("agent_full")
            if wrist is None or agent is None:
                continue
            row["group"] = "extra"
            found.append(row)
            print(f"  extra ep {episode} step {step} {row['label']}", flush=True)
    return found


def score_patches(policy, layer5, collect, pre, model, mean5, std5, n_features, packed_rows, passed):
    records = []
    rng = np.random.default_rng(0)
    frozen_sets = {
        "frozen_paper": np.asarray(FROZEN_PAPER, dtype=np.int64),
        "frozen_strict": np.asarray(FROZEN_STRICT, dtype=np.int64),
    }
    for frame, packed, device in packed_rows:
        noise = 5000 + int(frame["episode"]) * 1000 + int(frame["step"])
        source_tokens = packed["base"]["l5"]
        source_action = packed["base"]["action"]
        donor_tokens = packed["occluded"]["l5"]
        donor_action = packed["occluded"]["action"]
        agent, wrist = frame["built"]["images"]["base"]
        batch = cc.batch_from_images(collect, pre, agent, wrist, frame["state"], frame["sentence"], policy)
        code_delta = cc.encode_tokens(model, donor_tokens, mean5, std5) - cc.encode_tokens(
            model, source_tokens, mean5, std5
        )

        def run_patch(name: str, mode: str, payload_np: np.ndarray) -> None:
            _tokens, _tokens6, action = cc.forward_policy(
                policy, layer5, None, batch, noise, cc.to_gpu(payload_np, device), mode
            )
            metrics = cc.action_metrics(action, source_action, donor_action)
            records.append({"patch": name, "episode": int(frame["episode"]), "step": int(frame["step"]), **metrics})

        run_patch("full_tokens", "replace", donor_tokens.astype(np.float32))
        if passed.size:
            feature_payload = oc.decoded_patch(model, code_delta, passed, std5, None)
            pool = np.setdiff1d(np.arange(n_features), passed)
            random_ids = rng.choice(pool, size=int(passed.size), replace=False).astype(np.int64)
            moved = cc.remap_features(cc.keep_features(code_delta, passed), passed, random_ids)
            random_payload = cc.match_l2(cc.decode_delta(model, moved, std5), feature_payload)
            run_patch("occlusion_features", "add", feature_payload)
            run_patch("random_remap", "add", random_payload)
        for name, feature_ids in frozen_sets.items():
            usable = feature_ids[feature_ids < n_features]
            if usable.size == 0:
                continue
            run_patch(name, "add", oc.decoded_patch(model, code_delta, usable, std5, None))
    return records


def patch_means(records: list[dict]) -> list[dict]:
    names = []
    for row in records:
        if row["patch"] not in names:
            names.append(row["patch"])
    means = []
    for name in names:
        chosen = [row["fraction_gap_closed"] for row in records if row["patch"] == name]
        means.append({"patch": name, "mean_gap_closed": cc.mean_defined(chosen), "n": len(chosen)})
    return means


def print_patch_means(records: list[dict]) -> None:
    print(f"{'patch':<22}{'gap_closed':>12}{'n':>6}", flush=True)
    for row in patch_means(records):
        print(f"{row['patch']:<22}{pct(row['mean_gap_closed']):>12}{row['n']:>6}", flush=True)


def tile(image: np.ndarray, text: str) -> np.ndarray:
    picture = np.asarray(image)
    if picture.shape[0] != 256 or picture.shape[1] != 256:
        picture = cv2.resize(picture, (256, 256), interpolation=cv2.INTER_NEAREST)
    canvas = cv2.copyMakeBorder(picture, 22, 0, 0, 0, cv2.BORDER_CONSTANT, value=(255, 255, 255))
    cv2.putText(canvas, text, (4, 16), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 0), 1, cv2.LINE_AA)
    return canvas


def save_figure(figure_images: dict, path: Path) -> bool:
    strips = []
    for key in FIGURE_FRAMES:
        pictures = figure_images.get(key)
        if pictures is None:
            continue
        box = paper_gaps.zoom_box(pictures["leftover"], 36, pictures["base"].shape[0], pictures["base"].shape[1])
        zoom = paper_gaps.enlarge(paper_gaps.crop(highlight(pictures["base"], pictures["leftover"]), box), 4)
        tiles = [
            tile(pictures["base"], f"e{key[0]}s{key[1]} wrist"),
            tile(pictures["absent"], "absent"),
            tile(pictures["old"], "old cover"),
            tile(pictures["new"], "uncapped cover"),
            tile(zoom, "residue zoom"),
        ]
        strips.append(np.concatenate(tiles, axis=1))
    if not strips:
        return False
    width = max(strip.shape[1] for strip in strips)
    padded = []
    for strip in strips:
        if strip.shape[1] < width:
            pad = np.full((strip.shape[0], width - strip.shape[1], 3), 255, dtype=np.uint8)
            strip = np.concatenate([strip, pad], axis=1)
        padded.append(strip)
    sheet = np.concatenate(padded, axis=0)
    cv2.imwrite(str(path), cv2.cvtColor(sheet, cv2.COLOR_RGB2BGR))
    return True


def second_task_cover(collect, suite, pre, policy, layer5, layer6, scan_steps, get_libero_path, env_cls):
    if task_count(suite) < 2:
        return None
    task = suite.get_task(1)
    sentence = task.language
    demos = paper_gaps.read_demo_states(collect.find_demo_file(task), scan_steps)
    if not demos or len(demos[0]) == 0:
        return None
    bddl = os.path.join(get_libero_path("bddl_files"), task.problem_folder, task.bddl_file)
    env = env_cls(bddl_file_name=bddl, camera_heights=256, camera_widths=256)
    env.seed(0)
    try:
        row = cc.inspect_candidate(collect, env, collect.sim_of(env), sentence, 0, 0, demos[0][0])
        if row is None:
            return None
        images = row["built"]["images"]
        agent_mask = uncapped_change_mask(images["base"][0], images["recolor"][0])
        wrist_mask = uncapped_change_mask(images["base"][1], images["recolor"][1])
        present = (paint_camera(images["base"][0], agent_mask), paint_camera(images["base"][1], wrist_mask))
        absent = (paint_camera(images["absent"][0], agent_mask), paint_camera(images["absent"][1], wrist_mask))
        leftover = cc.changed_pixels(present[0], absent[0])
        leftover_wrist = cc.changed_pixels(present[1], absent[1])
        left_px, left_peak = diff_stats(present[0], absent[0])
        wrist_left, wrist_peak = diff_stats(present[1], absent[1])
        if wrist_left > left_px:
            left_px, left_peak = wrist_left, wrist_peak
        painted_present = (paint_camera(present[0], leftover), paint_camera(present[1], leftover_wrist))
        painted_absent = (paint_camera(absent[0], leftover), paint_camera(absent[1], leftover_wrist))

        def once(agent, wrist):
            batch = cc.batch_from_images(collect, pre, agent, wrist, row["state"], row["sentence"], policy)
            _tokens, _tokens6, action = cc.forward_policy(policy, layer5, layer6, batch, 6100, None, "base")
            return action

        new_rmse = action_rmse(once(present[0], present[1]), once(absent[0], absent[1]))
        painted_rmse = action_rmse(once(painted_present[0], painted_present[1]), once(painted_absent[0], painted_absent[1]))
        return {
            "task_id": 1,
            "task": sentence,
            "episode": 0,
            "step": 0,
            "new_wrist_pixels": 0 if wrist_mask is None else int(np.asarray(wrist_mask).sum()),
            "leftover_pixels": left_px,
            "leftover_peak": left_peak,
            "new_rmse": new_rmse,
            "painted_rmse": painted_rmse,
            "reading": residue_reading(left_px, left_peak, new_rmse),
        }
    finally:
        env.close()


def live_images(collect, env, sim, body: int, geoms: list[int], kind: EditName):
    raw = collect.render_after_edit(env)
    agent, wrist = cc.images_of(collect, raw)
    state = collect.libero_state(raw)
    match kind:
        case "base":
            return agent, wrist, state
        case "covered":
            recolor_agent, recolor_wrist = cc.render_rgba_images(collect, env, sim, geoms)
            raw = collect.render_after_edit(env)
            agent, wrist = cc.images_of(collect, raw)
            state = collect.libero_state(raw)
            agent_mask = uncapped_change_mask(agent, recolor_agent, min_pixels=5)
            wrist_mask = uncapped_change_mask(wrist, recolor_wrist, min_pixels=5)
            return paint_camera(agent, agent_mask), paint_camera(wrist, wrist_mask), state
        case "absent":
            qpos = np.array(sim.data.qpos, copy=True)
            qvel = np.array(sim.data.qvel, copy=True)
            if collect.move_object_away(sim, body) is None:
                return agent, wrist, state
            gone_agent, gone_wrist = cc.images_of(collect, collect.render_after_edit(env))
            sim.data.qpos[:] = qpos
            sim.data.qvel[:] = qvel
            sim.forward()
            raw = collect.render_after_edit(env)
            return gone_agent, gone_wrist, collect.libero_state(raw)
        case _ as unexpected:
            assert_never(unexpected)


def run_closed_loop(collect, env, policy, pre, layer5, layer6, sentence, demos, episodes: int, horizon: int, replan: int):
    edits: tuple[EditName, ...] = ("base", "covered", "absent")
    rows = []
    for episode in range(episodes):
        if episode >= len(demos) or len(demos[episode]) == 0:
            continue
        for edit in edits:
            collect.observe_state(env, demos[episode][0])
            sim = collect.sim_of(env)
            mapping = collect.candidate_bodies(sim, sentence)
            body = None
            geoms = None
            for candidate, candidate_geoms in mapping.items():
                if collect.object_qpos_span(sim, candidate) is not None:
                    body = int(candidate)
                    geoms = list(candidate_geoms)
                    break
            if body is None or not geoms:
                print(f"  ep {episode} {edit}: no free target body", flush=True)
                continue
            span = collect.object_qpos_span(sim, body)
            start = paper_gaps.object_xyz(sim, body, span)
            success_before = paper_gaps.task_succeeded(env)
            taken = 0
            succeeded = success_before
            error = ""
            try:
                while taken < horizon and not succeeded:
                    agent, wrist, state = live_images(collect, env, sim, body, geoms, edit)
                    batch = cc.batch_from_images(collect, pre, agent, wrist, state, sentence, policy)
                    noise = 7000 + episode * 1000 + taken
                    _tokens, _tokens6, chunk = cc.forward_policy(policy, layer5, layer6, batch, noise, None, "base")
                    acted = 0
                    for action in np.asarray(chunk)[:replan]:
                        if taken >= horizon:
                            break
                        paper_gaps.step_env(env, action)
                        taken += 1
                        acted += 1
                        sim = collect.sim_of(env)
                        if paper_gaps.task_succeeded(env):
                            break
                    if acted == 0:
                        break
                    succeeded = paper_gaps.task_succeeded(env)
                    print(f"    ep {episode} {edit} step {taken} success {succeeded}", flush=True)
            except Exception as exc:
                error = str(exc)
                print(f"  ep {episode} {edit} stopped: {error}", flush=True)
            end = paper_gaps.object_xyz(collect.sim_of(env), body, span)
            shift = None if start is None or end is None else float(np.linalg.norm(end - start))
            print(
                f"  ep {episode} {edit}: success {success_before} -> {succeeded} "
                f"steps {taken} bowl_shift_m {shift} {error}",
                flush=True,
            )
            rows.append(
                {
                    "episode": episode,
                    "edit": edit,
                    "success_before": success_before,
                    "success_after": succeeded,
                    "steps": taken,
                    "bowl_shift_m": shift,
                    "error": error,
                }
            )
    return rows


if __name__ == "__main__":
    main()
'''

scripts = LOCAL_REPO / "scripts"
scripts.mkdir(parents=True, exist_ok=True)
path = scripts / "paper_claim.py"
path.write_text(SCRIPT)
for required in ("controlled_contrasts.py", "occlusion_features.py", "collect_layer5_replay.py", "paper_gaps.py"):
    if not (scripts / required).is_file():
        raise SystemExit(
            f"scripts/{required} is missing. Run the collector cell, the contrast cell, the occlusion cell, and the paper-gaps cell first."
        )

env = os.environ.copy()
env["PYTHONPATH"] = str(scripts) + os.pathsep + env.get("PYTHONPATH", "")
env["LOCAL_REPO"] = str(LOCAL_REPO)
env["PYTHONUNBUFFERED"] = "1"
env["MAX_STEPS"] = env.get("CTRL_SCAN_STEPS", "200")

def stream(cmd):
    proc = subprocess.Popen(
        cmd,
        cwd=str(scripts),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    tail = []
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
        tail.append(line.rstrip())
        del tail[:-80]
    code = proc.wait()
    if code != 0:
        raise SystemExit(
            "paper_claim.py exited with code %s. Last lines:\n%s" % (code, "\n".join(tail))
        )

stream([str(PYTHON), "-u", str(path)])
print("summary", LOCAL_REPO / "outputs" / "permanence" / "paper_claim" / "summary.json")


In [ ]:
# Remaining gaps — both cameras, and a closed loop through the postprocessor
# Paste into the same runtime after the paper-claim cell. Does not retrain.
# Does not overwrite outputs/permanence/paper_claim/ or earlier directories.

import os, sys, subprocess
from pathlib import Path

os.environ.setdefault("CTRL_SCAN_STEPS", "200")
# The paper-claim cell may already have exported 120. This run uses 220.
os.environ["CTRL_ROLLOUT_HORIZON"] = "220"
os.environ.setdefault("CTRL_REPLAN", "10")
os.environ.setdefault("CTRL_ROLLOUT_EPISODES", "2")
os.environ.setdefault("CTRL_HARNESS_STEPS", "40")
os.environ.setdefault("SUITE", "libero_spatial")
os.environ.setdefault("TASK_IDS", "[0]")
os.environ.setdefault(
    "LAYER_NAME",
    "model.paligemma_with_expert.paligemma.model.language_model.layers.5",
)
os.environ.setdefault("MUJOCO_GL", "egl")
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

LOCAL_REPO = Path(os.environ.get("LOCAL_REPO", "/content/groot-run"))
PYTHON = Path("/content/lerobot-venv/bin/python")
if not PYTHON.exists():
    PYTHON = Path(sys.executable)

SCRIPT = r'''
"""Hide the bowl from both cameras, and step unnormalized actions.

The ramekin on the agent-camera ray left the wrist image unchanged. This run
searches a second free body along the wrist-camera ray, places both bodies in
one scene when that body exists, and always forwards a composite of the two
renders. The closed loop passes each chunk through the policy postprocessor
before env.step.

Does not retrain and does not rewrite earlier output directories.
"""

from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Literal, assert_never

import cv2
import numpy as np
import torch

import controlled_contrasts as cc
import paper_claim as pc
import paper_gaps

FRAMES = ((1, 0), (1, 44))
AGENT_FRACTION = 0.50
AGENT_SCALE = 3.0
WRIST_FRACTIONS = (0.30, 0.50, 0.70)
WRIST_SCALES = (1.5, 3.0, 6.0)
COVER_READY = 0.8
SCENE_LIMIT = 0.5
PlacementKind = Literal["agent_only", "wrist_only", "physical_pair"]


def choose_both(candidates: list[dict]) -> dict | None:
    """Prefer a tight placement that covers the bowl in both cameras."""
    usable = [
        row
        for row in candidates
        if np.isfinite(row.get("agent_cover", np.nan)) and np.isfinite(row.get("wrist_cover", np.nan))
    ]
    if not usable:
        return None
    qualified = [
        row
        for row in usable
        if float(row["agent_cover"]) >= COVER_READY
        and float(row["wrist_cover"]) >= COVER_READY
        and _scene(row) < SCENE_LIMIT
    ]
    pool = qualified or usable
    return max(pool, key=lambda row: min(float(row["agent_cover"]), float(row["wrist_cover"])))


def choose_wrist(candidates: list[dict]) -> dict | None:
    usable = [row for row in candidates if np.isfinite(row.get("wrist_cover", np.nan))]
    if not usable:
        return None
    qualified = [row for row in usable if float(row["wrist_cover"]) >= COVER_READY and _scene(row) < SCENE_LIMIT]
    pool = qualified or usable
    return max(pool, key=lambda row: float(row["wrist_cover"]))


def _scene(row: dict) -> float:
    value = float(row.get("scene_fraction", 1.0))
    if not np.isfinite(value):
        return 1.0
    return value


def second_body(names: list[tuple[int, str]], blocked: set[int]) -> int | None:
    pool = [(body, name) for body, name in names if body not in blocked]
    for needle in ("plate", "ramekin"):
        for body, name in pool:
            if needle in name:
                return body
    if not pool:
        return None
    return pool[0][0]


def short_name(blob: str) -> str:
    text = blob.lower()
    for needle in ("ramekin", "plate", "bowl"):
        if needle in text:
            return needle
    compact = " ".join(text.split())
    return compact[:48] or "body"


def view_scores(
    base_agent: np.ndarray,
    base_wrist: np.ndarray,
    agent: np.ndarray,
    wrist: np.ndarray,
    agent_mask: np.ndarray | None,
    wrist_mask: np.ndarray | None,
) -> dict:
    agent_scene = cc.changed_fraction(base_agent, agent)
    wrist_scene = cc.changed_fraction(base_wrist, wrist)
    return {
        "agent_cover": pc.mask_cover_fraction(base_agent, agent, agent_mask),
        "wrist_cover": pc.mask_cover_fraction(base_wrist, wrist, wrist_mask),
        "scene_fraction": float(max(agent_scene, wrist_scene)),
        "agent_scene": agent_scene,
        "wrist_scene": wrist_scene,
        "agent": np.asarray(agent),
        "wrist": np.asarray(wrist),
    }


def absence_gap_closed(edit_to_absent: float, base_to_absent: float) -> float | None:
    if not np.isfinite(base_to_absent) or float(base_to_absent) <= 1e-4:
        return None
    if not np.isfinite(edit_to_absent):
        return None
    return float((float(base_to_absent) - float(edit_to_absent)) / float(base_to_absent))


def both_reading(agent_cover: float, wrist_cover: float, scene: float, composite: bool) -> str:
    covered = (
        np.isfinite(agent_cover)
        and np.isfinite(wrist_cover)
        and float(agent_cover) >= COVER_READY
        and float(wrist_cover) >= COVER_READY
        and np.isfinite(scene)
        and float(scene) < SCENE_LIMIT
    )
    if composite and covered:
        return "The composite hides the bowl from both cameras. It is two renders combined."
    if covered:
        return "One scene hides the bowl from both cameras."
    if np.isfinite(agent_cover) and float(agent_cover) >= COVER_READY and (
        not np.isfinite(wrist_cover) or float(wrist_cover) < 0.2
    ):
        return "The agent camera is covered. The wrist camera still shows the bowl."
    if np.isfinite(wrist_cover) and float(wrist_cover) >= COVER_READY and (
        not np.isfinite(agent_cover) or float(agent_cover) < 0.2
    ):
        return "The wrist camera is covered. The agent camera still shows the bowl."
    return "The bowl stays partly visible in a camera."


def mean_abs(chunk: np.ndarray) -> float:
    array = np.asarray(chunk, dtype=np.float64)
    if array.size == 0:
        return float("nan")
    return float(np.mean(np.abs(array)))


def harness_reading(raw_abs: float, post_abs: float, base_arm: float, zero_arm: float) -> str:
    scale = max(1e-3, 0.05 * abs(float(raw_abs)))
    unchanged = abs(float(post_abs) - float(raw_abs)) <= scale
    separates = float(base_arm) > max(float(zero_arm) * 1.25, float(zero_arm) + 0.02)
    if not unchanged and separates:
        return "Unnormalized actions move the arm farther than a zero action."
    if unchanged and not separates:
        return "The postprocessor left the action scale unchanged, and the arm matches a zero action."
    if unchanged:
        return "The postprocessor left the action scale unchanged."
    return "The postprocessor changed the action scale."


def parse_action(value, steps: int, width: int) -> np.ndarray | None:
    if isinstance(value, (tuple, list)):
        if not value:
            return None
        value = value[0]
    if isinstance(value, dict):
        if "action" not in value:
            return None
        value = value["action"]
    if torch.is_tensor(value):
        value = value.detach().float().cpu().numpy()
    array = np.asarray(value, dtype=np.float32)
    while array.ndim > 2 and array.shape[0] == 1:
        array = array[0]
    if array.ndim == 1 and steps == 1 and array.shape[0] == width and np.isfinite(array).all():
        return array.reshape(1, width)
    if (
        array.ndim == 2
        and array.shape[1] == width
        and 1 <= array.shape[0] <= steps
        and np.isfinite(array).all()
    ):
        return array
    return None


def apply_post(post, chunk: np.ndarray, device: str | None = None) -> np.ndarray:
    """Unnormalize a chunk. Tries a batched dict, a batched tensor, then one step at a time."""
    raw = np.asarray(chunk, dtype=np.float32)
    if raw.ndim != 2 or raw.shape[0] < 1 or raw.shape[1] < 1:
        raise RuntimeError(f"Expected an action chunk [steps, dim], got {getattr(raw, 'shape', None)}")
    steps, width = int(raw.shape[0]), int(raw.shape[1])
    errors: list[str] = []
    for place in _devices(device):
        base = torch.from_numpy(np.ascontiguousarray(raw)).to(place)
        parsed = _call_post(post, {"action": base.unsqueeze(0).clone()}, steps, width, errors, f"{place} dict")
        if parsed is not None:
            return parsed
        parsed = _call_post(post, base.unsqueeze(0).clone(), steps, width, errors, f"{place} tensor")
        if parsed is not None:
            return parsed
        parsed = _call_steps(post, base, steps, width, errors, place)
        if parsed is not None:
            return parsed
    raise RuntimeError("The postprocessor did not return an action chunk. " + " | ".join(errors))


def _devices(device: str | None) -> list[str]:
    found = ["cpu"]
    if device and not str(device).startswith("cpu") and str(device) not in found:
        found.append(str(device))
    return found


def _call_post(post, payload, steps: int, width: int, errors: list[str], label: str) -> np.ndarray | None:
    try:
        out = post(payload)
    except Exception as exc:
        errors.append(f"{label}: {type(exc).__name__}: {exc}")
        return None
    parsed = parse_action(out, steps, width)
    if parsed is None:
        errors.append(f"{label}: returned a value that is not a finite action chunk")
        return None
    return np.asarray(parsed, dtype=np.float32)


def _call_steps(post, base: torch.Tensor, steps: int, width: int, errors: list[str], place: str) -> np.ndarray | None:
    rows = []
    for index in range(steps):
        try:
            out = post({"action": base[index].unsqueeze(0).clone()})
        except Exception as exc:
            errors.append(f"{place} step {index}: {type(exc).__name__}: {exc}")
            return None
        parsed = parse_action(out, 1, width)
        if parsed is None:
            errors.append(f"{place} step {index}: returned a value that is not a finite action")
            return None
        rows.append(parsed[0])
    return np.stack(rows, axis=0).astype(np.float32)


def plain_row(row: dict | None) -> dict | None:
    if row is None:
        return None
    return {key: value for key, value in row.items() if key not in ("agent", "wrist")}


def save_both_sheet(frames: list[dict], path: Path) -> bool:
    strips = []
    for frame in frames:
        pictures = frame.get("pictures")
        if not pictures:
            continue
        label = f"e{frame['episode']}s{frame['step']}"
        strips.append(
            np.concatenate(
                [
                    pc.tile(pictures["base_agent"], f"{label} base agent"),
                    pc.tile(pictures["physical_agent"], "physical agent"),
                    pc.tile(pictures["composite_agent"], "composite agent"),
                    pc.tile(pictures["absent_agent"], "absent agent"),
                ],
                axis=1,
            )
        )
        strips.append(
            np.concatenate(
                [
                    pc.tile(pictures["base_wrist"], f"{label} base wrist"),
                    pc.tile(pictures["physical_wrist"], "physical wrist"),
                    pc.tile(pictures["composite_wrist"], "composite wrist"),
                    pc.tile(pictures["absent_wrist"], "absent wrist"),
                ],
                axis=1,
            )
        )
    if not strips:
        return False
    width = max(strip.shape[1] for strip in strips)
    padded = []
    for strip in strips:
        if strip.shape[1] < width:
            pad = np.full((strip.shape[0], width - strip.shape[1], 3), 255, dtype=np.uint8)
            strip = np.concatenate([strip, pad], axis=1)
        padded.append(strip)
    sheet = np.concatenate(padded, axis=0)
    cv2.imwrite(str(path), cv2.cvtColor(sheet, cv2.COLOR_RGB2BGR))
    return True


def render_placed(collect, env, sim, placements: list[tuple[int, list[int], np.ndarray, float]]):
    qpos = np.array(sim.data.qpos, copy=True)
    qvel = np.array(sim.data.qvel, copy=True)
    sizes = np.array(sim.model.geom_size, copy=True)
    try:
        fronts: dict[int, np.ndarray] = {}
        for body, geoms, point, scale in placements:
            span = collect.object_qpos_span(sim, body)
            if span is None or not geoms:
                return None
            start, stop = span
            sim.data.qpos[start : start + 3] = np.asarray(point, dtype=np.float64)
            sim.data.qpos[start + 3 : stop] = np.array([1.0, 0.0, 0.0, 0.0])
            for geom in geoms:
                sim.model.geom_size[geom] = sizes[geom] * float(scale)
        sim.data.qvel[:] = 0
        sim.forward()
        agent, wrist = cc.images_of(collect, collect.render_after_edit(env))
        for body, geoms, _point, _scale in placements:
            fronts[int(body)] = np.mean(
                [np.asarray(sim.data.geom_xpos[geom], dtype=np.float64) for geom in geoms],
                axis=0,
            )
        return {"agent": agent, "wrist": wrist, "fronts": fronts}
    finally:
        sim.model.geom_size[:] = sizes
        sim.data.qpos[:] = qpos
        sim.data.qvel[:] = qvel
        sim.forward()
        collect.render_after_edit(env)


def _fmt(value: float | None, digits: int = 3) -> str:
    if value is None or not np.isfinite(value):
        return "undefined"
    return f"{float(value):.{digits}f}"


def _print_view(name: str, row: dict, base_to_view: float | None, base_to_absent: float | None, closed: float | None) -> None:
    line = (
        f"  {name} [{row.get('kind', '')}] {row.get('body', '')} fraction {_fmt(row.get('fraction'), 2)} "
        f"scale {_fmt(row.get('scale'), 1)}: "
        f"agent cover {_fmt(row.get('agent_cover'))} wrist cover {_fmt(row.get('wrist_cover'))} "
        f"scene {_fmt(row.get('scene_fraction'))} "
        f"depth {_fmt(row.get('depth_gap'), 4)} angle {_fmt(row.get('angle_ratio'), 3)} "
        f"wrist depth {_fmt(row.get('wrist_depth_gap'), 4)} wrist angle {_fmt(row.get('wrist_angle_ratio'), 3)}"
    )
    if base_to_view is not None or base_to_absent is not None:
        line += (
            f" | RMSE base→view {_fmt(base_to_view, 4)} base→absent {_fmt(base_to_absent, 4)} "
            f"absence gap closed {pc.pct(closed)}"
        )
    print(line, flush=True)


def policy_device(policy) -> str:
    try:
        return str(next(policy.parameters()).device)
    except StopIteration:
        return "cpu"


def _motion(sim, body: int, span, arm0: np.ndarray, bowl0: np.ndarray | None) -> tuple[float, float | None]:
    arm = float(np.linalg.norm(np.asarray(sim.data.qpos[:7], dtype=np.float64) - arm0))
    bowl = paper_gaps.object_xyz(sim, body, span)
    shift = None if bowl0 is None or bowl is None else float(np.linalg.norm(bowl - bowl0))
    return arm, shift


def run_edited(
    collect,
    env,
    policy,
    pre,
    post,
    layer5,
    layer6,
    sentence: str,
    body: int,
    geoms: list[int],
    edit: pc.EditName,
    episode: int,
    horizon: int,
    replan: int,
    device: str,
    noise_base: int,
) -> dict:
    sim = collect.sim_of(env)
    span = collect.object_qpos_span(sim, body)
    arm0 = np.asarray(sim.data.qpos[:7], dtype=np.float64).copy()
    bowl0 = paper_gaps.object_xyz(sim, body, span)
    success_before = paper_gaps.task_succeeded(env)
    taken = 0
    succeeded = success_before
    error = ""
    raw_abs = None
    post_abs = None
    action_width = None
    try:
        while taken < horizon and not succeeded:
            agent, wrist, state = pc.live_images(collect, env, sim, body, geoms, edit)
            batch = cc.batch_from_images(collect, pre, agent, wrist, state, sentence, policy)
            _tokens, _tokens6, raw = cc.forward_policy(
                policy, layer5, layer6, batch, noise_base + taken, None, "base"
            )
            chunk = apply_post(post, raw, device)
            if raw_abs is None:
                raw_abs = mean_abs(raw)
                post_abs = mean_abs(chunk)
                action_width = int(chunk.shape[-1])
                print(
                    f"    ep {episode} {edit} raw mean abs {raw_abs:.4f} post mean abs {post_abs:.4f} "
                    f"raw {tuple(np.asarray(raw).shape)} post {tuple(chunk.shape)}",
                    flush=True,
                )
            acted = 0
            for action in np.asarray(chunk)[:replan]:
                if taken >= horizon:
                    break
                paper_gaps.step_env(env, action)
                taken += 1
                acted += 1
                sim = collect.sim_of(env)
                succeeded = paper_gaps.task_succeeded(env)
                if taken % 40 == 0 or succeeded:
                    arm, shift = _motion(sim, body, span, arm0, bowl0)
                    print(
                        f"    ep {episode} {edit} step {taken} success {succeeded} "
                        f"arm_move {_fmt(arm, 5)} bowl_shift_m {_fmt(shift, 6)}",
                        flush=True,
                    )
                if succeeded:
                    break
            if acted == 0:
                break
    except RuntimeError:
        raise
    except Exception as exc:
        error = str(exc)
        print(f"  ep {episode} {edit} stopped: {error}", flush=True)
    sim = collect.sim_of(env)
    arm, shift = _motion(sim, body, span, arm0, bowl0)
    print(
        f"  ep {episode} {edit}: success {success_before} -> {succeeded} steps {taken} "
        f"bowl_shift_m {_fmt(shift, 6)} arm_move {_fmt(arm, 5)} {error}",
        flush=True,
    )
    return {
        "episode": episode,
        "edit": edit,
        "success_before": success_before,
        "success_after": succeeded,
        "steps": taken,
        "bowl_shift_m": shift,
        "arm_move": arm,
        "raw_mean_abs": raw_abs,
        "post_mean_abs": post_abs,
        "action_width": action_width,
        "error": error,
    }


def main() -> None:
    cc.configure_environment()
    if not torch.cuda.is_available():
        raise SystemExit("CUDA GPU required. Use the Colab L4 or A100 runtime.")
    os.environ["MAX_STEPS"] = os.environ.get("CTRL_SCAN_STEPS", "200")
    collect = cc.load_collect_module()
    # Simulator imports stay inside main so unit tests do not load LIBERO or MuJoCo.
    from libero.libero import benchmark, get_libero_path
    from libero.libero.envs import OffScreenRenderEnv

    try:
        import mujoco

        free_type = int(mujoco.mjtJoint.mjJNT_FREE)
    except Exception:
        free_type = 0

    horizon = cc.env_int("CTRL_ROLLOUT_HORIZON", 220)
    replan = cc.env_int("CTRL_REPLAN", 10)
    episodes = cc.env_int("CTRL_ROLLOUT_EPISODES", 2)
    harness_steps = cc.env_int("CTRL_HARNESS_STEPS", 40)
    scan_steps = cc.env_int("CTRL_SCAN_STEPS", 200)
    repo = Path(os.environ.get("LOCAL_REPO", "/content/groot-run"))
    dest = repo / "outputs" / "permanence" / "remaining_gaps"
    dest.mkdir(parents=True, exist_ok=True)

    print("Loading policy", flush=True)
    policy, pre, post, layer5 = collect.load_policy()
    if post is None:
        raise SystemExit("load_policy did not return a postprocessor.")
    policy.eval()
    layer6 = cc.find_layer6(policy, layer5)
    device = policy_device(policy)
    suite = benchmark.get_benchmark_dict()[collect.SUITE]()
    task = suite.get_task(0)
    sentence = task.language
    demos = paper_gaps.read_demo_states(collect.find_demo_file(task), scan_steps)
    bddl = os.path.join(get_libero_path("bddl_files"), task.problem_folder, task.bddl_file)
    env = OffScreenRenderEnv(bddl_file_name=bddl, camera_heights=256, camera_widths=256)
    env.seed(0)

    def target_body(sim):
        mapping = collect.candidate_bodies(sim, sentence)
        for candidate, candidate_geoms in mapping.items():
            if collect.object_qpos_span(sim, candidate) is not None and candidate_geoms:
                return int(candidate), list(candidate_geoms)
        return None, None

    def forward(agent, wrist, state, noise: int):
        batch = cc.batch_from_images(collect, pre, agent, wrist, state, sentence, policy)
        _tokens, _tokens6, action = cc.forward_policy(policy, layer5, layer6, batch, noise, None, "base")
        return action

    def bowl_views(sim):
        _body, geoms = target_body(sim)
        raw = collect.render_after_edit(env)
        agent, wrist = cc.images_of(collect, raw)
        state = collect.libero_state(raw)
        if not geoms:
            return agent, wrist, state, None, None
        recolor_agent, recolor_wrist = cc.render_rgba_images(collect, env, sim, geoms)
        raw = collect.render_after_edit(env)
        agent, wrist = cc.images_of(collect, raw)
        state = collect.libero_state(raw)
        return (
            agent,
            wrist,
            state,
            pc.uncapped_change_mask(agent, recolor_agent, min_pixels=5),
            pc.uncapped_change_mask(wrist, recolor_wrist, min_pixels=5),
        )

    frame_rows: list[dict] = []
    try:
        print("\nBOTH CAMERAS  agent ray, wrist ray, and a labeled composite", flush=True)
        for episode, step in FRAMES:
            if episode >= len(demos) or step >= len(demos[episode]):
                print(f"  ep {episode} step {step}: outside the demos", flush=True)
                continue
            try:
                frame_rows.append(
                    measure_frame(
                        collect,
                        env,
                        sim_state=demos[episode][step],
                        episode=episode,
                        step=step,
                        free_type=free_type,
                        target_body=target_body,
                        bowl_views=bowl_views,
                        forward=forward,
                    )
                )
            except Exception as exc:
                print(f"  ep {episode} step {step} occluder failed: {exc}", flush=True)
                frame_rows.append({"episode": episode, "step": step, "error": str(exc)})

        sheet = dest / "both_cameras.png"
        if save_both_sheet(frame_rows, sheet):
            print("sheet", sheet, flush=True)

        print("\nHARNESS  40 steps of base actions through the postprocessor, then zeros", flush=True)
        harness = None
        if demos and len(demos[0]) > 0:
            collect.observe_state(env, demos[0][0])
            sim = collect.sim_of(env)
            body, geoms = target_body(sim)
            if body is None or not geoms:
                print("  no free target body", flush=True)
            else:
                harness = run_harness(
                    collect,
                    env,
                    policy,
                    pre,
                    post,
                    layer5,
                    layer6,
                    sentence,
                    demos[0][0],
                    body,
                    geoms,
                    harness_steps,
                    replan,
                    device,
                )
                print(" ", harness["reading"], flush=True)

        print("\nCLOSED LOOP  unnormalized chunks, re-query every 10 actions", flush=True)
        print(
            f"Horizon {horizon}, episodes {episodes}. covered paints the uncapped mask gray. "
            "absent shows the teleported bowl and restores it before the step. "
            "Proprioception is the live arm.",
            flush=True,
        )
        rollouts = []
        edits: tuple[pc.EditName, ...] = ("base", "covered", "absent")
        for episode in range(episodes):
            if episode >= len(demos) or len(demos[episode]) == 0:
                continue
            for edit in edits:
                collect.observe_state(env, demos[episode][0])
                sim = collect.sim_of(env)
                body, geoms = target_body(sim)
                if body is None or not geoms:
                    print(f"  ep {episode} {edit}: no free target body", flush=True)
                    continue
                rollouts.append(
                    run_edited(
                        collect,
                        env,
                        policy,
                        pre,
                        post,
                        layer5,
                        layer6,
                        sentence,
                        body,
                        geoms,
                        edit,
                        episode,
                        horizon,
                        replan,
                        device,
                        8000 + episode * 1000,
                    )
                )

        summary = {
            "scope": (
                "Scope: lerobot/pi05_libero_finetuned, PaliGemma language-model layer 5, "
                "proprioception taken from the unedited scene on the frozen frames. "
                f"Both-camera frames: {list(FRAMES)}. "
                f"Closed loop: {episodes} episodes, {horizon} steps, re-query every {replan} actions, "
                "chunks passed through the policy postprocessor."
            ),
            "frames": [{key: value for key, value in row.items() if key != "pictures"} for row in frame_rows],
            "harness": harness,
            "rollouts": rollouts,
            "figure": str(sheet) if sheet.is_file() else None,
        }
        (dest / "summary.json").write_text(json.dumps(cc.json_ready(summary), indent=2))
        print("\n" + summary["scope"], flush=True)
        print("Wrote", dest / "summary.json", flush=True)
        print("RESULT_DIR", dest, flush=True)
    finally:
        env.close()


def measure_frame(collect, env, sim_state, episode: int, step: int, free_type: int, target_body, bowl_views, forward) -> dict:
    collect.observe_state(env, sim_state)
    sim = collect.sim_of(env)
    bowl, bowl_geoms = target_body(sim)
    if bowl is None or not bowl_geoms:
        print(f"  ep {episode} step {step}: no free target body", flush=True)
        return {"episode": episode, "step": step, "error": "no free target body"}
    bodies = pc.free_body_map(sim, free_type)
    names = pc.body_names(sim, bodies)
    named = dict(names)
    agent_body = pc.prefer_occluder(names, bowl)
    if agent_body is None or agent_body not in bodies:
        print(f"  ep {episode} step {step}: no occluder body", flush=True)
        return {"episode": episode, "step": step, "error": "no occluder body"}
    other = second_body(names, {bowl, agent_body})
    agent_cam = collect.camera_id(sim)
    wrist_cam = cc.camera_index(sim, "eye_in_hand")
    agent_pos = np.asarray(sim.data.cam_xpos[agent_cam], dtype=np.float64)
    wrist_pos = None if wrist_cam is None else np.asarray(sim.data.cam_xpos[wrist_cam], dtype=np.float64)
    bowl_center = np.mean([np.asarray(sim.data.geom_xpos[geom], dtype=np.float64) for geom in bowl_geoms], axis=0)
    distance = None if wrist_pos is None else float(np.linalg.norm(bowl_center - wrist_pos))
    base_agent, base_wrist, state, agent_mask, wrist_mask = bowl_views(sim)
    print(
        f"  ep {episode} step {step} agent {short_name(named.get(agent_body, ''))} "
        f"wrist body {short_name(named.get(other, '')) if other is not None else 'same object, separate render'} "
        f"wrist-camera-to-bowl_m {_fmt(distance, 4)}",
        flush=True,
    )

    def place(kind: PlacementKind, placements: list[tuple[int, list[int], np.ndarray, float]], fraction: float, scale: float, body: int):
        rendered = render_placed(collect, env, sim, placements)
        if rendered is None:
            return None
        row = view_scores(base_agent, base_wrist, rendered["agent"], rendered["wrist"], agent_mask, wrist_mask)
        depth, ratio = (float("nan"), float("nan"))
        wrist_depth, wrist_ratio = (float("nan"), float("nan"))
        match kind:
            case "agent_only":
                wrist_body_id = None
                if agent_body in rendered["fronts"]:
                    depth, ratio = pc.covering_ratio(
                        agent_pos, rendered["fronts"][agent_body], bowl_center, 0.04 * AGENT_SCALE
                    )
            case "wrist_only":
                wrist_body_id = body
            case "physical_pair":
                wrist_body_id = other
                if agent_body in rendered["fronts"]:
                    depth, ratio = pc.covering_ratio(
                        agent_pos, rendered["fronts"][agent_body], bowl_center, 0.04 * AGENT_SCALE
                    )
            case _ as unexpected:
                assert_never(unexpected)
        if wrist_pos is not None and wrist_body_id is not None and wrist_body_id in rendered["fronts"]:
            wrist_depth, wrist_ratio = pc.covering_ratio(
                wrist_pos, rendered["fronts"][wrist_body_id], bowl_center, 0.04 * float(scale)
            )
        row.update(
            {
                "kind": kind,
                "body": short_name(named.get(body, "")),
                "fraction": float(fraction),
                "scale": float(scale),
                "depth_gap": depth,
                "angle_ratio": ratio,
                "wrist_depth_gap": wrist_depth,
                "wrist_angle_ratio": wrist_ratio,
            }
        )
        return row

    candidates = []
    agent_point = pc.point_on_ray(agent_pos, bowl_center, AGENT_FRACTION)
    agent_row = place(
        "agent_only",
        [(agent_body, bodies[agent_body], agent_point, AGENT_SCALE)],
        AGENT_FRACTION,
        AGENT_SCALE,
        agent_body,
    )
    if agent_row is not None:
        candidates.append(agent_row)
    search_body = other if other is not None else agent_body
    if wrist_pos is not None and search_body in bodies:
        for fraction in WRIST_FRACTIONS:
            for scale in WRIST_SCALES:
                point = pc.point_on_ray(wrist_pos, bowl_center, fraction)
                row = place(
                    "wrist_only",
                    [(search_body, bodies[search_body], point, scale)],
                    fraction,
                    scale,
                    search_body,
                )
                if row is not None:
                    candidates.append(row)
    if other is not None and other in bodies and wrist_pos is not None:
        for fraction in WRIST_FRACTIONS:
            for scale in WRIST_SCALES:
                wrist_point = pc.point_on_ray(wrist_pos, bowl_center, fraction)
                row = place(
                    "physical_pair",
                    [
                        (agent_body, bodies[agent_body], agent_point, AGENT_SCALE),
                        (other, bodies[other], wrist_point, scale),
                    ],
                    fraction,
                    scale,
                    other,
                )
                if row is not None:
                    candidates.append(row)
    print(f"  searched {len(candidates)} placements", flush=True)
    physical = choose_both([row for row in candidates if row["kind"] in ("agent_only", "wrist_only", "physical_pair")])
    wrist_choice = choose_wrist([row for row in candidates if row["kind"] == "wrist_only"])
    if agent_row is None or wrist_choice is None or physical is None:
        print(f"  ep {episode} step {step}: placement search produced no image", flush=True)
        return {
            "episode": episode,
            "step": step,
            "camera_to_bowl_m": distance,
            "candidates": [plain_row(row) for row in candidates],
            "error": "placement search produced no image",
        }
    composite = view_scores(
        base_agent,
        base_wrist,
        agent_row["agent"],
        wrist_choice["wrist"],
        agent_mask,
        wrist_mask,
    )
    composite.update(
        {
            "kind": "composite",
            "body": "two renders",
            "fraction": wrist_choice["fraction"],
            "scale": wrist_choice["scale"],
            "depth_gap": agent_row.get("depth_gap"),
            "angle_ratio": agent_row.get("angle_ratio"),
            "wrist_depth_gap": wrist_choice.get("wrist_depth_gap"),
            "wrist_angle_ratio": wrist_choice.get("wrist_angle_ratio"),
        }
    )
    noise = 8000 + episode * 1000 + step
    base_action = forward(base_agent, base_wrist, state, noise)
    physical_action = forward(physical["agent"], physical["wrist"], state, noise)
    composite_action = forward(composite["agent"], composite["wrist"], state, noise)
    absent_agent, absent_wrist, _absent_state = pc.live_images(collect, env, sim, bowl, bowl_geoms, "absent")
    absent_action = forward(absent_agent, absent_wrist, state, noise)
    to_absent = pc.action_rmse(base_action, absent_action)
    physical_rmse = pc.action_rmse(base_action, physical_action)
    composite_rmse = pc.action_rmse(base_action, composite_action)
    physical_closed = absence_gap_closed(pc.action_rmse(physical_action, absent_action), to_absent)
    composite_closed = absence_gap_closed(pc.action_rmse(composite_action, absent_action), to_absent)
    _print_view("agent_only", agent_row, None, None, None)
    _print_view("wrist_only", wrist_choice, None, None, None)
    _print_view("physical", physical, physical_rmse, to_absent, physical_closed)
    _print_view("composite", composite, composite_rmse, to_absent, composite_closed)
    physical_text = both_reading(
        physical["agent_cover"], physical["wrist_cover"], physical["scene_fraction"], False
    )
    composite_text = both_reading(
        composite["agent_cover"], composite["wrist_cover"], composite["scene_fraction"], True
    )
    print(f"  physical: {physical_text}", flush=True)
    print(f"  composite: {composite_text}", flush=True)
    return {
        "episode": episode,
        "step": step,
        "camera_to_bowl_m": distance,
        "agent_only": plain_row(agent_row),
        "wrist_only": plain_row(wrist_choice),
        "physical": plain_row(physical),
        "composite": plain_row(composite),
        "rmse_base_to_absent": to_absent,
        "rmse_base_to_physical": physical_rmse,
        "rmse_base_to_composite": composite_rmse,
        "physical_absence_gap_closed": physical_closed,
        "composite_absence_gap_closed": composite_closed,
        "physical_reading": physical_text,
        "composite_reading": composite_text,
        "candidates": [plain_row(row) for row in candidates],
        "pictures": {
            "base_agent": base_agent,
            "base_wrist": base_wrist,
            "physical_agent": physical["agent"],
            "physical_wrist": physical["wrist"],
            "composite_agent": composite["agent"],
            "composite_wrist": composite["wrist"],
            "absent_agent": absent_agent,
            "absent_wrist": absent_wrist,
        },
    }


def run_harness(
    collect,
    env,
    policy,
    pre,
    post,
    layer5,
    layer6,
    sentence: str,
    state0,
    body: int,
    geoms: list[int],
    harness_steps: int,
    replan: int,
    device: str,
) -> dict:
    collect.observe_state(env, state0)
    base = run_edited(
        collect,
        env,
        policy,
        pre,
        post,
        layer5,
        layer6,
        sentence,
        body,
        geoms,
        "base",
        0,
        harness_steps,
        replan,
        device,
        7900,
    )
    collect.observe_state(env, state0)
    sim = collect.sim_of(env)
    span = collect.object_qpos_span(sim, body)
    arm0 = np.asarray(sim.data.qpos[:7], dtype=np.float64).copy()
    bowl0 = paper_gaps.object_xyz(sim, body, span)
    space = getattr(env, "action_space", None)
    width = int(base["action_width"] or 0)
    if width < 1 and space is not None and getattr(space, "shape", None):
        width = int(space.shape[0])
    if width < 1:
        width = 7
    chunk = np.zeros((max(replan, 1), width), dtype=np.float32)
    taken = 0
    while taken < harness_steps:
        for action in chunk:
            if taken >= harness_steps:
                break
            paper_gaps.step_env(env, action)
            taken += 1
    arm, shift = _motion(collect.sim_of(env), body, span, arm0, bowl0)
    print(
        f"  zeros: steps {taken} arm_move {_fmt(arm, 5)} bowl_shift_m {_fmt(shift, 6)}",
        flush=True,
    )
    reading = harness_reading(
        float(base["raw_mean_abs"] or np.nan),
        float(base["post_mean_abs"] or np.nan),
        float(base["arm_move"]),
        arm,
    )
    return {
        "base_arm_move": base["arm_move"],
        "base_bowl_shift_m": base["bowl_shift_m"],
        "zero_arm_move": arm,
        "zero_bowl_shift_m": shift,
        "raw_mean_abs": base["raw_mean_abs"],
        "post_mean_abs": base["post_mean_abs"],
        "reading": reading,
    }


if __name__ == "__main__":
    main()
'''

scripts = LOCAL_REPO / "scripts"
scripts.mkdir(parents=True, exist_ok=True)
path = scripts / "remaining_gaps.py"
path.write_text(SCRIPT)
for required in (
    "controlled_contrasts.py",
    "occlusion_features.py",
    "collect_layer5_replay.py",
    "paper_gaps.py",
    "paper_claim.py",
):
    if not (scripts / required).is_file():
        raise SystemExit(
            f"scripts/{required} is missing. Run the collector cell, the contrast cell, "
            "the occlusion cell, the paper-gaps cell, and the paper-claim cell first."
        )

env = os.environ.copy()
env["PYTHONPATH"] = str(scripts) + os.pathsep + env.get("PYTHONPATH", "")
env["LOCAL_REPO"] = str(LOCAL_REPO)
env["PYTHONUNBUFFERED"] = "1"
env["MAX_STEPS"] = env.get("CTRL_SCAN_STEPS", "200")

def stream(cmd):
    proc = subprocess.Popen(
        cmd,
        cwd=str(scripts),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    tail = []
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
        tail.append(line.rstrip())
        del tail[:-80]
    code = proc.wait()
    if code != 0:
        raise SystemExit(
            "remaining_gaps.py exited with code %s. Last lines:\n%s" % (code, "\n".join(tail))
        )

stream([str(PYTHON), "-u", str(path)])
print("summary", LOCAL_REPO / "outputs" / "permanence" / "remaining_gaps" / "summary.json")


In [ ]:
# Tight cover — smallest both-camera plate, and a closed loop that resets
# Paste into the same runtime after the remaining-gaps cell. Does not retrain.
# Does not overwrite outputs/permanence/remaining_gaps/ or earlier directories.

import os, sys, subprocess
from pathlib import Path

os.environ.setdefault("CTRL_SCAN_STEPS", "200")
os.environ["CTRL_ROLLOUT_HORIZON"] = "220"
os.environ["CTRL_ROLLOUT_EPISODES"] = "10"
os.environ.setdefault("CTRL_REPLAN", "10")
os.environ.setdefault("SUITE", "libero_spatial")
os.environ.setdefault("TASK_IDS", "[0]")
os.environ.setdefault(
    "LAYER_NAME",
    "model.paligemma_with_expert.paligemma.model.language_model.layers.5",
)
os.environ.setdefault("MUJOCO_GL", "egl")
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

LOCAL_REPO = Path(os.environ.get("LOCAL_REPO", "/content/groot-run"))
PYTHON = Path("/content/lerobot-venv/bin/python")
if not PYTHON.exists():
    PYTHON = Path(sys.executable)

SCRIPT = r'''
"""Score the tight both-camera hide, and roll it out from a fresh reset.

The previous selector kept the first full cover, so the plate that fills 42%
of the wrist was forwarded and the pair that fills 17% was not. This run
forwards the smallest full cover, including a closer and smaller plate on the
held frame, then rolls out base, gray paint, absence, and that tight plate.
Each episode is reset and the done flag is cleared before the first step.

Does not retrain and does not rewrite earlier output directories.
"""

from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Literal, assert_never

import cv2
import numpy as np
import torch

import controlled_contrasts as cc
import paper_claim as pc
import paper_gaps
import remaining_gaps as rg

FRAMES = ((1, 0), (1, 44))
LARGE_FRACTION = 0.30
LARGE_SCALE = 1.5
TIGHT_FRACTIONS = (0.55, 0.70, 0.80, 0.90)
TIGHT_SCALES = (0.75, 1.0, 1.5)
RollEdit = Literal["base", "covered", "absent", "physical"]


def choose_tight(candidates: list[dict]) -> dict | None:
    """Among full covers, keep the one that changes the least of either image."""
    usable = [
        row
        for row in candidates
        if np.isfinite(row.get("agent_cover", np.nan)) and np.isfinite(row.get("wrist_cover", np.nan))
    ]
    if not usable:
        return None
    full = [
        row
        for row in usable
        if float(row["agent_cover"]) >= rg.COVER_READY and float(row["wrist_cover"]) >= rg.COVER_READY
    ]
    pool = full or usable
    return min(pool, key=lambda row: (rg._scene(row), -min(float(row["agent_cover"]), float(row["wrist_cover"]))))


def tight_reading(agent_cover: float, wrist_cover: float, scene: float, distance: float | None) -> str:
    full = (
        np.isfinite(agent_cover)
        and np.isfinite(wrist_cover)
        and float(agent_cover) >= rg.COVER_READY
        and float(wrist_cover) >= rg.COVER_READY
    )
    if full and np.isfinite(scene) and float(scene) < rg.SCENE_LIMIT:
        return "A tight scene hides the bowl from both cameras."
    if full and np.isfinite(scene):
        dist = "an unknown distance" if distance is None or not np.isfinite(distance) else f"{float(distance):.3f} m"
        return (
            f"Both cameras lose the bowl and the plate fills {100 * float(scene):.0f}% of a frame. "
            f"The wrist camera is {dist} from the bowl."
        )
    if np.isfinite(agent_cover) and float(agent_cover) >= rg.COVER_READY and (
        not np.isfinite(wrist_cover) or float(wrist_cover) < 0.2
    ):
        return "The agent camera is covered. The wrist camera still shows the bowl."
    return "The bowl stays partly visible in a camera."


def owners_of(env) -> list:
    found = []
    seen: set[int] = set()
    pending = [env]
    while pending:
        owner = pending.pop()
        if owner is None or id(owner) in seen:
            continue
        seen.add(id(owner))
        found.append(owner)
        for attr in ("env", "unwrapped"):
            inner = getattr(owner, attr, None)
            if inner is not None and inner is not owner:
                pending.append(inner)
    return found


def clear_episode(env) -> None:
    """Clear the done flag and the step counter so the next edit can step."""
    for owner in owners_of(env):
        if hasattr(owner, "done"):
            owner.done = False
        for name in ("timestep", "_elapsed_steps"):
            if hasattr(owner, name):
                setattr(owner, name, 0)


def episode_done(env) -> bool:
    for owner in owners_of(env):
        if bool(getattr(owner, "done", False)):
            return True
    return False


def task_succeeded(env) -> bool:
    for owner in owners_of(env):
        for name in ("check_success", "_check_success"):
            fn = getattr(owner, name, None)
            if callable(fn) and bool(fn()):
                return True
    return False


def step_done(outcome) -> bool:
    if not isinstance(outcome, tuple):
        return False
    if len(outcome) == 5:
        return bool(outcome[2] or outcome[3])
    if len(outcome) == 4:
        return bool(outcome[2])
    return False


def step_action(env, action: np.ndarray) -> bool:
    vector = np.asarray(action, dtype=np.float64).reshape(-1)
    space = getattr(env, "action_space", None)
    width = int(space.shape[0]) if space is not None and getattr(space, "shape", None) else vector.size
    vector = vector[:width]
    outcome = env.step(vector)
    if not isinstance(outcome, tuple) or len(outcome) not in (4, 5):
        raise RuntimeError(f"env.step returned {type(outcome)}")
    return step_done(outcome) or episode_done(env)


def begin_episode(collect, env, state) -> None:
    if hasattr(env, "reset"):
        try:
            env.reset()
        except Exception as exc:
            print(f"  reset skipped: {exc}", flush=True)
    collect.observe_state(env, state)
    clear_episode(env)


def rate_rows(rows: list[dict]) -> list[dict]:
    edits: list[str] = []
    for row in rows:
        if row["edit"] not in edits:
            edits.append(row["edit"])
    summary = []
    for edit in edits:
        chosen = [row for row in rows if row["edit"] == edit]
        summary.append(
            {
                "edit": edit,
                "episodes": len(chosen),
                "successes": sum(1 for row in chosen if row["success_after"]),
                "no_step": sum(1 for row in chosen if int(row["steps"]) == 0),
            }
        )
    return summary


def plain_row(row: dict | None) -> dict | None:
    if row is None:
        return None
    return {key: value for key, value in row.items() if key not in ("agent", "wrist")}


def save_sheet(frames: list[dict], path: Path) -> bool:
    strips = []
    for frame in frames:
        pictures = frame.get("pictures")
        if not pictures:
            continue
        label = f"e{frame['episode']}s{frame['step']}"
        strips.append(
            np.concatenate(
                [
                    pc.tile(pictures["base_agent"], f"{label} base"),
                    pc.tile(pictures["large_agent"], "large agent"),
                    pc.tile(pictures["tight_agent"], "tight agent"),
                    pc.tile(pictures["absent_agent"], "absent agent"),
                ],
                axis=1,
            )
        )
        strips.append(
            np.concatenate(
                [
                    pc.tile(pictures["base_wrist"], f"{label} wrist"),
                    pc.tile(pictures["large_wrist"], "large wrist"),
                    pc.tile(pictures["tight_wrist"], "tight wrist"),
                    pc.tile(pictures["absent_wrist"], "absent wrist"),
                ],
                axis=1,
            )
        )
    if not strips:
        return False
    width = max(strip.shape[1] for strip in strips)
    padded = []
    for strip in strips:
        if strip.shape[1] < width:
            pad = np.full((strip.shape[0], width - strip.shape[1], 3), 255, dtype=np.uint8)
            strip = np.concatenate([strip, pad], axis=1)
        padded.append(strip)
    cv2.imwrite(str(path), cv2.cvtColor(np.concatenate(padded, axis=0), cv2.COLOR_RGB2BGR))
    return True


def _fmt(value: float | None, digits: int = 3) -> str:
    if value is None or not np.isfinite(value):
        return "undefined"
    return f"{float(value):.{digits}f}"


def _print_forward(name: str, row: dict, base_to_view: float, base_to_absent: float, closed: float | None) -> None:
    print(
        f"  {name} fraction {_fmt(row.get('fraction'), 2)} scale {_fmt(row.get('scale'), 2)}: "
        f"agent cover {_fmt(row.get('agent_cover'))} wrist cover {_fmt(row.get('wrist_cover'))} "
        f"scene {_fmt(row.get('scene_fraction'))} | "
        f"RMSE base→view {_fmt(base_to_view, 4)} base→absent {_fmt(base_to_absent, 4)} "
        f"absence gap closed {pc.pct(closed)}",
        flush=True,
    )


def main() -> None:
    cc.configure_environment()
    if not torch.cuda.is_available():
        raise SystemExit("CUDA GPU required. Use the Colab L4 or A100 runtime.")
    os.environ["MAX_STEPS"] = os.environ.get("CTRL_SCAN_STEPS", "200")
    collect = cc.load_collect_module()
    # Simulator imports stay inside main so unit tests do not load LIBERO or MuJoCo.
    from libero.libero import benchmark, get_libero_path
    from libero.libero.envs import OffScreenRenderEnv

    try:
        import mujoco

        free_type = int(mujoco.mjtJoint.mjJNT_FREE)
    except Exception:
        free_type = 0

    horizon = cc.env_int("CTRL_ROLLOUT_HORIZON", 220)
    replan = cc.env_int("CTRL_REPLAN", 10)
    episodes = cc.env_int("CTRL_ROLLOUT_EPISODES", 10)
    scan_steps = cc.env_int("CTRL_SCAN_STEPS", 200)
    repo = Path(os.environ.get("LOCAL_REPO", "/content/groot-run"))
    dest = repo / "outputs" / "permanence" / "tight_cover"
    dest.mkdir(parents=True, exist_ok=True)

    print("Loading policy", flush=True)
    policy, pre, post, layer5 = collect.load_policy()
    if post is None:
        raise SystemExit("load_policy did not return a postprocessor.")
    policy.eval()
    layer6 = cc.find_layer6(policy, layer5)
    device = rg.policy_device(policy)
    suite = benchmark.get_benchmark_dict()[collect.SUITE]()
    task = suite.get_task(0)
    sentence = task.language
    demos = paper_gaps.read_demo_states(collect.find_demo_file(task), scan_steps)
    bddl = os.path.join(get_libero_path("bddl_files"), task.problem_folder, task.bddl_file)
    env = OffScreenRenderEnv(bddl_file_name=bddl, camera_heights=256, camera_widths=256)
    env.seed(0)

    def target_body(sim):
        mapping = collect.candidate_bodies(sim, sentence)
        for candidate, candidate_geoms in mapping.items():
            if collect.object_qpos_span(sim, candidate) is not None and candidate_geoms:
                return int(candidate), list(candidate_geoms)
        return None, None

    def forward(agent, wrist, state, noise: int):
        batch = cc.batch_from_images(collect, pre, agent, wrist, state, sentence, policy)
        _tokens, _tokens6, action = cc.forward_policy(policy, layer5, layer6, batch, noise, None, "base")
        return action

    frame_rows: list[dict] = []
    tight_params = {"fraction": 0.70, "scale": 1.5}
    try:
        print("\nTIGHT COVER  smallest full cover of both cameras", flush=True)
        for episode, step in FRAMES:
            if episode >= len(demos) or step >= len(demos[episode]):
                print(f"  ep {episode} step {step}: outside the demos", flush=True)
                continue
            row = score_frame(
                collect,
                env,
                demos[episode][step],
                episode,
                step,
                free_type,
                target_body,
                forward,
            )
            frame_rows.append(row)
            if episode == 1 and step == 0 and row.get("tight"):
                tight_params = {
                    "fraction": float(row["tight"]["fraction"]),
                    "scale": float(row["tight"]["scale"]),
                }
        sheet = dest / "tight_cover.png"
        if save_sheet(frame_rows, sheet):
            print("sheet", sheet, flush=True)

        print("\nCLOSED LOOP  reset before every edit, unnormalized chunks", flush=True)
        print(
            f"Horizon {horizon}, episodes {min(episodes, len(demos))}, re-query every {replan}. "
            f"physical re-places the table-frame tight pair "
            f"(fraction {tight_params['fraction']:.2f}, scale {tight_params['scale']:.2f}) on the current rays. "
            "covered paints the uncapped mask gray. absent teleports the bowl and restores it before the step.",
            flush=True,
        )
        rollouts = run_rollouts(
            collect,
            env,
            policy,
            pre,
            post,
            layer5,
            layer6,
            sentence,
            demos,
            target_body,
            free_type,
            tight_params,
            min(episodes, len(demos)),
            horizon,
            replan,
            device,
        )
        rates = rate_rows(rollouts)
        print("\nSUCCESS", flush=True)
        for rate in rates:
            print(
                f"  {rate['edit']}: {rate['successes']}/{rate['episodes']} success, "
                f"{rate['no_step']} episodes with no step",
                flush=True,
            )
        summary = {
            "scope": (
                "Scope: lerobot/pi05_libero_finetuned, PaliGemma language-model layer 5. "
                f"Tight-cover frames: {list(FRAMES)}. "
                f"Rollout plate fraction {tight_params['fraction']:.2f}, scale {tight_params['scale']:.2f}. "
                f"Closed loop: {min(episodes, len(demos))} episodes, {horizon} steps, "
                f"re-query every {replan}, reset and done flag cleared before each edit."
            ),
            "frames": [{key: value for key, value in row.items() if key != "pictures"} for row in frame_rows],
            "tight_params": tight_params,
            "rollouts": rollouts,
            "rates": rates,
            "figure": str(sheet) if sheet.is_file() else None,
        }
        (dest / "summary.json").write_text(json.dumps(cc.json_ready(summary), indent=2))
        print("\n" + summary["scope"], flush=True)
        print("Wrote", dest / "summary.json", flush=True)
        print("RESULT_DIR", dest, flush=True)
    finally:
        env.close()


def score_frame(collect, env, sim_state, episode: int, step: int, free_type: int, target_body, forward) -> dict:
    begin_episode(collect, env, sim_state)
    sim = collect.sim_of(env)
    bowl, bowl_geoms = target_body(sim)
    if bowl is None or not bowl_geoms:
        print(f"  ep {episode} step {step}: no free target body", flush=True)
        return {"episode": episode, "step": step, "error": "no free target body"}
    bodies = pc.free_body_map(sim, free_type)
    names = dict(pc.body_names(sim, bodies))
    agent_body = pc.prefer_occluder(list(names.items()), bowl)
    other = rg.second_body(list(names.items()), {bowl, agent_body} if agent_body is not None else {bowl})
    if agent_body is None or other is None or agent_body not in bodies or other not in bodies:
        print(f"  ep {episode} step {step}: need a ramekin and a second body", flush=True)
        return {"episode": episode, "step": step, "error": "need a ramekin and a second body"}
    wrist_cam = cc.camera_index(sim, "eye_in_hand")
    if wrist_cam is None:
        print(f"  ep {episode} step {step}: wrist camera was not found", flush=True)
        return {"episode": episode, "step": step, "error": "wrist camera was not found"}
    agent_pos = np.asarray(sim.data.cam_xpos[collect.camera_id(sim)], dtype=np.float64)
    wrist_pos = np.asarray(sim.data.cam_xpos[wrist_cam], dtype=np.float64)
    bowl_center = np.mean([np.asarray(sim.data.geom_xpos[geom], dtype=np.float64) for geom in bowl_geoms], axis=0)
    distance = float(np.linalg.norm(bowl_center - wrist_pos))
    raw = collect.render_after_edit(env)
    base_agent, base_wrist = cc.images_of(collect, raw)
    state = collect.libero_state(raw)
    recolor_agent, recolor_wrist = cc.render_rgba_images(collect, env, sim, bowl_geoms)
    raw = collect.render_after_edit(env)
    base_agent, base_wrist = cc.images_of(collect, raw)
    state = collect.libero_state(raw)
    agent_mask = pc.uncapped_change_mask(base_agent, recolor_agent, min_pixels=5)
    wrist_mask = pc.uncapped_change_mask(base_wrist, recolor_wrist, min_pixels=5)
    print(
        f"  ep {episode} step {step} wrist-camera-to-bowl_m {_fmt(distance, 4)}",
        flush=True,
    )

    def one(fraction: float, scale: float) -> dict | None:
        rendered = rg.render_placed(
            collect,
            env,
            sim,
            [
                (agent_body, bodies[agent_body], pc.point_on_ray(agent_pos, bowl_center, rg.AGENT_FRACTION), rg.AGENT_SCALE),
                (other, bodies[other], pc.point_on_ray(wrist_pos, bowl_center, fraction), scale),
            ],
        )
        if rendered is None:
            return None
        row = rg.view_scores(base_agent, base_wrist, rendered["agent"], rendered["wrist"], agent_mask, wrist_mask)
        row.update({"fraction": float(fraction), "scale": float(scale), "kind": "physical_pair"})
        return row

    candidates = []
    large = one(LARGE_FRACTION, LARGE_SCALE)
    if large is not None:
        candidates.append(large)
    for fraction in TIGHT_FRACTIONS:
        for scale in TIGHT_SCALES:
            row = one(fraction, scale)
            if row is not None:
                candidates.append(row)
    tight = choose_tight(candidates)
    if large is None or tight is None:
        print(f"  ep {episode} step {step}: placement search produced no image", flush=True)
        return {"episode": episode, "step": step, "camera_to_bowl_m": distance, "error": "placement search produced no image"}
    noise = 8100 + episode * 1000 + step
    base_action = forward(base_agent, base_wrist, state, noise)
    large_action = forward(large["agent"], large["wrist"], state, noise)
    tight_action = forward(tight["agent"], tight["wrist"], state, noise)
    absent_agent, absent_wrist, _absent_state = pc.live_images(collect, env, sim, bowl, bowl_geoms, "absent")
    absent_action = forward(absent_agent, absent_wrist, state, noise)
    to_absent = pc.action_rmse(base_action, absent_action)
    large_rmse = pc.action_rmse(base_action, large_action)
    tight_rmse = pc.action_rmse(base_action, tight_action)
    large_closed = rg.absence_gap_closed(pc.action_rmse(large_action, absent_action), to_absent)
    tight_closed = rg.absence_gap_closed(pc.action_rmse(tight_action, absent_action), to_absent)
    _print_forward("large", large, large_rmse, to_absent, large_closed)
    _print_forward("tight", tight, tight_rmse, to_absent, tight_closed)
    text = tight_reading(tight["agent_cover"], tight["wrist_cover"], tight["scene_fraction"], distance)
    print(f"  tight: {text}", flush=True)
    return {
        "episode": episode,
        "step": step,
        "camera_to_bowl_m": distance,
        "large": plain_row(large),
        "tight": plain_row(tight),
        "rmse_base_to_absent": to_absent,
        "rmse_base_to_large": large_rmse,
        "rmse_base_to_tight": tight_rmse,
        "large_absence_gap_closed": large_closed,
        "tight_absence_gap_closed": tight_closed,
        "tight_reading": text,
        "candidates": [plain_row(row) for row in candidates],
        "pictures": {
            "base_agent": base_agent,
            "base_wrist": base_wrist,
            "large_agent": large["agent"],
            "large_wrist": large["wrist"],
            "tight_agent": tight["agent"],
            "tight_wrist": tight["wrist"],
            "absent_agent": absent_agent,
            "absent_wrist": absent_wrist,
        },
    }


def current_pair(collect, env, sim, agent_body, agent_geoms, wrist_body, wrist_geoms, bowl_geoms, fraction: float, scale: float):
    wrist_cam = cc.camera_index(sim, "eye_in_hand")
    if wrist_cam is None or not bowl_geoms:
        return None
    agent_pos = np.asarray(sim.data.cam_xpos[collect.camera_id(sim)], dtype=np.float64)
    wrist_pos = np.asarray(sim.data.cam_xpos[wrist_cam], dtype=np.float64)
    bowl_center = np.mean([np.asarray(sim.data.geom_xpos[geom], dtype=np.float64) for geom in bowl_geoms], axis=0)
    return rg.render_placed(
        collect,
        env,
        sim,
        [
            (agent_body, agent_geoms, pc.point_on_ray(agent_pos, bowl_center, rg.AGENT_FRACTION), rg.AGENT_SCALE),
            (wrist_body, wrist_geoms, pc.point_on_ray(wrist_pos, bowl_center, fraction), scale),
        ],
    )


def rollout_images(
    kind: RollEdit,
    collect,
    env,
    sim,
    body: int,
    geoms: list[int],
    pair,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    match kind:
        case "base" | "covered" | "absent":
            return pc.live_images(collect, env, sim, body, geoms, kind)
        case "physical":
            agent, wrist, state = pc.live_images(collect, env, sim, body, geoms, "base")
            if pair is None:
                return agent, wrist, state
            placed = current_pair(collect, env, sim, *pair)
            if placed is None:
                return agent, wrist, state
            return placed["agent"], placed["wrist"], state
        case _ as unexpected:
            assert_never(unexpected)


def run_rollouts(
    collect,
    env,
    policy,
    pre,
    post,
    layer5,
    layer6,
    sentence: str,
    demos,
    target_body,
    free_type: int,
    tight_params: dict,
    episodes: int,
    horizon: int,
    replan: int,
    device: str,
) -> list[dict]:
    edits: tuple[RollEdit, ...] = ("base", "covered", "absent", "physical")
    rows = []
    for episode in range(episodes):
        if episode >= len(demos) or len(demos[episode]) == 0:
            continue
        for edit in edits:
            begin_episode(collect, env, demos[episode][0])
            sim = collect.sim_of(env)
            body, geoms = target_body(sim)
            if body is None or not geoms:
                print(f"  ep {episode} {edit}: no free target body", flush=True)
                continue
            bodies = pc.free_body_map(sim, free_type)
            names = dict(pc.body_names(sim, bodies))
            agent_body = pc.prefer_occluder(list(names.items()), body)
            other = rg.second_body(list(names.items()), {body, agent_body} if agent_body is not None else {body})
            pair = None
            if edit == "physical" and (agent_body is None or other is None):
                print(f"  ep {episode} physical: no second body, using the live image", flush=True)
            if agent_body is not None and other is not None and agent_body in bodies and other in bodies:
                pair = (
                    agent_body,
                    bodies[agent_body],
                    other,
                    bodies[other],
                    geoms,
                    float(tight_params["fraction"]),
                    float(tight_params["scale"]),
                )
            span = collect.object_qpos_span(sim, body)
            arm0 = np.asarray(sim.data.qpos[:7], dtype=np.float64).copy()
            bowl0 = paper_gaps.object_xyz(sim, body, span)
            success_before = task_succeeded(env)
            taken = 0
            succeeded = success_before
            error = ""
            env_done = False
            raw_abs = None
            post_abs = None
            try:
                while taken < horizon and not succeeded and not episode_done(env):
                    sim = collect.sim_of(env)
                    agent, wrist, state = rollout_images(edit, collect, env, sim, body, geoms, pair)
                    batch = cc.batch_from_images(collect, pre, agent, wrist, state, sentence, policy)
                    _tokens, _tokens6, raw = cc.forward_policy(
                        policy, layer5, layer6, batch, 8200 + episode * 1000 + taken, None, "base"
                    )
                    chunk = rg.apply_post(post, raw, device)
                    if raw_abs is None:
                        raw_abs = rg.mean_abs(raw)
                        post_abs = rg.mean_abs(chunk)
                        print(
                            f"    ep {episode} {edit} raw mean abs {raw_abs:.4f} post mean abs {post_abs:.4f}",
                            flush=True,
                        )
                    acted = 0
                    for action in np.asarray(chunk)[:replan]:
                        if taken >= horizon or episode_done(env):
                            break
                        env_done = step_action(env, action)
                        taken += 1
                        acted += 1
                        sim = collect.sim_of(env)
                        succeeded = task_succeeded(env)
                        if taken % 40 == 0 or succeeded or env_done:
                            arm, shift = rg._motion(sim, body, span, arm0, bowl0)
                            print(
                                f"    ep {episode} {edit} step {taken} success {succeeded} "
                                f"env_done {env_done} arm_move {_fmt(arm, 5)} bowl_shift_m {_fmt(shift, 6)}",
                                flush=True,
                            )
                        if succeeded or env_done:
                            break
                    if acted == 0:
                        break
            except RuntimeError:
                raise
            except Exception as exc:
                error = str(exc)
                print(f"  ep {episode} {edit} stopped: {error}", flush=True)
            sim = collect.sim_of(env)
            arm, shift = rg._motion(sim, body, span, arm0, bowl0)
            print(
                f"  ep {episode} {edit}: success {success_before} -> {succeeded} steps {taken} "
                f"bowl_shift_m {_fmt(shift, 6)} arm_move {_fmt(arm, 5)} env_done {env_done} {error}",
                flush=True,
            )
            rows.append(
                {
                    "episode": episode,
                    "edit": edit,
                    "success_before": success_before,
                    "success_after": succeeded,
                    "steps": taken,
                    "bowl_shift_m": shift,
                    "arm_move": arm,
                    "env_done": env_done,
                    "raw_mean_abs": raw_abs,
                    "post_mean_abs": post_abs,
                    "error": error,
                }
            )
    return rows


if __name__ == "__main__":
    main()
'''

scripts = LOCAL_REPO / "scripts"
scripts.mkdir(parents=True, exist_ok=True)
path = scripts / "tight_cover.py"
path.write_text(SCRIPT)
for required in (
    "controlled_contrasts.py",
    "occlusion_features.py",
    "collect_layer5_replay.py",
    "paper_gaps.py",
    "paper_claim.py",
    "remaining_gaps.py",
):
    if not (scripts / required).is_file():
        raise SystemExit(
            f"scripts/{required} is missing. Run the collector cell through the remaining-gaps cell first."
        )

env = os.environ.copy()
env["PYTHONPATH"] = str(scripts) + os.pathsep + env.get("PYTHONPATH", "")
env["LOCAL_REPO"] = str(LOCAL_REPO)
env["PYTHONUNBUFFERED"] = "1"
env["MAX_STEPS"] = env.get("CTRL_SCAN_STEPS", "200")
env["CTRL_ROLLOUT_HORIZON"] = "220"
env["CTRL_ROLLOUT_EPISODES"] = os.environ.get("CTRL_ROLLOUT_EPISODES", "10")

def stream(cmd):
    proc = subprocess.Popen(
        cmd,
        cwd=str(scripts),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    tail = []
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
        tail.append(line.rstrip())
        del tail[:-80]
    code = proc.wait()
    if code != 0:
        raise SystemExit(
            "tight_cover.py exited with code %s. Last lines:\n%s" % (code, "\n".join(tail))
        )

stream([str(PYTHON), "-u", str(path)])
print("summary", LOCAL_REPO / "outputs" / "permanence" / "tight_cover" / "summary.json")


## Action-expert occlusion contrast

Separate experiment. Paste only the next cell into the runtime that already finished setup. It clones `feature/akhidre-transcoder-circuit-tracing` into `/content/pi05-run`, keeps the existing venv and caches, and writes `/content/pi05-run/outputs/occlusion_contrast/verdict.json`. If `HuggingFaceVLA/libero` is not already at the LeRobot path, it reuses the Hub snapshot or downloads episodes 0,1,2 and links it. It leaves `/content/groot-run` and `outputs/permanence/` in place.


In [ ]:
# Action-expert occlusion contrast
# Separate experiment. Paste this into the runtime that already finished setup.
# Clones feature/akhidre-transcoder-circuit-tracing into /content/pi05-run.
# Leaves /content/groot-run and outputs/permanence/ in place.

import os
import subprocess
import sys
from pathlib import Path

os.environ.setdefault("MUJOCO_GL", "egl")
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
os.environ.setdefault("SUITE", "libero_spatial")
os.environ.setdefault("GIT_TERMINAL_PROMPT", "0")

LOCAL_REPO = Path(os.environ.get("LOCAL_REPO", "/content/groot-run"))
WORK_REPO = Path(os.environ.get("PI05_RUN_REPO", "/content/pi05-run"))
BRANCH = "feature/akhidre-transcoder-circuit-tracing"
REMOTE = "https://github.com/MarquiseRosier/pi05-run.git"
PYTHON = Path("/content/lerobot-venv/bin/python")
if not PYTHON.exists():
    PYTHON = Path(sys.executable)

os.environ.setdefault("HF_HOME", "/content/hf_home")
os.environ.setdefault("HF_HUB_CACHE", "/content/hf_home/hub")
if LOCAL_REPO.is_dir():
    os.environ.setdefault("LIBERO_DATASET_DIR", str(LOCAL_REPO / "data" / "libero" / "datasets"))
    os.environ.setdefault("LIBERO_CONFIG_PATH", str(LOCAL_REPO / ".libero"))

DATASET_REPO = "HuggingFaceVLA/libero"
EPISODES = "0,1,2"
FEATURE_FILES = (
    "feature_candidates.json",
    "observations.jsonl",
    "feature_topk.pt",
    "feature_stats.pt",
)

REQUIRED = (
    "scripts/train_pi05_transcoders.py",
    "scripts/collect_pi05_transcoder_features.py",
    "scripts/make_pi05_feature_report.py",
    "scripts/trace_pi05_transcoder_circuit.py",
)


def git(cmd, cwd=None):
    print("+", " ".join(str(part) for part in cmd), flush=True)
    proc = subprocess.run(
        [str(part) for part in cmd],
        cwd=None if cwd is None else str(cwd),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if proc.stdout:
        print(proc.stdout, end="" if proc.stdout.endswith("\n") else "\n", flush=True)
    return proc


def stream(cmd, cwd, env):
    print("+", " ".join(str(part) for part in cmd), flush=True)
    proc = subprocess.Popen(
        [str(part) for part in cmd],
        cwd=str(cwd),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    tail = []
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
        tail.append(line.rstrip())
        del tail[:-80]
    code = proc.wait()
    if code != 0:
        raise SystemExit(
            "occlusion contrast exited with code %s. Last lines:\n%s" % (code, "\n".join(tail))
        )


def has_circuit_scripts(path: Path) -> bool:
    return all((path / name).is_file() for name in REQUIRED)


def ensure_circuit_repo() -> Path:
    # Clone the circuit-tracing branch beside groot-run. Do not git-status groot-run.
    if has_circuit_scripts(WORK_REPO):
        print("using existing", WORK_REPO, flush=True)
        git(["git", "rev-parse", "--short", "HEAD"], WORK_REPO)
        return WORK_REPO
    if WORK_REPO.exists() and any(WORK_REPO.iterdir()):
        raise SystemExit(
            "%s exists but is missing the circuit-tracing scripts. "
            "Move or rename that folder and rerun." % WORK_REPO
        )
    WORK_REPO.parent.mkdir(parents=True, exist_ok=True)
    cloned = git(["git", "clone", "--depth", "1", "--branch", BRANCH, REMOTE, str(WORK_REPO)])
    if cloned.returncode != 0 or not has_circuit_scripts(WORK_REPO):
        raise SystemExit("Could not clone %s into %s." % (BRANCH, WORK_REPO))
    git(["git", "rev-parse", "--short", "HEAD"], WORK_REPO)
    return WORK_REPO


def lerobot_dataset_dest() -> Path:
    return Path(os.environ.get("HF_HOME", "/content/hf_home")) / "lerobot" / DATASET_REPO


def find_lerobot_dataset():
    hits = [lerobot_dataset_dest()]
    if os.environ.get("LEROBOT_HOME"):
        hits.append(Path(os.environ["LEROBOT_HOME"]) / DATASET_REPO)
    hits.append(LOCAL_REPO / "data" / "lerobot" / DATASET_REPO)
    drive = Path("/content/drive/MyDrive/groot-run-shared-programmer908")
    hits.append(drive / "hf_home" / "lerobot" / DATASET_REPO)
    hits.append(drive / "lerobot" / DATASET_REPO)
    hf_home = Path(os.environ.get("HF_HOME", "/content/hf_home"))
    hub_roots = [
        Path(os.environ.get("HF_HUB_CACHE", str(hf_home / "hub"))),
        hf_home / "hub",
        hf_home / "lerobot" / "hub",
    ]
    if os.environ.get("LEROBOT_HOME"):
        hub_roots.append(Path(os.environ["LEROBOT_HOME"]) / "hub")
    for hub in hub_roots:
        snap_root = hub / "datasets--HuggingFaceVLA--libero" / "snapshots"
        if snap_root.is_dir():
            hits.extend(sorted(p for p in snap_root.iterdir() if p.is_dir()))
    for hit in hits:
        if (hit / "meta" / "info.json").is_file():
            return hit
    return None


def link_lerobot_dataset(found, dest):
    if (dest / "meta" / "info.json").is_file():
        print("using local LeRobot dataset", dest, flush=True)
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.is_symlink():
        dest.unlink()
    elif dest.exists() and dest.is_dir() and not any(dest.iterdir()):
        dest.rmdir()
    if dest.exists():
        raise SystemExit(
            "%s exists and is not a HuggingFaceVLA/libero dataset. Move or rename it and rerun." % dest
        )
    dest.symlink_to(found)
    print("linked LeRobot dataset", found, "->", dest, flush=True)
    return dest


def ensure_lerobot_libero(run_env):
    dest = lerobot_dataset_dest()
    found = find_lerobot_dataset()
    if found is not None:
        return link_lerobot_dataset(found, dest)
    print(
        "HuggingFaceVLA/libero is missing from the offline cache; downloading episodes %s" % EPISODES,
        flush=True,
    )
    download_env = dict(run_env)
    for key in ("HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE"):
        download_env.pop(key, None)
    download_env["OCCLUSION_EPISODES"] = EPISODES
    download_env.setdefault("HF_HOME", str(Path(os.environ.get("HF_HOME", "/content/hf_home"))))
    download_code = (
        "from lerobot.datasets.lerobot_dataset import LeRobotDataset\n"
        "import os\n"
        "episodes = [int(item) for item in os.environ['OCCLUSION_EPISODES'].split(',') if item]\n"
        "ds = LeRobotDataset('HuggingFaceVLA/libero', episodes=episodes)\n"
        "print('dataset root', ds.root, 'frames', len(ds), flush=True)\n"
    )
    stream([str(PYTHON), "-u", "-c", download_code], Path("/content"), download_env)
    found = find_lerobot_dataset()
    if found is None:
        raise SystemExit(
            "Could not materialize HuggingFaceVLA/libero under %s or the Hub snapshot cache." % dest
        )
    return link_lerobot_dataset(found, dest)


if not LOCAL_REPO.is_dir():
    print("LOCAL_REPO %s is missing; caches still come from the venv and Drive." % LOCAL_REPO, flush=True)

repo = ensure_circuit_repo()
print("circuit repo", repo, flush=True)

SCRIPT = r'''
"""Occlusion contrast for one frozen action-expert transcoder feature.

The transcoders, the feature report, and the circuit tracer already exist.
This script only replays LIBERO demonstrations, scores one grasp feature on
held_visible / held_hidden / gone, and, when that feature passes, asks the
existing tracer for a compact circuit and compares one ablation against a
random feature of similar firing rate.

Pi0.5 is never allowed to move the arm. The simulator is set to saved
demonstration states, and the policy is only a probe.
"""

from __future__ import annotations

import argparse
import json
import os
import re
import subprocess
import sys
from pathlib import Path
from typing import Any, Literal

import h5py
import numpy as np
import torch

FAIL_SENTENCE = (
    "no action-expert feature keeps the object when it is hidden and drops it when it is gone."
)
FIRING_FLOOR = 1e-4
GONE_FRACTION = 0.25
GRIPPER_R2 = 0.5
RESIDUAL_FRACTION = 0.25
ARM_ATOL = 1e-6
DEPTH_MARGIN = 0.02
GRIP_RADIUS = 0.03
Label = Literal["held_visible", "held_hidden", "not_held"]


def user_feature_id(layer: int, timestep: float, feature: int) -> str:
    return f"L{int(layer)}/tau{float(timestep):.4g}.F{int(feature)}"


def tracer_feature_key(layer: int, timestep: float, feature: int) -> str:
    return f"L{int(layer):02d}:tau{float(timestep):.4g}:F{int(feature)}"


def task_text(value: Any) -> str:
    if isinstance(value, (list, tuple)):
        return " ".join(task_text(item) for item in value)
    if value is None:
        return ""
    return str(value)


def same_language(left: str, right: str) -> bool:
    def norm(text: str) -> str:
        return " ".join(re.findall(r"[a-z0-9]+", text.lower()))

    a, b = norm(left), norm(right)
    if not a or not b:
        return False
    return a == b or a in b or b in a


def demo_index(rows: list[dict], episode_index: int) -> int | None:
    """Rank of one dataset episode among episodes that share its task text."""
    episode_index = int(episode_index)
    task = None
    for row in rows:
        if int(row["episode_index"]) == episode_index:
            task = task_text(row.get("task"))
            break
    if task is None:
        return None
    same = sorted(int(row["episode_index"]) for row in rows if task_text(row.get("task")) == task)
    return same.index(episode_index)


def episode_rows_from_meta(episodes: Any) -> list[dict]:
    """Read episode index and task text from a LeRobot metadata object."""
    if isinstance(episodes, dict):
        tasks = episodes.get("tasks", episodes.get("task"))
        length = len(episodes.get("dataset_from_index", tasks or []))
        rows = []
        for index in range(length):
            task = "" if tasks is None else tasks[index]
            rows.append({"episode_index": index, "task": task_text(task)})
        return rows
    if not hasattr(episodes, "__getitem__") or not hasattr(episodes, "__len__"):
        return []
    try:
        length = len(episodes)
    except TypeError:
        return []
    rows = []
    for index in range(length):
        item = episodes[index]
        if not isinstance(item, dict):
            continue
        rows.append(
            {
                "episode_index": int(item.get("episode_index", index)),
                "task": task_text(item.get("tasks", item.get("task", ""))),
            }
        )
    return rows


def label_from_measurement(
    contact_known: bool,
    touching: bool,
    angle_known: bool,
    hidden: bool,
) -> Label | None:
    """Drop a frame when contact or the held-frame camera angle cannot be measured."""
    if not contact_known:
        return None
    if not touching:
        return "not_held"
    if not angle_known:
        return None
    if hidden:
        return "held_hidden"
    return "held_visible"


def arm_joints(qpos: np.ndarray, span: tuple[int, int] | None) -> np.ndarray:
    values = np.asarray(qpos, dtype=np.float64).reshape(-1)
    if span is None:
        return values.copy()
    start, stop = span
    return np.concatenate([values[:start], values[stop:]])


def arm_unchanged(before: np.ndarray, after: np.ndarray, atol: float = ARM_ATOL) -> bool:
    before = np.asarray(before, dtype=np.float64)
    after = np.asarray(after, dtype=np.float64)
    if before.shape != after.shape:
        return False
    return float(np.max(np.abs(before - after))) <= atol


def choose_grasp(rows: list[dict]) -> dict | None:
    """Pick the report feature whose top frames are most often a grasp.

    A grasp frame is one where a finger is touching the object. Ties go to the
    feature the report already ranked higher.
    """
    usable = [row for row in rows if int(row["top_count"]) > 0 and int(row["held_top"]) > 0]
    if not usable:
        return None
    usable.sort(key=lambda row: (-int(row["held_top"]) / int(row["top_count"]), int(row["rank"])))
    return usable[0]


def _fit_line(x: np.ndarray, y: np.ndarray) -> tuple[float, float, float]:
    x = np.asarray(x, dtype=np.float64).reshape(-1)
    y = np.asarray(y, dtype=np.float64).reshape(-1)
    if x.size < 2 or np.allclose(x, x[0]):
        mean = float(y.mean()) if y.size else 0.0
        return mean, 0.0, 0.0
    design = np.column_stack([np.ones(x.size), x])
    coef, _, _, _ = np.linalg.lstsq(design, y, rcond=None)
    predicted = design @ coef
    total = float(np.sum((y - y.mean()) ** 2))
    residual = float(np.sum((y - predicted) ** 2))
    r2 = 0.0 if total <= 1e-12 else 1.0 - residual / total
    return float(coef[0]), float(coef[1]), float(r2)


def gripper_correlation_failed(scores: np.ndarray, grippers: np.ndarray, held: np.ndarray) -> bool:
    """True when gripper opening alone accounts for the held-versus-open firing."""
    scores = np.asarray(scores, dtype=np.float64).reshape(-1)
    grippers = np.asarray(grippers, dtype=np.float64).reshape(-1)
    held = np.asarray(held, dtype=bool).reshape(-1)
    if scores.size < 4 or int(held.sum()) == 0 or int((~held).sum()) == 0:
        return False
    intercept, slope, r2 = _fit_line(grippers, scores)
    residual = scores - (intercept + slope * grippers)
    gap = abs(float(scores[held].mean() - scores[~held].mean()))
    residual_gap = abs(float(residual[held].mean() - residual[~held].mean()))
    if gap < 1e-8:
        return r2 >= GRIPPER_R2
    return r2 >= GRIPPER_R2 and residual_gap <= RESIDUAL_FRACTION * gap


def occlusion_passes(
    held_visible: float | None,
    held_hidden: float | None,
    gone: float | None,
    gripper_failed: bool,
) -> bool:
    if gripper_failed:
        return False
    rates = (held_visible, held_hidden, gone)
    if any(rate is None or not np.isfinite(rate) for rate in rates):
        return False
    assert held_visible is not None and held_hidden is not None and gone is not None
    if held_visible <= FIRING_FLOOR or held_hidden <= FIRING_FLOOR:
        return False
    return gone <= GONE_FRACTION * min(held_visible, held_hidden)


def similar_feature(frequencies: np.ndarray, target: int, seed: int) -> int:
    """Pick one other feature whose firing rate is close to the target."""
    freq = np.asarray(frequencies, dtype=np.float64).reshape(-1)
    if freq.size < 2:
        raise ValueError("Need at least two features to choose a random control.")
    distance = np.abs(freq - freq[int(target)])
    distance[int(target)] = np.inf
    order = np.argsort(distance, kind="mergesort")
    closest = float(distance[int(order[0])])
    pool = [int(index) for index in order if abs(float(distance[int(index)]) - closest) <= 1e-12]
    if not pool:
        raise ValueError("No control feature is available.")
    return int(np.random.default_rng(seed).choice(np.asarray(pool, dtype=np.int64)))


def action_rmse(source: np.ndarray, edited: np.ndarray) -> float:
    source = np.asarray(source, dtype=np.float64)
    edited = np.asarray(edited, dtype=np.float64)
    return float(np.sqrt(np.mean((source - edited) ** 2)))


def real_feature_counts(real_delta: float, random_delta: float) -> bool:
    return float(real_delta) > float(random_delta)


def circuit_path(graph: dict) -> str | None:
    """Strongest incoming edge, walked back from the traced target."""
    nodes = {str(node["node_key"]): node for node in graph.get("nodes", [])}
    if not nodes:
        return None
    target = str((graph.get("config") or {}).get("target") or "")
    if target not in nodes:
        target = max(nodes, key=lambda key: int(nodes[key].get("depth") or 0))
    incoming: dict[str, list[dict]] = {}
    for edge in graph.get("edges", []):
        incoming.setdefault(str(edge["target_key"]), []).append(edge)
    chain = [target]
    seen = {target}
    while chain[-1] in incoming and len(chain) < 8:
        options = incoming[chain[-1]]
        best = max(
            options,
            key=lambda edge: float(
                edge.get("edge_score")
                or edge.get("mean_abs_contribution")
                or edge.get("edge_influence")
                or 0.0
            ),
        )
        source = str(best["source_key"])
        if source in seen:
            break
        chain.append(source)
        seen.add(source)
    chain.reverse()
    return " -> ".join(_node_user_id(key, nodes) for key in chain)


def _node_user_id(key: str, nodes: dict[str, dict]) -> str:
    node = nodes.get(key)
    if node is not None and "layer" in node and "feature" in node:
        return user_feature_id(int(node["layer"]), float(node["timestep"]), int(node["feature"]))
    parts = key.split(":")
    if len(parts) == 3 and parts[0].startswith("L") and parts[1].startswith("tau") and parts[2].startswith("F"):
        return user_feature_id(int(parts[0][1:]), float(parts[1][3:]), int(parts[2][1:]))
    return key


def mean_or_none(values: list[float]) -> float | None:
    if not values:
        return None
    return float(np.mean(np.asarray(values, dtype=np.float64)))


def verdict_document(
    *,
    passed: bool,
    feature_id: str | None,
    firing_rates: dict[str, float | None],
    gripper_correlation_failed_flag: bool | None,
    circuit_path_text: str | None,
    real_delta: float | None,
    random_delta: float | None,
    frame_counts: dict[str, int] | None = None,
) -> dict:
    real_counts = None
    delta = None
    if real_delta is not None and random_delta is not None:
        real_counts = real_feature_counts(real_delta, random_delta)
        delta = {"real": real_delta, "random": random_delta, "real_counts": real_counts}
    if not passed:
        text = FAIL_SENTENCE
        circuit_path_text = None
        delta = None
    elif real_counts:
        text = (
            f"{feature_id} fires while the object is hidden and drops when it is gone, "
            "and ablating it moves the action more than a random feature."
        )
    else:
        text = (
            f"{feature_id} fires while the object is hidden and drops when it is gone. "
            "Ablating it does not move the action more than a random feature."
        )
    document = {
        "verdict": text,
        "feature_id": feature_id,
        "firing_rates": {
            "held_visible": firing_rates.get("held_visible"),
            "held_hidden": firing_rates.get("held_hidden"),
            "gone": firing_rates.get("gone"),
        },
        "gripper_correlation_failed": gripper_correlation_failed_flag,
        "circuit_path": circuit_path_text,
        "real_versus_random_action_delta": delta,
    }
    if frame_counts is not None:
        document["frame_counts"] = frame_counts
    return document


def spread_frames(frames: list[dict], limit: int) -> list[dict]:
    """Take frames round-robin across episodes, up to limit."""
    if limit <= 0:
        return []
    buckets: dict[int, list[dict]] = {}
    for frame in frames:
        buckets.setdefault(int(frame["episode"]), []).append(frame)
    ordered = [buckets[key] for key in sorted(buckets)]
    chosen: list[dict] = []
    while ordered and len(chosen) < limit:
        nxt: list[list[dict]] = []
        for bucket in ordered:
            if bucket and len(chosen) < limit:
                chosen.append(bucket.pop(0))
            if bucket:
                nxt.append(bucket)
        ordered = nxt
    return chosen


def top_examples_for_candidate(topk: dict, observations: dict[int, dict], candidate: dict, limit: int) -> list[dict]:
    name = candidate["layer_name"]
    timestep = candidate["timestep_key"]
    feature = int(candidate["feature"])
    store = topk["topk"][name][timestep]
    scores = store["scores"][feature]
    obs_ids = store["observation_ids"][feature]
    rows = []
    width = min(limit, int(scores.shape[0]))
    for rank in range(width):
        score = float(scores[rank])
        if not np.isfinite(score):
            continue
        observation_id = int(obs_ids[rank])
        if observation_id < 0:
            continue
        observation = dict(observations.get(observation_id, {"observation_id": observation_id}))
        observation["activation"] = score
        rows.append(observation)
    return rows


def main() -> None:
    # LIBERO, MuJoCo, and LeRobot are imported here on purpose. The unit tests
    # import this module without a simulator or a policy install.
    # LeRobot and the action-expert package are imported here so unit tests can
    # load this module without that install. LIBERO is imported the same way.
    from lerobot.datasets.factory import make_dataset
    from lerobot.policies import make_policy

    from collect_pi05_transcoder_features import _freeze_policy, _load_transcoders
    from pi05_mi.patch_pi05 import Pi05TranscoderContext, install_pi05_action_expert_wrappers
    from train_pi05_transcoders import (
        _configure_train_config,
        _make_preprocessor,
        patch_pi05_checkpoint_key_compat,
        patch_transformers_causal_mask_compat,
        resolve_device,
        resolve_policy_dtype,
    )

    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--checkpoint", type=Path, required=True)
    parser.add_argument("--feature-dir", type=Path, required=True)
    parser.add_argument("--output-dir", type=Path, required=True)
    parser.add_argument("--suite", default="libero_spatial")
    parser.add_argument("--seed", type=int, default=0)
    parser.add_argument("--policy-path", default="lerobot/pi05_libero_finetuned")
    args = parser.parse_args()

    if not torch.cuda.is_available():
        raise SystemExit("This contrast needs the Colab GPU.")

    feature_dir = args.feature_dir
    output_dir = args.output_dir
    output_dir.mkdir(parents=True, exist_ok=True)
    candidates = _candidate_rows(json.loads((feature_dir / "feature_candidates.json").read_text()))
    observations = _read_jsonl(feature_dir / "observations.jsonl")
    topk = torch.load(feature_dir / "feature_topk.pt", map_location="cpu", weights_only=False)
    stats = torch.load(feature_dir / "feature_stats.pt", map_location="cpu", weights_only=False)

    episode_rows = _episode_rows(args, resolve_device, resolve_policy_dtype, _configure_train_config, make_dataset)
    labeled_candidates = []
    with _suite_env(args.suite) as environment:
        for candidate in candidates:
            examples = top_examples_for_candidate(topk, observations, candidate, limit=5)
            held_top = 0
            measured = 0
            task_hits: dict[str, int] = {}
            demo_ids: list[int] = []
            for example in examples:
                located = _locate_example(example, episode_rows, environment["suite"])
                if located is None:
                    continue
                task_index, demo_index_value, frame_index, sentence = located
                label = _label_demo_frame(environment, task_index, demo_index_value, frame_index)
                if label is None:
                    continue
                measured += 1
                if label.startswith("held_"):
                    held_top += 1
                    task_hits[sentence] = task_hits.get(sentence, 0) + 1
                    demo_ids.append(demo_index_value)
            labeled_candidates.append(
                {
                    **candidate,
                    "held_top": held_top,
                    "top_count": measured,
                    "task_hits": task_hits,
                    "demo_ids": demo_ids,
                }
            )
            print(
                f"report L{candidate['layer']}/tau{float(candidate['timestep']):.4g}.F{candidate['feature']} "
                f"rank {candidate['rank']}: grasp frames {held_top}/{measured}",
                flush=True,
            )
        chosen = choose_grasp(labeled_candidates)
        if chosen is None:
            document = verdict_document(
                passed=False,
                feature_id=None,
                firing_rates={"held_visible": None, "held_hidden": None, "gone": None},
                gripper_correlation_failed_flag=None,
                circuit_path_text=None,
                real_delta=None,
                random_delta=None,
                frame_counts={"held_visible": 0, "held_hidden": 0, "not_held": 0, "gone": 0, "dropped": 0},
            )
            _write_verdict(output_dir, document)
            print(document["verdict"], flush=True)
            return

        feature_id = user_feature_id(int(chosen["layer"]), float(chosen["timestep"]), int(chosen["feature"]))
        print(f"chosen grasp feature {feature_id}", flush=True)
        sentence, task_index = _chosen_task(chosen, environment["suite"])
        demos = _chosen_demos(chosen)
        print(f"replaying {args.suite} task {task_index}: {sentence}", flush=True)
        print(f"demos {demos}", flush=True)
        catalog = _catalog_frames(environment, task_index, demos, sentence)
        print(
            "labels "
            f"held_visible={len(catalog['held_visible'])} "
            f"held_hidden={len(catalog['held_hidden'])} "
            f"not_held={len(catalog['not_held'])} "
            f"dropped={catalog['dropped']}",
            flush=True,
        )
        selected = {
            "held_visible": spread_frames(catalog["held_visible"], 8),
            "held_hidden": spread_frames(catalog["held_hidden"], 8),
            "not_held": spread_frames(catalog["not_held"], 8),
        }
        policy, preprocessor, context, ablate = _load_probe(
            args,
            make_policy,
            make_dataset,
            _freeze_policy,
            _load_transcoders,
            Pi05TranscoderContext,
            install_pi05_action_expert_wrappers,
            _configure_train_config,
            _make_preprocessor,
            patch_transformers_causal_mask_compat,
            patch_pi05_checkpoint_key_compat,
            resolve_device,
            resolve_policy_dtype,
            str(chosen["layer_name"]),
        )
        scored = _score_selected(
            environment,
            policy,
            preprocessor,
            context,
            ablate,
            selected,
            sentence,
            task_index=task_index,
            layer=int(chosen["layer"]),
            timestep=float(chosen["timestep"]),
            feature=int(chosen["feature"]),
            seed=args.seed,
        )
        rates = {
            "held_visible": mean_or_none(scored["held_visible"]),
            "held_hidden": mean_or_none(scored["held_hidden"]),
            "gone": mean_or_none(scored["gone"]),
        }
        gripper_failed = gripper_correlation_failed(scored["grip_scores"], scored["grip_values"], scored["grip_held"])
        passed = occlusion_passes(rates["held_visible"], rates["held_hidden"], rates["gone"], gripper_failed)
        counts = {
            "held_visible": len(scored["held_visible"]),
            "held_hidden": len(scored["held_hidden"]),
            "not_held": len(scored["not_held"]),
            "gone": len(scored["gone"]),
            "dropped": int(catalog["dropped"]),
        }
        print(
            f"firing held_visible={rates['held_visible']} held_hidden={rates['held_hidden']} "
            f"gone={rates['gone']} gripper_correlation_failed={gripper_failed}",
            flush=True,
        )
        if not passed:
            document = verdict_document(
                passed=False,
                feature_id=feature_id,
                firing_rates=rates,
                gripper_correlation_failed_flag=gripper_failed,
                circuit_path_text=None,
                real_delta=None,
                random_delta=None,
                frame_counts=counts,
            )
            _write_verdict(output_dir, document)
            print(document["verdict"], flush=True)
            return

        _release_policy(policy)
        path_text = _trace_circuit(args, chosen, feature_dir, output_dir)
        policy, preprocessor, context, ablate = _load_probe(
            args,
            make_policy,
            make_dataset,
            _freeze_policy,
            _load_transcoders,
            Pi05TranscoderContext,
            install_pi05_action_expert_wrappers,
            _configure_train_config,
            _make_preprocessor,
            patch_transformers_causal_mask_compat,
            patch_pi05_checkpoint_key_compat,
            resolve_device,
            resolve_policy_dtype,
            str(chosen["layer_name"]),
        )
        frequency = stats["stats"][chosen["layer_name"]][chosen["timestep_key"]]["firing_frequency"]
        control = similar_feature(frequency.detach().float().cpu().numpy(), int(chosen["feature"]), args.seed)
        print(f"random control feature {control}", flush=True)
        real_delta, random_delta = _ablate(
            environment,
            policy,
            preprocessor,
            context,
            ablate,
            selected["held_hidden"],
            sentence,
            task_index=task_index,
            feature=int(chosen["feature"]),
            control=control,
            seed=args.seed,
        )
        document = verdict_document(
            passed=True,
            feature_id=feature_id,
            firing_rates=rates,
            gripper_correlation_failed_flag=gripper_failed,
            circuit_path_text=path_text,
            real_delta=real_delta,
            random_delta=random_delta,
            frame_counts=counts,
        )
        _write_verdict(output_dir, document)
        print(json.dumps(document, indent=2, sort_keys=True), flush=True)


def _candidate_rows(payload: Any) -> list[dict]:
    if isinstance(payload, dict):
        return list(payload["candidates"])
    return list(payload)


def _read_jsonl(path: Path) -> dict[int, dict]:
    rows: dict[int, dict] = {}
    with path.open() as handle:
        for line in handle:
            row = json.loads(line)
            rows[int(row["observation_id"])] = row
    return rows


def _write_verdict(output_dir: Path, document: dict) -> None:
    path = output_dir / "verdict.json"
    path.write_text(json.dumps(document, indent=2, sort_keys=True) + "\n")
    print(f"verdict {path}", flush=True)


def _episode_rows(args, resolve_device, resolve_policy_dtype, configure, make_dataset) -> list[dict]:
    try:
        device = resolve_device("cpu")
        namespace = _namespace(args, device, resolve_policy_dtype("auto", device))
        cfg = configure(namespace, episodes=None)
        dataset = make_dataset(cfg)
        rows = episode_rows_from_meta(dataset.meta.episodes)
        print(f"dataset episodes with task text: {len(rows)}", flush=True)
        return rows
    except Exception as exc:
        print(f"dataset episode map unavailable ({exc}); demo index will follow episode_index", flush=True)
        return []


def _namespace(args, device, dtype: str):
    return argparse.Namespace(
        policy_path=args.policy_path,
        batch_size=1,
        num_workers=0,
        local_files_only=os.environ.get("HF_HUB_OFFLINE") == "1",
        resolved_device=device,
        resolved_policy_dtype=dtype,
    )


def _suite_env(suite_name: str):
    return _Suite(suite_name)


class _Suite:
    """One LIBERO suite kept open while frames are labeled and probed."""

    def __init__(self, suite_name: str):
        self.suite_name = suite_name
        self.suite = None
        self.env = None
        self.task_index = None

    def __enter__(self):
        # LIBERO is imported on use so tests do not require the simulator.
        from libero.libero import benchmark, get_libero_path
        from libero.libero.envs import OffScreenRenderEnv

        self._benchmark = benchmark
        self._get_libero_path = get_libero_path
        self._env_cls = OffScreenRenderEnv
        suite_cls = benchmark.get_benchmark_dict()[self.suite_name]
        self.suite = suite_cls()
        return {"suite": self, "name": self.suite_name}

    def __exit__(self, *_exc):
        self.close()

    def close(self) -> None:
        if self.env is not None:
            self.env.close()
            self.env = None

    def task_count(self) -> int:
        count = self.suite.n_tasks
        return int(count() if callable(count) else count)

    def task(self, index: int):
        return self.suite.get_task(int(index))

    def ensure(self, task_index: int):
        if self.env is not None and self.task_index == int(task_index):
            return self.env
        self.close()
        task = self.task(task_index)
        bddl = os.path.join(self._get_libero_path("bddl_files"), task.problem_folder, task.bddl_file)
        self.env = self._env_cls(bddl_file_name=bddl, camera_heights=256, camera_widths=256)
        self.task_index = int(task_index)
        return self.env


def _locate_example(example: dict, episode_rows: list[dict], suite: _Suite):
    if "episode_index" not in example or "frame_index" not in example:
        return None
    episode = int(example["episode_index"])
    frame = int(example["frame_index"])
    language = task_text(example.get("task"))
    if episode_rows:
        mapped = demo_index(episode_rows, episode)
        demo = episode if mapped is None else mapped
    else:
        demo = episode
    match = _match_task(suite, language)
    if match is None:
        return None
    task_index, sentence = match
    return task_index, demo, frame, sentence


def _match_task(suite: _Suite, language: str):
    for index in range(suite.task_count()):
        task = suite.task(index)
        if same_language(task.language, language):
            return index, task.language
    return None


def _label_demo_frame(environment: dict, task_index: int, demo_index_value: int, frame_index: int) -> str | None:
    suite: _Suite = environment["suite"]
    try:
        _actions, states = _load_demo(suite, task_index, demo_index_value)
    except (IndexError, FileNotFoundError, KeyError, OSError):
        return None
    if frame_index < 0 or frame_index >= len(states):
        return None
    env = suite.ensure(task_index)
    raw = _observe_state(env, states[frame_index])
    return _label_raw(env.sim, suite.task(task_index).language, raw)


def _chosen_task(chosen: dict, suite: _Suite) -> tuple[str, int]:
    hits: dict[str, int] = chosen.get("task_hits") or {}
    if hits:
        sentence = max(hits, key=hits.get)
        match = _match_task(suite, sentence)
        if match is not None:
            return match[1], match[0]
    task = suite.task(0)
    return task.language, 0


def _chosen_demos(chosen: dict) -> list[int]:
    demos = []
    for demo in chosen.get("demo_ids") or []:
        if int(demo) not in demos:
            demos.append(int(demo))
    if not demos:
        demos = [0, 1, 2]
    return demos[:3]


def _catalog_frames(environment: dict, task_index: int, demos: list[int], sentence: str) -> dict:
    suite: _Suite = environment["suite"]
    stride = int(os.environ.get("OCCLUSION_STRIDE", "8"))
    max_steps = int(os.environ.get("OCCLUSION_MAX_STEPS", "80"))
    catalog: dict[str, Any] = {"held_visible": [], "held_hidden": [], "not_held": [], "dropped": 0}
    for demo in demos:
        try:
            _actions, states = _load_demo(suite, task_index, demo)
        except (IndexError, FileNotFoundError, KeyError, OSError) as exc:
            print(f"skip demo {demo}: {exc}", flush=True)
            continue
        env = suite.ensure(task_index)
        for frame_index in range(0, min(len(states), max_steps), stride):
            raw = _observe_state(env, states[frame_index])
            label = _label_raw(env.sim, sentence, raw)
            if label is None:
                catalog["dropped"] += 1
                continue
            catalog[label].append(
                {
                    "episode": int(demo),
                    "step": int(frame_index),
                    "label": label,
                    "state": np.asarray(states[frame_index], dtype=np.float64),
                }
            )
    return catalog


def _load_demo(suite: _Suite, task_index: int, demo_index_value: int):
    task = suite.task(task_index)
    path = _find_demo_file(task)
    with h5py.File(path, "r") as handle:
        names = sorted(handle["data"].keys(), key=lambda name: int(re.search(r"(\d+)$", name).group(1)))
        if demo_index_value >= len(names):
            raise IndexError(f"{path} has {len(names)} demos, asked for {demo_index_value}")
        group = handle["data"][names[demo_index_value]]
        actions = np.asarray(group["actions"])
        states = np.asarray(group["states"])
    if len(states) == len(actions) + 1:
        states = states[:-1]
    return actions, states


def _find_demo_file(task) -> Path:
    name = task.name
    folder = task.problem_folder
    roots = []
    if os.environ.get("LIBERO_DATASET_DIR"):
        roots.append(Path(os.environ["LIBERO_DATASET_DIR"]))
    # LIBERO dataset root is resolved only when a demonstration is opened.
    from libero.libero import get_libero_path

    roots.append(Path(get_libero_path("datasets")))
    exact_names = [f"{name}_demo.hdf5", f"{name}.hdf5"]
    tried = []
    for root in roots:
        for exact in exact_names:
            candidate = root / folder / exact
            tried.append(candidate)
            if candidate.is_file():
                return candidate
        if root.is_dir():
            for hit in sorted(root.rglob("*.hdf5")):
                if name in hit.stem:
                    return hit
    raise FileNotFoundError("No LIBERO demonstration file found:\n" + "\n".join(str(path) for path in tried[:8]))


def _observe_state(env, state: np.ndarray) -> dict:
    state = np.asarray(state, dtype=np.float64)
    if hasattr(env, "set_init_state"):
        return env.set_init_state(state)
    env.reset()
    env.sim.set_state_from_flattened(state)
    env.sim.forward()
    inner = env.env if hasattr(env, "env") else env
    return inner._get_observations()


def _label_raw(sim, sentence: str, raw: dict) -> str | None:
    del raw
    bodies = _candidate_bodies(sim, sentence)
    fingers = _finger_geoms(sim)
    touching, body = _contact_body(sim, fingers, bodies)
    if touching is None:
        return label_from_measurement(False, False, False, False)
    if not touching:
        return label_from_measurement(True, False, False, False)
    geoms = bodies.get(body) if body is not None else None
    measured = _occlusion(sim, geoms or [], _camera_id(sim), _site_id(sim))
    if measured is None:
        return label_from_measurement(True, True, False, False)
    hidden, _depth, _ratio = measured
    return label_from_measurement(True, True, True, hidden)


def _candidate_bodies(sim, sentence: str) -> dict[int, list[int]]:
    # MuJoCo is imported on use so tests do not require the simulator.
    import mujoco

    model = sim.model
    free = int(mujoco.mjtJoint.mjJNT_FREE)
    movable = set()
    for body in range(int(model.nbody)):
        start = int(model.body_jntadr[body])
        count = int(model.body_jntnum[body])
        for joint in range(start, start + count):
            if joint >= 0 and int(model.jnt_type[joint]) == free:
                movable.add(body)
                break
    by_body: dict[int, list[int]] = {}
    for geom in range(int(model.ngeom)):
        body = int(model.geom_bodyid[geom])
        if body in movable:
            by_body.setdefault(body, []).append(geom)
    words = _target_words(sentence)
    named = {}
    for body, geoms in by_body.items():
        blob = " ".join((model.geom_id2name(geom) or "") for geom in geoms).lower()
        blob += " " + (model.body_id2name(body) or "").lower()
        if any(word in blob for word in words):
            named[body] = geoms
    return named or by_body


def _target_words(sentence: str) -> list[str]:
    low = sentence.lower()
    verbs = ("pick up", "pick", "grasp", "take", "put", "place", "move", "open", "close", "turn")
    rels = (" between ", " next to ", " on top of ", " on the ", " in the ", " and ", " near ", " into ", " onto ")
    segment = low
    for verb in verbs:
        at = segment.find(verb)
        if at >= 0:
            segment = segment[at + len(verb) :]
            break
    cut = len(segment)
    for rel in rels:
        at = segment.find(rel)
        if 0 <= at < cut:
            cut = at
    stop = {"the", "a", "an", "and", "that", "with", "your", "from", "into", "onto"}
    words = [word for word in re.findall(r"[a-z]+", segment[:cut]) if len(word) > 3 and word not in stop]
    return words or [word for word in re.findall(r"[a-z]+", low) if len(word) > 3 and word not in stop]


def _finger_geoms(sim) -> list[int]:
    model = sim.model
    return [index for index in range(int(model.ngeom)) if "finger" in (model.geom_id2name(index) or "").lower()]


def _contact_body(sim, fingers: list[int], bodies: dict[int, list[int]]):
    if not fingers or not bodies:
        return None, None
    geom_to_body = {geom: body for body, geoms in bodies.items() for geom in geoms}
    finger_set = set(fingers)
    data = sim.data
    for index in range(int(data.ncon)):
        contact = data.contact[index]
        left, right = int(contact.geom1), int(contact.geom2)
        if left in finger_set and right in geom_to_body:
            return True, geom_to_body[right]
        if right in finger_set and left in geom_to_body:
            return True, geom_to_body[left]
    return False, None


def _site_id(sim) -> int | None:
    for name in ("gripper0_grip_site", "gripper0_eef", "robot0_eef", "grip_site"):
        try:
            return int(sim.model.site_name2id(name))
        except Exception:
            continue
    return None


def _camera_id(sim) -> int:
    model = sim.model
    for index in range(int(model.ncam)):
        if (model.camera_id2name(index) or "") == "agentview":
            return index
    return 0


def _occlusion(sim, geoms: list[int], cam: int, site: int | None):
    if not geoms or site is None:
        return None
    data = sim.data
    camera = np.asarray(data.cam_xpos[cam], dtype=np.float64)
    obj = np.mean([np.asarray(data.geom_xpos[geom], dtype=np.float64) for geom in geoms], axis=0)
    grip = np.asarray(data.site_xpos[site], dtype=np.float64)
    to_obj = obj - camera
    to_grip = grip - camera
    obj_dist = float(np.linalg.norm(to_obj))
    grip_dist = float(np.linalg.norm(to_grip))
    if obj_dist < 1e-6 or grip_dist < 1e-6:
        return None
    cosine = float(np.clip(np.dot(to_obj, to_grip) / (obj_dist * grip_dist), -1.0, 1.0))
    angle = float(np.arccos(cosine))
    grip_angle = float(np.arctan2(GRIP_RADIUS, grip_dist))
    depth_gap = obj_dist - grip_dist
    hidden = depth_gap > DEPTH_MARGIN and angle < grip_angle
    return hidden, depth_gap, (angle / grip_angle if grip_angle > 1e-9 else float("inf"))


def _object_span(sim, body: int) -> tuple[int, int] | None:
    # MuJoCo is imported on use so tests do not require the simulator.
    import mujoco

    model = sim.model
    free = int(mujoco.mjtJoint.mjJNT_FREE)
    start = int(model.body_jntadr[body])
    count = int(model.body_jntnum[body])
    for joint in range(start, start + count):
        if joint >= 0 and int(model.jnt_type[joint]) == free:
            address = int(model.jnt_qposadr[joint])
            return address, address + 7
    return None


def _move_object_away(sim, body: int):
    span = _object_span(sim, body)
    if span is None:
        return None, None
    saved = np.array(sim.data.qpos, copy=True)
    start, stop = span
    sim.data.qpos[start : start + 3] = np.array([5.0, 5.0, -1.0])
    sim.data.qpos[start + 3 : stop] = np.array([1.0, 0.0, 0.0, 0.0])
    sim.data.qvel[:] = 0
    sim.forward()
    return saved, span


def _restore_qpos(sim, saved: np.ndarray) -> None:
    sim.data.qpos[:] = saved
    sim.data.qvel[:] = 0
    sim.forward()


def _render_after_edit(env) -> dict:
    inner = env.env if hasattr(env, "env") else env
    try:
        return inner._get_observations(force_update=True)
    except TypeError:
        if hasattr(inner, "_update_observables"):
            inner._update_observables(force=True)
        return inner._get_observations()


def _load_probe(
    args,
    make_policy,
    make_dataset,
    freeze_policy,
    load_transcoders,
    context_cls,
    install,
    configure,
    make_preprocessor,
    patch_mask,
    patch_keys,
    resolve_device,
    resolve_policy_dtype,
    layer_name: str,
):
    patch_mask()
    patch_keys()
    device = resolve_device("cuda")
    dtype = resolve_policy_dtype("auto", device)
    namespace = _namespace(args, device, dtype)
    cfg = configure(namespace, episodes="0")
    dataset = make_dataset(cfg)
    preprocessor = make_preprocessor(cfg, dataset, args.policy_path)
    cfg.policy.device = str(device)
    cfg.policy.dtype = dtype
    cfg.policy.pretrained_path = Path(args.policy_path)
    cfg.policy.compile_model = False
    cfg.policy.gradient_checkpointing = False
    print("loading frozen Pi0.5 policy", flush=True)
    policy = make_policy(cfg.policy, ds_meta=dataset.meta, rename_map=cfg.rename_map)
    freeze_policy(policy)
    transcoders = load_transcoders(args.checkpoint, device=device)
    context = context_cls(
        mode="probe",
        capture_records=False,
        capture_latents=True,
        latent_top_k=0,
        save_full_latents=False,
        store_latent_summaries=False,
    )
    _context, _names = install(policy, context=context, transcoders=transcoders, mode="probe")
    ablate = {"index": None}
    module = dict(policy.named_modules())[layer_name]
    original = module.forward

    def forward(x):
        if module.context.mode != "replace" or ablate["index"] is None:
            return original(x)
        timestep = module.context.timestep_for(x)
        _y_hat, latent, preactivation = module.transcoder(x, timestep, return_preactivation=True)
        module.context.record_trace(module.name, module.layer_index, preactivation, latent, timestep)
        module.context.record_latent(module.name, module.layer_index, latent, timestep)
        latent = latent.clone()
        latent[..., int(ablate["index"])] = 0
        y_hat = module.transcoder.decoder(latent.to(dtype=module.transcoder.decoder.weight.dtype))
        return y_hat.to(dtype=x.dtype)

    module.forward = forward
    return policy, preprocessor, context, ablate


def _release_policy(policy) -> None:
    del policy
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _score_selected(
    environment, policy, preprocessor, context, ablate, selected, sentence, task_index, layer, timestep, feature, seed
):
    suite: _Suite = environment["suite"]
    ablate["index"] = None
    context.mode = "probe"
    scored: dict[str, Any] = {
        "held_visible": [],
        "held_hidden": [],
        "not_held": [],
        "gone": [],
        "grip_scores": [],
        "grip_values": [],
        "grip_held": [],
    }
    for label, frames in selected.items():
        for frame in frames:
            env = suite.ensure(task_index)
            raw = _observe_state(env, frame["state"])
            again = _label_raw(env.sim, sentence, raw)
            if again != label:
                continue
            value = _probe_feature(
                policy,
                preprocessor,
                context,
                raw,
                sentence,
                layer,
                timestep,
                feature,
                seed + int(frame["episode"]) * 1000 + int(frame["step"]),
            )
            if value is None:
                continue
            scored[label].append(value)
            scored["grip_scores"].append(value)
            scored["grip_values"].append(_gripper_value(raw))
            scored["grip_held"].append(label.startswith("held_"))
            frame["raw_seed"] = seed + int(frame["episode"]) * 1000 + int(frame["step"])
    gone_limit = int(os.environ.get("OCCLUSION_GONE", "4"))
    gone_frames = spread_frames(selected["held_hidden"], gone_limit)
    if len(gone_frames) < gone_limit:
        gone_frames = gone_frames + spread_frames(selected["held_visible"], gone_limit - len(gone_frames))
    for frame in gone_frames:
        env = suite.ensure(task_index)
        _observe_state(env, frame["state"])
        gone = _gone_probe(
            env,
            policy,
            preprocessor,
            context,
            sentence,
            layer,
            timestep,
            feature,
            int(frame.get("raw_seed", seed + int(frame["episode"]) * 1000 + int(frame["step"]))),
        )
        if gone is not None:
            scored["gone"].append(gone)
    return scored


def _gone_probe(env, policy, preprocessor, context, sentence, layer, timestep, feature, seed):
    sim = env.sim
    bodies = _candidate_bodies(sim, sentence)
    _touching, body = _contact_body(sim, _finger_geoms(sim), bodies)
    if body is None:
        return None
    before = arm_joints(np.array(sim.data.qpos, copy=True), _object_span(sim, body))
    saved, span = _move_object_away(sim, body)
    if saved is None:
        return None
    try:
        after = arm_joints(np.array(sim.data.qpos, copy=True), span)
        if not arm_unchanged(before, after):
            return None
        raw = _render_after_edit(env)
        after_render = arm_joints(np.array(sim.data.qpos, copy=True), span)
        if not arm_unchanged(before, after_render):
            return None
        return _probe_feature(policy, preprocessor, context, raw, sentence, layer, timestep, feature, seed)
    finally:
        _restore_qpos(sim, saved)


def _probe_feature(policy, preprocessor, context, raw, sentence, layer, timestep, feature, seed) -> float | None:
    box: dict[str, float] = {}

    def callback(name, layer_index, latent, step_time):
        del name
        if int(layer_index) != int(layer):
            return
        current = float(step_time.detach().float().reshape(-1)[0].cpu())
        if abs(current - float(timestep)) > 1e-3:
            return
        values = latent.detach().float()
        if values.ndim == 3:
            score = float(values[0, :, int(feature)].max().cpu())
        elif values.ndim == 2:
            score = float(values[0, int(feature)].cpu())
        else:
            return
        box["score"] = max(score, box.get("score", score))

    context.mode = "probe"
    context.latent_callback = callback
    context.clear_records()
    _seed_torch(seed)
    with torch.no_grad():
        _predict(policy, preprocessor, raw, sentence)
    context.latent_callback = None
    context.clear_records()
    if "score" not in box:
        return None
    return float(box["score"])


def _ablate(environment, policy, preprocessor, context, ablate, frames, sentence, task_index, feature, control, seed):
    suite: _Suite = environment["suite"]
    real_deltas = []
    random_deltas = []
    context.mode = "replace"
    for frame in frames:
        env = suite.ensure(task_index)
        raw = _observe_state(env, frame["state"])
        frame_seed = seed + int(frame["episode"]) * 1000 + int(frame["step"])
        base = _action_chunk(policy, preprocessor, context, ablate, raw, sentence, None, frame_seed)
        real = _action_chunk(policy, preprocessor, context, ablate, raw, sentence, feature, frame_seed)
        other = _action_chunk(policy, preprocessor, context, ablate, raw, sentence, control, frame_seed)
        real_deltas.append(action_rmse(base, real))
        random_deltas.append(action_rmse(base, other))
    if not real_deltas:
        raise RuntimeError("The passing feature has no held_hidden frame to ablate.")
    return float(np.mean(real_deltas)), float(np.mean(random_deltas))


def _action_chunk(policy, preprocessor, context, ablate, raw, sentence, feature_index, seed):
    ablate["index"] = feature_index
    context.mode = "replace"
    context.latent_callback = None
    context.clear_records()
    _seed_torch(seed)
    with torch.no_grad():
        chunk = _predict(policy, preprocessor, raw, sentence)
    ablate["index"] = None
    return _chunk_array(chunk)


def _predict(policy, preprocessor, raw, sentence):
    batch = {
        "observation.images.image": _image_tensor(raw, "agentview_image", "agentview_rgb"),
        "observation.images.image2": _image_tensor(
            raw, "robot0_eye_in_hand_image", "eye_in_hand_rgb", "robot0_eye_in_hand_rgb"
        ),
        "observation.state": torch.from_numpy(_libero_state(raw)).unsqueeze(0),
        "task": [sentence],
    }
    features = getattr(policy.config, "input_features", {}) or {}
    reference = batch["observation.images.image"]
    for key in features:
        if key.startswith("observation.images.") and key not in batch:
            batch[key] = torch.zeros_like(reference)
    prepared = preprocessor(batch)
    return policy.predict_action_chunk(prepared, num_steps=10)


def _image_tensor(raw: dict, *keys: str):
    image = None
    for key in keys:
        if key in raw:
            image = np.ascontiguousarray(raw[key])
            break
    if image is None:
        raise KeyError(f"None of {keys} are in the simulator observation. Keys: {sorted(raw)}")
    image = image[::-1, ::-1].copy()
    return torch.from_numpy(image).permute(2, 0, 1).float().div(255.0).unsqueeze(0)


def _libero_state(raw: dict) -> np.ndarray:
    state = np.concatenate(
        [
            np.asarray(raw["robot0_eef_pos"], dtype=np.float32).reshape(3),
            _quat_xyzw_to_axisangle(raw["robot0_eef_quat"]),
            np.asarray(raw["robot0_gripper_qpos"], dtype=np.float32).reshape(2),
        ]
    )
    if state.shape != (8,):
        raise RuntimeError(f"Expected an 8-number arm state, got {state.shape}")
    return state


def _quat_xyzw_to_axisangle(quat: np.ndarray) -> np.ndarray:
    quat = np.asarray(quat, dtype=np.float64)
    norm = np.linalg.norm(quat)
    if norm < 1e-8:
        return np.zeros(3, dtype=np.float32)
    quat = quat / norm
    w = float(np.clip(quat[3], -1.0, 1.0))
    den = float(np.sqrt(max(1.0 - w * w, 0.0)))
    if den < 1e-8:
        return np.zeros(3, dtype=np.float32)
    return (quat[:3] * 2.0 * np.arccos(w) / den).astype(np.float32)


def _gripper_value(raw: dict) -> float:
    return float(np.sum(np.asarray(raw["robot0_gripper_qpos"], dtype=np.float64).reshape(-1)))


def _chunk_array(chunk) -> np.ndarray:
    if isinstance(chunk, (tuple, list)):
        chunk = chunk[0]
    if isinstance(chunk, dict):
        chunk = chunk["action"]
    action = chunk.detach().float().cpu().numpy()
    if action.ndim == 3:
        action = action[0]
    if action.ndim != 2:
        raise RuntimeError(f"Expected an action chunk, got shape {action.shape}")
    return action


def _seed_torch(seed: int) -> None:
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))


def _trace_circuit(args, chosen: dict, feature_dir: Path, output_dir: Path) -> str | None:
    target = tracer_feature_key(int(chosen["layer"]), float(chosen["timestep"]), int(chosen["feature"]))
    circuit_dir = output_dir / "circuit"
    script = Path(__file__).resolve().parent / "trace_pi05_transcoder_circuit.py"
    command = [
        sys.executable,
        "-u",
        str(script),
        "--checkpoint",
        str(args.checkpoint),
        "--feature-dir",
        str(feature_dir),
        "--output-dir",
        str(circuit_dir),
        "--target",
        target,
        "--trace-mode",
        "diffract-frontier",
        "--max-nodes",
        "5",
        "--min-attribution",
        "0.005",
        "--node-cumulative-threshold",
        "0.03",
        "--edge-cumulative-threshold",
        "0.7",
        "--source-policy",
        "all-earlier",
        "--num-inference-steps",
        "10",
        "--top-examples",
        "5",
        "--device",
        "cuda",
    ]
    if os.environ.get("HF_HUB_OFFLINE") == "1":
        command.append("--local-files-only")
    print("tracing " + " ".join(command), flush=True)
    env = os.environ.copy()
    root = Path(__file__).resolve().parents[1]
    env["PYTHONPATH"] = os.pathsep.join([str(root / "src"), str(root / "scripts"), env.get("PYTHONPATH", "")])
    env["PYTHONUNBUFFERED"] = "1"
    process = subprocess.Popen(
        command,
        cwd=str(root / "scripts"),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    code = process.wait()
    if code != 0:
        raise RuntimeError(f"trace_pi05_transcoder_circuit.py exited with code {code}")
    graph = json.loads((circuit_dir / "graph.json").read_text())
    return circuit_path(graph)


if __name__ == "__main__":
    main()
'''

scripts = repo / "scripts"
path = scripts / "occlusion_contrast.py"
path.write_text(SCRIPT if SCRIPT.endswith("\n") else SCRIPT + "\n")
print("wrote", path, flush=True)


def find_checkpoint():
    roots = [
        repo,
        LOCAL_REPO,
        Path("/content/drive/MyDrive/groot-run-shared-programmer908"),
    ]
    preferred = []
    others = []
    for root in roots:
        if not root.exists():
            continue
        for hit in root.rglob("step_*.pt"):
            if "permanence" in hit.parts:
                continue
            if hit.name == "step_027233.pt":
                preferred.append(hit)
            else:
                others.append(hit)
    pool = preferred or others
    if not pool:
        return None
    return max(pool, key=lambda item: item.stat().st_mtime)


checkpoint = find_checkpoint()
train_dir = repo / "outputs" / "transcoders" / "pi05_libero" / "occlusion_contrast"
feature_dir = repo / "outputs" / "features" / "pi05_libero" / "occlusion_contrast"
verdict_dir = repo / "outputs" / "occlusion_contrast"
env = os.environ.copy()
env["PYTHONPATH"] = os.pathsep.join(
    [str(repo / "src"), str(repo / "scripts"), env.get("PYTHONPATH", "")]
)
env["PYTHONUNBUFFERED"] = "1"
env.setdefault("HF_HOME", str(Path(os.environ.get("HF_HOME", "/content/hf_home"))))
env.setdefault("HF_HUB_CACHE", str(Path(os.environ.get("HF_HUB_CACHE", "/content/hf_home/hub"))))
offline = ["--local-files-only"] if env.get("HF_HUB_OFFLINE") == "1" else []
ensure_lerobot_libero(env)

if checkpoint is None:
    print("No step_*.pt checkpoint found. Training once with scripts/train_pi05_transcoders.py.", flush=True)
    stream(
        [
            str(PYTHON),
            "-u",
            str(scripts / "train_pi05_transcoders.py"),
            "--policy-path",
            "lerobot/pi05_libero_finetuned",
            "--output-dir",
            str(train_dir),
            "--num-feed-forwards",
            "100",
            "--batch-size",
            "1",
            "--episodes",
            EPISODES,
            "--device",
            "cuda",
            "--collection-mode",
            "random-timestep",
            "--lambda-l1",
            "1e-4",
            "--expansion-factor",
            "16",
            "--save-every",
            "100",
            *offline,
        ],
        repo,
        env,
    )
    checkpoint = find_checkpoint()
    if checkpoint is None:
        raise SystemExit("Training finished without a step_*.pt checkpoint.")
else:
    print("loading existing checkpoint", checkpoint, flush=True)

if all((feature_dir / name).is_file() for name in FEATURE_FILES):
    print("reusing feature dir", feature_dir, flush=True)
else:
    stream(
        [
            str(PYTHON),
            "-u",
            str(scripts / "collect_pi05_transcoder_features.py"),
            "--checkpoint",
            str(checkpoint),
            "--output-dir",
            str(feature_dir),
            "--episodes",
            EPISODES,
            "--batch-size",
            "1",
            "--collection-mode",
            "inference",
            "--num-inference-steps",
            "10",
            "--top-k",
            "5",
            "--device",
            "cuda",
            *offline,
        ],
        repo,
        env,
    )
stream(
    [
        str(PYTHON),
        "-u",
        str(scripts / "make_pi05_feature_report.py"),
        "--feature-dir",
        str(feature_dir),
        "--max-features",
        "30",
        "--top-examples",
        "5",
        "--sort-by",
        "interesting",
    ],
    repo,
    env,
)
stream(
    [
        str(PYTHON),
        "-u",
        str(path),
        "--checkpoint",
        str(checkpoint),
        "--feature-dir",
        str(feature_dir),
        "--output-dir",
        str(verdict_dir),
        "--suite",
        env.get("SUITE", "libero_spatial"),
        "--seed",
        "0",
    ],
    scripts,
    env,
)
print("verdict", verdict_dir / "verdict.json")


## Conclusion

Paste only the next cell into the runtime that already finished tight cover. It reuses those two frames and plate placements, counts leftover bowl pixels, and writes `outputs/permanence/conclusion/`. If `scripts/tight_cover.py` is missing, this cell writes it and does not rewrite `outputs/permanence/tight_cover/`.


In [ ]:
# Conclusion — leftover bowl pixels after the tight hide
# Paste into the same runtime after the tight-cover cell. Does not retrain.
# Does not overwrite outputs/permanence/tight_cover/ or earlier directories.
# Writes scripts/tight_cover.py if that file is missing.

import os, sys, subprocess
from pathlib import Path

os.environ.setdefault("CTRL_SCAN_STEPS", "200")
os.environ.setdefault("SUITE", "libero_spatial")
os.environ.setdefault("TASK_IDS", "[0]")
os.environ.setdefault(
    "LAYER_NAME",
    "model.paligemma_with_expert.paligemma.model.language_model.layers.5",
)
os.environ.setdefault("MUJOCO_GL", "egl")
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

LOCAL_REPO = Path(os.environ.get("LOCAL_REPO", "/content/groot-run"))
PYTHON = Path("/content/lerobot-venv/bin/python")
if not PYTHON.exists():
    PYTHON = Path(sys.executable)

SCRIPT = r'''
"""Score leftover bowl pixels after the tight hide, then stop.

The tight-cover run hid the table bowl from both cameras and split closed-loop
success. This run reuses those two frames and those plate placements. It counts
bowl pixels the plate did not cover, paints them, and compares that action with
full gray paint and with removal. It does not retrain, does not roll out, and
does not rewrite earlier output directories.
"""

from __future__ import annotations

import json
import os
from pathlib import Path

import cv2
import numpy as np
import torch

import controlled_contrasts as cc
import paper_claim as pc
import paper_gaps
import remaining_gaps as rg
import tight_cover as tc

FRAMES = ((1, 0), (1, 44))
FALLBACK = {
    (1, 0): {"fraction": 0.90, "scale": 0.75},
    (1, 44): {"fraction": 0.80, "scale": 0.75},
}


def leftover_split(base: np.ndarray, edited: np.ndarray, bowl_mask: np.ndarray | None) -> dict:
    """Bowl pixels the edit left in place, and changed pixels outside the bowl."""
    changed = cc.changed_pixels(base, edited)
    if bowl_mask is None:
        leftover = np.zeros(changed.shape, dtype=bool)
        extra = changed
        bowl_n = 0
        known = False
    else:
        bowl = np.asarray(bowl_mask, dtype=bool)
        leftover = np.logical_and(bowl, np.logical_not(changed))
        extra = np.logical_and(changed, np.logical_not(bowl))
        bowl_n = int(bowl.sum())
        known = True
    return {
        "leftover": leftover,
        "extra": extra,
        "leftover_pixels": int(leftover.sum()),
        "extra_pixels": int(extra.sum()),
        "bowl_pixels": bowl_n,
        "changed_pixels": int(changed.sum()),
        "known": known,
    }


def hide_reading(
    leftover_agent: int,
    leftover_wrist: int,
    known: bool,
    paint_to_cover: float,
    cover_to_absent: float,
    tight_to_cover: float,
) -> str:
    if not known:
        return "The bowl mask could not be measured."
    leftover = int(leftover_agent) + int(leftover_wrist)
    if leftover == 0:
        if (
            np.isfinite(tight_to_cover)
            and tight_to_cover < 0.05
            and np.isfinite(cover_to_absent)
            and cover_to_absent >= 0.05
        ):
            return (
                "The bowl is hidden as fully as gray paint. "
                "The remaining action versus removal is the plate, not a hidden bowl."
            )
        if np.isfinite(cover_to_absent) and cover_to_absent < 0.05:
            return "The hide matches removal."
        return (
            "The bowl pixels are gone. "
            "The remaining action versus removal is the extra object, not a hidden bowl."
        )
    if np.isfinite(paint_to_cover) and paint_to_cover < 1e-3:
        return "A visible bowl rim remains. Painting it matches the full cover."
    if np.isfinite(paint_to_cover) and paint_to_cover < 0.05:
        return "A visible bowl rim remains. Painting it nearly matches the full cover."
    return "A visible bowl rim remains, and painting it still leaves an action gap to the full cover."


def placements_from_summary(summary: dict | None) -> dict[tuple[int, int], dict]:
    found = dict(FALLBACK)
    if not summary:
        return found
    for row in summary.get("frames") or []:
        tight = row.get("tight") or {}
        if "fraction" not in tight or "scale" not in tight:
            continue
        found[(int(row["episode"]), int(row["step"]))] = {
            "fraction": float(tight["fraction"]),
            "scale": float(tight["scale"]),
        }
    return found


def rate_line(rates: list[dict] | None) -> str:
    if not rates:
        return "Closed-loop rates from the tight-cover run were not found."
    parts = []
    for row in rates:
        parts.append(f"{row['edit']} {row['successes']}/{row['episodes']}")
    return "; ".join(parts)


def final_claim(table: dict | None, held: dict | None, rates: list[dict] | None) -> str:
    table_text = table["reading"] if table else "the table hide was not scored"
    held_text = held["reading"] if held else "the held hide was not scored"
    return (
        "The next action follows the visible bowl pixels. "
        f"Table hide: {table_text} "
        f"Held hide: {held_text} "
        f"Closed loop: {rate_line(rates)}. "
        "Layer-5 prefix replace already closed 93% of a paint edit; passing features closed 1.8%."
    )


def load_summary(path: Path) -> dict | None:
    if not path.is_file():
        return None
    return json.loads(path.read_text())


def plain_split(row: dict) -> dict:
    return {key: value for key, value in row.items() if key not in ("leftover", "extra")}


def save_sheet(frames: list[dict], path: Path) -> bool:
    strips = []
    for frame in frames:
        pictures = frame.get("pictures")
        if not pictures:
            continue
        label = f"e{frame['episode']}s{frame['step']}"
        strips.append(
            np.concatenate(
                [
                    pc.tile(pictures["base_agent"], f"{label} base"),
                    pc.tile(pictures["tight_agent"], "tight"),
                    pc.tile(pc.highlight(pictures["tight_agent"], pictures["leftover_agent"]), "leftover"),
                    pc.tile(pictures["paint_agent"], "paint rim"),
                    pc.tile(pictures["cover_agent"], "gray"),
                    pc.tile(pictures["absent_agent"], "absent"),
                ],
                axis=1,
            )
        )
        strips.append(
            np.concatenate(
                [
                    pc.tile(pictures["base_wrist"], f"{label} wrist"),
                    pc.tile(pictures["tight_wrist"], "tight"),
                    pc.tile(pc.highlight(pictures["tight_wrist"], pictures["leftover_wrist"]), "leftover"),
                    pc.tile(pictures["paint_wrist"], "paint rim"),
                    pc.tile(pictures["cover_wrist"], "gray"),
                    pc.tile(pictures["absent_wrist"], "absent"),
                ],
                axis=1,
            )
        )
    if not strips:
        return False
    width = max(strip.shape[1] for strip in strips)
    padded = []
    for strip in strips:
        if strip.shape[1] < width:
            pad = np.full((strip.shape[0], width - strip.shape[1], 3), 255, dtype=np.uint8)
            strip = np.concatenate([strip, pad], axis=1)
        padded.append(strip)
    cv2.imwrite(str(path), cv2.cvtColor(np.concatenate(padded, axis=0), cv2.COLOR_RGB2BGR))
    return True


def _fmt(value: float | None, digits: int = 3) -> str:
    if value is None or not np.isfinite(value):
        return "undefined"
    return f"{float(value):.{digits}f}"


def main() -> None:
    cc.configure_environment()
    if not torch.cuda.is_available():
        raise SystemExit("CUDA GPU required. Use the Colab L4 or A100 runtime.")
    os.environ["MAX_STEPS"] = os.environ.get("CTRL_SCAN_STEPS", "200")
    collect = cc.load_collect_module()
    # Simulator imports stay inside main so unit tests do not load LIBERO or MuJoCo.
    from libero.libero import benchmark, get_libero_path
    from libero.libero.envs import OffScreenRenderEnv

    try:
        import mujoco

        free_type = int(mujoco.mjtJoint.mjJNT_FREE)
    except Exception:
        free_type = 0

    scan_steps = cc.env_int("CTRL_SCAN_STEPS", 200)
    repo = Path(os.environ.get("LOCAL_REPO", "/content/groot-run"))
    dest = repo / "outputs" / "permanence" / "conclusion"
    dest.mkdir(parents=True, exist_ok=True)
    prior = load_summary(repo / "outputs" / "permanence" / "tight_cover" / "summary.json")
    placements = placements_from_summary(prior)
    rates = None if prior is None else prior.get("rates")

    print("Loading policy", flush=True)
    policy, pre, post, layer5 = collect.load_policy()
    del post
    policy.eval()
    layer6 = cc.find_layer6(policy, layer5)
    suite = benchmark.get_benchmark_dict()[collect.SUITE]()
    task = suite.get_task(0)
    sentence = task.language
    demos = paper_gaps.read_demo_states(collect.find_demo_file(task), scan_steps)
    bddl = os.path.join(get_libero_path("bddl_files"), task.problem_folder, task.bddl_file)
    env = OffScreenRenderEnv(bddl_file_name=bddl, camera_heights=256, camera_widths=256)
    env.seed(0)

    def target_body(sim):
        mapping = collect.candidate_bodies(sim, sentence)
        for candidate, candidate_geoms in mapping.items():
            if collect.object_qpos_span(sim, candidate) is not None and candidate_geoms:
                return int(candidate), list(candidate_geoms)
        return None, None

    def forward(agent, wrist, state, noise: int):
        batch = cc.batch_from_images(collect, pre, agent, wrist, state, sentence, policy)
        _tokens, _tokens6, action = cc.forward_policy(policy, layer5, layer6, batch, noise, None, "base")
        return action

    print("\nCONCLUSION  leftover bowl pixels after the tight hide", flush=True)
    print(f"Closed loop from tight_cover: {rate_line(rates)}", flush=True)
    frame_rows: list[dict] = []
    try:
        for episode, step in FRAMES:
            if episode >= len(demos) or step >= len(demos[episode]):
                print(f"  ep {episode} step {step}: outside the demos", flush=True)
                continue
            place = placements.get((episode, step), FALLBACK[(episode, step)])
            row = score_frame(
                collect,
                env,
                demos[episode][step],
                episode,
                step,
                free_type,
                target_body,
                forward,
                place,
            )
            frame_rows.append(row)
        sheet = dest / "conclusion.png"
        if save_sheet(frame_rows, sheet):
            print("sheet", sheet, flush=True)
        table = next((row for row in frame_rows if row.get("episode") == 1 and row.get("step") == 0), None)
        held = next((row for row in frame_rows if row.get("episode") == 1 and row.get("step") == 44), None)
        claim = final_claim(table, held, rates)
        print("\nCLAIM", flush=True)
        print(claim, flush=True)
        summary = {
            "scope": (
                "Scope: lerobot/pi05_libero_finetuned, PaliGemma language-model layer 5, "
                "proprioception from the unedited scene. "
                f"Frames {list(FRAMES)}. Reuses the tight-cover placements. No closed loop."
            ),
            "rates": rates,
            "frames": [{key: value for key, value in row.items() if key != "pictures"} for row in frame_rows],
            "claim": claim,
            "figure": str(sheet) if sheet.is_file() else None,
        }
        (dest / "summary.json").write_text(json.dumps(cc.json_ready(summary), indent=2))
        print("\n" + summary["scope"], flush=True)
        print("Wrote", dest / "summary.json", flush=True)
        print("RESULT_DIR", dest, flush=True)
    finally:
        env.close()


def score_frame(collect, env, sim_state, episode: int, step: int, free_type: int, target_body, forward, place: dict) -> dict:
    tc.begin_episode(collect, env, sim_state)
    sim = collect.sim_of(env)
    bowl, bowl_geoms = target_body(sim)
    if bowl is None or not bowl_geoms:
        print(f"  ep {episode} step {step}: no free target body", flush=True)
        return {"episode": episode, "step": step, "error": "no free target body"}
    bodies = pc.free_body_map(sim, free_type)
    names = dict(pc.body_names(sim, bodies))
    agent_body = pc.prefer_occluder(list(names.items()), bowl)
    other = rg.second_body(list(names.items()), {bowl, agent_body} if agent_body is not None else {bowl})
    if agent_body is None or other is None or agent_body not in bodies or other not in bodies:
        print(f"  ep {episode} step {step}: need a ramekin and a second body", flush=True)
        return {"episode": episode, "step": step, "error": "need a ramekin and a second body"}
    wrist_cam = cc.camera_index(sim, "eye_in_hand")
    if wrist_cam is None:
        print(f"  ep {episode} step {step}: wrist camera was not found", flush=True)
        return {"episode": episode, "step": step, "error": "wrist camera was not found"}
    agent_pos = np.asarray(sim.data.cam_xpos[collect.camera_id(sim)], dtype=np.float64)
    wrist_pos = np.asarray(sim.data.cam_xpos[wrist_cam], dtype=np.float64)
    bowl_center = np.mean([np.asarray(sim.data.geom_xpos[geom], dtype=np.float64) for geom in bowl_geoms], axis=0)
    distance = float(np.linalg.norm(bowl_center - wrist_pos))
    recolor_agent, recolor_wrist = cc.render_rgba_images(collect, env, sim, bowl_geoms)
    raw = collect.render_after_edit(env)
    base_agent, base_wrist = cc.images_of(collect, raw)
    state = collect.libero_state(raw)
    agent_mask = pc.uncapped_change_mask(base_agent, recolor_agent, min_pixels=5)
    wrist_mask = pc.uncapped_change_mask(base_wrist, recolor_wrist, min_pixels=5)
    fraction = float(place["fraction"])
    scale = float(place["scale"])
    placed = rg.render_placed(
        collect,
        env,
        sim,
        [
            (agent_body, bodies[agent_body], pc.point_on_ray(agent_pos, bowl_center, rg.AGENT_FRACTION), rg.AGENT_SCALE),
            (other, bodies[other], pc.point_on_ray(wrist_pos, bowl_center, fraction), scale),
        ],
    )
    if placed is None:
        print(f"  ep {episode} step {step}: placement produced no image", flush=True)
        return {"episode": episode, "step": step, "error": "placement produced no image"}
    cover_agent, cover_wrist, _cover_state = pc.live_images(collect, env, sim, bowl, bowl_geoms, "covered")
    absent_agent, absent_wrist, _absent_state = pc.live_images(collect, env, sim, bowl, bowl_geoms, "absent")
    agent_split = leftover_split(base_agent, placed["agent"], agent_mask)
    wrist_split = leftover_split(base_wrist, placed["wrist"], wrist_mask)
    paint_agent = pc.paint_camera(placed["agent"], agent_split["leftover"])
    paint_wrist = pc.paint_camera(placed["wrist"], wrist_split["leftover"])
    scores = rg.view_scores(base_agent, base_wrist, placed["agent"], placed["wrist"], agent_mask, wrist_mask)
    noise = 8300 + episode * 1000 + step
    base_action = forward(base_agent, base_wrist, state, noise)
    tight_action = forward(placed["agent"], placed["wrist"], state, noise)
    paint_action = forward(paint_agent, paint_wrist, state, noise)
    cover_action = forward(cover_agent, cover_wrist, state, noise)
    absent_action = forward(absent_agent, absent_wrist, state, noise)
    tight_to_cover = pc.action_rmse(tight_action, cover_action)
    paint_to_cover = pc.action_rmse(paint_action, cover_action)
    cover_to_absent = pc.action_rmse(cover_action, absent_action)
    tight_to_absent = pc.action_rmse(tight_action, absent_action)
    base_to_absent = pc.action_rmse(base_action, absent_action)
    reading = hide_reading(
        agent_split["leftover_pixels"],
        wrist_split["leftover_pixels"],
        agent_split["known"] and wrist_split["known"],
        paint_to_cover,
        cover_to_absent,
        tight_to_cover,
    )
    print(
        f"  ep {episode} step {step} wrist-camera-to-bowl_m {_fmt(distance, 4)} "
        f"plate fraction {_fmt(fraction, 2)} scale {_fmt(scale, 2)}",
        flush=True,
    )
    print(
        f"    leftover bowl px agent {agent_split['leftover_pixels']} wrist {wrist_split['leftover_pixels']} "
        f"| plate extra px agent {agent_split['extra_pixels']} wrist {wrist_split['extra_pixels']} "
        f"| cover agent {_fmt(scores.get('agent_cover'))} wrist {_fmt(scores.get('wrist_cover'))} "
        f"scene {_fmt(scores.get('scene_fraction'))}",
        flush=True,
    )
    print(
        f"    RMSE base→tight {_fmt(pc.action_rmse(base_action, tight_action), 4)} "
        f"tight→cover {_fmt(tight_to_cover, 4)} paint-rim→cover {_fmt(paint_to_cover, 4)} "
        f"cover→absent {_fmt(cover_to_absent, 4)} tight→absent {_fmt(tight_to_absent, 4)} "
        f"base→absent {_fmt(base_to_absent, 4)}",
        flush=True,
    )
    print(f"    {reading}", flush=True)
    return {
        "episode": episode,
        "step": step,
        "camera_to_bowl_m": distance,
        "fraction": fraction,
        "scale": scale,
        "agent": plain_split(agent_split),
        "wrist": plain_split(wrist_split),
        "agent_cover": scores.get("agent_cover"),
        "wrist_cover": scores.get("wrist_cover"),
        "scene_fraction": scores.get("scene_fraction"),
        "rmse_base_to_tight": pc.action_rmse(base_action, tight_action),
        "rmse_tight_to_cover": tight_to_cover,
        "rmse_paint_to_cover": paint_to_cover,
        "rmse_cover_to_absent": cover_to_absent,
        "rmse_tight_to_absent": tight_to_absent,
        "rmse_base_to_absent": base_to_absent,
        "reading": reading,
        "pictures": {
            "base_agent": base_agent,
            "base_wrist": base_wrist,
            "tight_agent": placed["agent"],
            "tight_wrist": placed["wrist"],
            "leftover_agent": agent_split["leftover"],
            "leftover_wrist": wrist_split["leftover"],
            "paint_agent": paint_agent,
            "paint_wrist": paint_wrist,
            "cover_agent": cover_agent,
            "cover_wrist": cover_wrist,
            "absent_agent": absent_agent,
            "absent_wrist": absent_wrist,
        },
    }


if __name__ == "__main__":
    main()
'''

TIGHT_COVER = r'''
"""Score the tight both-camera hide, and roll it out from a fresh reset.

The previous selector kept the first full cover, so the plate that fills 42%
of the wrist was forwarded and the pair that fills 17% was not. This run
forwards the smallest full cover, including a closer and smaller plate on the
held frame, then rolls out base, gray paint, absence, and that tight plate.
Each episode is reset and the done flag is cleared before the first step.

Does not retrain and does not rewrite earlier output directories.
"""

from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Literal, assert_never

import cv2
import numpy as np
import torch

import controlled_contrasts as cc
import paper_claim as pc
import paper_gaps
import remaining_gaps as rg

FRAMES = ((1, 0), (1, 44))
LARGE_FRACTION = 0.30
LARGE_SCALE = 1.5
TIGHT_FRACTIONS = (0.55, 0.70, 0.80, 0.90)
TIGHT_SCALES = (0.75, 1.0, 1.5)
RollEdit = Literal["base", "covered", "absent", "physical"]


def choose_tight(candidates: list[dict]) -> dict | None:
    """Among full covers, keep the one that changes the least of either image."""
    usable = [
        row
        for row in candidates
        if np.isfinite(row.get("agent_cover", np.nan)) and np.isfinite(row.get("wrist_cover", np.nan))
    ]
    if not usable:
        return None
    full = [
        row
        for row in usable
        if float(row["agent_cover"]) >= rg.COVER_READY and float(row["wrist_cover"]) >= rg.COVER_READY
    ]
    pool = full or usable
    return min(pool, key=lambda row: (rg._scene(row), -min(float(row["agent_cover"]), float(row["wrist_cover"]))))


def tight_reading(agent_cover: float, wrist_cover: float, scene: float, distance: float | None) -> str:
    full = (
        np.isfinite(agent_cover)
        and np.isfinite(wrist_cover)
        and float(agent_cover) >= rg.COVER_READY
        and float(wrist_cover) >= rg.COVER_READY
    )
    if full and np.isfinite(scene) and float(scene) < rg.SCENE_LIMIT:
        return "A tight scene hides the bowl from both cameras."
    if full and np.isfinite(scene):
        dist = "an unknown distance" if distance is None or not np.isfinite(distance) else f"{float(distance):.3f} m"
        return (
            f"Both cameras lose the bowl and the plate fills {100 * float(scene):.0f}% of a frame. "
            f"The wrist camera is {dist} from the bowl."
        )
    if np.isfinite(agent_cover) and float(agent_cover) >= rg.COVER_READY and (
        not np.isfinite(wrist_cover) or float(wrist_cover) < 0.2
    ):
        return "The agent camera is covered. The wrist camera still shows the bowl."
    return "The bowl stays partly visible in a camera."


def owners_of(env) -> list:
    found = []
    seen: set[int] = set()
    pending = [env]
    while pending:
        owner = pending.pop()
        if owner is None or id(owner) in seen:
            continue
        seen.add(id(owner))
        found.append(owner)
        for attr in ("env", "unwrapped"):
            inner = getattr(owner, attr, None)
            if inner is not None and inner is not owner:
                pending.append(inner)
    return found


def clear_episode(env) -> None:
    """Clear the done flag and the step counter so the next edit can step."""
    for owner in owners_of(env):
        if hasattr(owner, "done"):
            owner.done = False
        for name in ("timestep", "_elapsed_steps"):
            if hasattr(owner, name):
                setattr(owner, name, 0)


def episode_done(env) -> bool:
    for owner in owners_of(env):
        if bool(getattr(owner, "done", False)):
            return True
    return False


def task_succeeded(env) -> bool:
    for owner in owners_of(env):
        for name in ("check_success", "_check_success"):
            fn = getattr(owner, name, None)
            if callable(fn) and bool(fn()):
                return True
    return False


def step_done(outcome) -> bool:
    if not isinstance(outcome, tuple):
        return False
    if len(outcome) == 5:
        return bool(outcome[2] or outcome[3])
    if len(outcome) == 4:
        return bool(outcome[2])
    return False


def step_action(env, action: np.ndarray) -> bool:
    vector = np.asarray(action, dtype=np.float64).reshape(-1)
    space = getattr(env, "action_space", None)
    width = int(space.shape[0]) if space is not None and getattr(space, "shape", None) else vector.size
    vector = vector[:width]
    outcome = env.step(vector)
    if not isinstance(outcome, tuple) or len(outcome) not in (4, 5):
        raise RuntimeError(f"env.step returned {type(outcome)}")
    return step_done(outcome) or episode_done(env)


def begin_episode(collect, env, state) -> None:
    if hasattr(env, "reset"):
        try:
            env.reset()
        except Exception as exc:
            print(f"  reset skipped: {exc}", flush=True)
    collect.observe_state(env, state)
    clear_episode(env)


def rate_rows(rows: list[dict]) -> list[dict]:
    edits: list[str] = []
    for row in rows:
        if row["edit"] not in edits:
            edits.append(row["edit"])
    summary = []
    for edit in edits:
        chosen = [row for row in rows if row["edit"] == edit]
        summary.append(
            {
                "edit": edit,
                "episodes": len(chosen),
                "successes": sum(1 for row in chosen if row["success_after"]),
                "no_step": sum(1 for row in chosen if int(row["steps"]) == 0),
            }
        )
    return summary


def plain_row(row: dict | None) -> dict | None:
    if row is None:
        return None
    return {key: value for key, value in row.items() if key not in ("agent", "wrist")}


def save_sheet(frames: list[dict], path: Path) -> bool:
    strips = []
    for frame in frames:
        pictures = frame.get("pictures")
        if not pictures:
            continue
        label = f"e{frame['episode']}s{frame['step']}"
        strips.append(
            np.concatenate(
                [
                    pc.tile(pictures["base_agent"], f"{label} base"),
                    pc.tile(pictures["large_agent"], "large agent"),
                    pc.tile(pictures["tight_agent"], "tight agent"),
                    pc.tile(pictures["absent_agent"], "absent agent"),
                ],
                axis=1,
            )
        )
        strips.append(
            np.concatenate(
                [
                    pc.tile(pictures["base_wrist"], f"{label} wrist"),
                    pc.tile(pictures["large_wrist"], "large wrist"),
                    pc.tile(pictures["tight_wrist"], "tight wrist"),
                    pc.tile(pictures["absent_wrist"], "absent wrist"),
                ],
                axis=1,
            )
        )
    if not strips:
        return False
    width = max(strip.shape[1] for strip in strips)
    padded = []
    for strip in strips:
        if strip.shape[1] < width:
            pad = np.full((strip.shape[0], width - strip.shape[1], 3), 255, dtype=np.uint8)
            strip = np.concatenate([strip, pad], axis=1)
        padded.append(strip)
    cv2.imwrite(str(path), cv2.cvtColor(np.concatenate(padded, axis=0), cv2.COLOR_RGB2BGR))
    return True


def _fmt(value: float | None, digits: int = 3) -> str:
    if value is None or not np.isfinite(value):
        return "undefined"
    return f"{float(value):.{digits}f}"


def _print_forward(name: str, row: dict, base_to_view: float, base_to_absent: float, closed: float | None) -> None:
    print(
        f"  {name} fraction {_fmt(row.get('fraction'), 2)} scale {_fmt(row.get('scale'), 2)}: "
        f"agent cover {_fmt(row.get('agent_cover'))} wrist cover {_fmt(row.get('wrist_cover'))} "
        f"scene {_fmt(row.get('scene_fraction'))} | "
        f"RMSE base→view {_fmt(base_to_view, 4)} base→absent {_fmt(base_to_absent, 4)} "
        f"absence gap closed {pc.pct(closed)}",
        flush=True,
    )


def main() -> None:
    cc.configure_environment()
    if not torch.cuda.is_available():
        raise SystemExit("CUDA GPU required. Use the Colab L4 or A100 runtime.")
    os.environ["MAX_STEPS"] = os.environ.get("CTRL_SCAN_STEPS", "200")
    collect = cc.load_collect_module()
    # Simulator imports stay inside main so unit tests do not load LIBERO or MuJoCo.
    from libero.libero import benchmark, get_libero_path
    from libero.libero.envs import OffScreenRenderEnv

    try:
        import mujoco

        free_type = int(mujoco.mjtJoint.mjJNT_FREE)
    except Exception:
        free_type = 0

    horizon = cc.env_int("CTRL_ROLLOUT_HORIZON", 220)
    replan = cc.env_int("CTRL_REPLAN", 10)
    episodes = cc.env_int("CTRL_ROLLOUT_EPISODES", 10)
    scan_steps = cc.env_int("CTRL_SCAN_STEPS", 200)
    repo = Path(os.environ.get("LOCAL_REPO", "/content/groot-run"))
    dest = repo / "outputs" / "permanence" / "tight_cover"
    dest.mkdir(parents=True, exist_ok=True)

    print("Loading policy", flush=True)
    policy, pre, post, layer5 = collect.load_policy()
    if post is None:
        raise SystemExit("load_policy did not return a postprocessor.")
    policy.eval()
    layer6 = cc.find_layer6(policy, layer5)
    device = rg.policy_device(policy)
    suite = benchmark.get_benchmark_dict()[collect.SUITE]()
    task = suite.get_task(0)
    sentence = task.language
    demos = paper_gaps.read_demo_states(collect.find_demo_file(task), scan_steps)
    bddl = os.path.join(get_libero_path("bddl_files"), task.problem_folder, task.bddl_file)
    env = OffScreenRenderEnv(bddl_file_name=bddl, camera_heights=256, camera_widths=256)
    env.seed(0)

    def target_body(sim):
        mapping = collect.candidate_bodies(sim, sentence)
        for candidate, candidate_geoms in mapping.items():
            if collect.object_qpos_span(sim, candidate) is not None and candidate_geoms:
                return int(candidate), list(candidate_geoms)
        return None, None

    def forward(agent, wrist, state, noise: int):
        batch = cc.batch_from_images(collect, pre, agent, wrist, state, sentence, policy)
        _tokens, _tokens6, action = cc.forward_policy(policy, layer5, layer6, batch, noise, None, "base")
        return action

    frame_rows: list[dict] = []
    tight_params = {"fraction": 0.70, "scale": 1.5}
    try:
        print("\nTIGHT COVER  smallest full cover of both cameras", flush=True)
        for episode, step in FRAMES:
            if episode >= len(demos) or step >= len(demos[episode]):
                print(f"  ep {episode} step {step}: outside the demos", flush=True)
                continue
            row = score_frame(
                collect,
                env,
                demos[episode][step],
                episode,
                step,
                free_type,
                target_body,
                forward,
            )
            frame_rows.append(row)
            if episode == 1 and step == 0 and row.get("tight"):
                tight_params = {
                    "fraction": float(row["tight"]["fraction"]),
                    "scale": float(row["tight"]["scale"]),
                }
        sheet = dest / "tight_cover.png"
        if save_sheet(frame_rows, sheet):
            print("sheet", sheet, flush=True)

        print("\nCLOSED LOOP  reset before every edit, unnormalized chunks", flush=True)
        print(
            f"Horizon {horizon}, episodes {min(episodes, len(demos))}, re-query every {replan}. "
            f"physical re-places the table-frame tight pair "
            f"(fraction {tight_params['fraction']:.2f}, scale {tight_params['scale']:.2f}) on the current rays. "
            "covered paints the uncapped mask gray. absent teleports the bowl and restores it before the step.",
            flush=True,
        )
        rollouts = run_rollouts(
            collect,
            env,
            policy,
            pre,
            post,
            layer5,
            layer6,
            sentence,
            demos,
            target_body,
            free_type,
            tight_params,
            min(episodes, len(demos)),
            horizon,
            replan,
            device,
        )
        rates = rate_rows(rollouts)
        print("\nSUCCESS", flush=True)
        for rate in rates:
            print(
                f"  {rate['edit']}: {rate['successes']}/{rate['episodes']} success, "
                f"{rate['no_step']} episodes with no step",
                flush=True,
            )
        summary = {
            "scope": (
                "Scope: lerobot/pi05_libero_finetuned, PaliGemma language-model layer 5. "
                f"Tight-cover frames: {list(FRAMES)}. "
                f"Rollout plate fraction {tight_params['fraction']:.2f}, scale {tight_params['scale']:.2f}. "
                f"Closed loop: {min(episodes, len(demos))} episodes, {horizon} steps, "
                f"re-query every {replan}, reset and done flag cleared before each edit."
            ),
            "frames": [{key: value for key, value in row.items() if key != "pictures"} for row in frame_rows],
            "tight_params": tight_params,
            "rollouts": rollouts,
            "rates": rates,
            "figure": str(sheet) if sheet.is_file() else None,
        }
        (dest / "summary.json").write_text(json.dumps(cc.json_ready(summary), indent=2))
        print("\n" + summary["scope"], flush=True)
        print("Wrote", dest / "summary.json", flush=True)
        print("RESULT_DIR", dest, flush=True)
    finally:
        env.close()


def score_frame(collect, env, sim_state, episode: int, step: int, free_type: int, target_body, forward) -> dict:
    begin_episode(collect, env, sim_state)
    sim = collect.sim_of(env)
    bowl, bowl_geoms = target_body(sim)
    if bowl is None or not bowl_geoms:
        print(f"  ep {episode} step {step}: no free target body", flush=True)
        return {"episode": episode, "step": step, "error": "no free target body"}
    bodies = pc.free_body_map(sim, free_type)
    names = dict(pc.body_names(sim, bodies))
    agent_body = pc.prefer_occluder(list(names.items()), bowl)
    other = rg.second_body(list(names.items()), {bowl, agent_body} if agent_body is not None else {bowl})
    if agent_body is None or other is None or agent_body not in bodies or other not in bodies:
        print(f"  ep {episode} step {step}: need a ramekin and a second body", flush=True)
        return {"episode": episode, "step": step, "error": "need a ramekin and a second body"}
    wrist_cam = cc.camera_index(sim, "eye_in_hand")
    if wrist_cam is None:
        print(f"  ep {episode} step {step}: wrist camera was not found", flush=True)
        return {"episode": episode, "step": step, "error": "wrist camera was not found"}
    agent_pos = np.asarray(sim.data.cam_xpos[collect.camera_id(sim)], dtype=np.float64)
    wrist_pos = np.asarray(sim.data.cam_xpos[wrist_cam], dtype=np.float64)
    bowl_center = np.mean([np.asarray(sim.data.geom_xpos[geom], dtype=np.float64) for geom in bowl_geoms], axis=0)
    distance = float(np.linalg.norm(bowl_center - wrist_pos))
    raw = collect.render_after_edit(env)
    base_agent, base_wrist = cc.images_of(collect, raw)
    state = collect.libero_state(raw)
    recolor_agent, recolor_wrist = cc.render_rgba_images(collect, env, sim, bowl_geoms)
    raw = collect.render_after_edit(env)
    base_agent, base_wrist = cc.images_of(collect, raw)
    state = collect.libero_state(raw)
    agent_mask = pc.uncapped_change_mask(base_agent, recolor_agent, min_pixels=5)
    wrist_mask = pc.uncapped_change_mask(base_wrist, recolor_wrist, min_pixels=5)
    print(
        f"  ep {episode} step {step} wrist-camera-to-bowl_m {_fmt(distance, 4)}",
        flush=True,
    )

    def one(fraction: float, scale: float) -> dict | None:
        rendered = rg.render_placed(
            collect,
            env,
            sim,
            [
                (agent_body, bodies[agent_body], pc.point_on_ray(agent_pos, bowl_center, rg.AGENT_FRACTION), rg.AGENT_SCALE),
                (other, bodies[other], pc.point_on_ray(wrist_pos, bowl_center, fraction), scale),
            ],
        )
        if rendered is None:
            return None
        row = rg.view_scores(base_agent, base_wrist, rendered["agent"], rendered["wrist"], agent_mask, wrist_mask)
        row.update({"fraction": float(fraction), "scale": float(scale), "kind": "physical_pair"})
        return row

    candidates = []
    large = one(LARGE_FRACTION, LARGE_SCALE)
    if large is not None:
        candidates.append(large)
    for fraction in TIGHT_FRACTIONS:
        for scale in TIGHT_SCALES:
            row = one(fraction, scale)
            if row is not None:
                candidates.append(row)
    tight = choose_tight(candidates)
    if large is None or tight is None:
        print(f"  ep {episode} step {step}: placement search produced no image", flush=True)
        return {"episode": episode, "step": step, "camera_to_bowl_m": distance, "error": "placement search produced no image"}
    noise = 8100 + episode * 1000 + step
    base_action = forward(base_agent, base_wrist, state, noise)
    large_action = forward(large["agent"], large["wrist"], state, noise)
    tight_action = forward(tight["agent"], tight["wrist"], state, noise)
    absent_agent, absent_wrist, _absent_state = pc.live_images(collect, env, sim, bowl, bowl_geoms, "absent")
    absent_action = forward(absent_agent, absent_wrist, state, noise)
    to_absent = pc.action_rmse(base_action, absent_action)
    large_rmse = pc.action_rmse(base_action, large_action)
    tight_rmse = pc.action_rmse(base_action, tight_action)
    large_closed = rg.absence_gap_closed(pc.action_rmse(large_action, absent_action), to_absent)
    tight_closed = rg.absence_gap_closed(pc.action_rmse(tight_action, absent_action), to_absent)
    _print_forward("large", large, large_rmse, to_absent, large_closed)
    _print_forward("tight", tight, tight_rmse, to_absent, tight_closed)
    text = tight_reading(tight["agent_cover"], tight["wrist_cover"], tight["scene_fraction"], distance)
    print(f"  tight: {text}", flush=True)
    return {
        "episode": episode,
        "step": step,
        "camera_to_bowl_m": distance,
        "large": plain_row(large),
        "tight": plain_row(tight),
        "rmse_base_to_absent": to_absent,
        "rmse_base_to_large": large_rmse,
        "rmse_base_to_tight": tight_rmse,
        "large_absence_gap_closed": large_closed,
        "tight_absence_gap_closed": tight_closed,
        "tight_reading": text,
        "candidates": [plain_row(row) for row in candidates],
        "pictures": {
            "base_agent": base_agent,
            "base_wrist": base_wrist,
            "large_agent": large["agent"],
            "large_wrist": large["wrist"],
            "tight_agent": tight["agent"],
            "tight_wrist": tight["wrist"],
            "absent_agent": absent_agent,
            "absent_wrist": absent_wrist,
        },
    }


def current_pair(collect, env, sim, agent_body, agent_geoms, wrist_body, wrist_geoms, bowl_geoms, fraction: float, scale: float):
    wrist_cam = cc.camera_index(sim, "eye_in_hand")
    if wrist_cam is None or not bowl_geoms:
        return None
    agent_pos = np.asarray(sim.data.cam_xpos[collect.camera_id(sim)], dtype=np.float64)
    wrist_pos = np.asarray(sim.data.cam_xpos[wrist_cam], dtype=np.float64)
    bowl_center = np.mean([np.asarray(sim.data.geom_xpos[geom], dtype=np.float64) for geom in bowl_geoms], axis=0)
    return rg.render_placed(
        collect,
        env,
        sim,
        [
            (agent_body, agent_geoms, pc.point_on_ray(agent_pos, bowl_center, rg.AGENT_FRACTION), rg.AGENT_SCALE),
            (wrist_body, wrist_geoms, pc.point_on_ray(wrist_pos, bowl_center, fraction), scale),
        ],
    )


def rollout_images(
    kind: RollEdit,
    collect,
    env,
    sim,
    body: int,
    geoms: list[int],
    pair,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    match kind:
        case "base" | "covered" | "absent":
            return pc.live_images(collect, env, sim, body, geoms, kind)
        case "physical":
            agent, wrist, state = pc.live_images(collect, env, sim, body, geoms, "base")
            if pair is None:
                return agent, wrist, state
            placed = current_pair(collect, env, sim, *pair)
            if placed is None:
                return agent, wrist, state
            return placed["agent"], placed["wrist"], state
        case _ as unexpected:
            assert_never(unexpected)


def run_rollouts(
    collect,
    env,
    policy,
    pre,
    post,
    layer5,
    layer6,
    sentence: str,
    demos,
    target_body,
    free_type: int,
    tight_params: dict,
    episodes: int,
    horizon: int,
    replan: int,
    device: str,
) -> list[dict]:
    edits: tuple[RollEdit, ...] = ("base", "covered", "absent", "physical")
    rows = []
    for episode in range(episodes):
        if episode >= len(demos) or len(demos[episode]) == 0:
            continue
        for edit in edits:
            begin_episode(collect, env, demos[episode][0])
            sim = collect.sim_of(env)
            body, geoms = target_body(sim)
            if body is None or not geoms:
                print(f"  ep {episode} {edit}: no free target body", flush=True)
                continue
            bodies = pc.free_body_map(sim, free_type)
            names = dict(pc.body_names(sim, bodies))
            agent_body = pc.prefer_occluder(list(names.items()), body)
            other = rg.second_body(list(names.items()), {body, agent_body} if agent_body is not None else {body})
            pair = None
            if edit == "physical" and (agent_body is None or other is None):
                print(f"  ep {episode} physical: no second body, using the live image", flush=True)
            if agent_body is not None and other is not None and agent_body in bodies and other in bodies:
                pair = (
                    agent_body,
                    bodies[agent_body],
                    other,
                    bodies[other],
                    geoms,
                    float(tight_params["fraction"]),
                    float(tight_params["scale"]),
                )
            span = collect.object_qpos_span(sim, body)
            arm0 = np.asarray(sim.data.qpos[:7], dtype=np.float64).copy()
            bowl0 = paper_gaps.object_xyz(sim, body, span)
            success_before = task_succeeded(env)
            taken = 0
            succeeded = success_before
            error = ""
            env_done = False
            raw_abs = None
            post_abs = None
            try:
                while taken < horizon and not succeeded and not episode_done(env):
                    sim = collect.sim_of(env)
                    agent, wrist, state = rollout_images(edit, collect, env, sim, body, geoms, pair)
                    batch = cc.batch_from_images(collect, pre, agent, wrist, state, sentence, policy)
                    _tokens, _tokens6, raw = cc.forward_policy(
                        policy, layer5, layer6, batch, 8200 + episode * 1000 + taken, None, "base"
                    )
                    chunk = rg.apply_post(post, raw, device)
                    if raw_abs is None:
                        raw_abs = rg.mean_abs(raw)
                        post_abs = rg.mean_abs(chunk)
                        print(
                            f"    ep {episode} {edit} raw mean abs {raw_abs:.4f} post mean abs {post_abs:.4f}",
                            flush=True,
                        )
                    acted = 0
                    for action in np.asarray(chunk)[:replan]:
                        if taken >= horizon or episode_done(env):
                            break
                        env_done = step_action(env, action)
                        taken += 1
                        acted += 1
                        sim = collect.sim_of(env)
                        succeeded = task_succeeded(env)
                        if taken % 40 == 0 or succeeded or env_done:
                            arm, shift = rg._motion(sim, body, span, arm0, bowl0)
                            print(
                                f"    ep {episode} {edit} step {taken} success {succeeded} "
                                f"env_done {env_done} arm_move {_fmt(arm, 5)} bowl_shift_m {_fmt(shift, 6)}",
                                flush=True,
                            )
                        if succeeded or env_done:
                            break
                    if acted == 0:
                        break
            except RuntimeError:
                raise
            except Exception as exc:
                error = str(exc)
                print(f"  ep {episode} {edit} stopped: {error}", flush=True)
            sim = collect.sim_of(env)
            arm, shift = rg._motion(sim, body, span, arm0, bowl0)
            print(
                f"  ep {episode} {edit}: success {success_before} -> {succeeded} steps {taken} "
                f"bowl_shift_m {_fmt(shift, 6)} arm_move {_fmt(arm, 5)} env_done {env_done} {error}",
                flush=True,
            )
            rows.append(
                {
                    "episode": episode,
                    "edit": edit,
                    "success_before": success_before,
                    "success_after": succeeded,
                    "steps": taken,
                    "bowl_shift_m": shift,
                    "arm_move": arm,
                    "env_done": env_done,
                    "raw_mean_abs": raw_abs,
                    "post_mean_abs": post_abs,
                    "error": error,
                }
            )
    return rows


if __name__ == "__main__":
    main()
'''

scripts = LOCAL_REPO / "scripts"
scripts.mkdir(parents=True, exist_ok=True)
path = scripts / "conclusion.py"
path.write_text(SCRIPT if SCRIPT.endswith("\n") else SCRIPT + "\n")
print("wrote", path, flush=True)
tight_path = scripts / "tight_cover.py"
if tight_path.is_file():
    print("using existing", tight_path, flush=True)
else:
    tight_path.write_text(TIGHT_COVER if TIGHT_COVER.endswith("\n") else TIGHT_COVER + "\n")
    print("wrote", tight_path, flush=True)
for required in (
    "controlled_contrasts.py",
    "occlusion_features.py",
    "collect_layer5_replay.py",
    "paper_gaps.py",
    "paper_claim.py",
    "remaining_gaps.py",
    "tight_cover.py",
):
    if not (scripts / required).is_file():
        raise SystemExit(
            "scripts/%s is missing. Run the collector cell through the remaining-gaps cell first." % required
        )

env = os.environ.copy()
env["PYTHONPATH"] = str(scripts) + os.pathsep + env.get("PYTHONPATH", "")
env["LOCAL_REPO"] = str(LOCAL_REPO)
env["PYTHONUNBUFFERED"] = "1"
env["MAX_STEPS"] = env.get("CTRL_SCAN_STEPS", "200")

def stream(cmd):
    proc = subprocess.Popen(
        cmd,
        cwd=str(scripts),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    tail = []
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
        tail.append(line.rstrip())
        del tail[:-80]
    code = proc.wait()
    if code != 0:
        raise SystemExit(
            "conclusion.py exited with code %s. Last lines:\n%s" % (code, "\n".join(tail))
        )

stream([str(PYTHON), "-u", str(path)])
print("summary", LOCAL_REPO / "outputs" / "permanence" / "conclusion" / "summary.json")


## Research zip

Paste only the next cell into the runtime that already finished the runs. It packs the notebook, source, tests, figures, and summary JSON into `/content/pi05_permanence_research.zip`. It skips the venv, HuggingFace cache, LIBERO archives, secrets, and policy weights. It does not overwrite `outputs/permanence/`.


In [ ]:
# Pack one research zip. Paste into the same runtime after the runs.
# Does not overwrite outputs/permanence/. Does not retrain.
# Writes /content/pi05_permanence_research.zip and copies it to Drive when that folder is mounted.

import os
import sys
import subprocess
from pathlib import Path

try:
    from google.colab import files as colab_files
except ImportError:
    colab_files = None

LOCAL_REPO = Path(os.environ.get("LOCAL_REPO", "/content/groot-run"))
PYTHON = Path("/content/lerobot-venv/bin/python")
if not PYTHON.exists():
    PYTHON = Path(sys.executable)

SCRIPT = r'''
"""Pack the π0.5 permanence study into one research zip.

Keeps the Colab notebook, experiment source, tests, figures, and summary JSON.
Skips caches, public model weights, LIBERO archives, secrets, and raw dumps.
Does not rewrite outputs/permanence/.
"""

from __future__ import annotations

import argparse
import hashlib
import json
import os
import zipfile
from pathlib import Path
from typing import Literal, assert_never

Decision = Literal["keep", "skip"]

SOURCE_NAMES = (
    "collect_layer5_replay.py",
    "controlled_contrasts.py",
    "feature_count_sweep.py",
    "token_position_patch.py",
    "token_subspace.py",
    "occlusion_features.py",
    "paper_gaps.py",
    "cover_autopsy.py",
    "paper_claim.py",
    "remaining_gaps.py",
    "tight_cover.py",
    "conclusion.py",
    "occlusion_contrast.py",
    "pack_research.py",
)

NOTEBOOK_NAMES = (
    "final_smaj.ipynb",
    "controlled_contrasts.ipynb",
    "feature_count_sweep.ipynb",
    "token_position_patch.ipynb",
    "token_subspace.ipynb",
)

RESULT_SUFFIXES = {".json", ".png", ".txt", ".csv", ".md"}
WEIGHT_KEEP = "transcoder.pt"
DEFAULT_ZIP_NAME = "pi05_permanence_research.zip"
MAX_KEEP_BYTES = 250 * 1024 * 1024


def skip_roots() -> list[tuple[str, str]]:
    return [
        ("lerobot-venv", "runtime virtualenv"),
        (".venv", "runtime virtualenv"),
        ("hf_home", "HuggingFace cache; Pi0.5 and PaliGemma stay on the Hub"),
        ("huggingface", "HuggingFace cache"),
        ("datasets--", "Hub snapshot; not a paper figure"),
        (".git", "version control internals"),
        ("__pycache__", "bytecode"),
        ("secrets", "tokens and keys"),
        ("libero_hdf5", "public LIBERO demonstration archive"),
        ("libero-assets", "public LIBERO assets"),
        ("uploads", "scratch upload folder, not the paper drop"),
    ]


def skip_suffixes() -> tuple[str, ...]:
    return (
        ".tar",
        ".tar.gz",
        ".tgz",
        ".hdf5",
        ".h5",
        ".safetensors",
        ".bin",
        ".ckpt",
        ".npz",
        ".pyc",
        ".whl",
    )


def is_smaj_notebook(path: Path) -> bool:
    name = path.name.lower()
    return path.suffix == ".ipynb" and "smaj" in name


def under_permanence_results(path: Path) -> bool:
    parts = set(path.parts)
    return "outputs" in parts and "permanence" in parts


def under_occlusion_results(path: Path) -> bool:
    parts = path.parts
    return "outputs" in parts and "occlusion_contrast" in parts


def skip_root_reason(path: Path) -> str | None:
    parts = [part.lower() for part in path.parts]
    for name, reason in skip_roots():
        key = name.lower()
        if key.endswith("--"):
            if any(part.startswith(key) for part in parts):
                return reason
        elif key in parts:
            return reason
    lowered = path.name.lower()
    if lowered in {"hf_token.txt", ".env"} or lowered.startswith("hf_token"):
        return "secret"
    if lowered.endswith(skip_suffixes()):
        if lowered.endswith(".npz"):
            return "raw replay or feature dump; the JSON summaries are enough"
        return "cache, archive, or public checkpoint"
    if lowered.endswith(".pt") and lowered != WEIGHT_KEEP:
        return "policy or action-expert weight; Pi0.5 is on HuggingFace, the Drive checkpoint stays on Drive"
    return None


def keep_reason(path: Path) -> str | None:
    name = path.name
    if is_smaj_notebook(path):
        return "Colab run notebook"
    if name in NOTEBOOK_NAMES:
        return "experiment notebook"
    if name == "README.md":
        return "how to run the study"
    if name in SOURCE_NAMES:
        return "experiment source"
    if path.parent.name == "tests" and name.startswith("test_") and name.endswith(".py"):
        return "unit test"
    if name == WEIGHT_KEEP and under_permanence_results(path):
        return "trained layer-5 dictionary used by the paper figures"
    if under_permanence_results(path) and path.suffix.lower() in RESULT_SUFFIXES:
        return "permanence result"
    if under_occlusion_results(path) and path.suffix.lower() in RESULT_SUFFIXES:
        return "action-expert verdict or figure"
    return None


def classify(path: Path) -> tuple[Decision, str]:
    if not path.is_file():
        return "skip", "not a file"
    skipped = skip_root_reason(path)
    if skipped is not None:
        if path.suffix.lower() in RESULT_SUFFIXES and (
            under_permanence_results(path) or under_occlusion_results(path) or is_smaj_notebook(path)
        ):
            pass
        else:
            return "skip", skipped
    kept = keep_reason(path)
    if kept is None:
        return "skip", "not a paper artifact"
    size = path.stat().st_size
    if size > MAX_KEEP_BYTES:
        return "skip", f"larger than {MAX_KEEP_BYTES} bytes"
    return "keep", kept


def search_roots(cwd: Path | None = None) -> list[Path]:
    here = cwd if cwd is not None else Path.cwd()
    drive = os.environ.get("DRIVE_ROOT", "/content/drive/MyDrive/groot-run-shared-programmer908")
    roots = [
        here,
        Path("/content"),
        Path("/content/groot-run"),
        Path("/content/pi05-run"),
        Path(drive),
    ]
    found: list[Path] = []
    seen: set[Path] = set()
    for root in roots:
        try:
            resolved = root.resolve()
        except OSError:
            continue
        if not resolved.exists() or resolved in seen:
            continue
        seen.add(resolved)
        found.append(resolved)
    return found


def iter_candidates(roots: list[Path]) -> list[Path]:
    files: list[Path] = []
    seen: set[Path] = set()

    def add(path: Path) -> None:
        try:
            resolved = path.resolve()
        except OSError:
            return
        if resolved in seen or not resolved.is_file():
            return
        if "uploads" in resolved.parts:
            return
        seen.add(resolved)
        files.append(resolved)

    for root in roots:
        for name in SOURCE_NAMES:
            add(root / name)
            add(root / "scripts" / name)
        add(root / "README.md")
        tests = root / "tests"
        if tests.is_dir():
            for path in sorted(tests.glob("test_*.py")):
                add(path)
        for name in NOTEBOOK_NAMES:
            add(root / name)
        try:
            for path in list(root.glob("*smaj*.ipynb")) + list(root.glob("*/*smaj*.ipynb")):
                add(path)
        except OSError:
            pass
        permanence = root / "outputs" / "permanence"
        if permanence.is_dir():
            for path in permanence.rglob("*"):
                if path.is_file():
                    add(path)
        occlusion = root / "outputs" / "occlusion_contrast"
        if occlusion.is_dir():
            for path in occlusion.rglob("*"):
                if path.is_file():
                    add(path)
        scripts = root / "scripts"
        if scripts.is_dir():
            for name in SOURCE_NAMES:
                add(scripts / name)
    return files


def zip_name_for(path: Path) -> str:
    name = path.name
    if is_smaj_notebook(path) or name in NOTEBOOK_NAMES:
        return f"notebooks/{name}"
    if name in SOURCE_NAMES:
        return f"source/{name}"
    if path.parent.name == "tests" and name.startswith("test_"):
        return f"tests/{name}"
    if name == "README.md":
        return "README.md"
    parts = list(path.parts)
    if "permanence" in parts:
        index = parts.index("permanence")
        rel = "/".join(parts[index:])
        return f"results/{rel}"
    if "occlusion_contrast" in parts:
        index = parts.index("occlusion_contrast")
        rel = "/".join(parts[index:])
        return f"results/{rel}"
    digest = hashlib.sha1(str(path).encode("utf-8")).hexdigest()[:8]
    return f"other/{digest}_{name}"


def inventory(paths: list[Path]) -> list[dict]:
    rows: list[dict] = []
    used: dict[str, Path] = {}
    for path in paths:
        decision, reason = classify(path)
        row = {
            "path": str(path),
            "name": path.name,
            "bytes": path.stat().st_size if path.is_file() else 0,
            "decision": decision,
            "reason": reason,
            "zip_name": None,
        }
        if decision == "keep":
            entry = zip_name_for(path)
            taken = used.get(entry)
            if taken is not None and taken != path:
                if taken.stat().st_mtime >= path.stat().st_mtime:
                    row["decision"] = "skip"
                    row["reason"] = f"duplicate of {taken}"
                else:
                    for old in rows:
                        if old.get("zip_name") == entry and old["decision"] == "keep":
                            old["decision"] = "skip"
                            old["reason"] = f"duplicate of {path}"
                            break
                    row["zip_name"] = entry
                    used[entry] = path
            else:
                row["zip_name"] = entry
                used[entry] = path
        elif decision == "skip":
            pass
        else:
            assert_never(decision)
        rows.append(row)
    return rows


def manifest_text(rows: list[dict], zip_path: Path) -> str:
    kept = [row for row in rows if row["decision"] == "keep"]
    skipped = [row for row in rows if row["decision"] == "skip"]
    lines = [
        "π0.5 permanence research zip",
        f"archive: {zip_path.name}",
        "",
        "KEEP these files. They are the paper: notebook, source, tests, figures, summary JSON.",
        "",
    ]
    for row in kept:
        lines.append(f"KEEP  {row['zip_name']}  ({row['bytes']} bytes)  {row['reason']}")
    lines.extend(
        [
            "",
            "SKIP these. They are runtime, public data, or secrets. Do not put them in a paper drop.",
            "",
        ]
    )
    for name, reason in skip_roots():
        lines.append(f"SKIP  {name}/  {reason}")
    lines.append("SKIP  *.pt except transcoder.pt  policy weights stay on HuggingFace or Drive")
    lines.append("SKIP  *.npz  raw replay dumps")
    lines.append("SKIP  *.hdf5 *.tar  LIBERO archives")
    extra = [
        row
        for row in skipped
        if row["reason"] not in {reason for _name, reason in skip_roots()}
        and "duplicate" not in row["reason"]
        and row["reason"] != "not a paper artifact"
    ]
    if extra:
        lines.append("")
        lines.append("Skipped files that were considered and rejected:")
        for row in extra[:80]:
            lines.append(f"SKIP  {row['path']}  {row['reason']}")
    lines.extend(
        [
            "",
            "How to read this zip",
            "- notebooks/ holds the Colab run, including names like 3final_smaj (1).ipynb.",
            "- source/ is the experiment code. tests/ checks the scoring rules without LIBERO.",
            "- results/permanence/*/summary.json and *.png are the numbers and figures.",
            "- results/occlusion_contrast/verdict.json is the separate action-expert check.",
            "- Pi0.5 weights: HuggingFace lerobot/pi05_libero_finetuned.",
            "- Demonstrations: HuggingFaceVLA/libero and the public LIBERO suite.",
            "",
        ]
    )
    return "\n".join(lines) + "\n"


def write_zip(rows: list[dict], zip_path: Path) -> Path:
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    if zip_path.exists():
        zip_path.unlink()
    kept = [row for row in rows if row["decision"] == "keep"]
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
        bundle.writestr("MANIFEST.txt", manifest_text(rows, zip_path))
        bundle.writestr(
            "inventory.json",
            json.dumps({"files": rows, "kept": len(kept)}, indent=2) + "\n",
        )
        for row in kept:
            bundle.write(row["path"], arcname=row["zip_name"])
    return zip_path


def print_inventory(rows: list[dict], zip_path: Path | None) -> None:
    kept = [row for row in rows if row["decision"] == "keep"]
    print("KEEP")
    for row in kept:
        print(f"  {row['zip_name']}  {row['reason']}")
    print("SKIP roots")
    for name, reason in skip_roots():
        print(f"  {name}/  {reason}")
    print(f"kept {len(kept)} files")
    if zip_path is not None:
        print("ZIP", zip_path, "bytes", zip_path.stat().st_size)


def parse_args(argv: list[str] | None = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Pack the permanence study into one research zip.")
    parser.add_argument("--cwd", type=Path, default=None)
    parser.add_argument("--zip", type=Path, default=None)
    parser.add_argument("--dry-run", action="store_true")
    return parser.parse_args(argv)


def main(argv: list[str] | None = None) -> int:
    args = parse_args(argv)
    cwd = args.cwd.resolve() if args.cwd is not None else Path.cwd()
    roots = search_roots(cwd)
    rows = inventory(iter_candidates(roots))
    if args.dry_run:
        print_inventory(rows, None)
        return 0
    zip_path = args.zip
    if zip_path is None:
        colab = Path("/content")
        zip_path = (colab if colab.is_dir() else cwd) / DEFAULT_ZIP_NAME
    zip_path = zip_path.resolve()
    write_zip(rows, zip_path)
    print_inventory(rows, zip_path)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
'''

dest = LOCAL_REPO / "scripts" / "pack_research.py"
dest.parent.mkdir(parents=True, exist_ok=True)
dest.write_text(SCRIPT)
print("wrote", dest)

zip_path = Path("/content/pi05_permanence_research.zip")
subprocess.run([str(PYTHON), str(dest), "--zip", str(zip_path)], check=True)
print("ZIP", zip_path, "bytes", zip_path.stat().st_size)

drive_root = Path(os.environ.get("DRIVE_ROOT", "/content/drive/MyDrive/groot-run-shared-programmer908"))
if drive_root.is_dir():
    copied = drive_root / zip_path.name
    copied.write_bytes(zip_path.read_bytes())
    print("copied", copied)

if colab_files is not None:
    colab_files.download(str(zip_path))
else:
    print("download from the Files sidebar:", zip_path)
